### 0. Import

In [ ]:
# Import all the necessary libraries
import os
import glob
import pandas as pd
import shutil
import re
from openpyxl import load_workbook
from openpyxl.styles import Alignment, Font, PatternFill, Border, Side
from openpyxl.styles.colors import Color
from openpyxl.utils import get_column_letter
from openpyxl.utils.dataframe import dataframe_to_rows
import xlwings as xw
import time
import numpy as np
import win32com.client as win32
from dateutil import parser
from typing import List
from pytrends.request import TrendReq
from openpyxl.cell.cell import MergedCell
from pathlib import Path
from datetime import datetime
import warnings as wr
wr.simplefilter("ignore",category=FutureWarning)
wr.simplefilter("ignore",category=UserWarning)
wr.filterwarnings("ignore",category=SyntaxWarning)

### 1a. Working Space

In [ ]:
# Set the main working folder (Update the path to your local working folder)
main_working_folder = r'ENTER YOUR WORKING FOLDER HERE'
# Ensures file operations occur in the correct location
os.chdir(main_working_folder)

# Set up the master DataFrame inside the master file, Guidelines sheet, from row 3 excel
master_files = glob.glob("*UniqueClientName - * - DeliverData - Q*")
master_file = master_files[-1] # Don’t use pandas’ built-in list of NA strings, only treat an explicit empty string as missing, but preserve “N/A” as data
print("Detected", master_file)
df_master = pd.read_excel(master_file, sheet_name="Guidelines", header=2, engine='openpyxl', keep_default_na=False, na_values=['']) # At this point “N/A” remains the literal string "N/A" 
header_DeliverData = list(pd.read_excel(master_file, sheet_name="DeliverData", nrows=0, engine='openpyxl').columns)

### 1b. Build Dictionaries

In [ ]:
# Build all the dictionaries from df_master

country_codes_expand = "AU CN HK HK-UniqueClientName HK-VIK ID ID-UniqueClientName ID-VIK IN JP KR MY NZ PH SG TH VN".split()

# Set up the important taxonomy keys
required_campaign_taxonomy_keys = ['MK','CN','FS','MT','AC','PCB','PM','DN','SP'] 
required_placement_taxonomy_keys = ['CH','PB','IT','PD','PRM']

# Define Countries for the two groups:
# Group A: Expected Local Currency 
group_A = {"Australia", "China", "Hong Kong", "India", "Japan", "Korea", "Malaysia", "New Zealand", "Singapore", "Thailand"}

# Group B: Expected USD Currency
group_B = {"Indonesia", "Philippines", "Vietnam"}

# Build a map of column name -> Excel format string
excel_format_map = {
"Date": "yyyy-mm-dd",

"Plan ID": "@",
"Start Date": "dd-mmm-yyyy",
"End Date": "dd-mmm-yyyy",
"Week/Month Beginning": "dd-mmm-yyyy",
"Week/Month Ending": "dd-mmm-yyyy",
"Media Cost (LC)": "#,##0.00",
"Media Cost (USD)": "#,##0.00",
"Paid Impressions": "#,##0",
"Organic Impressions": "#,##0",
"Clicks": "#,##0",
"Video Views": "#,##0",
"Video Completes": "#,##0",
"TV National GRPs": "#,##0.00",
"TV Duration (seconds)": "#,##0",
"CPM/CPC/CPP": "#,##0.00",
}

country_code_map_id = {"Australia": "01", "China": "02", "Hong Kong": "03", "Indonesia": "04", "India": "05", "Japan": "06", "Korea": "07", "Malaysia": "08", "New Zealand": "09", "Philippines": "10", "Singapore": "11", "Thailand": "12", "Vietnam": "13"}

# For "Country" mapping extraction from "Campaign Taxonomy"
country_mapping = dict(zip(df_master["Country Code"], df_master["Country Full"]))

# For "Country" mapping extraction from Short Country Code to Full Country Name in UniqueCloud2
country_code_to_full = dict(zip(df_master["Country Short Code"], df_master["Country Full"]))

# For "Funding Source" mapping extraction from "Campaign Taxonomy"
funding_source_mapping = dict(zip(df_master["Funding Source Code"], df_master["Funding Source Full"]))

# For "Initiative" mapping extraction from "Campaign Taxonomy"
initiative_mapping = dict(zip(df_master["Initiative Code"], df_master["Initiative Full"]))

# For "Profitable Consumer Behaviour" mapping extraction from "Campaign Taxonomy"
profitable_consumer_behaviour_mapping = dict(zip(df_master["Profitable Consumer Behaviour Code"], df_master["Profitable Consumer Behaviour Full"]))

# For "Product Message" mapping extraction from "Campaign Taxonomy"
product_message_mapping = dict(zip(df_master["Product Message Code"], df_master["Product Message Full"]))

# For "Destination" mapping extraction from "Placement Taxonomy"
destination_mapping = dict(zip(df_master["Destination Code"], df_master["Destination Full"]))

# For "Sponsorship" mapping extraction from "Campaign Taxonomy"
sponsorship_mapping = dict(zip(df_master["Sponsorship Code"], df_master["Sponsorship Full"]))

# For "Passion Pillar (Campaign-Level)" mapping extraction from "Campaign Taxonomy"
passion_pillar_campaign_mapping = dict(zip(df_master["Passion Pillar (Campaign-Level) Code"], df_master["Passion Pillar (Campaign-Level) Full"]))

# For "Initiative (Campaign-Level)" mapping extraction from "Campaign Taxonomy"
initiative_campaign_mapping = dict(zip(df_master["Initiative (Campaign-Level) Code"], df_master["Initiative (Campaign-Level) Full"]))

# For "Campaign Objective" mapping extraction from "Placement Taxonomy"
campaignobjective_mapping = dict(zip(df_master["Campaign Objective Code"], df_master["Campaign Objective Full"]))

# For "Cohorts" mapping extraction from "Placement Taxonomy"
cohorts_mapping = dict(zip(df_master["Cohorts Code"], df_master["Cohorts Full"]))

# For "Publisher/Vendor" mapping extraction from "Placement Taxonomy" or "Media Buy Name" (in UniqueCloud2) only for Online dictionary
publisher_mapping = dict(zip(df_master["Publisher/Vendor (Online) Code"], df_master["Publisher/Vendor (Online) Full"]))

# For "Vehicle Name" mapping only for Online dictionary
vehicle_name_mapping = dict(zip(df_master["Vehicle Name (Online) Code"], df_master["Vehicle Name (Online) Full"]))

# For "Media Channel" mapping extraction from "Placement Taxonomy"
media_channel_mapping = dict(zip(df_master["Media Channel Code"], df_master["Media Channel Full"]))

# For "Inventory (Distribution) Type" mapping extraction from "Placement Taxonomy"
inventory_distribution_type_mapping = dict(zip(df_master["Inventory (Distribution) Type Code"], df_master["Inventory (Distribution) Type Full"]))

# For "Primary Target Strategy" mapping extraction from "Placement Taxonomy"
primary_target_strategy_mapping = dict(zip(df_master["Primary Target Strategy Code"], df_master["Primary Target Strategy Full"]))

# For "Promotion Incentive" mapping extraction from "Placement Taxonomy"
promotion_incentive_mapping = dict(zip(df_master["Promotion Incentive Code"], df_master["Promotion Incentive Full"]))

# For "Passion Pillar (Placement-Level)" mapping extraction from "Placement Taxonomy"
passion_pillar_placement_mapping = dict(zip(df_master["Passion Pillar (Placement-Level) Code"], df_master["Passion Pillar (Placement-Level) Full"]))

# For "Initiative (Placement-Level)" mapping extraction from "Placement Taxonomy"
initiative_placement_mapping = dict(zip(df_master["Initiative (Placement-Level) Code"], df_master["Initiative (Placement-Level) Full"]))

# For "Passion Pillar (Creative-Level)" mapping extraction from "Creative Taxonomy"
passion_pillar_creative_mapping = dict(zip(df_master["Passion Pillar (Creative-Level) Code"], df_master["Passion Pillar (Creative-Level) Full"]))

# For "Initiative (Creative-Level)" mapping extraction from "Creative Taxonomy"
initiative_creative_mapping = dict(zip(df_master["Initiative (Creative-Level) Code"], df_master["Initiative (Creative-Level) Full"]))

# For "Currency" mapping extraction from Full Country Name
currency_dict = dict(zip(df_master["Country Full"], df_master["Currency Code"]))

# For "Exchange Rate" mapping extraction from Full Country Name
exchange_dict = dict(zip(df_master["Country Full"], df_master["Exchange Rate"]))

### 1c. VBA Scripts

In [ ]:
Module_Macro_1 = """
Sub Refresh_Selected_Cells()
    Dim cell As Range
    ' Disable screen updating and automatic calculations to improve performance
    Application.ScreenUpdating = False
    Application.Calculation = xlCalculationManual
    Application.EnableEvents = False ' Prevent any event-triggered macros
    ' Check if anything is selected
    If TypeName(Selection) = "Range" Then
        ' Loop through each selected cell
        For Each cell In Selection
            ' Re-enter the value to refresh format or calculation
            cell.Value = cell.Value
        Next cell
    Else
        MsgBox "Please select a range of cells to refresh"
    End If
    ' Re-enable screen updating and calculation after processing
    Application.Calculation = xlCalculationAutomatic
    Application.ScreenUpdating = True
    Application.EnableEvents = True
    MsgBox "Selected cells are refreshed!", vbInformation, "Complete"
End Sub

Public Sub SubFunc_CreateNewSheet(sheetName As String, ws As Worksheet)
    ' Helper function to create or clear a sheet by name
    On Error Resume Next
    Set ws = ThisWorkbook.Worksheets(sheetName)
    On Error GoTo 0
    If ws Is Nothing Then
        Set ws = ThisWorkbook.Worksheets.Add(After:=ThisWorkbook.Worksheets(ThisWorkbook.Worksheets.Count))
        ws.Name = sheetName
    Else
        ws.Cells.Clear
    End If
End Sub

Public Sub SubFunc_PivotRepeatLabel(pt As PivotTable, Condition As Boolean)
    ' Helper function to Design - Report Layout - Repeat all item labels
    Dim pf As PivotField
    For Each pf In pt.RowFields
        pf.RepeatLabels = Condition
    Next pf
End Sub

Public Sub SubFunc_SetZoomLevel(wb_temp As Workbook)
    Dim ws As Worksheet
    ' Helper function to set zoom level
    Set wb = wb_temp
    ' Loop through every worksheet in the workbook
    For Each ws In wb.Worksheets
        ' Skip sheets that are hidden (xlSheetHidden or xlSheetVeryHidden)
        If ws.Visible = xlSheetVisible Then
            ws.Activate
            ActiveWindow.Zoom = 100 ' Set zoom level to 100%
        End If
    Next ws
End Sub

Public Sub SubFunc_CreateNewColumnRight(colPrevious As Long, colCurrent As Long, colName As String, ws As Worksheet)
    ' Helper: Check if the column exists. If not, create a new one to the right side
    Dim findCol As Variant
    lastRow = ws.Cells(Rows.Count, 1).End(xlUp).Row
    Set headerRow = ws.Rows(1)
    ' Check if colCurrent exists
    findCol = Application.Match(colName, headerRow, 0)
    If IsError(findCol) Then
        ws.Columns(colPrevious + 1).Insert Shift:=xlToRight
        colCurrent = colPrevious + 1
        ws.Cells(1, colCurrent).Value = colName
    Else
        colCurrent = findCol
        ws.Range(ws.Cells(2, colCurrent), ws.Cells(lastRow, colCurrent)).Clear
    End If
End Sub

Public Sub SubFunc_UnifyMetricCol(ws As Worksheet)
    ' Helper function to unify data format type for certain metric columns
    Dim colName As Variant, DateTypeColArray As Variant, FloatTypeColArray As Variant, IntegerTypeColArray As Variant
    Dim lastRow As Long, colIndex As Long
    Dim headerRow As Range

    lastRow = ws.Cells(Rows.Count, 1).End(xlUp).Row
    Set headerRow = ws.Rows(1)

    DateTypeColArray = Array("Start Date", "End Date", "Week/Month Beginning", "Week/Month Ending", "Week Beginning", "Month")
    FloatTypeColArray = Array("Media Cost (LC)", "Media Cost (USD)", "TV National GRPs", "GRPs", "CPM/CPC/CPP")
    IntegerTypeColArray = Array("Paid Impressions", "Organic Impressions", "Impressions", _
    "Clicks", "Video Views", "Video Completes", "TV Duration (seconds)", "Spend", "Spend (LC)", "Spend (USD)")
    TextTypeColArray = Array("Plan ID")

    ' Format date type columns
    For Each colName In DateTypeColArray
        colIndex = 0
        On Error Resume Next
        colIndex = Application.Match(colName, headerRow, 0) ' Find column by header name
        If colIndex > 0 Then
            With ws.Range(ws.Cells(2, colIndex), ws.Cells(lastRow, colIndex))
                .Value = .Value
                .NumberFormat = "dd-mmm-yyyy"
            End With
        End If
        On Error GoTo 0
    Next colName

    ' Format float type columns
    For Each colName In FloatTypeColArray
        colIndex = 0
        On Error Resume Next
        colIndex = Application.Match(colName, headerRow, 0)
        If colIndex > 0 Then
            With ws.Range(ws.Cells(2, colIndex), ws.Cells(lastRow, colIndex))
                .Value = .Value
                .NumberFormat = "#,##0.00"
            End With
        End If
        On Error GoTo 0
    Next colName

    ' Format integer type columns
    For Each colName In IntegerTypeColArray
        colIndex = 0
        On Error Resume Next
        colIndex = Application.Match(colName, headerRow, 0)
        If colIndex > 0 Then
            With ws.Range(ws.Cells(2, colIndex), ws.Cells(lastRow, colIndex))
                .Value = .Value
                .NumberFormat = "#,##0"
            End With
        End If
        On Error GoTo 0
    Next colName
    
    ' Format text type columns
    For Each colName In TextTypeColArray
        colIndex = 0
        On Error Resume Next
        colIndex = Application.Match(colName, headerRow, 0)
        If colIndex > 0 Then
            With ws.Range(ws.Cells(2, colIndex), ws.Cells(lastRow, colIndex))
                .Value = .Value
                .NumberFormat = "@"
            End With
        End If
        On Error GoTo 0
    Next colName
    ws.UsedRange.Font.Name = "Arial"
    ws.UsedRange.Font.Size = 8
    ws.Columns.AutoFit
End Sub

Public Sub SubFunc_SheetVisible(targetSheetArray As Variant)
    ' Helper function to set all sheets in the array visible and everything outside else hidden
    Dim ws As Worksheet
    For Each ws In ThisWorkbook.Worksheets
        If IsInArray(ws, targetSheetArray) Then
            ws.Visible = xlSheetVisible
        Else
            ws.Visible = xlSheetHidden
        ' Clear object reference before next iteration
        Set ws = Nothing
    Next ws
End Sub

Public Function SubFunc_GetFileName(ThisWorkbook As Workbook) As String
    ' Helper function to get filename without extension for cleaner matching
    Dim FileNameOnly As String
    FileNameOnly = Replace(ThisWorkbook.Name, ".xlsm", "")
    FileNameOnly = Replace(FileNameOnly, ".xlsx", "")
    FileNameOnly = Replace(FileNameOnly, ".xls", "")
    SubFunc_GetFileName = FileNameOnly
End Function

Public Sub SubFunc_ExtractFileDateRange(ByRef output_WeekStartRange As Date, ByRef output_WeekEndRange As Date, ByRef output_DayStartRange As Date, ByRef output_DayEndRange As Date)
    ' Helper function to extract the period range inside file name
    Dim FileNameOnly As String
    Dim regEx As Object, matches As Object
    Dim quarterNum As Integer, yearNum As Integer, startMonth As Integer, endMonth As Integer, startDayinsideWeek As Integer, endDayinsideWeek As Integer, fullYear As Integer
    Dim startDayoftheMonth As Date, endDayoftheMonth As Date
    Application.ScreenUpdating = False
    ' Reset prior value to avoid unintended reuse
    output_WeekStartRange = 0
    output_WeekEndRange = 0
    FileNameOnly = SubFunc_GetFileName(ThisWorkbook)
    ' Create regex object
    Set regEx = CreateObject("VBScript.RegExp")
    regEx.Global = True
    regEx.IgnoreCase = True
    regEx.Pattern = "Q([1-4])FY(\d{2})"
    Set matches = regEx.Execute(FileNameOnly)
    If matches.Count = 0 Then
        MsgBox "No Q#FY## pattern found in filename", vbExclamation
        Application.ScreenUpdating = True
        Exit Sub
    End If
    ' Case 1: Single Q#FY## pattern
    If matches.Count = 1 Then
        quarterNum = CInt(matches(0).SubMatches(0))
        yearNum = CInt(matches(0).SubMatches(1))
        fullYear = 2000 + yearNum
        ' Calculate both output_WeekStartRange and output_WeekEndRange
        Call SubFunc_GetSpecificRanges(quarterNum, fullYear, output_WeekStartRange, output_WeekEndRange, output_DayStartRange, output_DayEndRange)
    ' Case 2: Two Q#FY## patterns (range)
    ElseIf matches.Count >= 2 Then
        ' First pair - for output_WeekStartRange only
        quarterNum = CInt(matches(0).SubMatches(0))
        yearNum = CInt(matches(0).SubMatches(1))
        fullYear = 2000 + yearNum
        Call SubFunc_GetSpecificRanges(quarterNum, fullYear, output_WeekStartRange, endDayoftheMonth, output_DayStartRange, endDayoftheMonth)
        ' Second pair - for output_WeekEndRange only
        quarterNum = CInt(matches(1).SubMatches(0))
        yearNum = CInt(matches(1).SubMatches(1))
        fullYear = 2000 + yearNum
        Call SubFunc_GetSpecificRanges(quarterNum, fullYear, startDayoftheMonth, output_WeekEndRange, startDayoftheMonth, output_DayEndRange)
    End If
    'Debug.Print "Start Range: " & Format(output_WeekStartRange, "dd-mmm-yyyy")
    'Debug.Print "End Range: " & Format(output_WeekEndRange, "dd-mmm-yyyy")
    Application.ScreenUpdating = True
End Sub

Private Sub SubFunc_GetSpecificRanges(quarterNum As Integer, fullYear As Integer, _
                                  ByRef output_WeekStartRange As Date, ByRef output_WeekEndRange As Date, ByRef output_DayStartRange As Date, ByRef output_DayEndRange As Date)
    ' Helper function to get the period ending
    Dim startMonth As Integer, endMonth As Integer, startDayinsideWeek As Integer, endDayinsideWeek As Integer
    Dim startDayoftheMonth As Date, endDayoftheMonth As Date
    ' Determine start and end months based on quarter
    Select Case quarterNum ' Assuming fiscal quarter
        Case 1: startMonth = 1: endMonth = 3 ' Jan-Mar
        Case 2: startMonth = 4: endMonth = 6 ' Apr-Jun
        Case 3: startMonth = 7: endMonth = 9 ' Jul-Sep
        Case 4: startMonth = 10: endMonth = 12 ' Oct-Dec
    End Select
    ' Adjust start and end months by subtracting 3
    startMonth = startMonth - 3
    endMonth = endMonth - 3
    ' Handle negative months (wrap to previous year)
    If startMonth <= 0 Then
        startMonth = startMonth + 12
        fullYear = fullYear - 1
    End If
    If endMonth <= 0 Then endMonth = endMonth + 12
    ' Set start day to first of the month
    startDayoftheMonth = DateSerial(fullYear, startMonth, 1)
    ' Set the first of the month as the output to daily level limit
    output_DayStartRange = startDayoftheMonth
    ' Set end day to last day of the end month
    endDayoftheMonth = DateSerial(fullYear, endMonth + 1, 0)
    ' Set the last of the month as the output to daily level limit
    output_DayEndRange = endDayoftheMonth
    ' Get day of week (1=Sunday, 2=Monday, etc.)
    startDayinsideWeek = Weekday(startDayoftheMonth, vbSunday)
    endDayinsideWeek = Weekday(endDayoftheMonth, vbSunday)
    ' Calculate Monday of the start week
    If startDayinsideWeek = 2 Then
        output_WeekStartRange = startDayoftheMonth
    Else
        ' Find the Monday of that week
        If startDayinsideWeek = 1 Then  ' Sunday
            output_WeekStartRange = startDayoftheMonth + 1
        Else  ' Tuesday-Saturday
            output_WeekStartRange = startDayoftheMonth - (startDayinsideWeek - 2)
        End If
    End If
    ' Calculate Sunday of the end week
    If endDayinsideWeek = 1 Then
        output_WeekEndRange = endDayoftheMonth ' If Sunday, end range is the same day
    ElseIf endDayinsideWeek = 2 Then
        output_WeekEndRange = endDayoftheMonth + 6 ' If Monday, end range is 6 days later
    Else
        ' For any other day, calculate days until Sunday
        output_WeekEndRange = endDayoftheMonth + (7 - endDayinsideWeek + 1)
    End If
End Sub

Public Sub SubFunc_GetFixedQuarter(ByRef fixed_quarter As String)
    ' Helper function to get the fixed quarter value, no calculation
    Dim FileNameOnly As String, quarterNum As Integer
    Dim regEx As Object, matches As Object, rangeRegEx As Object
    fixed_quarter = vbNullString
    FileNameOnly = SubFunc_GetFileName(ThisWorkbook)
    Set regEx = CreateObject("VBScript.RegExp")
    regEx.Global = True
    regEx.IgnoreCase = True
    regEx.Pattern = "Q([1-4])FY(\d{2})"
    Set matches = regEx.Execute(FileNameOnly)
    If matches.Count = 0 Then
        MsgBox "No Q# pattern found in filename", vbExclamation
        Application.ScreenUpdating = True
        Exit Sub
    End If
    If matches.Count = 1 Then
        Set rangeRegEx = CreateObject("VBScript.RegExp")
        rangeRegEx.Global = False
        rangeRegEx.IgnoreCase = True
        rangeRegEx.Pattern = "Q([1-4])FY(\d{2})\s*-\s*?"
        ' \s*: Matches zero or more space or tab characters before the hyphen
        ' Matches the literal hyphen character -
        ' \s*?: Matches zero or more space characters after the hyphen, using lazy matching so it avoids eating spaces needed by subsequent tokens
        If rangeRegEx.Test(FileNameOnly) Then
            fixed_quarter = "(All)"
        Else
            quarterNum = CInt(matches(0).SubMatches(0))
            fixed_quarter = "Q" & quarterNum
        End If
    ElseIf matches.Count >= 2 Then
        fixed_quarter = "(All)"
    End If
    Debug.Print "Fixed Quarter Value: " & fixed_quarter
End Sub

Public Sub SubFunc_GetFixedCountry(ByRef fixed_country As String)
    ' Helper function to get the fixed country value, no calculation
    Dim wsGuidelines As Worksheet, headerRowGuidelines As Range, lastRowCountryFull As Long
    Dim FileNameOnly As String, mapKey As String, mapValue As String, i As Long
    Dim colCountryShortCode As Long, colCountryFull As Long, CountryCodeMap As Object
    fixed_country = None
    FileNameOnly = SubFunc_GetFileName(ThisWorkbook)
    Set wsGuidelines = ThisWorkbook.Worksheets("Guidelines")
    wsGuidelines.AutoFilterMode = False
    Set headerRowGuidelines = wsGuidelines.Rows(3)
    colCountryShortCode = Application.Match("Country Short Code", headerRowGuidelines, 0)
    colCountryFull = Application.Match("Country Full", headerRowGuidelines, 0)
    lastRowCountryFull = wsGuidelines.Cells(wsGuidelines.Rows.Count, colCountryFull).End(xlUp).Row
    Set CountryCodeMap = CreateObject("Scripting.Dictionary")
    With CountryCodeMap
        For i = 4 To lastRowCountryFull
            mapKey = Trim(CStr(wsGuidelines.Cells(i, colCountryShortCode).Value))
            mapValue = Trim(CStr(wsGuidelines.Cells(i, colCountryFull).Value))
            If mapKey <> "" Then
                If Not .Exists(mapKey) Then ' Duplicate key guard
                    .Add mapKey, mapValue
                End If
            End If
        Next i
    End With
    'Call DebugPrintDict(CountryCodeMap)
    For Each key In CountryCodeMap.Keys
        ' Case-sensitive
        If InStr(1, FileNameOnly, key, vbBinaryCompare) > 0 Then
            If key = "KR" Then
                fixed_country = "South Korea"
                Exit For
            Else
                fixed_country = CountryCodeMap(key)
                Exit For
            End If
        Else
            fixed_country = "(All)"
        End If
    Next key
    'Debug.Print "Fixed Country Value: " & fixed_country
End Sub
"""

In [ ]:
Module_Macro_2 = """
' Refresh UniqueCloud1 sheet by fetching data from ExportUrl via QueryTable
Sub Refresh_UniqueCloud1()
    Dim wsMT As Worksheet, queTab As QueryTable

    ' Refresh Link whole FY26
    Const ExportUrl = "https://"

    ' Set up a clean UniqueCloud1 sheet
    Call SubFunc_CreateNewSheet("UniqueCloud1", wsMT)

    ' Delete all existing QueryTables
    For n = wsMT.QueryTables.Count To 1 Step -1 ' Loops through all QueryTables in reverse order
    ' Goes backward to avoid indexing issues when removing items from a collection
      wsMT.QueryTables(n).Delete ' Deletes each one
    Next n

    ' Delete all workbook connections
    For n = ThisWorkbook.Connections.Count To 1 Step -1 ' Loops through all data connections in the entire workbook in reverse order
      On Error Resume Next ' Add Error Handling to skip connections that can't be deleted
      ThisWorkbook.Connections.item(n).Delete ' Deletes each connection (links to external data sources)
      On Error GoTo 0
    Next n

    ' Always add the web query configuration (loop above ensures Count = 0)
    Set queTab = wsMT.QueryTables.Add(Connection:="URL;" & ExportUrl, Destination:=wsMT.Range("A1"))

    ' Configure the query
    queTab.AdjustColumnWidth = True
    queTab.RefreshStyle = xlOverwriteCells
    queTab.RefreshOnFileOpen = False
    queTab.BackgroundQuery = False
    queTab.Name = "RefreshMTQueryTable"
    queTab.WebPreFormattedTextToColumns = True
    queTab.WebFormatting = xlWebFormattingNone
    queTab.WebConsecutiveDelimitersAsOne = True
    queTab.WebDisableRedirections = False
    queTab.WebSingleBlockTextImport = False
    queTab.WebDisableDateRecognition = False
    queTab.WebSelectionType = xlEntirePage

    ' Attempt refresh with error handling
    On Error Resume Next
    queTab.Refresh
    On Error GoTo 0
    If Err.Number <> 0 Then
        MsgBox "Connection failed. Check service availability.", vbCritical
        Err.Clear
        Exit Sub
    End If

    MsgBox "Wait for the data table to load, then click 'Import'" & vbCrLf & "Only works if your account has access to UniqueCloud1", vbInformation, "Complete"

    ' Activate the import dialog
    Application.Goto wsMT.Cells(1, 1) ' Go to cell A1
    Application.SendKeys ("+{F10}") ' Trigger mouse right click
    Application.SendKeys ("EE~") ' Press E two times then Enter
    ActiveWindow.Zoom = 100
End Sub

Public Function SubFunc_GetDictionary(chosenColName As String) As Object
    ' Helper function to build dictionary from a header name in Guidelines worksheet
    Dim wsGuidelines As Worksheet, headerRow As Range, dict As Object, chosen_col As Long, last_row As Long, i As Long, validValue As String
    Set wsGuidelines = ThisWorkbook.Worksheets("Guidelines")
    Set headerRow = wsGuidelines.Rows(3) ' Default header row 3 for all dictionaries
    Set dict = CreateObject("Scripting.Dictionary")
    ' Find the chosen column
    chosen_col = Application.Match(chosenColName, headerRow, 0)
    ' Find last row in the chosen column
    last_row = wsGuidelines.Cells(wsGuidelines.Rows.Count, chosen_col).End(xlUp).Row
    ' Build dictionary from the column
    For i = 4 To last_row
        validValue = UCase(Trim(CStr(wsGuidelines.Cells(i, chosen_col).Value)))
        If validValue <> "" Then
            If Not dict.Exists(validValue) Then
                dict.Add validValue, True
            End If
        End If
    Next i
    ' Return the dictionary
    Set SubFunc_GetDictionary = dict
End Function

Public Function ERR_HIG() As Long
    ' Helper function to set the strong red error cell highlight color
    ERR_HIG = RGB(255, 102, 102)
End Function

Public Function TAB_HIG() As Long
    ' Helper function to set the light yellow tab highlight color
    TAB_HIG = RGB(255, 255, 153)
End Function

Public Function DebugPrintDict(targDict As Object)
    ' Helper function to view a dictionary
    Dim key As Variant
    For Each key In targDict.Keys
        ' Prints "Key: [key value] | Item: [item value]" to the Immediate Window
        Debug.Print "Key: " & key & " | Item: " & targDict(key)
    Next key
End Function

Public Function IsInArray(valToFind As Variant, SearchArray As Variant) As Boolean
    ' Helper function to check if value exists in array (case-insensitive)
    Dim i As Long
    IsInArray = False
    For i = LBound(SearchArray) To UBound(SearchArray)
        If UCase(SearchArray(i)) = UCase(valToFind) Then
            IsInArray = True
            Exit Function
        End If
    Next i
End Function

Public Function SubFunc_GetColIndex(rngTable As Range) As Object
    ' Helper function to get pivot table header name and their corresponding index number
    Dim colIndex As Object, c As Long, headerName As String
    Set colIndex = CreateObject("Scripting.Dictionary") ' Dictionary of headerName and their index
    For c = 1 To rngTable.Columns.Count ' Check all columns
        headerName = Trim(CStr(rngTable.Cells(1, c).Value)) ' Only row 1 header
        If headerName <> "" Then colIndex(headerName) = c
    Next c
    ' Output the dictionary
    Set SubFunc_GetColIndex = colIndex
End Function

Public Function SubFunc_GetColLetter(ByVal colNum As Long) As String
    ' Helper function to convert a numeric column index to its letter (e.g. 3 -> "C")
    SubFunc_GetColLetter = Split(Cells(1, colNum).Address(True, False), "$")(0)
End Function

Public Sub SubFunc_EnforceCorrectDate(ws As Worksheet, rowNum As Long, InputCol As Long, OutputDateValue As Date)
    ' Helper function to Parse date from cell, enforcing dd/mm/yyyy format regardless of locale
    Dim rawVal As Variant, sep As String, parts() As String
    rawVal = ws.Cells(rowNum, InputCol).Value
    If IsError(rawVal) Or Len(rawVal) = 0 Then
        OutputDateValue = NA
        Exit Sub
    End If
    ' Parse dd/mm/yyyy regardless of locale
    If VarType(rawVal) = vbString Then
        ' Read the raw cell (rawVal) and, if it's a string, split on "-" or "/" to enforce dd/mm/yyyy parsing.
        sep = IIf(InStr(rawVal, "-") > 0, "-", "/")
        parts = Split(rawVal, sep)
        If UBound(parts) = 2 Then
            OutputDateValue = DateSerial(CInt(parts(2)), CInt(parts(1)), CInt(parts(0)))
        Else
            OutputDateValue = NA
        End If
    ElseIf IsDate(rawVal) Then
        OutputDateValue = CDate(rawVal)
    End If
End Sub

Public Sub SubFunc_RenameColHeaders(ws As Worksheet)
    ' Helper function to rename column headers
    Dim RenameMap As Object, colIndex As Long
    Dim headerName As Variant, FoundPosition As Variant
    ' Create dictionary for old header -> new header mapping
    Set RenameMap = CreateObject("Scripting.Dictionary")
    With RenameMap
        .Add "Funding Source", "Spend For"
        .Add "Initiative", "Strategy"
        .Add "Profitable Consumer Behaviour", "PCB"
        .Add "Product Message", "Product Msg."
        .Add "Destination", "Txn. Type"
        .Add "Media Channel", "Channel Execution"
        .Add "Inventory (Distribution) Type", "Media Buy"
        .Add "Promotion Incentive", "Promotion"
        .Add "Sum of Media Cost (LC)", "Spend"
        .Add "Sum of Media Cost (USD)", "Spend (USD)"
        .Add "Sum of Paid Impressions", "Impressions"
        .Add "Sum of Organic Impressions", "Organic Impressions"
        .Add "Sum of Viral Impressions", "Viral Impressions"
        .Add "Sum of Clicks", "Clicks"
        .Add "Sum of TV National GRPs", "GRPs"
        .Add "Average of TV Duration (seconds)", "TV Duration (seconds)"
        .Add "Average of CPM/CPC/CPP", "CPM/CPC/CPP"
    End With
    ' Loop through each key (old header name) in the dictionary
    For Each headerName In RenameMap.Keys
        ' Match returns column index if found, or error if not
        On Error Resume Next
        FoundPosition = Application.Match(headerName, ws.Rows(1), 0)
        On Error GoTo 0
        ' If a match was found, rename the header
        If Not IsError(FoundPosition) And Not IsEmpty(FoundPosition) Then
            colIndex = CLng(FoundPosition)
            ws.Cells(1, colIndex).Value = RenameMap(headerName)
        End If
    Next headerName
End Sub
"""

In [ ]:
Module_Macro_3 = """
Sub Apply_DeliverData_Format()
    'Call SubFunc_AddButtons(False)
    Call SubFunc_CustomizeWholeSheet(Array("UniqueCloud1", "DeliverData", "DeliverData_Check_Impressions", "Regional DeliverData Dashboards"))
    Call SubFunc_CustomizeGreyOut
    Call SubFunc_SetZoomLevel(ThisWorkbook)
    On Error Resume Next
    Dim wsDeliverData As Worksheet, wsImp As Worksheet
    Set wsDeliverData = ThisWorkbook.Worksheets("DeliverData")
    Call SubFunc_UnifyMetricCol(wsDeliverData)
    Set wsImp = ThisWorkbook.Worksheets("DeliverData_Check_Impressions")
    Call SubFunc_UnifyMetricCol(wsImp)
    On Error GoTo 0
    Call DeliverData_Data_Validation
    wsDeliverData.Select
End Sub

Public Sub SubFunc_CustomizeWholeSheet(SheetArray As Variant)
    Dim wb As Workbook, ws As Worksheet
    Dim sheetName As Variant, lastUsedColumn As Long
    Dim headerRange As Range, lastFoundCell As Range, sheetIsEmpty As Boolean
    Application.ScreenUpdating = False
    Application.EnableEvents = False
    Set wb = ThisWorkbook
    ' Loop through explicitly-named sheets
    For Each sheetName In SheetArray
        On Error Resume Next
        Set ws = wb.Worksheets(CStr(sheetName))
        On Error GoTo 0
        If Not ws Is Nothing Then
            With ws
                ' Determine whether the sheet contains any data at all
                sheetIsEmpty = (Application.WorksheetFunction.CountA(.Cells) = 0)
                If sheetIsEmpty Then
                    ' If sheet totally empty, default to column 1 (A)
                    lastUsedColumn = 1
                Else
                    ws.AutoFilterMode = False
                    ' Find the last used cell by searching by columns from the end
                    ' This is robust for sheets where some rows/columns may be empty
                    Set lastFoundCell = .Cells.Find(What:="*", LookIn:=xlFormulas, LookAt:=xlPart, SearchOrder:=xlByColumns, SearchDirection:=xlPrevious)
                    If Not lastFoundCell Is Nothing Then
                        lastUsedColumn = lastFoundCell.Column
                    Else
                        lastUsedColumn = 1 ' Fallback: ensure at least column 1 is used
                    End If
                End If
                ' Define header range only from A1 to the last used column in row 1
                Set headerRange = .Range(.Cells(1, 1), .Cells(1, lastUsedColumn))
                ' a) All-cells font & alignment (applies to entire sheet)
                With .Cells
                    .FormatConditions.Delete
                    .ClearFormats ' Clear old formats & fills
                    .Font.Name = "Arial"
                    .Font.Size = 8
                    .HorizontalAlignment = xlCenter
                    .VerticalAlignment = xlCenter
                End With
                ' b) Header row formatting (only A1:lastUsedColumn, row 1)
                With headerRange.Interior
                    .ThemeColor = xlThemeColorAccent1
                    .TintAndShade = 0.80 ' 80%
                End With
                ' Tab color
                With .Tab
                    .ThemeColor = xlThemeColorAccent1
                    .TintAndShade = 0.80 ' 80%
                End With
                .Columns.AutoFit
                ' Clear object references for this iteration
                Set headerRange = Nothing
                Set lastFoundCell = Nothing
            End With
        End If
        ' Clear worksheet reference before next iteration
        Set ws = Nothing
    Next sheetName
    Application.EnableEvents = True
    Application.ScreenUpdating = True
End Sub

Sub DeliverData_Data_Validation()
    Dim wsDeliverData As Worksheet, wsSource As Worksheet, wsGuidelines As Worksheet, wsImp As Worksheet
    Dim srcMap As Object
    Dim headerName As Variant, parts As Variant
    Dim headerCell As Range, targetRange As Range
    Dim lastRowDeliverData As Long, lastColDeliverData As Long, lastRowSrc As Long, lastRowTarget As Long
    Dim formula1 As String
    Application.ScreenUpdating = False
    Set wsDeliverData = ThisWorkbook.Worksheets("DeliverData")
    Set wsGuidelines = ThisWorkbook.Worksheets("Guidelines")
    wsGuidelines.AutoFilterMode = False

    lastRowDeliverData = wsDeliverData.Cells(wsDeliverData.Rows.Count, "A").End(xlUp).Row
    lastColDeliverData = wsDeliverData.Cells(1, wsDeliverData.Columns.Count).End(xlToLeft).Column
    
    ' Helper section to combine the columns
    Dim lastRow_PubOnline As Long, lastRow_PubOffline As Long, lastRow_VehOnline As Long, lastRow_VehOffline As Long, lastRow_PubMerge As Long, lastRow_VehMerge As Long
    Dim col_PubMerge As Long, col_VehMerge As Long, col_PubOnline As Long, col_PubOffline As Long, col_VehOnline As Long, col_VehOffline As Long
    Set headerRowGuidelines = wsGuidelines.Rows(3)
    col_PubOnline = Application.Match("Publisher/Vendor (Online) Full", headerRowGuidelines, 0)
    col_PubOffline = Application.Match("Publisher/Vendor (Offline) Full", headerRowGuidelines, 0)
    col_PubMerge = Application.Match("Publisher/Vendor (Online+Offline) Full", headerRowGuidelines, 0)
    col_VehOnline = Application.Match("Vehicle Name (Online) Full", headerRowGuidelines, 0)
    col_VehOffline = Application.Match("Vehicle Name (Offline) Full", headerRowGuidelines, 0)
    col_VehMerge = Application.Match("Vehicle Name (Online+Offline) Full", headerRowGuidelines, 0)
    lastRow_PubOnline = wsGuidelines.Cells(wsGuidelines.Rows.Count, col_PubOnline).End(xlUp).Row
    lastRow_PubOffline = wsGuidelines.Cells(wsGuidelines.Rows.Count, col_PubOffline).End(xlUp).Row
    lastRow_PubMerge = wsGuidelines.Cells(wsGuidelines.Rows.Count, col_PubMerge).End(xlUp).Row
    lastRow_VehOnline = wsGuidelines.Cells(wsGuidelines.Rows.Count, col_VehOnline).End(xlUp).Row
    lastRow_VehOffline = wsGuidelines.Cells(wsGuidelines.Rows.Count, col_VehOffline).End(xlUp).Row
    lastRow_VehMerge = wsGuidelines.Cells(wsGuidelines.Rows.Count, col_VehMerge).End(xlUp).Row

    ' Clear col_PubMerge and col_VehMerge first
    If lastRow_PubMerge > 3 Then wsGuidelines.Range(wsGuidelines.Cells(4, col_PubMerge), wsGuidelines.Cells(lastRow_PubMerge, col_PubMerge)).Clear
    If lastRow_VehMerge > 3 Then wsGuidelines.Range(wsGuidelines.Cells(4, col_VehMerge), wsGuidelines.Cells(lastRow_VehMerge, col_VehMerge)).Clear

    ' For Publisher/Vendor (Online+Offline) Full column
    wsGuidelines.Range(wsGuidelines.Cells(4, col_PubOnline), wsGuidelines.Cells(lastRow_PubOnline, col_PubOnline)).Copy wsGuidelines.Cells(4, col_PubMerge)
    lastRow_PubMerge = lastRow_PubOnline + 1
    wsGuidelines.Range(wsGuidelines.Cells(4, col_PubOffline), wsGuidelines.Cells(lastRow_PubOffline, col_PubOffline)).Copy wsGuidelines.Cells(lastRow_PubMerge, col_PubMerge)
    ' For Vehicle Name (Online+Offline) Full column
    wsGuidelines.Range(wsGuidelines.Cells(4, col_VehOnline), wsGuidelines.Cells(lastRow_VehOnline, col_VehOnline)).Copy wsGuidelines.Cells(4, col_VehMerge)
    lastRow_VehMerge = lastRow_VehOnline + 1
    wsGuidelines.Range(wsGuidelines.Cells(4, col_VehOffline), wsGuidelines.Cells(lastRow_VehOffline, col_VehOffline)).Copy wsGuidelines.Cells(lastRow_VehMerge, col_VehMerge)

    ' Build a dictionary of:
    ' key = the DeliverData header name, value = Array(SourceSheetName, SourceColLetter, SourceStartRow)
    Set srcMap = CreateObject("Scripting.Dictionary")
    With srcMap
        .Add "Country", Array("Guidelines", "D", 4)
        .Add "Plan ID", Array("UniqueCloud1", "C", 2)
        .Add "Plan Name", Array("UniqueCloud1", "D", 2)
        .Add "UniqueClientName Campaign Lead", Array("Guidelines", "M", 4)
        .Add "Funding Source", Array("Guidelines", "N", 4)
        .Add "Fund", Array("Guidelines", "P", 4)
        .Add "Initiative", Array("Guidelines", "Q", 4)
        .Add "Profitable Consumer Behaviour", Array("Guidelines", "S", 4)
        .Add "Product Message", Array("Guidelines", "U", 4)
        .Add "Destination", Array("Guidelines", "W", 4)
        .Add "Sponsorship", Array("Guidelines", "Y", 4)
        .Add "Passion Pillar (Campaign-Level)", Array("Guidelines", "AA", 4)
        .Add "Initiative (Campaign-Level)", Array("Guidelines", "AC", 4)
        .Add "YQ", Array("Guidelines", "AE", 4)
        .Add "Campaign Objective", Array("Guidelines", "AJ", 4)
        .Add "Cohorts", Array("Guidelines", "AL", 4)
        .Add "Publisher/Vendor", Array("Guidelines", "AQ", 4)
        .Add "Vehicle Name", Array("Guidelines", "AU", 4)
        .Add "Media Channel", Array("Guidelines", "AW", 4)
        .Add "Inventory (Distribution) Type", Array("Guidelines", "AZ", 4)
        .Add "Primary Target Strategy", Array("Guidelines", "BB", 4)
        .Add "Promotion Incentive", Array("Guidelines", "BD", 4)
        .Add "Passion Pillar (Placement-Level)", Array("Guidelines", "BF", 4)
        .Add "Initiative (Placement-Level)", Array("Guidelines", "BH", 4)
        .Add "Media Market", Array("Guidelines", "BJ", 4)
        .Add "Passion Pillar (Creative-Level)", Array("Guidelines", "BK", 4)
        .Add "Initiative (Creative-Level)", Array("Guidelines", "BM", 4)
        .Add "Category", Array("Guidelines", "BO", 6)
        .Add "Does Media Cost include Production Fees?", Array("Guidelines", "BP", 4)
        .Add "Currency", Array("Guidelines", "BQ", 4)
        .Add "TV Duration (seconds)", Array("Guidelines", "CB", 4)
    End With

    ' Loop through each DeliverData header, find it, and apply validation
    For Each headerName In srcMap.Keys
        ' locate the header cell in row 1 of DeliverData
        Set headerCell = wsDeliverData.Rows(1).Find(What:=headerName, LookIn:=xlValues, LookAt:=xlWhole, MatchCase:=False)
        If Not headerCell Is Nothing Then
            parts = srcMap(headerName) ' parts(0)=sheet name, (1)=column letter, (2)=start row
            ' Find last row in the source column
            Set wsSource = ThisWorkbook.Worksheets(parts(0))
            lastRowSrc = wsSource.Cells(wsSource.Rows.Count, parts(1)).End(xlUp).Row
            ' Build the dynamic Formula1 reference
            formula1 = "=" & parts(0) & "!$" & parts(1) & "$" & parts(2) & ":$" & parts(1) & "$" & lastRowSrc
            ' Find last row in the target DeliverData column
            lastRowTarget = wsDeliverData.Cells(wsDeliverData.Rows.Count, headerCell.Column).End(xlUp).Row
            ' Define and clear the target range (row 2 -> lastRowTarget)
            Set targetRange = wsDeliverData.Range(wsDeliverData.Cells(2, headerCell.Column), wsDeliverData.Cells(lastRowTarget, headerCell.Column))
            On Error Resume Next
            targetRange.Validation.Delete
            On Error GoTo 0
            ' Apply the dropdown
            With targetRange.Validation
                .Add Type:=xlValidateList, AlertStyle:=xlValidAlertStop, Operator:=xlBetween, formula1:=formula1
                .IgnoreBlank = True
                .InCellDropdown = True
                .InputTitle = "Select Value"
                .ErrorTitle = "Invalid Entry"
            End With
        End If
    Next headerName

    With wsGuidelines.Tab
        .ThemeColor = xlThemeColorAccent3
        .TintAndShade = 0.8  ' 80%
    End With
    Application.ScreenUpdating = True
End Sub

Private Sub SubFunc_CustomizeGreyOut()
    Dim wb As Workbook, sht As Worksheet, wsDR As Worksheet
    Dim headerList As Variant, hdr As Variant, DRsheet_Array As Variant, DRsheet As Variant
    Dim found As Range, headerRangeDR As Range
    Dim col As Long, lastRow As Long, lastColDR As Long
    Set wb = ThisWorkbook
    On Error Resume Next
    Set sht = wb.Worksheets("DeliverData") ' In the DeliverData sheet, “grey-fill” certain columns
    On Error GoTo 0
    If sht Is Nothing Then Exit Sub
    headerList = Array("Campaign", "Campaign Taxonomy", "Placement Taxonomy", _
                       "Creative Taxonomy", "Week/Month Beginning", "Week/Month Ending", "Channel", "Social?", "Media Cost (USD)", "CPM/CPC/CPP")
    lastRow = sht.Cells(sht.Rows.Count, "A").End(xlUp).Row
    ' Loop through each column in row 1 to find matches
    For col = 1 To sht.Cells(1, sht.Columns.Count).End(xlToLeft).Column
        For Each hdr In headerList
            If Trim(sht.Cells(1, col).Value) = hdr Then
                ' Format header separately (optional)
                sht.Cells(1, col).Interior.Color = RGB(153, 153, 153)
                Exit For ' Move to next column once match is found
            End If
        Next hdr
    Next col

    ' UniqueCloud1
    On Error Resume Next
    Set wsMT = ThisWorkbook.Sheets("UniqueCloud1")
    lastColMT = wsMT.Cells(1, Columns.Count).End(xlToLeft).Column
    Set headerRangeMT = wsMT.Range(wsMT.Cells(1, 1), wsMT.Cells(1, lastColMT))
    With headerRangeMT.Interior
        .ThemeColor = xlThemeColorAccent5
        .TintAndShade = 0.80 ' 80%
    End With
    With wsMT.Tab
        .ThemeColor = xlThemeColorAccent5
        .TintAndShade = 0.80 ' 80%
    End With
    On Error GoTo 0
End Sub

Public Sub SubFunc_AddButtons(keep0 As Boolean)
    Dim btn0 As Button, btn1 As Button, btn2 As Button, btn3 As Button, btn4 As Button
    Dim rng0 As Range, rng1 As Range, rng2 As Range, rng3 As Range, rng4 As Range
    Set wsGuidelines = ThisWorkbook.Worksheets("Guidelines")
    ' Clean up old buttons if they exist to avoid duplicates
    On Error Resume Next
    wsGuidelines.Shapes("btnRun0").Delete
    wsGuidelines.Shapes("btnRun1").Delete
    wsGuidelines.Shapes("btnRun2").Delete
    wsGuidelines.Shapes("btnRun3").Delete
    wsGuidelines.Shapes("btnRun4").Delete
    On Error GoTo 0
    ' Set the button locations
    Set rng0 = wsGuidelines.Range(wsGuidelines.Cells(60, 1), wsGuidelines.Cells(61, 3))
    Set rng1 = wsGuidelines.Range(wsGuidelines.Cells(16, 1), wsGuidelines.Cells(17, 3))
    Set rng2 = wsGuidelines.Range(wsGuidelines.Cells(22, 1), wsGuidelines.Cells(23, 3))
    Set rng3 = wsGuidelines.Range(wsGuidelines.Cells(34, 1), wsGuidelines.Cells(35, 3))
    Set rng4 = wsGuidelines.Range(wsGuidelines.Cells(48, 1), wsGuidelines.Cells(49, 3))
    
    If keep0 = True Then
        With wsGuidelines
            ' Add Button 0 at rng0
            Set btn0 = wsGuidelines.Buttons.Add(rng0.Left, rng0.Top, rng0.Width, rng0.Height)
            btn0.Name = "btnRun0"
            btn0.OnAction = "DeliverData_All" ' Assign macro to run
            btn0.Characters.Text = "Refresh DeliverData All" ' Caption shown on button
        End With
    End If

    With wsGuidelines
        Set btn1 = wsGuidelines.Buttons.Add(rng1.Left, rng1.Top, rng1.Width, rng1.Height)
        btn1.Name = "btnRun1"
        btn1.OnAction = "Refresh_UniqueCloud1"
        btn1.Characters.Text = "Refresh UniqueCloud1 data"

        Set btn2 = wsGuidelines.Buttons.Add(rng2.Left, rng2.Top, rng2.Width, rng2.Height)
        btn2.Name = "btnRun2"
        btn2.OnAction = "MT_vs_DeliverData_Plan"
        btn2.Characters.Text = "Refresh UniqueCloud1 vs DeliverData Plan"

        Set btn3 = wsGuidelines.Buttons.Add(rng3.Left, rng3.Top, rng3.Width, rng3.Height)
        btn3.Name = "btnRun3"
        btn3.OnAction = "MT_vs_DeliverData_Spend"
        btn3.Characters.Text = "Refresh UniqueCloud1 vs DeliverData Spend"

        Set btn4 = wsGuidelines.Buttons.Add(rng4.Left, rng4.Top, rng4.Width, rng4.Height)
        btn4.Name = "btnRun4"
        btn4.OnAction = "MT_vs_DeliverData_Classification"
        btn4.Characters.Text = "Refresh UniqueCloud1 vs DeliverData Classification"
    End With
End Sub
"""

In [ ]:
Module_Macro_4 = """
Sub MT_vs_DeliverData_Plan()
    Dim wsMT As Worksheet, wsDeliverData As Worksheet, wsNew As Worksheet
    Dim ptMT As PivotTable, ptDeliverData As PivotTable
    Dim pcMT As PivotCache, pcDeliverData As PivotCache
    Dim dataMT As Range, dataDeliverData As Range
    Dim lastRowMT As Long, lastColMT As Long, i As Long, j As Long
    Dim lastRowDeliverData As Long, lastColDeliverData As Long
    Dim pivotStartCellMT As Range, pivotStartCellDeliverData As Range
    Dim dictMT_PI As Object, dictDeliverData_PI As Object, dictMT_PN As Object, dictDeliverData_PN As Object
    Dim keyMT As String, keyMT2 As String, keyDeliverData As String, keyDeliverData2 As String, fixed_quarter As String, fixed_country As String
    Dim k As Variant, z As Variant
    Dim rngMT As Range, rngDeliverData As Range, errorMess1 As String, errorMess2 As String, errorList1 As New Collection, errorList2 As New Collection

    BenchMark = Timer
    Application.ScreenUpdating = False
    Set wsMT = ThisWorkbook.Sheets("UniqueCloud1")
    Set wsDeliverData = ThisWorkbook.Sheets("DeliverData")
    wsMT.AutoFilterMode = False
    wsDeliverData.AutoFilterMode = False

    Call SubFunc_CreateNewSheet("MT vs DeliverData Plan", wsNew)

    wsNew.Range("A1").Value = "UniqueCloud1 (Overview)"
    wsNew.Range("C1").Value = "DeliverData (Overview)"

    errorList1.Add "MT vs DeliverData Plan"
    errorList1.Add "UniqueCloud1 (Overview)"

    errorList2.Add "MT vs DeliverData Plan"
    errorList2.Add "DeliverData (Overview)"

    ' Define data range for UniqueCloud1
    lastRowMT = wsMT.Cells(Rows.Count, 1).End(xlUp).Row
    lastColMT = wsMT.Cells(1, Columns.Count).End(xlToLeft).Column
    Set dataMT = wsMT.Range(wsMT.Cells(1, 1), wsMT.Cells(lastRowMT, lastColMT))

    ' Define data range for DeliverData
    lastRowDeliverData = wsDeliverData.Cells(Rows.Count, 1).End(xlUp).Row
    lastColDeliverData = wsDeliverData.Cells(1, Columns.Count).End(xlToLeft).Column
    Set dataDeliverData = wsDeliverData.Range(wsDeliverData.Cells(1, 1), wsDeliverData.Cells(lastRowDeliverData, lastColDeliverData))

    ' Define Overview PivotTable start positions
    Set pivotStartCellMT = wsNew.Range("A6")
    Set pivotStartCellDeliverData = wsNew.Range("C6")

    ' Create PivotTable for UniqueCloud1
    Set pcMT = ThisWorkbook.PivotCaches.Create(SourceType:=xlDatabase, SourceData:=dataMT)
    Set ptMT = pcMT.CreatePivotTable(TableDestination:=pivotStartCellMT, TableName:="PivotUniqueCloud1")
    Call SubFunc_GetFixedQuarter(fixed_quarter)
    Call SubFunc_GetFixedCountry(fixed_country)

    DoEvents
    With ptMT
        With .PivotFields("Quarter")
            .Orientation = xlPageField
            On Error Resume Next
            .CurrentPage = fixed_quarter
            If Err.Number <> 0 Then
                Err.Clear
                On Error GoTo 0
                wsMT.Select
                MsgBox fixed_quarter & " not found in Quarter column, UniqueCloud1 sheet", vbExclamation, "Date Validation Error"
                Exit Sub
            End If
            On Error GoTo 0
        End With
        With .PivotFields("Country")
            .Orientation = xlPageField
            .CurrentPage = fixed_country
        End With
        With .PivotFields("Plan ID")
             .Orientation = xlRowField
             .Subtotals = Array(False, False, False, False, False, False, False, False, False, False, False, False)
             .Caption = "Plan ID"
        End With
        With .PivotFields("Plan Name")
             .Orientation = xlRowField
             .Subtotals = Array(False, False, False, False, False, False, False, False, False, False, False, False)
        End With
    .RowAxisLayout xlTabularRow
    .TableStyle2 = "PivotStyleLight20"
    End With
    FileNameOnly = SubFunc_GetFileName(ThisWorkbook)
    If InStr(1, FileNameOnly, "-VIK - DeliverData -", vbTextCompare) > 0 Then
        With ptMT.PivotFields("Funding Source")
            .Orientation = xlPageField
            On Error Resume Next
            .CurrentPage = "Value In Kind (VIK)"
            If Err.Number <> 0 Then
                .CurrentPage = "(All)"
                Err.Clear
            End If
            On Error GoTo 0
        End With
    ElseIf InStr(1, FileNameOnly, "-UniqueClientName - DeliverData -", vbTextCompare) > 0 Then
        With ptMT.PivotFields("Funding Source")
            .Orientation = xlPageField
            On Error Resume Next
            .CurrentPage = "UniqueClientName/Core Marketing"
            If Err.Number <> 0 Then
                .CurrentPage = "(All)"
                Err.Clear
            End If
            On Error GoTo 0
        End With
    End If
    Call SubFunc_PivotRepeatLabel(ptMT, True) ' On/Show
    ' Create PivotTable for DeliverData
    Set pcDeliverData = ThisWorkbook.PivotCaches.Create(SourceType:=xlDatabase, SourceData:=dataDeliverData)
    Set ptDeliverData = pcDeliverData.CreatePivotTable(TableDestination:=pivotStartCellDeliverData, TableName:="PivotDeliverData")
    DoEvents
    With ptDeliverData
        With .PivotFields("Plan ID")
             .Orientation = xlRowField
             .Subtotals = Array(False, False, False, False, False, False, False, False, False, False, False, False)
             .Caption = "Plan ID"
        End With
        With .PivotFields("Plan Name")
             .Orientation = xlRowField
             .Subtotals = Array(False, False, False, False, False, False, False, False, False, False, False, False)
        End With
        With .PivotFields("Campaign")
             .Orientation = xlRowField
             .Subtotals = Array(False, False, False, False, False, False, False, False, False, False, False, False)
        End With
        With .PivotFields("Category")
             .Orientation = xlRowField
             .Subtotals = Array(False, False, False, False, False, False, False, False, False, False, False, False)
        End With
        With .PivotFields("Paid Impressions")
             .Orientation = xlDataField
             .Function = xlSum
             .NumberFormat = "#,##0"
        End With
    .RowAxisLayout xlTabularRow
    .RefreshTable
    End With
    Call SubFunc_PivotRepeatLabel(ptDeliverData, True) ' On/Show

    ' Set the pivot table ranges
    Set rngMT = ptMT.TableRange1
    Set rngDeliverData = ptDeliverData.TableRange1

    ' Create a dictionary to store UniqueCloud1 pivot table cost cell references keyed by "PlanID"
    Set dictMT_PI = CreateObject("Scripting.Dictionary")
    Set dictMT_PN = CreateObject("Scripting.Dictionary")
    Set dictDeliverData_PI = CreateObject("Scripting.Dictionary")
    Set dictDeliverData_PN = CreateObject("Scripting.Dictionary")

    ' Loop through the UniqueCloud1 pivot table data rows (starting from row 2 to skip headers)
    For i = 2 To rngMT.Rows.Count - 1
        keyMT = UCase(Trim(CStr(rngMT.Cells(i, 1).Value))) ' Plan ID
        keyMT2 = UCase(Trim(CStr(rngMT.Cells(i, 2).Value))) ' Plan Name
        ' Use Set when storing a Range object in the dictionary
        Set dictMT_PI(keyMT) = rngMT.Cells(i, 1)
        Set dictMT_PN(keyMT2) = rngMT.Cells(i, 2)
    Next i

    ' Loop through the DeliverData pivot table data rows (starting from row 2 to skip headers)
    For j = 2 To rngDeliverData.Rows.Count - 1
        keyDeliverData = UCase(Trim(CStr(rngDeliverData.Cells(j, 1).Value))) ' Plan ID
        keyDeliverData2 = UCase(Trim(CStr(rngDeliverData.Cells(j, 2).Value))) ' Plan Name
        Set dictDeliverData_PI(keyDeliverData) = rngDeliverData.Cells(j, 1)
        Set dictDeliverData_PN(keyDeliverData2) = rngDeliverData.Cells(j, 2)
    Next j

    ' Flag Missing Keys Section
    ' a) For each key in UniqueCloud1 (dictMT_PI) that is not found in DeliverData (dictDeliverData_PI), highlight UniqueCloud1 Plan ID
    For Each k In dictMT_PI.Keys
        If Not dictDeliverData_PI.Exists(k) Then 
            dictMT_PI(k).Interior.Color = ERR_HIG
            errorMess1 = "Plan ID, Cell " & dictMT_PI(k).Address & ": " & "Plan ID must match with Plan ID in DeliverData (Overview)"
            errorList1.Add errorMess1
        End If
    Next k

    ' b) For each key in UniqueCloud1 (dictMT_PN) that is not found in DeliverData (dictDeliverData_PN), highlight UniqueCloud1 Plan Name
    For Each k In dictMT_PN.Keys
        If Not dictDeliverData_PN.Exists(k) Then 
            dictMT_PN(k).Interior.Color = ERR_HIG
            errorMess1 = "Plan Name, Cell " & dictMT_PN(k).Address & ": " & "Plan Name must match with Plan Name in DeliverData (Overview)"
            errorList1.Add errorMess1
        End If
    Next k

    ' c) For each key in DeliverData (dictDeliverData_PI) that is not found in UniqueCloud1 (dictMT_PI), highlight  DeliverData Plan ID
    For Each z In dictDeliverData_PI.Keys
        If Not dictMT_PI.Exists(z) Then 
            dictDeliverData_PI(z).Interior.Color = ERR_HIG
            errorMess2 = "Plan ID, Cell " & dictDeliverData_PI(z).Address & ": " & "Plan ID must match with Plan ID in MT (Overview)"
            errorList2.Add errorMess2
        End If
    Next z

    ' d) For each key in DeliverData (dictDeliverData_PN) that is not found in UniqueCloud1 (dictMT_PN), highlight  DeliverData Plan Name
    For Each z In dictDeliverData_PN.Keys
        If Not dictMT_PN.Exists(z) Then 
            dictDeliverData_PN(z).Interior.Color = ERR_HIG
            errorMess2 = "Plan Name, Cell " & dictDeliverData_PN(z).Address & ": " & "Plan Name must match with Plan Name in MT (Overview)"
            errorList2.Add errorMess2
        End If
    Next z

    For r = 2 To rngMT.Rows.Count - 1
        Set cellName_PlanID = rngMT.Cells(r, 1)
        Set cellName_PlanName = rngMT.Cells(r, 2)
        cellValue_PlanID = UCase(Trim(CStr(cellName_PlanID.Value)))
        cellValue_PlanName = UCase(Trim(CStr(cellName_PlanName.Value)))
        ' Find Plan Name value blank, empty
        If cellValue_PlanID = "" Or cellValue_PlanID = "(BLANK)" Or cellValue_PlanID = "#N/A" Or cellValue_PlanID = "N/A" Then
            cellName_PlanID.Interior.Color = ERR_HIG
            errorMess1 = "Plan ID, Cell " & cellName_PlanID.Address & ": " & "Plan ID cannot be blank"
            errorList1.Add errorMess1
        End If
        ' Find Plan Name value blank, empty
        If cellValue_PlanName = "" Or cellValue_PlanName = "(BLANK)" Or cellValue_PlanName = "#N/A" Or cellValue_PlanName = "N/A" Then
            cellName_PlanName.Interior.Color = ERR_HIG
            errorMess1 = "Plan Name, Cell " & cellName_PlanName.Address & ": " & "Plan Name cannot be blank"
            errorList1.Add errorMess1
        End If
    Next r

    For r = 2 To rngDeliverData.Rows.Count - 1
        Set cellName_PlanID = rngDeliverData.Cells(r, 1)
        Set cellName_PlanName = rngDeliverData.Cells(r, 2)
        cellValue_PlanID = UCase(Trim(CStr(cellName_PlanID.Value)))
        cellValue_PlanName = UCase(Trim(CStr(cellName_PlanName.Value)))
        ' Find Plan Name value blank, empty
        If cellValue_PlanID = "" Or cellValue_PlanID = "(BLANK)" Or cellValue_PlanID = "#N/A" Or cellValue_PlanID = "N/A" Then
            cellName_PlanID.Interior.Color = ERR_HIG
            errorMess2 = "Plan ID, Cell " & cellName_PlanID.Address & ": " & "Plan ID cannot be blank"
            errorList2.Add errorMess2
        End If
        ' Find Plan Name value blank, empty
        If cellValue_PlanName = "" Or cellValue_PlanName = "(BLANK)" Or cellValue_PlanName = "#N/A" Or cellValue_PlanName = "N/A" Then
            cellName_PlanName.Interior.Color = ERR_HIG
            errorMess2 = "Plan Name, Cell " & cellName_PlanName.Address & ": " & "Plan Name cannot be blank"
            errorList2.Add errorMess2
        End If
    Next r

    Call SubFunc_WriteErrorList(errorList1, 1, True) ' Col 1, start block true
    Call SubFunc_WriteErrorList(errorList2, 2, True) ' Col 2, start block true

    ' Appy additional complex conditional formatting logic
    Call SubFunc_ConFor_Plan_MT_Overview(ptMT)

    wsNew.UsedRange.Font.Name = "Arial"
    wsNew.UsedRange.Font.Size = 8
    wsNew.Columns.AutoFit
    Call SubFunc_SetZoomLevel(ThisWorkbook)
    wsNew.Select
    totalRunTime = Round((Timer - BenchMark) / 60, 2)
    Application.ScreenUpdating = True
    ' Show the MsgBox by default, when no showMsg indicated
    If Not showMsg Then MsgBox "Compare UniqueCloud1 vs DeliverData Plan" & vbCrLf & _
           "Total run time: " & totalRunTime & " minutes", vbInformation, "Complete"
End Sub

' Conditional Formatting Logic for Plan MT Overview table
Private Sub SubFunc_ConFor_Plan_MT_Overview(ptMT As PivotTable)
    Dim countryShortCode As String, errorMess As String, errorList As New Collection
    Dim underscoreCount As Long, firstUnderscore As Long, secondUnderscore As Long

    ' Get the Country Short Code dictionary
    Set dictCountry = SubFunc_GetDictionary("Country Short Code")
    Set rngMT = ptMT.TableRange1
    ' Start from row 2 (skip the header) to last row (skip the Grand Total)
    For r = 2 To rngMT.Rows.Count - 1
        cellValue2 = Trim(CStr(rngMT.Cells(r, 2).Value)) ' Plan Name
        errorMess = ""
        ' Check for 8 underscores limit
        underscoreCount = Len(cellValue2) - Len(Replace(cellValue2, "_", ""))
        If underscoreCount > 8 Then
            rngMT.Cells(r, 2).Interior.Color = ERR_HIG
            errorMess = "Plan Name has more than 8 underscores"
        ElseIf underscoreCount < 8 Then
            rngMT.Cells(r, 2).Interior.Color = ERR_HIG
            errorMess = "Plan Name has less than 8 underscores"
        End If

        ' Check Country Classification code
        ' Find position of first and second underscore
        firstUnderscore = InStr(1, cellValue2, "_")
        If firstUnderscore > 0 Then
            ' Find second underscore after the first one
            secondUnderscore = InStr(firstUnderscore + 1, cellValue2, "_")
            If secondUnderscore > 0 Then
                ' Extract country code between first and second underscore
                countryShortCode = Mid(cellValue2, firstUnderscore + 1, secondUnderscore - firstUnderscore - 1)
                countryShortCode_Upper = UCase(Trim(CStr(countryShortCode))) ' Normalize to uppercase
                ' Check if country code exists in dictionary
                If Not dictCountry.Exists(countryShortCode_Upper) Then
                    rngMT.Cells(r, 2).Interior.Color = ERR_HIG
                    ' Append to error messages with line break
                    If Len(errorMess) > 0 Then
                        errorMess = errorMess & ", " & countryShortCode & " not exist in Dictionary under Country Short Code"
                    Else
                        errorMess = countryShortCode & " not exist in Dictionary under Country Short Code"
                    End If
                End If
            Else
                ' Second underscore not found
                errorMess = errorMess & ", " & "one underscore"
            End If
        Else
            ' First underscore not found
            errorMess = errorMess & ", " & "zero underscore"
        End If
        ' Write combined error messages
        If Len(errorMess) > 0 Then
            errorMess = "Plan Name, Cell " & rngMT.Cells(r, 2).Address & ": " & errorMess
            errorList.Add errorMess
        End If
    Next r
    Call SubFunc_WriteErrorList(errorList, 1, False) ' Col 1, start block false
End Sub

' Calculation for Spend UniqueCloud1 Breakdown table Exchange Rate
Public Sub SubFunc_ExchangeRateLogic(ptMT As PivotTable)
    Dim wsGuidelines As Worksheet, rngMT As Range, headerRowGuidelines As Range, errorMess As String, errorList As New Collection
    Dim col_GL_CountryFull As Long, col_GL_ExchangeRate As Long, i As Long, j As Long, r As Long
    Dim dictGLCountryExchangeRate As Object, MatchingExchangeRate As Double

    errorList.Add "MT vs DeliverData Spend"
    errorList.Add "UniqueCloud1 (Breakdown)"

    Set wsGuidelines = ThisWorkbook.Sheets("Guidelines")
    Set headerRowGuidelines = wsGuidelines.Rows(3)
    col_GL_CountryFull = Application.Match("Country Full", headerRowGuidelines, 0)
    col_GL_ExchangeRate = Application.Match("Exchange Rate", headerRowGuidelines, 0)
    lastRowCountryFull = wsGuidelines.Cells(Rows.Count, col_GL_CountryFull).End(xlUp).Row

    ' Create dictionary: Country Full-Exchange Rate
    Set dictGLCountryExchangeRate = CreateObject("Scripting.Dictionary")
    For j = 4 To lastRowCountryFull
        Dim countryFull As String, exRate As Variant
        countryFull = UCase(Trim(CStr(wsGuidelines.Cells(j, col_GL_CountryFull).Value)))
        exRate = wsGuidelines.Cells(j, col_GL_ExchangeRate).Value
        If Not dictGLCountryExchangeRate.Exists(countryFull) Then
            dictGLCountryExchangeRate.Add countryFull, exRate
        End If
    Next j
    ' Overwrite specific countries with exchange rate of 1
    dictGLCountryExchangeRate("INDONESIA") = 1
    dictGLCountryExchangeRate("PHILIPPINES") = 1
    dictGLCountryExchangeRate("VIETNAM") = 1
    'Call DebugPrintDict(dictGLCountryExchangeRate)

    Set rngMT = ptMT.TableRange1
    Set colIndex = SubFunc_GetColIndex(rngMT)
    For i = 2 To rngMT.Rows.Count - 1
        ' Calculate Exchange Rate = Media Cost (LC) / Media Cost (USD)
        cellValue_MediaCostLC = Trim(rngMT.Cells(i, colIndex("Sum of Media Cost (LC)")).Value)
        cellValue_MediaCostUSD = Trim(rngMT.Cells(i, colIndex("Sum of Media Cost (USD)")).Value)
        If IsNumeric(cellValue_MediaCostLC) Then ' For numeric Media Cost (LC)
            If IsNumeric(cellValue_MediaCostUSD) And cellValue_MediaCostUSD <> 0 Then
                cellValue_ExchangeRate = cellValue_MediaCostLC / cellValue_MediaCostUSD
            Else
                cellValue_ExchangeRate = "#DIV/0!" ' Write error
            End If
        Else ' For non-numeric Media Cost (LC)
            cellValue_ExchangeRate = "#N/A" ' Write error
        End If
        rngMT.Cells(i, colIndex("Sum of Media Cost (USD)") + 1).Value = cellValue_ExchangeRate
    Next i

    For r = 2 To rngMT.Rows.Count - 1
        Set cellName_Country = rngMT.Cells(r, colIndex("Country"))
        Set cellName_ExchangeRate = rngMT.Cells(r, colIndex("Sum of Media Cost (USD)") + 1)
        cellValue_Country = UCase(Trim(CStr(cellName_Country.Value)))
        cellValue_ExchangeRate = cellName_ExchangeRate.Value

        ' Match MT Country value South Korea to Korea
        If cellValue_Country = "SOUTH KOREA" Then cellValue_Country = "KOREA"
        
        If cellValue_Country <> "GRAND TOTAL" Then
            If dictGLCountryExchangeRate.Exists(cellValue_Country) Then
                MatchingExchangeRate = dictGLCountryExchangeRate(cellValue_Country)
                ' Check for empty Exchange Rate
                If Not IsNumeric(cellValue_ExchangeRate) Then
                    cellName_ExchangeRate.Interior.Color = ERR_HIG
                    errorMess = "Exchange Rate, Cell " & cellName_ExchangeRate.Address & ": " & "Exchange Rate must be numeric"
                    errorList.Add errorMess
                    GoTo NextIteration
                End If
                ' Skip if Exchange Rate is 0
                If cellValue_ExchangeRate = 0 Then
                    cellName_ExchangeRate.Interior.Color = ERR_HIG
                    errorMess = "Exchange Rate, Cell " & cellName_ExchangeRate.Address & ": " & "Exchange Rate cannot be zero"
                    errorList.Add errorMess
                    GoTo NextIteration
                End If
                ' Compare exchange rates (highlight if difference > 0.01%)
                Dim currentER As Double, guidelineER As Double, percentDiff As Double
                currentER = Round(CDbl(cellValue_ExchangeRate), 5)
                guidelineER = Round(CDbl(MatchingExchangeRate), 5)
                If guidelineER <> 0 Then
                    percentDiff = Abs((currentER - guidelineER) / guidelineER)
                    If percentDiff > 0.0001 Then
                        cellName_ExchangeRate.Interior.Color = ERR_HIG
                        errorMess = "Exchange Rate, Cell " & cellName_ExchangeRate.Address & ": " & "Exchange Rate cannot be more than 0.01% the Guidelines sheet Exchange Rate"
                        errorList.Add errorMess
                    End If
                End If
            Else
                cellName_ExchangeRate.Interior.Color = ERR_HIG
            End If
        End If
NextIteration:
    Next r
    Call SubFunc_WriteErrorList(errorList, 7, True) ' Col 7, start block True
End Sub

Public Sub SubFunc_WriteErrorList(errorList As Collection, writeCol_Index As Long, start_block As Boolean)
    Dim ws As Worksheet, wsNew As Worksheet, i As Long
    Dim WorksheetExists As Boolean, lastRow As Long

    WorksheetExists = False
    ' Check if "Issue List" worksheet exists
    For Each ws In ThisWorkbook.Worksheets
        If UCase(ws.Name) = UCase("Issue List") Then
            WorksheetExists = True
            Exit For
        End If
    Next ws

    ' If worksheet not found, create a brand new empty sheet
    If Not WorksheetExists Then
        Call SubFunc_CreateNewSheet("Issue List", wsNew)
    Else ' If worksheet found, set and continue
        Set wsNew = ThisWorkbook.Sheets("Issue List")
    End If
    wsNew.AutoFilterMode = False

    ' Get the current write column last row
    lastRow = wsNew.Cells(Rows.Count, writeCol_Index).End(xlUp).Row

    ' start_block is True for logic at the beginning of the column
    If start_block = True Then
        ' Always clear the Write column first
        wsNew.Range(wsNew.Cells(1, writeCol_Index), wsNew.Cells(lastRow, writeCol_Index)).Clear
        ' If there is error message list input in
        If Not errorList Is Nothing Then
            If errorList.Count > 0 Then
                For i = 1 To errorList.Count
                    wsNew.Cells(i, writeCol_Index).Value = errorList(i)
                Next i
            End If
        End If
        ' Format for the first 2 row header
        With wsNew.Range(wsNew.Cells(1, writeCol_Index), wsNew.Cells(2, writeCol_Index))
            .Font.Bold = True
            .Interior.ThemeColor = xlThemeColorAccent1
            .Interior.TintAndShade = 0.8  ' 80%
        End With
    ' start_block is False for logic continuing the last row of the column
    ElseIf start_block = False Then
        If Not errorList Is Nothing Then
            If errorList.Count > 0 Then
                For i = 1 To errorList.Count
                    ' start from the last row + 1
                    wsNew.Cells(lastRow + i, writeCol_Index).Value = errorList(i)
                Next i
            End If
        End If
    End If
    
    ' Check if the count of non-empty cells in Column 1 is zero
    'If Application.WorksheetFunction.CountA(wsNew.Columns(1)) = 0 Then
        'wsNew.Columns(1).Delete Shift:=xlToLeft
    'End If

    ' Remove the default $ sign from Excel address print
    wsNew.UsedRange.Replace What:="$", Replacement:="", LookAt:=xlPart, MatchCase:=False

    wsNew.UsedRange.Font.Name = "Arial"
    wsNew.UsedRange.Font.Size = 8
    wsNew.Columns.AutoFit
End Sub
"""

In [ ]:
Module_Macro_5 = """
Sub MT_vs_DeliverData_Spend()
    Dim wsMT As Worksheet, wsDeliverData As Worksheet, wsNew As Worksheet
    Dim ptMT As PivotTable, ptDeliverData As PivotTable, ptMT_2 As PivotTable, ptDeliverData_2 As PivotTable
    Dim pcMT As PivotCache, pcDeliverData As PivotCache, pcMT_2 As PivotCache, pcDeliverData_2 As PivotCache
    Dim dataMT As Range, dataDeliverData As Range
    Dim lastRowMT As Long, lastColMT As Long, lastRowDeliverData As Long, lastColDeliverData As Long, i As Long, j As Long
    Dim pivotStartCellMT As Range, pivotStartCellDeliverData As Range, pivotStartCellMT_2 As Range, pivotStartCellDeliverData_2 As Range
    Dim dictMT As Object, dictDeliverData As Object, colIndex As Object, dictYQ As Object
    Dim keyMT As String, keyDeliverData As String, fixed_quarter As String, fixed_country As String
    Dim rngMT As Range, rngDeliverData As Range, errorMess1 As String, errorMess2 As String, errorList1 As New Collection, errorList2 As New Collection
    Dim ColumnToSearchBlank As Variant, k As Variant, cell_StartDate As Variant, cell_EndDate As Variant
    Dim MT_cost_LC As Double, DeliverData_cost_LC As Double, perc As Double

    BenchMark = Timer
    Application.ScreenUpdating = False
    Set wsMT = ThisWorkbook.Sheets("UniqueCloud1")
    Set wsDeliverData = ThisWorkbook.Sheets("DeliverData")
    wsMT.AutoFilterMode = False
    wsDeliverData.AutoFilterMode = False

    Call SubFunc_CreateNewSheet("MT vs DeliverData Spend", wsNew)

    wsNew.Range("A1").Value = "UniqueCloud1 (Overview)"
    wsNew.Range("D1").Value = "DeliverData (Overview)"

    errorList1.Add "MT vs DeliverData Spend"
    errorList1.Add "UniqueCloud1 (Overview)"

    errorList2.Add "MT vs DeliverData Spend"
    errorList2.Add "DeliverData (Overview)"

    ' Define data range for UniqueCloud1
    lastRowMT = wsMT.Cells(Rows.Count, 1).End(xlUp).Row
    lastColMT = wsMT.Cells(1, Columns.Count).End(xlToLeft).Column
    Set dataMT = wsMT.Range(wsMT.Cells(1, 1), wsMT.Cells(lastRowMT, lastColMT))

    ' Define data range for DeliverData
    lastRowDeliverData = wsDeliverData.Cells(Rows.Count, 1).End(xlUp).Row
    lastColDeliverData = wsDeliverData.Cells(1, Columns.Count).End(xlToLeft).Column
    Set headerRowDeliverData = wsDeliverData.Rows(1)
    Set dataDeliverData = wsDeliverData.Range(wsDeliverData.Cells(1, 1), wsDeliverData.Cells(lastRowDeliverData, lastColDeliverData))

    ' Safe guard check for Date columns inside DeliverData sheet
    col_StartDate = Application.Match("Start Date", headerRowDeliverData, 0)
    col_EndDate = Application.Match("End Date", headerRowDeliverData, 0)
    For r = 2 To lastRowDeliverData
        Set cell_StartDate = wsDeliverData.Cells(r, col_StartDate)
        Set cell_EndDate = wsDeliverData.Cells(r, col_EndDate)
        ' Check if Start Date / End Date is not a valid date format
        If Not IsDate(cell_StartDate.Value) Then
            cell_StartDate.Interior.Color = ERR_HIG
            wsDeliverData.Select
            MsgBox "DeliverData Cell " & cell_StartDate.Address & ": " & "Start Date is invalid!", vbExclamation, "Date Validation Error"
            Exit Sub
        End If
        If Not IsDate(cell_EndDate.Value) Then
            cell_EndDate.Interior.Color = ERR_HIG
            wsDeliverData.Select
            MsgBox "DeliverData Cell " & cell_EndDate.Address & ": " & "End Date is invalid!", vbExclamation, "Date Validation Error"
            Exit Sub
        End If
        ' Check if both are valid dates and End Date < Start Date
        If CDate(cell_EndDate.Value) < CDate(cell_StartDate.Value) Then
            cell_EndDate.Interior.Color = ERR_HIG
            wsDeliverData.Select
            MsgBox "DeliverData Cell " & cell_EndDate.Address & ": " & "End Date is smaller than Start Date!", vbExclamation, "Date Validation Error"
            Exit Sub
        End If
    Next r

    ' Define Overview PivotTable start positions
    Set pivotStartCellMT = wsNew.Range("A6")
    Set pivotStartCellDeliverData = wsNew.Range("D6")

    ' Create PivotTable for UniqueCloud1
    Set pcMT = ThisWorkbook.PivotCaches.Create(SourceType:=xlDatabase, SourceData:=dataMT)
    Set ptMT = pcMT.CreatePivotTable(TableDestination:=pivotStartCellMT, TableName:="PivotUniqueCloud1")
    Call SubFunc_GetFixedQuarter(fixed_quarter)
    Call SubFunc_GetFixedCountry(fixed_country)

    DoEvents
    On Error Resume Next
    With ptMT
        With .PivotFields("Quarter")
            .Orientation = xlPageField
            On Error Resume Next
            .CurrentPage = fixed_quarter
            If Err.Number <> 0 Then
                Err.Clear
                On Error GoTo 0
                wsMT.Select
                MsgBox fixed_quarter & " not found in Quarter column, UniqueCloud1 sheet", vbExclamation, "Date Validation Error"
                Exit Sub
            End If
            On Error GoTo 0
        End With
        With .PivotFields("Country")
            .Orientation = xlPageField
            .CurrentPage = fixed_country
        End With
        With .PivotFields("Plan Name")
             .Orientation = xlRowField
             .Subtotals = Array(True, False, False, False, False, False, False, False, False, False, False, False)
             .Caption = "Plan Name"
        End With
        With .PivotFields("Publisher/Vendor")
             .Orientation = xlRowField
        End With
        With .PivotFields("Media Cost (LC)")
             .Orientation = xlDataField
             .Function = xlSum
             .NumberFormat = "#,##0"
        End With
    .RowAxisLayout xlTabularRow
    .TableStyle2 = "PivotStyleLight20"
    End With
    FileNameOnly = SubFunc_GetFileName(ThisWorkbook)
    If InStr(1, FileNameOnly, "-VIK - DeliverData -", vbTextCompare) > 0 Then
        With ptMT.PivotFields("Funding Source")
            .Orientation = xlPageField
            On Error Resume Next
            .CurrentPage = "Value In Kind (VIK)"
            If Err.Number <> 0 Then
                .CurrentPage = "(All)"
                Err.Clear
            End If
            On Error GoTo 0
        End With
    ElseIf InStr(1, FileNameOnly, "-UniqueClientName - DeliverData -", vbTextCompare) > 0 Then
        With ptMT.PivotFields("Funding Source")
            .Orientation = xlPageField
            On Error Resume Next
            .CurrentPage = "UniqueClientName/Core Marketing"
            If Err.Number <> 0 Then
                .CurrentPage = "(All)"
                Err.Clear
            End If
            On Error GoTo 0
        End With
    End If
    Call SubFunc_PivotRepeatLabel(ptMT, True) ' On/Show
    On Error GoTo 0
    ' Create PivotTable for DeliverData
    Set pcDeliverData = ThisWorkbook.PivotCaches.Create(SourceType:=xlDatabase, SourceData:=dataDeliverData)
    Set ptDeliverData = pcDeliverData.CreatePivotTable(TableDestination:=pivotStartCellDeliverData, TableName:="PivotDeliverData")
    DoEvents
    On Error Resume Next
    With ptDeliverData
        With .PivotFields("Plan Name")
             .Orientation = xlRowField
             .Subtotals = Array(True, False, False, False, False, False, False, False, False, False, False, False)
             .Caption = "Plan Name"
        End With
        With .PivotFields("Publisher/Vendor")
             .Orientation = xlRowField
             .Subtotals = Array(False, False, False, False, False, False, False, False, False, False, False, False)
        End With
        With .PivotFields("Category")
             .Orientation = xlRowField
             .Subtotals = Array(False, False, False, False, False, False, False, False, False, False, False, False)
        End With
        With .PivotFields("YQ")
             .Orientation = xlRowField
             .Subtotals = Array(False, False, False, False, False, False, False, False, False, False, False, False)
        End With
        With .PivotFields("Start Date")
             .Orientation = xlDataField
             .Function = xlMin
             .NumberFormat = "dd-mmm-yyyy"
             .Name = "Earliest Start Date"
        End With
        With .PivotFields("End Date")
             .Orientation = xlDataField
             .Function = xlMax
             .NumberFormat = "dd-mmm-yyyy"
             .Name = "Latest End Date"
        End With
        With .PivotFields("Start Date")
             .Orientation = xlDataField
             .Function = xlCount
             .NumberFormat = "#,##0"
             .Name = "Count of Start Date"
        End With
        With .PivotFields("Media Cost (LC)")
             .Orientation = xlDataField
             .Function = xlSum
             .NumberFormat = "#,##0"
        End With
        .RowAxisLayout xlTabularRow
    End With
    Call SubFunc_PivotRepeatLabel(ptDeliverData, True) ' On/Show
    On Error GoTo 0

    ' Set the pivot table ranges
    Set rngMT = ptMT.TableRange1
    Set rngDeliverData = ptDeliverData.TableRange1

    ' Create a dictionary to store UniqueCloud1 pivot table cost cell references keyed by "PlanName|Publisher/Vendor"
    Set dictMT = CreateObject("Scripting.Dictionary")
    Set dictDeliverData = CreateObject("Scripting.Dictionary")

    ' Loop through the UniqueCloud1 pivot table data rows (starting from row 2 to skip headers)
    For i = 2 To rngMT.Rows.Count
        keyMT = UCase(Trim(CStr(rngMT.Cells(i, 1).Value))) & "|" & UCase(Trim(CStr(rngMT.Cells(i, 2).Value)))
        ' Use Set when storing a Range object in the dictionary
        Set dictMT(keyMT) = rngMT.Cells(i, 3) ' Media Cost (LC)
    Next i

    ' Loop through the DeliverData pivot table data rows (starting from row 2 to skip headers)
    For j = 2 To rngDeliverData.Rows.Count
        keyDeliverData = UCase(Trim(CStr(rngDeliverData.Cells(j, 1).Value))) & "|" & UCase(Trim(CStr(rngDeliverData.Cells(j, 2).Value)))
        Set dictDeliverData(keyDeliverData) = rngDeliverData.Cells(j, 8) ' Media Cost (LC)
        If dictMT.Exists(keyDeliverData) Then
            ' Retrieve the values from the pivot table cells
            MT_cost_LC = dictMT(keyDeliverData).Value ' Sum of Media Cost (LC) from UniqueCloud1
            DeliverData_cost_LC = rngDeliverData.Cells(j, 8).Value ' Sum of Media Cost (LC) from DeliverData
            ' Calculate the percentage difference using MT_cost_LC as the base
            If MT_cost_LC <> 0 Then
                perc = Abs(MT_cost_LC - DeliverData_cost_LC) / Abs(MT_cost_LC)
            Else
                If DeliverData_cost_LC <> 0 Then
                    perc = 1
                Else
                    perc = 0
                End If
            End If
            ' If the absolute difference is greater than 5% of the UniqueCloud1 value, then highlight
            If perc > 0.05 Then
                'rngDeliverData.Cells(j, 9).Value = perc ' Print the percentage difference
                'rngDeliverData.Cells(j, 9).NumberFormat = "0.00%"

                dictMT(keyDeliverData).Interior.Color = ERR_HIG ' UniqueCloud1 cost cell
                errorMess1 = "Media Cost (LC), Cell " & dictMT(keyDeliverData).Address & ": " & "Media Cost (LC) must be within 5% variance against Media Cost (LC) in DeliverData (Overview) at Plan Name+Publisher/Vendor level"
                errorList1.Add errorMess1

                rngDeliverData.Cells(j, 8).Interior.Color = ERR_HIG ' DeliverData cost cell
                errorMess2 = "Media Cost (LC), Cell " & rngDeliverData.Cells(j, 8).Address & ": " & "Media Cost (LC) must be within 5% variance against Media Cost (LC) in UniqueCloud1 (Overview) at Plan Name+Publisher/Vendor level"
                errorList2.Add errorMess2
            End If
        End If
    Next j

    ' Flag Missing Keys Section
    ' a) For each key in UniqueCloud1 (dictMT) that is not found in DeliverData (dictDeliverData), highlight  UniqueCloud1 cell
    For Each k In dictMT.Keys
        If Not dictDeliverData.Exists(k) Then 
            dictMT(k).Interior.Color = ERR_HIG
            errorMess1 = "Media Cost (LC), Cell " & dictMT(k).Address & ": " & "Media Cost (LC) must match with Media Cost (LC) in DeliverData (Overview) at Plan Name+Publisher/Vendor level"
            errorList1.Add errorMess1
        End If
    Next k
    ' b) For each key in DeliverData (dictDeliverData) that is not found in UniqueCloud1 (dictMT), highlight  DeliverData cell
    For Each k In dictDeliverData.Keys
        If Not dictMT.Exists(k) Then 
            dictDeliverData(k).Interior.Color = ERR_HIG
            errorMess2 = "Media Cost (LC), Cell " & dictDeliverData(k).Address & ": " & "Media Cost (LC) must match with Media Cost (LC) in UniqueCloud1 (Overview) at Plan Name+Publisher/Vendor level"
            errorList2.Add errorMess2
        End If
    Next k

    Call SubFunc_WriteErrorList(errorList1, 4, True) ' Col 4, start block true
    Call SubFunc_WriteErrorList(errorList2, 5, True) ' Col 5, start block true

    ' Create 2 Breakdown pivot tables
    wsNew.Range("M1").Value = "UniqueCloud1 (Breakdown)"
    With wsNew.Range("V6") ' Set up the Exchange Rate column for MT Breakdown
        .Value = "Exchange Rate"
        .Font.Bold = True
        .Interior.ThemeColor = xlThemeColorAccent5
        .Interior.TintAndShade = 0.8  ' 80%
    End With
    wsNew.Range("W1").Value = "DeliverData (Breakdown)"
    With wsNew.Range("AY6") ' Set up the CPM column for DeliverData Breakdown
        .Value = "CPM/CPC/CPP"
        .Font.Bold = True
        .Interior.ThemeColor = xlThemeColorAccent1
        .Interior.TintAndShade = 0.8  ' 80%
    End With

    ' Define Breakdown PivotTable start positions
    Set pivotStartCellMT_2 = wsNew.Range("M6")
    Set pivotStartCellDeliverData_2 = wsNew.Range("W6")

    ' Create Breakdown PivotTable for UniqueCloud1
    Set pcMT_2 = ThisWorkbook.PivotCaches.Create(SourceType:=xlDatabase, SourceData:=dataMT)
    Set ptMT_2 = pcMT_2.CreatePivotTable(TableDestination:=pivotStartCellMT_2, TableName:="PivotUniqueCloud1_2")
    ' Ensure PivotTable is ready before accessing fields
    DoEvents
    With ptMT_2
        With .PivotFields("Quarter")
            .Orientation = xlPageField
            On Error Resume Next
            .CurrentPage = fixed_quarter
            If Err.Number <> 0 Then
                Err.Clear
                On Error GoTo 0
                wsMT.Select
                MsgBox fixed_quarter & " not found in Quarter column, UniqueCloud1 sheet", vbExclamation, "Date Validation Error"
                Exit Sub
            End If
            On Error GoTo 0
        End With
        With .PivotFields("Country")
             .Orientation = xlRowField
             .Subtotals = Array(False, False, False, False, False, False, False, False, False, False, False, False)
        End With
        With .PivotFields("Plan Name")
             .Orientation = xlRowField
             .Subtotals = Array(False, False, False, False, False, False, False, False, False, False, False, False)
        End With
        With .PivotFields("Publisher/Vendor")
             .Orientation = xlRowField
             .Subtotals = Array(False, False, False, False, False, False, False, False, False, False, False, False)
        End With
        With .PivotFields("Vehicle Name")
             .Orientation = xlRowField
             .Subtotals = Array(False, False, False, False, False, False, False, False, False, False, False, False)
        End With
        With .PivotFields("Media Channel")
             .Orientation = xlRowField
             .Subtotals = Array(False, False, False, False, False, False, False, False, False, False, False, False)
        End With
        With .PivotFields("Inventory (Distribution) Type")
             .Orientation = xlRowField
             .Subtotals = Array(False, False, False, False, False, False, False, False, False, False, False, False)
        End With

        With .PivotFields("Promotion Incentive")
             .Orientation = xlRowField
             .Subtotals = Array(False, False, False, False, False, False, False, False, False, False, False, False)
        End With
        With .PivotFields("Media Cost (LC)")
             .Orientation = xlDataField
             .Function = xlSum
             .NumberFormat = "#,##0"
        End With
        With .PivotFields("Media Cost (USD)")
             .Orientation = xlDataField
             .Function = xlSum
             .NumberFormat = "#,##0"
        End With
        .RowAxisLayout xlTabularRow
        .TableStyle2 = "PivotStyleLight20"
    End With
    If InStr(1, FileNameOnly, "-VIK - DeliverData -", vbTextCompare) > 0 Then
        With ptMT_2.PivotFields("Funding Source")
            .Orientation = xlPageField
            On Error Resume Next
            .CurrentPage = "Value In Kind (VIK)"
            If Err.Number <> 0 Then
                .CurrentPage = "(All)"
                Err.Clear
            End If
            On Error GoTo 0
        End With

    ElseIf InStr(1, FileNameOnly, "-UniqueClientName - DeliverData -", vbTextCompare) > 0 Then
        With ptMT_2.PivotFields("Funding Source")
            .Orientation = xlPageField
            On Error Resume Next
            .CurrentPage = "UniqueClientName/Core Marketing"
            If Err.Number <> 0 Then
                .CurrentPage = "(All)"
                Err.Clear
            End If
            On Error GoTo 0
        End With
    End If

    ' Filter Country inside the pivot table itself, not an outside filter
    With ptMT_2.PivotFields("Country")
        For Each pi In .PivotItems
            pi.Visible = True
        Next pi
        For Each pi In .PivotItems
            If fixed_country <> "(All)" And pi.Value <> fixed_country Then
                pi.Visible = False
            End If
        Next pi
    End With

    Call SubFunc_PivotRepeatLabel(ptMT_2, True) ' On/Show
    ' Create PivotTable for DeliverData (Full Breakdown)
    Set pcDeliverData_2 = ThisWorkbook.PivotCaches.Create(SourceType:=xlDatabase, SourceData:=dataDeliverData)
    Set ptDeliverData_2 = pcDeliverData_2.CreatePivotTable(TableDestination:=pivotStartCellDeliverData_2, TableName:="PivotDeliverData_2")
    DoEvents ' Ensure PivotTable is ready before accessing fields
    With ptDeliverData_2
        With .PivotFields("Country")
             .Orientation = xlRowField
             .Subtotals = Array(False, False, False, False, False, False, False, False, False, False, False, False)
        End With
        With .PivotFields("Plan Name")
             .Orientation = xlRowField
             .Subtotals = Array(False, False, False, False, False, False, False, False, False, False, False, False)
        End With
        With .PivotFields("Campaign")
             .Orientation = xlRowField
             .Subtotals = Array(False, False, False, False, False, False, False, False, False, False, False, False)
        End With
        With .PivotFields("Campaign Objective")
             .Orientation = xlRowField
             .Subtotals = Array(False, False, False, False, False, False, False, False, False, False, False, False)
        End With
        With .PivotFields("Cohorts")
             .Orientation = xlRowField
             .Subtotals = Array(False, False, False, False, False, False, False, False, False, False, False, False)
        End With
        With .PivotFields("Publisher/Vendor")
             .Orientation = xlRowField
             .Subtotals = Array(False, False, False, False, False, False, False, False, False, False, False, False)
        End With
        With .PivotFields("Vehicle Name")
             .Orientation = xlRowField
             .Subtotals = Array(False, False, False, False, False, False, False, False, False, False, False, False)
        End With
        With .PivotFields("Media Channel")
             .Orientation = xlRowField
             .Subtotals = Array(False, False, False, False, False, False, False, False, False, False, False, False)
        End With
        With .PivotFields("Inventory (Distribution) Type")
             .Orientation = xlRowField
             .Subtotals = Array(False, False, False, False, False, False, False, False, False, False, False, False)
        End With
        With .PivotFields("Primary Target Strategy")
             .Orientation = xlRowField
             .Subtotals = Array(False, False, False, False, False, False, False, False, False, False, False, False)
        End With
        With .PivotFields("Promotion Incentive")
             .Orientation = xlRowField
             .Subtotals = Array(False, False, False, False, False, False, False, False, False, False, False, False)
        End With
        With .PivotFields("Passion Pillar (Placement-Level)")
             .Orientation = xlRowField
             .Subtotals = Array(False, False, False, False, False, False, False, False, False, False, False, False)
        End With
        With .PivotFields("Initiative (Placement-Level)")
             .Orientation = xlRowField
             .Subtotals = Array(False, False, False, False, False, False, False, False, False, False, False, False)
        End With
        With .PivotFields("Media Market")
             .Orientation = xlRowField
             .Subtotals = Array(False, False, False, False, False, False, False, False, False, False, False, False)
        End With
        With .PivotFields("Passion Pillar (Creative-Level)")
             .Orientation = xlRowField
             .Subtotals = Array(False, False, False, False, False, False, False, False, False, False, False, False)
        End With
        With .PivotFields("Initiative (Creative-Level)")
             .Orientation = xlRowField
             .Subtotals = Array(False, False, False, False, False, False, False, False, False, False, False, False)
        End With
        With .PivotFields("Category")
             .Orientation = xlRowField
             .Subtotals = Array(False, False, False, False, False, False, False, False, False, False, False, False)
        End With
        With .PivotFields("Notes")
             .Orientation = xlRowField
             .Subtotals = Array(False, False, False, False, False, False, False, False, False, False, False, False)
        End With
        With .PivotFields("Does Media Cost include Production Fees?")
             .Orientation = xlRowField
             .Subtotals = Array(False, False, False, False, False, False, False, False, False, False, False, False)
        End With
        With .PivotFields("Currency")
             .Orientation = xlRowField
             .Subtotals = Array(False, False, False, False, False, False, False, False, False, False, False, False)
        End With
        With .PivotFields("Media Cost (LC)")
             .Orientation = xlDataField
             .Function = xlSum
             .NumberFormat = "#,##0"
        End With
        With .PivotFields("Paid Impressions")
             .Orientation = xlDataField
             .Function = xlSum
             .NumberFormat = "#,##0"
        End With
        With .PivotFields("Organic Impressions")
             .Orientation = xlDataField
             .Function = xlSum
             .NumberFormat = "#,##0"
        End With
        With .PivotFields("Clicks")
             .Orientation = xlDataField
             .Function = xlSum
             .NumberFormat = "#,##0"
        End With
        With .PivotFields("Video Views")
             .Orientation = xlDataField
             .Function = xlSum
             .NumberFormat = "#,##0"
        End With
        With .PivotFields("Video Completes")
             .Orientation = xlDataField
             .Function = xlSum
             .NumberFormat = "#,##0"
        End With
        With .PivotFields("TV National GRPs")
             .Orientation = xlDataField
             .Function = xlSum
             .NumberFormat = "#,##0.00"
        End With
        With .PivotFields("TV Duration (seconds)")
             .Orientation = xlDataField
             .Function = xlAverage
             .NumberFormat = "#,##0"
        End With
        .RowAxisLayout xlTabularRow
    End With
    Call SubFunc_PivotRepeatLabel(ptDeliverData_2, True) ' On/Show

    ' Appy additional complex conditional formatting logic
    Call SubFunc_ConFor_Spend_DeliverData_Overview(ptDeliverData) ' Few conditions for DeliverData Overview table
    Call SubFunc_ExchangeRateLogic(ptMT_2) ' Calculate the Exchange Rate value for MT Breakdown table
    Call SubFunc_ConFor_Spend_DeliverData_Breakdown(ptDeliverData_2) ' Lots of conditions for DeliverData Breakdown table
    Call SubFunc_ConFor_Spend_DeliverData_Breakdown_IDT(ptDeliverData_2) ' Extra condition for DeliverData Breakdown table, Inventory Type

    Call SubFunc_PivotRepeatLabel(ptMT, False) ' Off/Hide
    Call SubFunc_PivotRepeatLabel(ptDeliverData, False) ' Off/Hide
    wsNew.Columns(22).NumberFormat = "#,##0.00000" ' Apply extra format for MT Breakdown, Exchange Rate column
    wsNew.Columns(44).NumberFormat = "#,##0.00" ' Apply extra format for DeliverData Breakdown, CPM column
    wsNew.UsedRange.Font.Name = "Arial"
    wsNew.UsedRange.Font.Size = 8
    wsNew.Columns.AutoFit
    Call SubFunc_SetZoomLevel(ThisWorkbook)
    wsNew.Select
    totalRunTime = Round((Timer - BenchMark) / 60, 2)
    Application.ScreenUpdating = True
    ' Show the MsgBox by default, when no showMsg indicated
    If Not showMsg Then MsgBox "Compare UniqueCloud1 vs DeliverData Spend" & vbCrLf & _
           "Total run time: " & totalRunTime & " minutes", vbInformation, "Complete"
End Sub

' Conditional Formatting Logic for Spend DeliverData Overview table
Public Sub SubFunc_ConFor_Spend_DeliverData_Overview(ptDeliverData As PivotTable)
    Dim rngDeliverData As Range, errorMess As String, errorList As New Collection
    ' Get Min and Max Date Range from the file name
    Dim output_WeekStartRange As Date, output_WeekEndRange As Date, output_DayStartRange As Date, output_DayEndRange As Date
    Call SubFunc_ExtractFileDateRange(output_WeekStartRange, output_WeekEndRange, output_DayStartRange, output_DayEndRange)

    ' Get the YQ dictionary
    Set dictYQ = SubFunc_GetDictionary("YQ Full")

    Set rngDeliverData = ptDeliverData.TableRange1
    Set colIndex = SubFunc_GetColIndex(rngDeliverData) ' Initilize the column index of the table range
    ' Start from row 2 (skip the header) to last row (skip the Grand Total)
    For r = 2 To rngDeliverData.Rows.Count - 1

        ' Set up the cell name by row and column
        Set cellName_Category = rngDeliverData.Cells(r, colIndex("Category"))
        Set cellName_YQ = rngDeliverData.Cells(r, colIndex("YQ"))
        Set cellName_EarliestStartDate = rngDeliverData.Cells(r, colIndex("Earliest Start Date"))
        Set cellName_LatestEndDate = rngDeliverData.Cells(r, colIndex("Latest End Date"))
        Set cellName_CountofStartDate = rngDeliverData.Cells(r, colIndex("Count of Start Date"))

        ' Set up the cell value with some cleaning
        cellValue_Category = UCase(Trim(CStr(cellName_Category.Value)))
        cellValue_YQ = UCase(Trim(CStr(cellName_YQ.Value)))
        cellValue_EarliestStartDate = cellName_EarliestStartDate.Value
        cellValue_LatestEndDate = cellName_LatestEndDate.Value
        cellValue_CountofStartDate = Trim(cellName_CountofStartDate.Value)

        ' Display the Min and Max Date Range pulled earlier on top of the columns
        'rngDeliverData.Cells(0, colIndex("Earliest Start Date")) = "Min Range: " & Format(output_WeekStartRange, "dd-mmm-yyyy")
        'rngDeliverData.Cells(0, colIndex("Latest End Date")) = "Max Range: " & Format(output_WeekEndRange, "dd-mmm-yyyy")

        ' Find YQ value not empty and not exist in dictionary
        If Not dictYQ.Exists(cellValue_YQ) And cellValue_YQ <> "" Then 
            cellName_YQ.Interior.Color = ERR_HIG
            errorMess = "YQ, Cell " & cellName_YQ.Address & ": " & "YQ must exist in Dictionary under YQ"
            errorList.Add errorMess
        End If

        ' Find Start Date value smaller than Earliest Start Date
        If cellValue_EarliestStartDate < output_WeekStartRange Then 
            cellName_EarliestStartDate.Interior.Color = ERR_HIG
            errorMess = "Earliest Start Date, Cell " & cellName_EarliestStartDate.Address & ": " & "Min Start Date cannot be smaller than Earliest Start Date"
            errorList.Add errorMess
        End If

        ' Find End Date value larger than Latest End Date
        If cellValue_LatestEndDate > output_WeekEndRange Then 
            cellName_LatestEndDate.Interior.Color = ERR_HIG
            errorMess = "Latest End Date, Cell " & cellName_LatestEndDate.Address & ": " & "Max End Date cannot be bigger than Latest End Date"
            errorList.Add errorMess
        End If
        
        ' Find Category Offline Monthly, valid dates, Start Date & End Date difference > 3 months
        'If cellValue_Category = "OFFLINE-MONTHLY" And IsDate(cellValue_EarliestStartDate) And IsDate(cellValue_LatestEndDate) Then
            'If DateDiff("m", DateSerial(Year(cellValue_EarliestStartDate), Month(cellValue_EarliestStartDate), 1), DateSerial(Year(cellValue_LatestEndDate), Month(cellValue_LatestEndDate), 1)) > 3 Then
                'cellName_LatestEndDate.Interior.Color = ERR_HIG
                'errorMess = "Latest End Date, Cell " & cellName_LatestEndDate.Address & ": " & "If Category is Offline-Monthly, Start Date and End Date difference cannot be more than 3 months"
                'errorList.Add errorMess
            'End If
        'End If
        
        ' Find Count of Start Date = 1
        If cellValue_CountofStartDate = 1 Then 
            cellName_CountofStartDate.Interior.Color = ERR_HIG
            errorMess = "Count of Start Date, Cell " & cellName_CountofStartDate.Address & ": " & "Count of Start Date cannot be 1"
            errorList.Add errorMess
        End If
    Next r
    Call SubFunc_WriteErrorList(errorList, 5, False) ' Col 5, start block false
End Sub

' Conditional Formatting Logic for Spend DeliverData Breakdown table
Public Sub SubFunc_ConFor_Spend_DeliverData_Breakdown(ptDeliverData As PivotTable)
    Dim rngDeliverData As Range, headerRowGuidelines As Range, errorMess As String, errorList As New Collection
    Dim wsGuidelines As Worksheet, lastRowMediaMarket As Long, lastRowCurrency As Long
    Dim i As Long, j As Long, mapKey1 As String, mapValue1 As String

    errorList.Add "MT vs DeliverData Spend"
    errorList.Add "DeliverData (Breakdown)"

    ' Set up all the dictionaries from the Guidelines sheet, based on the header name
    Set dictCountry = SubFunc_GetDictionary("Country Full")
    Set dictCampaignObjective = SubFunc_GetDictionary("Campaign Objective Full")
    Set dictCohorts = SubFunc_GetDictionary("Cohorts Full")
    Set dictPublisherOnline = SubFunc_GetDictionary("Publisher/Vendor (Online) Full")
    Set dictPublisherOffline = SubFunc_GetDictionary("Publisher/Vendor (Offline) Full")
    Set dictVehicleNameOnline = SubFunc_GetDictionary("Vehicle Name (Online) Full")
    Set dictVehicleNameOffline = SubFunc_GetDictionary("Vehicle Name (Offline) Full")
    Set dictMediaChannel = SubFunc_GetDictionary("Media Channel Full")
    Set dictInventoryDistributionType = SubFunc_GetDictionary("Inventory (Distribution) Type Full")
    Set dictPrimaryTargetStrategy = SubFunc_GetDictionary("Primary Target Strategy Full")
    Set dictPromotionIncentive = SubFunc_GetDictionary("Promotion Incentive Full")
    Set dictPassionPillarPlacement = SubFunc_GetDictionary("Passion Pillar (Placement-Level) Full")
    Set dictInitiativePlacement = SubFunc_GetDictionary("Initiative (Placement-Level) Full")
    Set dictMediaMarket = SubFunc_GetDictionary("Media Market Full")
    Set dictPassionPillarCreative = SubFunc_GetDictionary("Passion Pillar (Creative-Level) Full")
    Set dictInitiativeCreative = SubFunc_GetDictionary("Initiative (Creative-Level) Full")
    Set dictCategory = SubFunc_GetDictionary("Category Full")
    Set dictCurrency = SubFunc_GetDictionary("Currency Code")

    ' Set up the Currency dictionary and its corresponding Country Rules
    Set wsGuidelines = ThisWorkbook.Sheets("Guidelines")
    Set headerRowGuidelines = wsGuidelines.Rows(3)
    colCurrencyCode = Application.Match("Currency Code", headerRowGuidelines, 0)
    colCurrencyCodeRule = Application.Match("Country Full", headerRowGuidelines, 0)
    lastRowCurrency = wsGuidelines.Cells(wsGuidelines.Rows.Count, colCurrencyCode).End(xlUp).Row
    Set CurrencyMap = CreateObject("Scripting.Dictionary")
    With CurrencyMap
        For j = 4 To lastRowCurrency ' From row 4
            mapKey2 = UCase(Trim(CStr(wsGuidelines.Cells(j, colCurrencyCode).Value)))
            mapValue2 = UCase(Trim(CStr(wsGuidelines.Cells(j, colCurrencyCodeRule).Value)))
            If mapKey2 <> "" Then
                If Not .Exists(mapKey2) Then ' Add in the unique key first
                    .Add mapKey2, mapValue2
                Else ' For duplicate Currency values, concatenate their country rules together, instead of skip
                    .item(mapKey2) = .item(mapKey2) & ", " & mapValue2
                End If
            End If
        Next j
    End With

    ' Set up the Social Publisher/Vehicle Name List
    SocialPublisher_List = Array("META FACEBOOK", "META INSTAGRAM", "LINKEDIN", "THREADS", "TIKTOK", "SNAPCHAT", "WECHAT", "X (TWITTER)", "KAKAO", "KAKAO BIZBOARD", "KAKAO MOMENT", "KAKAOTALK")

    ' Set up the Programmatic Publisher/Vehicle Name List
    ProgrammaticPublisher_List = Array("APEX", "GOOGLE OPEN EXCHANGE (DV360)", "SOJERN", "TEADS", "THE TRADE DESK")

    ' Set up the Category Offline Monthly Media Channel List, no TV
    OfflineMonthlyMediaChannel_List = Array("AIRPORT", "OOH-OTHER", "CINEMA", "RADIO", "MAGAZINES", "NEWSPAPERS", "INSTORE/POS", "DIGITAL OOH")

    ' Set up the Social Media Channel List
    SocialMediaChannel_List = Array("PAID SOCIAL DISPLAY", "PAID SOCIAL VIDEO", "INFLUENCER/CREATOR DISPLAY", "INFLUENCER/CREATOR VIDEO")

    Set rngDeliverData = ptDeliverData.TableRange1
    Set colIndex = SubFunc_GetColIndex(rngDeliverData) ' Initilize the column index of the table range
    ' Start from row 2 (skip the header) to last row (skip the Grand Total)
    For r = 2 To rngDeliverData.Rows.Count - 1

        ' Set up the cell name by row and column
        Set cellName_Country = rngDeliverData.Cells(r, colIndex("Country"))
        Set cellName_PlanName = rngDeliverData.Cells(r, colIndex("Plan Name"))
        Set cellName_CampaignObjective = rngDeliverData.Cells(r, colIndex("Campaign Objective"))
        Set cellName_Cohorts = rngDeliverData.Cells(r, colIndex("Cohorts"))
        Set cellName_Publisher = rngDeliverData.Cells(r, colIndex("Publisher/Vendor"))
        Set cellName_VehicleName = rngDeliverData.Cells(r, colIndex("Vehicle Name"))
        Set cellName_MediaChannel = rngDeliverData.Cells(r, colIndex("Media Channel"))
        Set cellName_InventoryType = rngDeliverData.Cells(r, colIndex("Inventory (Distribution) Type"))
        Set cellName_PrimaryTargetStrategy = rngDeliverData.Cells(r, colIndex("Primary Target Strategy"))
        Set cellName_PromotionIncentive = rngDeliverData.Cells(r, colIndex("Promotion Incentive"))
        Set cellName_PassionPillarPlacement = rngDeliverData.Cells(r, colIndex("Passion Pillar (Placement-Level)"))
        Set cellName_InitiativePlacement = rngDeliverData.Cells(r, colIndex("Initiative (Placement-Level)"))
        Set cellName_MediaMarket = rngDeliverData.Cells(r, colIndex("Media Market"))
        Set cellName_PassionPillarCreative = rngDeliverData.Cells(r, colIndex("Passion Pillar (Creative-Level)"))
        Set cellName_InitiativeCreative = rngDeliverData.Cells(r, colIndex("Initiative (Creative-Level)"))
        Set cellName_Category = rngDeliverData.Cells(r, colIndex("Category"))
        Set cellName_Notes = rngDeliverData.Cells(r, colIndex("Notes"))
        Set cellName_DMCIPF = rngDeliverData.Cells(r, colIndex("Does Media Cost include Production Fees?"))
        Set cellName_Currency = rngDeliverData.Cells(r, colIndex("Currency"))
        Set cellName_MediaCostLC = rngDeliverData.Cells(r, colIndex("Sum of Media Cost (LC)"))
        Set cellName_PaidImpressions = rngDeliverData.Cells(r, colIndex("Sum of Paid Impressions"))
        Set cellName_Clicks = rngDeliverData.Cells(r, colIndex("Sum of Clicks"))
        Set cellName_TVGRP = rngDeliverData.Cells(r, colIndex("Sum of TV National GRPs"))
        Set cellName_TVDuration = rngDeliverData.Cells(r, colIndex("Average of TV Duration (seconds)"))
        ' Determine the CPM column after the TV Duration column
        Set cellName_CPMCPCCPP = rngDeliverData.Cells(r, colIndex("Average of TV Duration (seconds)") + 1)

        ' Set up the cell value with some cleaning
        cellValue_Country = UCase(Trim(CStr(cellName_Country.Value)))
        cellValue_PlanName = UCase(Trim(CStr(cellName_PlanName.Value)))
        cellValue_CampaignObjective = UCase(Trim(CStr(cellName_CampaignObjective.Value)))
        cellValue_Cohorts = UCase(Trim(CStr(cellName_Cohorts.Value)))
        cellValue_Publisher = UCase(Trim(CStr(cellName_Publisher.Value)))
        cellValue_VehicleName = UCase(Trim(CStr(cellName_VehicleName.Value)))
        cellValue_MediaChannel = UCase(Trim(CStr(cellName_MediaChannel.Value)))
        cellValue_InventoryType = UCase(Trim(CStr(cellName_InventoryType.Value)))
        cellValue_PrimaryTargetStrategy = UCase(Trim(CStr(cellName_PrimaryTargetStrategy.Value)))
        cellValue_PromotionIncentive = UCase(Trim(CStr(cellName_PromotionIncentive.Value)))
        cellValue_PassionPillarPlacement = UCase(Trim(CStr(cellName_PassionPillarPlacement.Value)))
        cellValue_InitiativePlacement = UCase(Trim(CStr(cellName_InitiativePlacement.Value)))
        cellValue_MediaMarket = UCase(Trim(CStr(cellName_MediaMarket.Value)))
        cellValue_PassionPillarCreative = UCase(Trim(CStr(cellName_PassionPillarCreative.Value)))
        cellValue_InitiativeCreative = UCase(Trim(CStr(cellName_InitiativeCreative.Value)))
        cellValue_Category = UCase(Trim(CStr(cellName_Category.Value)))
        cellValue_Notes = UCase(Trim(CStr(cellName_Notes.Value)))
        cellValue_DMCIPF = UCase(Trim(CStr(cellName_DMCIPF.Value)))
        cellValue_Currency = UCase(Trim(CStr(cellName_Currency.Value)))

        ' Set up the metric cell value with some cleaning
        cellValue_MediaCostLC = Trim(cellName_MediaCostLC.Value)
        cellValue_PaidImpressions = Trim(cellName_PaidImpressions.Value)
        cellValue_Clicks = Trim(cellName_Clicks.Value)
        cellValue_TVGRP = Trim(cellName_TVGRP.Value)
        cellValue_TVDuration = Trim(cellName_TVDuration.Value)
        cellValue_CPMCPCCPP = Trim(cellName_CPMCPCCPP.Value)

        ' Find Country value not exist in dictionary
        If Not dictCountry.Exists(cellValue_Country) Then 
            cellName_Country.Interior.Color = ERR_HIG
            errorMess = "Country, Cell " & cellName_Country.Address & ": " & "Country must exist in Dictionary under Country"
            errorList.Add errorMess
        End If

        ' Find Plan Name value blank, empty
        If cellValue_PlanName = "" Or cellValue_PlanName = "(BLANK)" Or cellValue_PlanName = "#N/A" Or cellValue_PlanName = "N/A" Then 
            cellName_PlanName.Interior.Color = ERR_HIG
            errorMess = "Plan Name, Cell " & cellName_PlanName.Address & ": " & "Plan Name cannot be blank"
            errorList.Add errorMess
        End If

        ' Find Campaign Objective value not exist in dictionary
        If Not dictCampaignObjective.Exists(cellValue_CampaignObjective) Then 
            cellName_CampaignObjective.Interior.Color = ERR_HIG
            errorMess = "Campaign Objective, Cell " & cellName_CampaignObjective.Address & ": " & "Campaign Objective must exist in Dictionary under Campaign Objective"
            errorList.Add errorMess
        End If

        ' Find Cohorts value not exist in dictionary
        If Not dictCohorts.Exists(cellValue_Cohorts) Then 
            cellName_Cohorts.Interior.Color = ERR_HIG
            errorMess = "Cohorts, Cell " & cellName_Cohorts.Address & ": " & "Cohorts must exist in Dictionary under Cohorts"
            errorList.Add errorMess
        End If

        ' Find Catergory Online, Publisher & Vehicle Name value not exist in dictionary
        If InStr(1, cellValue_Category, "ONLINE", vbTextCompare) > 0 Then
            If Not dictPublisherOnline.Exists(cellValue_Publisher) Then 
                cellName_Publisher.Interior.Color = ERR_HIG
                errorMess = "Publisher/Vendor, Cell " & cellName_Publisher.Address & ": " & "Publisher/Vendor must exist in Dictionary under Publisher/Vendor (Online) if Category contains Online"
                errorList.Add errorMess
            End If
        End If

        ' Find Catergory Offline, Publisher & Vehicle Name value not exist in dictionary
        If InStr(1, cellValue_Category, "OFFLINE", vbTextCompare) > 0 Then
            If Not dictPublisherOffline.Exists(cellValue_Publisher) Then 
                cellName_Publisher.Interior.Color = ERR_HIG
                errorMess = "Publisher/Vendor, Cell " & cellName_Publisher.Address & ": " & "Publisher/Vendor must exist in Dictionary under Publisher/Vendor (Offline) if Category contains Offline"
                errorList.Add errorMess
            End If
        End If

        ' Find Media Channel = TV, Publisher not Pay TV/Cable and Broadcast
        If cellValue_MediaChannel = "TV" Then
            If cellValue_Publisher <> "PAY TV/CABLE" And cellValue_Publisher <> "BROADCAST" Then
                cellName_Publisher.Interior.Color = ERR_HIG
                errorMess = "Publisher/Vendor, Cell " & cellName_Publisher.Address & ": " & "Publisher/Vendor must be Pay TV/Cable or Broadcast if Media Channel is TV"
                errorList.Add errorMess
            End If
        End If

        ' Find Media Channel = Airport, Publisher & Vehicle Name do not contain Airport
        If cellValue_MediaChannel = "AIRPORT" Then
            If InStr(1, cellValue_Publisher, "AIRPORT", vbTextCompare) = 0 And InStr(1, cellValue_VehicleName, "AIRPORT", vbTextCompare) = 0 Then
                cellName_Publisher.Interior.Color = ERR_HIG
                errorMess = "Publisher/Vendor, Cell " & cellName_Publisher.Address & ": " & "Publisher/Vendor or Vehicle Name must contain Airport if Media Channel is Airport"
                errorList.Add errorMess
            End If
        End If

        ' Find Catergory Online, Publisher & Vehicle Name value not exist in dictionary
        If InStr(1, cellValue_Category, "ONLINE", vbTextCompare) > 0 Then
            If Not dictVehicleNameOnline.Exists(cellValue_VehicleName) Then 
                cellName_VehicleName.Interior.Color = ERR_HIG
                errorMess = "Vehicle Name, Cell " & cellName_VehicleName.Address & ": " & "Vehicle Name must exist in Dictionary under Vehicle Name (Online) if Category contains Online"
                errorList.Add errorMess
            End If
        End If

        ' Find Catergory Offline, Publisher & Vehicle Name value not exist in dictionary
        If InStr(1, cellValue_Category, "OFFLINE", vbTextCompare) > 0 Then
            If Not dictVehicleNameOffline.Exists(cellValue_VehicleName) Then 
                cellName_VehicleName.Interior.Color = ERR_HIG
                errorMess = "Vehicle Name, Cell " & cellName_VehicleName.Address & ": " & "Vehicle Name must exist in Dictionary under Vehicle Name (Offline) if Category contains Offline"
                errorList.Add errorMess
            End If
        End If

        ' Find Vehicle Name value = Publisher value
        If UCase(cellValue_VehicleName) = UCase(cellValue_Publisher) Then
            cellName_VehicleName.Interior.Color = ERR_HIG
            errorMess = "Vehicle Name, Cell " & cellName_VehicleName.Address & ": " & "Vehicle Name cannot be the same as Publisher/Vendor"
            errorList.Add errorMess
        End If

        ' Find Media Channel value not exist in dictionary
        If Not dictMediaChannel.Exists(cellValue_MediaChannel) Then 
            cellName_MediaChannel.Interior.Color = ERR_HIG
            errorMess = "Media Channel, Cell " & cellName_MediaChannel.Address & ": " & "Media Channel must exist in Dictionary under Media Channel"
            errorList.Add errorMess
        End If

        ' Find Publisher or Vehicle Name in SocialPublisher_List
        If IsInArray(cellValue_Publisher, SocialPublisher_List) Or IsInArray(cellValue_VehicleName, SocialPublisher_List) Then
            ' Find Media Channel not in SocialMediaChannel_List
            If Not IsInArray(cellValue_MediaChannel, SocialMediaChannel_List) Then
                cellName_MediaChannel.Interior.Color = ERR_HIG
                errorMess = "Media Channel, Cell " & cellName_MediaChannel.Address & ": " & "Media Channel must be Paid Social Display, Paid Social Video, Influencer/Creator Display or Influencer/Creator Video if Publisher/Vendor or Vehicle Name is Meta Facebook, Meta Instagram, Kakao, Kakao Bizboard, Kakao Moment, Kakaotalk, LinkedIn, Snapchat, Threads, TikTok, WeChat or X (Twitter)"
                errorList.Add errorMess
            End If
        End If

        ' Find Publisher = Pay TV/Cable or Broadcast, Media Channel not TV
        If cellValue_Publisher = "PAY TV/CABLE" Or cellValue_Publisher = "BROADCAST" Then
            If cellValue_MediaChannel <> "TV" Then
                cellName_MediaChannel.Interior.Color = ERR_HIG
                errorMess = "Media Channel, Cell " & cellName_MediaChannel.Address & ": " & "Media Channel must be TV if Publisher/Vendor is Pay TV/Cable or Broadcast"
                errorList.Add errorMess
            End If
        End If

        ' Find Inventory Type = Social, Media Channel not in SocialMediaChannel_List
        If cellValue_InventoryType = "SOCIAL" Then
            If Not IsInArray(cellValue_MediaChannel, SocialMediaChannel_List) Then
                cellName_MediaChannel.Interior.Color = ERR_HIG
                errorMess = "Media Channel, Cell " & cellName_MediaChannel.Address & ": " & "Media Channel must be Paid Social Display, Paid Social Video, Influencer/Creator Display or Influencer/Creator Video if Inventory (Distribution) Type is Social"
                errorList.Add errorMess
            End If
        End If

        ' Find Inventory Type = Search, Media Channel not Search
        If cellValue_InventoryType = "SEARCH" Then
            If cellValue_MediaChannel <> "SEARCH" Then
                cellName_MediaChannel.Interior.Color = ERR_HIG
                errorMess = "Media Channel, Cell " & cellName_MediaChannel.Address & ": " & "Media Channel must be Search if Inventory (Distribution) Type is Search"
                errorList.Add errorMess
            End If
        End If

        ' Find Category = Offline-Weekly, Media Channel not TV
        If cellValue_Category = "OFFLINE-WEEKLY" Then
            If cellValue_MediaChannel <> "TV" Then
                cellName_MediaChannel.Interior.Color = ERR_HIG
                errorMess = "Media Channel, Cell " & cellName_MediaChannel.Address & ": " & "Media Channel must be TV if Category is Offline-Weekly"
                errorList.Add errorMess
            End If
        End If

        ' Find Category contains Offline
        If InStr(1, cellValue_Category, "OFFLINE", vbTextCompare) > 0 Then
            ' Find Media Channel not in OfflineMonthlyMediaChannel_List and not TV
            If Not IsInArray(cellValue_MediaChannel, OfflineMonthlyMediaChannel_List) And cellValue_MediaChannel <> "TV" Then
                cellName_MediaChannel.Interior.Color = ERR_HIG
                errorMess = "Media Channel, Cell " & cellName_MediaChannel.Address & ": " & "Media Channel must be Airport, OOH-Other, Cinema, Radio, Magazines, Newspapers, Instore/POS, TV or Digital OOH if Category is Offline-Weekly or Offline-Monthly"
                errorList.Add errorMess
            End If
        End If

        ' Find Category contains Online
        Select Case cellValue_Category
            Case "ONLINE-DAILY-PLATFORM", "ONLINE-DAILY-UniqueCloud3", "ONLINE-DAILY-OTHER"
                ' Find Media Channel value in OfflineMonthlyMediaChannel_List
                If IsInArray(cellValue_MediaChannel, OfflineMonthlyMediaChannel_List) Then
                    cellName_MediaChannel.Interior.Color = ERR_HIG
                    errorMess = "Media Channel, Cell " & cellName_MediaChannel.Address & ": " & "Media Channel must not be Airport, OOH-Other, Cinema, Radio, Magazines, Newspapers, Instore/POS, TV or Digital OOH if Category is Online-Daily-Platform, Online-Daily-UniqueCloud3 or Online-Daily-Other"
                    errorList.Add errorMess
                End If ' Find Media Channel = TV
                If cellValue_MediaChannel = "TV" Then
                    cellName_MediaChannel.Interior.Color = ERR_HIG
                    errorMess = "Media Channel, Cell " & cellName_MediaChannel.Address & ": " & "Media Channel must not be Airport, OOH-Other, Cinema, Radio, Magazines, Newspapers, Instore/POS, TV or Digital OOH if Category is Online-Daily-Platform, Online-Daily-UniqueCloud3 or Online-Daily-Other"
                    errorList.Add errorMess
                End If
        End Select

        ' Find Inventory Type value not exist in dictionary
        If Not dictInventoryDistributionType.Exists(cellValue_InventoryType) Then 
            cellName_InventoryType.Interior.Color = ERR_HIG
            errorMess = "Inventory (Distribution) Type, Cell " & cellName_InventoryType.Address & ": " & "Inventory (Distribution) Type must exist in Dictionary under Inventory (Distribution) Type"
            errorList.Add errorMess
        End If

        ' Find Media Channel = Paid Social Display or Paid Social Video, Inventory Type not Social
        If cellValue_MediaChannel = "PAID SOCIAL DISPLAY" Or cellValue_MediaChannel = "PAID SOCIAL VIDEO" Then
            If cellValue_InventoryType <> "SOCIAL" Then
                cellName_InventoryType.Interior.Color = ERR_HIG
                errorMess = "Inventory (Distribution) Type, Cell " & cellName_InventoryType.Address & ": " & "Inventory (Distribution) Type must be Social if  Media Channel is Paid Social Display or Paid Social Video"
                errorList.Add errorMess
            End If
        End If

        ' Find Media Channel not in SocialMediaChannel_List, Inventory Type = Social
        If Not IsInArray(cellValue_MediaChannel, SocialMediaChannel_List) Then
            If cellValue_InventoryType = "SOCIAL" Then
                cellName_InventoryType.Interior.Color = ERR_HIG
                errorMess = "Inventory (Distribution) Type, Cell " & cellName_InventoryType.Address & ": " & "Inventory (Distribution) Type cannot be Social if Media Channel is not Paid Social Display, Paid Social Video, Influencer/Creator Display or Influencer/Creator Video."
                errorList.Add errorMess
            End If
        End If

        ' Find Media Channel = Search, Inventory Type not Search
        If cellValue_MediaChannel = "SEARCH" Then
            If cellValue_InventoryType <> "SEARCH" Then
                cellName_InventoryType.Interior.Color = ERR_HIG
                errorMess = "Inventory (Distribution) Type, Cell " & cellName_InventoryType.Address & ": " & "Inventory (Distribution) Type must be Search if Media Channel is Search"
                errorList.Add errorMess
            End If
        End If

        ' Find Media Channel not Search, Inventory Type Search
        If cellValue_MediaChannel <> "SEARCH" Then
            If cellValue_InventoryType = "SEARCH" Then
                cellName_InventoryType.Interior.Color = ERR_HIG
                errorMess = "Inventory (Distribution) Type, Cell " & cellName_InventoryType.Address & ": " & "Inventory (Distribution) Type cannot be Search if Media Channel is not Search"
                errorList.Add errorMess
            End If
        End If

        ' Find Publisher or Vehicle Name in SocialPublisher_List, Inventory Type not Social
        If IsInArray(cellValue_Publisher, SocialPublisher_List) Or IsInArray(cellValue_VehicleName, SocialPublisher_List) Then
            If cellValue_InventoryType <> "SOCIAL" Then
                cellName_InventoryType.Interior.Color = ERR_HIG
                errorMess = "Inventory (Distribution) Type, Cell " & cellName_InventoryType.Address & ": " & "Inventory (Distribution) Type must be Social if Publisher/Vendor or Vehicle Name is Meta Facebook, Meta Instagram, Kakao, Kakao Bizboard, Kakao Moment, Kakaotalk, LinkedIn, Snapchat, Threads, TikTok, WeChat or X (Twitter)"
                errorList.Add errorMess
            End If
        End If

        ' Find Publisher or Vehicle Name not in SocialPublisher_List, Inventory Type Social
        If Not IsInArray(cellValue_Publisher, SocialPublisher_List) And Not IsInArray(cellValue_VehicleName, SocialPublisher_List) Then
            If cellValue_InventoryType = "SOCIAL" Then
                cellName_InventoryType.Interior.Color = ERR_HIG
                errorMess = "Inventory (Distribution) Type, Cell " & cellName_InventoryType.Address & ": " & "Inventory (Distribution) Type cannot be Social if Publisher/Vendor or Vehicle Name is not Meta Facebook, Meta Instagram, Kakao, Kakao Bizboard, Kakao Moment, Kakaotalk, LinkedIn, Snapchat, Threads, TikTok, WeChat or X (Twitter)"
                errorList.Add errorMess
            End If
        End If

        ' Find Publisher or Vehicle Name in ProgrammaticPublisher_List, Inventory Type not contain Programmatic
        If IsInArray(cellValue_Publisher, ProgrammaticPublisher_List) Or IsInArray(cellValue_VehicleName, ProgrammaticPublisher_List) Then
            If InStr(1, cellValue_InventoryType, "PROGRAMMATIC", vbTextCompare) = 0 Then
                cellName_InventoryType.Interior.Color = ERR_HIG
                errorMess = "Inventory (Distribution) Type, Cell " & cellName_InventoryType.Address & ": " & "Inventory (Distribution) Type must contain Programmatic if Publisher/Vendor or Vehicle Name is APEX, Google Open Exchange (DV360), Sojern, Teads or The Trade Desk (Adara can be Direct or Programmatic)"
                errorList.Add errorMess
            End If
        End If

        ' Find Publisher & Vehicle Name not in ProgrammaticPublisher_List
        If Not IsInArray(cellValue_Publisher, ProgrammaticPublisher_List) And Not IsInArray(cellValue_VehicleName, ProgrammaticPublisher_List) Then
            ' Find Publisher & Vehicle Name not Adara
            If cellValue_Publisher <> "ADARA" And cellValue_VehicleName <> "ADARA" Then
                ' Find Inventory Type contains Programmatic
                If InStr(1, cellValue_InventoryType, "PROGRAMMATIC", vbTextCompare) > 0 Then
                    cellName_InventoryType.Interior.Color = ERR_HIG
                    errorMess = "Inventory (Distribution) Type, Cell " & cellName_InventoryType.Address & ": " & "Inventory (Distribution) Type cannot contain Programmatic if Publisher/Vendor or Vehicle Name is not Adara, APEX, Google Open Exchange (DV360), Sojern, Teads or The Trade Desk"
                    errorList.Add errorMess
                End If
            End If
        End If

        ' Find Primary Target Strategy value not exist in dictionary
        If Not dictPrimaryTargetStrategy.Exists(cellValue_PrimaryTargetStrategy) Then 
            cellName_PrimaryTargetStrategy.Interior.Color = ERR_HIG
            errorMess = "Primary Target Strategy, Cell " & cellName_PrimaryTargetStrategy.Address & ": " & "Primary Target Strategy must exist in Dictionary under Primary Target Strategy"
            errorList.Add errorMess
        End If

        ' Find Promotion Incentive value not exist in dictionary
        If Not dictPromotionIncentive.Exists(cellValue_PromotionIncentive) Then 
            cellName_PromotionIncentive.Interior.Color = ERR_HIG
            errorMess = "Promotion Incentive, Cell " & cellName_PromotionIncentive.Address & ": " & "Promotion Incentive must exist in Dictionary under Promotion Incentive"
            errorList.Add errorMess
        End If

        ' Find Passion Pillar Placement value not exist in dictionary
        If Not dictPassionPillarPlacement.Exists(cellValue_PassionPillarPlacement) Then 
            cellName_PassionPillarPlacement.Interior.Color = ERR_HIG
            errorMess = "Passion Pillar Placement, Cell " & cellName_PassionPillarPlacement.Address & ": " & "Passion Pillar (Placement-Level) must exist in Dictionary under Passion Pillar (Placement-Level)"
            errorList.Add errorMess
        End If

        ' Find Initiative Placement value not exist in dictionary
        If Not dictInitiativePlacement.Exists(cellValue_InitiativePlacement) Then 
            cellName_InitiativePlacement.Interior.Color = ERR_HIG
            errorMess = "Initiative Placement, Cell " & cellName_InitiativePlacement.Address & ": " & "Initiative (Placement-Level) must exist in Dictionary under Initiative (Placement-Level)"
            errorList.Add errorMess
        End If

        ' Find Media Market value not exist in dictionary
        If Not dictMediaMarket.Exists(cellValue_MediaMarket) Then 
            cellName_MediaMarket.Interior.Color = ERR_HIG
            errorMess = "Media Market, Cell " & cellName_MediaMarket.Address & ": " & "Media Market must exist in Dictionary under Media Market"
            errorList.Add errorMess
        End If

        ' Find Passion Pillar Creative value not exist in dictionary
        If Not dictPassionPillarCreative.Exists(cellValue_PassionPillarCreative) Then 
            cellName_PassionPillarCreative.Interior.Color = ERR_HIG
            errorMess = "Passion Pillar Creative, Cell " & cellName_PassionPillarCreative.Address & ": " & "Passion Pillar (Creative-Level) must exist in Dictionary under Passion Pillar (Creative-Level)"
            errorList.Add errorMess
        End If

        ' Find Initiative Creative value not exist in dictionary
        If Not dictInitiativeCreative.Exists(cellValue_InitiativeCreative) Then 
            cellName_InitiativeCreative.Interior.Color = ERR_HIG
            errorMess = "Initiative Creative, Cell " & cellName_InitiativeCreative.Address & ": " & "Initiative (Creative-Level) must exist in Dictionary under Initiative (Creative-Level)"
            errorList.Add errorMess
        End If

        ' Find Category value not exist in dictionary
        If Not dictCategory.Exists(cellValue_Category) Then 
            cellName_Category.Interior.Color = ERR_HIG
            errorMess = "Category, Cell " & cellName_Category.Address & ": " & "Category must exist in Dictionary under Category"
            errorList.Add errorMess
        End If

        ' Find Media Channel = TV, Category not contain Offline
        If cellValue_MediaChannel = "TV" Then
            If InStr(1, cellValue_Category, "OFFLINE", vbTextCompare) = 0 Then
                cellName_Category.Interior.Color = ERR_HIG
                errorMess = "Category, Cell " & cellName_Category.Address & ": " & "Category must be Offline if Media Channel is TV"
                errorList.Add errorMess
            End If
        End If

        ' Find Media Channel in OfflineMonthlyMediaChannel_List, Category not Offline-Monthly
        If IsInArray(cellValue_MediaChannel, OfflineMonthlyMediaChannel_List) Then
            If cellValue_Category <> "OFFLINE-MONTHLY" Then
                cellName_Category.Interior.Color = ERR_HIG
                errorMess = "Category, Cell " & cellName_Category.Address & ": " & "Category must be Offline-Monthly if Media Channel is Airport, OOH-Other, Cinema, Radio, Magazines, Newspapers, Instore/POS, TV or Digital OOH"
                errorList.Add errorMess
            End If
        End If

        ' Find Media Channel = Direct Mail or Email, Category not contain Online
        If cellValue_MediaChannel = "DIRECT MAIL" Or cellValue_MediaChannel = "EMAIL" Then
            If InStr(1, cellValue_Category, "ONLINE", vbTextCompare) = 0 Then
                cellName_Category.Interior.Color = ERR_HIG
                errorMess = "Category, Cell " & cellName_Category.Address & ": " & "Category must be Online if Media Channel is Direct Mail or Email"
                errorList.Add errorMess
            End If
        End If

        ' Find Media Channel is OTHER (I.E. EVENT SIGN.), Notes value blank, empty
        If cellValue_MediaChannel = "OTHER (I.E. EVENT SIGN.)" Then
            If cellValue_Notes = "" Or cellValue_Notes = "(BLANK)" Or cellValue_Notes = "#N/A" Or cellValue_Notes = "N/A" Then 
                cellName_Notes.Interior.Color = ERR_HIG
                errorMess = "Notes, Cell " & cellName_Notes.Address & ": " & "Notes cannot be blank if Media Channel is OTHER (I.E. EVENT SIGN.). Media Channel must be provided."
                errorList.Add errorMess
            End If
        End If

        ' Find Does Media Cost Include Production Fees? value not Yes or No
        If (cellValue_DMCIPF <> "YES" And cellValue_DMCIPF <> "NO") Then 
            cellName_DMCIPF.Interior.Color = ERR_HIG
            errorMess = "Does Media Cost include Production Fees?, Cell " & cellName_DMCIPF.Address & ": " & "Does Media Cost include Production Fees? Must exist in Dictionary under Does Media Cost include Production Fees?"
            errorList.Add errorMess
        End If

        ' Find Currency value not exist in dictionary
        If Not dictCurrency.Exists(cellValue_Currency) Then 
            cellName_Currency.Interior.Color = ERR_HIG
            errorMess = "Currency, Cell " & cellName_Currency.Address & ": " & "Currency must exist in Dictionary under Currency"
            errorList.Add errorMess
        End If

        ' Find Currency value does exist in dictionary
        If CurrencyMap.Exists(cellValue_Currency) Then
            countryRuleValue2 = CurrencyMap(cellValue_Currency) ' Pull the Country Rule
            ' Find the Country value not inside the rule
            If InStr(1, countryRuleValue2, cellValue_Country, vbTextCompare) = 0 Then
                cellName_Currency.Interior.Color = ERR_HIG
                errorMess = "Currency, Cell " & cellName_Currency.Address & ": " & "Currency must match with the correct Country"
                errorList.Add errorMess
            End If
        End If

        ' Find Notes not contain Bonus, Media Cost (LC) value blank
        If InStr(1, cellValue_Notes, "BONUS", vbTextCompare) = 0 Then
            If cellValue_MediaCostLC = "" Or cellValue_MediaCostLC = 0 Or Not IsNumeric(cellValue_MediaCostLC) Then 
                cellName_MediaCostLC.Interior.Color = ERR_HIG
                errorMess = "Media Cost (LC), Cell " & cellName_MediaCostLC.Address & ": " & "Media Cost (LC) cannot be blank if Notes does not contain Bonus"
                errorList.Add errorMess
            End If
        End If

        ' Find Media Channel not Search or TV, Paid Impressions value blank
        If cellValue_MediaChannel <> "TV" And cellValue_MediaChannel <> "SEARCH" Then
            If cellValue_PaidImpressions = "" Or cellValue_PaidImpressions = 0 Or Not IsNumeric(cellValue_PaidImpressions) Then
                cellName_PaidImpressions.Interior.Color = ERR_HIG
                errorMess = "Paid Impressions, Cell " & cellName_PaidImpressions.Address & ": " & "Paid Impressions cannot be blank if Media Channel is not Search or TV"
                errorList.Add errorMess
            End If
        End If

        ' Find Media Channel = Search, Clicks value blank
        If cellValue_MediaChannel = "SEARCH" Or cellValue_InventoryType = "SEARCH" Then
            If cellValue_Clicks = "" Or cellValue_Clicks = 0 Or Not IsNumeric(cellValue_Clicks) Then
                cellName_Clicks.Interior.Color = ERR_HIG
                errorMess = "Clicks, Cell " & cellName_Clicks.Address & ": " & "Clicks cannot be blank if Media Channel is Search"
                errorList.Add errorMess
            End If
        End If

        ' Find Media Channel = TV, TV National GRPs value blank
        If cellValue_MediaChannel = "TV" Or cellValue_Publisher = "PAY TV/CABLE" Or cellValue_Publisher = "BROADCAST" Then
            If cellValue_TVGRP = "" Or cellValue_TVGRP = 0 Or Not IsNumeric(cellValue_TVGRP) Then
                cellName_TVGRP.Interior.Color = ERR_HIG
                errorMess = "TV National GRPs, Cell " & cellName_TVGRP.Address & ": " & "TV National GRPs cannot be blank if Media Channel is TV"
                errorList.Add errorMess
            End If
        End If

        ' Find Media Channel = TV, TV Duration (seconds) value blank
        If cellValue_MediaChannel = "TV" Or cellValue_Publisher = "PAY TV/CABLE" Or cellValue_Publisher = "BROADCAST" Then
            If cellValue_TVDuration = "" Or cellValue_TVDuration = 0 Or Not IsNumeric(cellValue_TVDuration) Then
                cellName_TVDuration.Interior.Color = ERR_HIG
                errorMess = "TV Duration (seconds), Cell " & cellName_TVDuration.Address & ": " & "TV Duration (seconds) cannot be blank if Media Channel is TV"
                errorList.Add errorMess
            End If
        End If

        ' Calculation logic for CPM/CPC/CPP column
        If UCase(cellValue_MediaChannel) = "TV" Then ' Media Channel = TV
            If IsNumeric(cellValue_MediaCostLC) Then ' For numeric Media Cost (LC)
                If IsNumeric(cellValue_TVGRP) And cellValue_TVGRP <> 0 Then ' For numeric and non-zero GRP
                    ' CPP = Media Cost (LC) / TV National GRPs
                    cellValue_CPMCPCCPP = cellValue_MediaCostLC / cellValue_TVGRP
                Else ' For non-numeric and zero GRP
                    cellValue_CPMCPCCPP = "#DIV/0!" ' Write error
                End If
            Else ' For non-numeric Media Cost (LC)
                cellValue_CPMCPCCPP = "#N/A" ' Write error
            End If
        ElseIf UCase(cellValue_MediaChannel) = "SEARCH" Then ' Media Channel = Search
            If IsNumeric(cellValue_MediaCostLC) Then ' For numeric Media Cost (LC)
                If IsNumeric(cellValue_Clicks) And cellValue_Clicks <> 0 Then ' For numeric and non-zero Clicks
                    ' CPC = Media Cost (LC) / Clicks
                    cellValue_CPMCPCCPP = cellValue_MediaCostLC / cellValue_Clicks
                Else ' For non-numeric and zero Clicks
                    cellValue_CPMCPCCPP = "#DIV/0!" ' Write error
                End If
            Else ' For non-numeric Media Cost (LC)
                cellValue_CPMCPCCPP = "#N/A" ' Write error
            End If
        Else ' Everything else
            If IsNumeric(cellValue_MediaCostLC) Then ' For numeric Media Cost (LC)
                If IsNumeric(cellValue_PaidImpressions) And cellValue_PaidImpressions <> 0 Then ' For numeric and non-zero Paid Impressions
                    ' CPM = (Media Cost (LC) / Paid Impressions) * 1000
                    cellValue_CPMCPCCPP = (cellValue_MediaCostLC / cellValue_PaidImpressions) * 1000
                Else ' For non-numeric and zero Paid Impressions
                    cellValue_CPMCPCCPP = "#DIV/0!" ' Write error
                End If
            Else ' For non-numeric Media Cost (LC)
                cellValue_CPMCPCCPP = "#N/A" ' Write error
            End If
        End If
        ' Write the final CPM value back into the cell
        rngDeliverData.Cells(r, colIndex("Average of TV Duration (seconds)") + 1).Value = cellValue_CPMCPCCPP

        ' Find faulty CPM/CPC/CPP
        If cellValue_CPMCPCCPP = "#DIV/0!" Or cellValue_CPMCPCCPP = "#N/A" Then
            cellName_CPMCPCCPP.Interior.Color = ERR_HIG
            errorMess = "CPM/CPC/CPP, Cell " & cellName_CPMCPCCPP.Address & ": " & "Cannot calculate CPM/CPC/CPP value"
            errorList.Add errorMess
        End If

        ' Find Media Channel is Search, CPM/CPC/CPP > 10
        If cellValue_MediaChannel = "SEARCH" And cellValue_CPMCPCCPP > 10 Then
            cellName_CPMCPCCPP.Interior.Color = ERR_HIG
            errorMess = "CPM/CPC/CPP, Cell " & cellName_CPMCPCCPP.Address & ": " & "CPM/CPC/CPP cannot be more than 10 if Media Channel is Search"
            errorList.Add errorMess
        End If

        ' Find Media Channel is Digital Display (Non-Paid Social), CPM/CPC/CPP > 20
        If cellValue_MediaChannel = "DIGITAL DISPLAY (NON-PAID SOCIAL)" And cellValue_CPMCPCCPP > 20 Then
            cellName_CPMCPCCPP.Interior.Color = ERR_HIG
            errorMess = "CPM/CPC/CPP, Cell " & cellName_CPMCPCCPP.Address & ": " & "CPM/CPC/CPP cannot be more than 20 if Media Channel is Digital Display (Non-Paid Social)"
            errorList.Add errorMess
        End If

        ' Find Media Channel is Email or Radio, CPM/CPC/CPP > 25
        If cellValue_MediaChannel = "RADIO" Or cellValue_MediaChannel = "EMAIL" Then
            If cellValue_CPMCPCCPP > 25 Then 
                cellName_CPMCPCCPP.Interior.Color = ERR_HIG
                errorMess = "CPM/CPC/CPP, Cell " & cellName_CPMCPCCPP.Address & ": " & "CPM/CPC/CPP cannot be more than 25 if Media Channel is Email or Radio"
                errorList.Add errorMess
            End If
        End If

        ' Find Media Channel is Paid Social Display, CPM/CPC/CPP > 30
        If cellValue_MediaChannel = "PAID SOCIAL DISPLAY" And cellValue_CPMCPCCPP > 30 Then
            cellName_CPMCPCCPP.Interior.Color = ERR_HIG
            errorMess = "CPM/CPC/CPP, Cell " & cellName_CPMCPCCPP.Address & ": " & "CPM/CPC/CPP cannot be more than 30 if Media Channel is Paid Social Display"
            errorList.Add errorMess
        End If

        ' Find Media Channel is Paid Social Video, Digital Audio or Newspapers, CPM/CPC/CPP > 35
        If cellValue_MediaChannel = "PAID SOCIAL VIDEO" Or cellValue_MediaChannel = "DIGITAL AUDIO" Or cellValue_MediaChannel = "NEWSPAPERS" Then
            If cellValue_CPMCPCCPP > 35 Then 
                cellName_CPMCPCCPP.Interior.Color = ERR_HIG
                errorMess = "CPM/CPC/CPP, Cell " & cellName_CPMCPCCPP.Address & ": " & "CPM/CPC/CPP cannot be more than 35 if Media Channel is Paid Social Video, Digital Audio or Newspapers"
                errorList.Add errorMess
            End If
        End If
        
        ' Find Media Channel is Influencer/Creator Display or Magazines, CPM/CPC/CPP > 40
        If cellValue_MediaChannel = "MAGAZINES" Or cellValue_MediaChannel = "INFLUENCER/CREATOR DISPLAY" Then
            If cellValue_CPMCPCCPP > 40 Then 
                cellName_CPMCPCCPP.Interior.Color = ERR_HIG
                errorMess = "CPM/CPC/CPP, Cell " & cellName_CPMCPCCPP.Address & ": " & "CPM/CPC/CPP cannot be more than 40 if Media Channel is Influencer/Creator Display or Magazines"
                errorList.Add errorMess
            End If
        End If

        ' Find Media Channel is Cinema, Digital OOH or Digital Video (Non-CTV, OTT, Paid Social), CPM/CPC/CPP > 45
        If cellValue_MediaChannel = "DIGITAL OOH" Or cellValue_MediaChannel = "DIGITAL VIDEO (NON-CTV, OTT, PAID SOCIAL)" Or cellValue_MediaChannel = "CINEMA" Then
            If cellValue_CPMCPCCPP > 45 Then 
                cellName_CPMCPCCPP.Interior.Color = ERR_HIG
                errorMess = "CPM/CPC/CPP, Cell " & cellName_CPMCPCCPP.Address & ": " & "CPM/CPC/CPP cannot be more than 45 if Media Channel is Cinema, Digital OOH or Digital Video (Non-CTV, OTT, Paid Social)"
                errorList.Add errorMess
            End If
        End If

        ' Find Media Channel is OTHER (I.E. EVENT SIGN.), OOH-Other or Instore/POS, CPM/CPC/CPP > 50
        If cellValue_MediaChannel = "OOH-OTHER" Or cellValue_MediaChannel = "INSTORE/POS" Or cellValue_MediaChannel = "OTHER (I.E. EVENT SIGN.)" Then
            If cellValue_CPMCPCCPP > 50 Then 
                cellName_CPMCPCCPP.Interior.Color = ERR_HIG
                errorMess = "CPM/CPC/CPP, Cell " & cellName_CPMCPCCPP.Address & ": " & "CPM/CPC/CPP cannot be more than 50 if Media Channel is OTHER (I.E. EVENT SIGN.), OOH-Other or Instore/POS"
                errorList.Add errorMess
            End If
        End If

        ' Find Media Channel is Influencer/Creator Video or Airport, CPM/CPC/CPP > 60
        If cellValue_MediaChannel = "INFLUENCER/CREATOR VIDEO" Or cellValue_MediaChannel = "AIRPORT" Then
            If cellValue_CPMCPCCPP > 60 Then 
                cellName_CPMCPCCPP.Interior.Color = ERR_HIG
                errorMess = "CPM/CPC/CPP, Cell " & cellName_CPMCPCCPP.Address & ": " & "CPM/CPC/CPP cannot be more than 60 if Media Channel is Influencer/Creator Video or Airport"
                errorList.Add errorMess
            End If
        End If

        ' Find Media Channel is Non-Linear TV (VOD, OTT & CTV), CPM/CPC/CPP > 70
        If cellValue_MediaChannel = "NON-LINEAR TV (VOD, OTT & CTV)" And cellValue_CPMCPCCPP > 70 Then
            cellName_CPMCPCCPP.Interior.Color = ERR_HIG
            errorMess = "CPM/CPC/CPP, Cell " & cellName_CPMCPCCPP.Address & ": " & "CPM/CPC/CPP cannot be more than 70 if Media Channel is Non-Linear TV (VOD, OTT & CTV)"
            errorList.Add errorMess
        End If

        ' Find Media Channel is Direct Mail, CPM/CPC/CPP > 80
        If cellValue_MediaChannel = "DIRECT MAIL" And cellValue_CPMCPCCPP > 80 Then 
            cellName_CPMCPCCPP.Interior.Color = ERR_HIG
            errorMess = "CPM/CPC/CPP, Cell " & cellName_CPMCPCCPP.Address & ": " & "CPM/CPC/CPP cannot be more than 80 if Media Channel is Direct Mail"
            errorList.Add errorMess
        End If

        ' Find Media Channel is TV, CPM/CPC/CPP > 1000
        If cellValue_MediaChannel = "TV" And cellValue_CPMCPCCPP > 1000 Then 
            cellName_CPMCPCCPP.Interior.Color = ERR_HIG
            errorMess = "CPM/CPC/CPP, Cell " & cellName_CPMCPCCPP.Address & ": " & "CPM/CPC/CPP cannot be more than 1000 if Media Channel is TV"
            errorList.Add errorMess
        End If
    Next r
    Call SubFunc_WriteErrorList(errorList, 8, True) ' Col 8, start block true
End Sub

' Conditional Formatting Logic for Spend DeliverData Breakdown table, Inventory (Distribution) Type
Public Sub SubFunc_ConFor_Spend_DeliverData_Breakdown_IDT(ptDeliverData As PivotTable)
    ' Within each Category (identified by Online/ Offline), each Publisher should have all the same Inventory (Distribution) Type (identified by Direct/ Programmatic/ Search/ Social)
    Dim rngDeliverData As Range, dictGroups As Object, inventoryTypeSet As Object, errorMess As String, errorList As New Collection
    Dim groupKey As String, normalizedInventory As String, normalizedCategory As String

    Set rngDeliverData = ptDeliverData.TableRange1
    Set colIndex = SubFunc_GetColIndex(rngDeliverData) ' Initilize the column index of the table range
    ' Initialize dictionary
    Set dictGroups = CreateObject("Scripting.Dictionary")
    ' First pass: Build the dictionary of groups and their inventory types
    ' Start from row 2 (skip the header) to last row (skip the Grand Total)
    For r = 2 To rngDeliverData.Rows.Count - 1

        ' Set up the cell value with some cleaning
        cellValue_Publisher = UCase(Trim(CStr(rngDeliverData.Cells(r, colIndex("Publisher/Vendor")).Value))) ' Upper Case
        cellValue_InventoryType = Trim(CStr(rngDeliverData.Cells(r, colIndex("Inventory (Distribution) Type")).Value))
        cellValue_Category = Trim(CStr(rngDeliverData.Cells(r, colIndex("Category")).Value))

        ' Normalize Category: Identify only Online and Offline
        If InStr(1, cellValue_Category, "ONLINE", vbTextCompare) > 0 Then
            normalizedCategory = "ONLINE"
        ElseIf InStr(1, cellValue_Category, "OFFLINE", vbTextCompare) > 0 Then
            normalizedCategory = "OFFLINE"
        Else ' Upper Case
            normalizedCategory = UCase(cellValue_Category)
        End If

        ' Normalize Inventory Type: Identify only Programmatic
        If InStr(1, cellValue_InventoryType, "PROGRAMMATIC", vbTextCompare) > 0 Then
            normalizedInventory = "PROGRAMMATIC"
        Else ' Upper Case
            normalizedInventory = UCase(cellValue_InventoryType)
        End If

        ' Create unique group key only Upper Case Category & Publisher
        groupKey = normalizedCategory & "|" & cellValue_Publisher

        ' If group does not exist, create new set for this group
        If Not dictGroups.Exists(groupKey) Then
            Set inventoryTypeSet = CreateObject("Scripting.Dictionary")
            inventoryTypeSet(normalizedInventory) = True
            Set dictGroups(groupKey) = inventoryTypeSet
        Else ' If group already exists, concatenate Inventory Type
            Set inventoryTypeSet = dictGroups(groupKey)
            inventoryTypeSet(normalizedInventory) = True
        End If
    Next r

    ' Second pass: find the group with more than 1 Inventory Type
    ' Start from row 2 (skip the header) to last row (skip the Grand Total)
    For r = 2 To rngDeliverData.Rows.Count - 1

        ' Set up the cell value with some cleaning
        Set cellName_InventoryType = rngDeliverData.Cells(r, colIndex("Inventory (Distribution) Type"))
        cellValue_Publisher = UCase(Trim(CStr(rngDeliverData.Cells(r, colIndex("Publisher/Vendor")).Value))) ' Upper Case
        cellValue_Category = Trim(CStr(rngDeliverData.Cells(r, colIndex("Category")).Value))

        ' Normalize Category: Identify only Online and Offline
        If InStr(1, cellValue_Category, "ONLINE", vbTextCompare) > 0 Then
            normalizedCategory = "ONLINE"
        ElseIf InStr(1, cellValue_Category, "OFFLINE", vbTextCompare) > 0 Then
            normalizedCategory = "OFFLINE"
        Else ' Upper Case
            normalizedCategory = UCase(cellValue_Category)
        End If

        ' Create unique group key only Upper Case Category & Publisher
        groupKey = normalizedCategory & "|" & cellValue_Publisher

        ' Find groupkey exists in the dictGroups created earlier
        If dictGroups.Exists(groupKey) Then
            Set inventoryTypeSet = dictGroups(groupKey) ' Pull the Inventory Type out
            If inventoryTypeSet.Count > 1 Then ' If there are more than 1 unique Inventory Type, highlight all Inventory Types
                'Call DebugPrintDict(inventoryTypeSet)
                cellName_InventoryType.Interior.Color = ERR_HIG
                errorMess = "Inventory (Distribution) Type, Cell " & cellName_InventoryType.Address & ": " & "Inventory (Distribution) Type must only have one unique value (Direct or Programmatic or Search or Social) for each Publisher/Vendor within each Online and Offline Category"
                errorList.Add errorMess
            End If
        End If
    Next r
    Call SubFunc_WriteErrorList(errorList, 8, False) ' Col 8, start block false
End Sub
"""

In [ ]:
Module_Macro_6 = """
Sub MT_vs_DeliverData_Classification()
    Dim wsDeliverData As Worksheet, wsNew As Worksheet, wsMT As Worksheet
    Dim ptDeliverData As PivotTable, ptMT As PivotTable
    Dim pcDeliverData As PivotCache, pcMT As PivotCache
    Dim pi As PivotItem
    Dim dataDeliverData As Range, dataMT As Range
    Dim lastRowDeliverData As Long, lastColDeliverData As Long, lastRowMT As Long, lastColMT As Long, idx As Long
    Dim pivotStartCellMT As Range, pivotStartCellDeliverData As Range
    Dim rngMT As Range, rngDeliverData As Range, errorMess1 As String, errorMess2 As String, errorList1 As New Collection, errorList2 As New Collection
    Dim keyMT As String, keyDeliverData As String, valMT As String, valDeliverData As String, fixed_quarter As String, fixed_country As String
    Dim k As Variant, p As Variant, SubValues_MT As Variant, SubValues_DeliverData As Variant
    Dim colIndex As Object
    BenchMark = Timer
    Application.ScreenUpdating = False
    Set wsMT = ThisWorkbook.Sheets("UniqueCloud1")
    Set wsDeliverData = ThisWorkbook.Sheets("DeliverData")
    wsMT.AutoFilterMode = False
    wsDeliverData.AutoFilterMode = False

    Call SubFunc_CreateNewSheet("MT vs DeliverData Classification", wsNew)

    wsNew.Range("A1").Value = "UniqueCloud1 (Overview)"
    wsNew.Range("H1").Value = "DeliverData (Overview)"

    errorList1.Add "MT vs DeliverData Classification"
    errorList1.Add "UniqueCloud1 (Overview)"

    errorList2.Add "MT vs DeliverData Classification"
    errorList2.Add "DeliverData (Overview)"

    ' Define data range for UniqueCloud1
    lastRowMT = wsMT.Cells(Rows.Count, 1).End(xlUp).Row
    lastColMT = wsMT.Cells(1, Columns.Count).End(xlToLeft).Column
    Set dataMT = wsMT.Range(wsMT.Cells(1, 1), wsMT.Cells(lastRowMT, lastColMT))

    ' Define data range for DeliverData
    lastRowDeliverData = wsDeliverData.Cells(Rows.Count, 1).End(xlUp).Row
    lastColDeliverData = wsDeliverData.Cells(1, Columns.Count).End(xlToLeft).Column
    Set dataDeliverData = wsDeliverData.Range(wsDeliverData.Cells(1, 1), wsDeliverData.Cells(lastRowDeliverData, lastColDeliverData))

    ' Define PivotTable start positions
    Set pivotStartCellMT = wsNew.Range("A6")
    Set pivotStartCellDeliverData = wsNew.Range("H6")

    ' Create PivotTable for UniqueCloud1 (Classification)
    Set pcMT = ThisWorkbook.PivotCaches.Create(SourceType:=xlDatabase, SourceData:=dataMT)
    Set ptMT = pcMT.CreatePivotTable(TableDestination:=pivotStartCellMT, tableName:="PivotUniqueCloud1")

    ' Create PivotTable for DeliverData (Classification)
    Set pcDeliverData = ThisWorkbook.PivotCaches.Create(SourceType:=xlDatabase, SourceData:=dataDeliverData)
    Set ptDeliverData = pcDeliverData.CreatePivotTable(TableDestination:=pivotStartCellDeliverData, tableName:="PivotDeliverData")
    Call SubFunc_GetFixedQuarter(fixed_quarter)
    Call SubFunc_GetFixedCountry(fixed_country)
    
    DoEvents
    With ptMT
        With .PivotFields("Quarter")
            .Orientation = xlPageField
            On Error Resume Next
            .CurrentPage = fixed_quarter
            If Err.Number <> 0 Then
                Err.Clear
                On Error GoTo 0
                wsMT.Select
                MsgBox fixed_quarter & " not found in Quarter column, UniqueCloud1 sheet", vbExclamation, "Date Validation Error"
                Exit Sub
            End If
            On Error GoTo 0
        End With
        With .PivotFields("Plan Name")
             .Orientation = xlRowField
             .Subtotals = Array(False, False, False, False, False, False, False, False, False, False, False, False)
             .Caption = "Plan Name"
        End With
        With .PivotFields("Country")
             .Orientation = xlRowField
             .Subtotals = Array(False, False, False, False, False, False, False, False, False, False, False, False)
        End With
        With .PivotFields("Funding Source")
             .Orientation = xlRowField
             .Subtotals = Array(False, False, False, False, False, False, False, False, False, False, False, False)
        End With
        With .PivotFields("Initiative")
             .Orientation = xlRowField
             .Subtotals = Array(False, False, False, False, False, False, False, False, False, False, False, False)
        End With
        With .PivotFields("Product Message")
             .Orientation = xlRowField
             .Subtotals = Array(False, False, False, False, False, False, False, False, False, False, False, False)
        End With
        With .PivotFields("Destination")
             .Orientation = xlRowField
             .Subtotals = Array(False, False, False, False, False, False, False, False, False, False, False, False)
        End With
        With .PivotFields("Sponsorship")
             .Orientation = xlRowField
             .Subtotals = Array(False, False, False, False, False, False, False, False, False, False, False, False)
        End With
        .RowAxisLayout xlTabularRow
        .TableStyle2 = "PivotStyleLight20"
    End With
    FileNameOnly = SubFunc_GetFileName(ThisWorkbook)
    If InStr(1, FileNameOnly, "-VIK - DeliverData -", vbTextCompare) > 0 Then
        With ptMT.PivotFields("Funding Source")
            For Each pi In .PivotItems
                pi.Visible = True ' First make all items visible
            Next pi
            For Each pi In .PivotItems
                If pi.Value <> "Value In Kind (VIK)" Then
                    pi.Visible = False ' Then hide everything except the target value
                End If
            Next pi
        End With
    ElseIf InStr(1, FileNameOnly, "-UniqueClientName - DeliverData -", vbTextCompare) > 0 Then
        With ptMT.PivotFields("Funding Source")
            For Each pi In .PivotItems
                pi.Visible = True
            Next pi
            For Each pi In .PivotItems
                If pi.Value <> "UniqueClientName/Core Marketing" Then
                    pi.Visible = False
                End If
            Next pi
        End With
    End If
    ' Filter current Country from the file name
    With ptMT.PivotFields("Country")
        For Each pi In .PivotItems
            pi.Visible = True
        Next pi
        For Each pi In .PivotItems
            If fixed_country <> "(All)" And pi.Value <> fixed_country Then
                pi.Visible = False
            End If
        Next pi
    End With
    Call SubFunc_PivotRepeatLabel(ptMT, True) ' On/Show

    With ptDeliverData
        With .PivotFields("Plan Name")
             .Orientation = xlRowField
             .Subtotals = Array(False, False, False, False, False, False, False, False, False, False, False, False)
             .Caption = "Plan Name"
        End With
        With .PivotFields("Country")
             .Orientation = xlRowField
             .Subtotals = Array(False, False, False, False, False, False, False, False, False, False, False, False)
        End With
        With .PivotFields("UniqueClientName Campaign Lead")
             .Orientation = xlRowField
             .Subtotals = Array(False, False, False, False, False, False, False, False, False, False, False, False)
        End With
        With .PivotFields("Funding Source")
             .Orientation = xlRowField
             .Subtotals = Array(False, False, False, False, False, False, False, False, False, False, False, False)
        End With
        With .PivotFields("Fund")
             .Orientation = xlRowField
             .Subtotals = Array(False, False, False, False, False, False, False, False, False, False, False, False)
        End With
        With .PivotFields("Initiative")
             .Orientation = xlRowField
             .Subtotals = Array(False, False, False, False, False, False, False, False, False, False, False, False)
        End With
        With .PivotFields("Profitable Consumer Behaviour")
             .Orientation = xlRowField
             .Subtotals = Array(False, False, False, False, False, False, False, False, False, False, False, False)
        End With
        With .PivotFields("Product Message")
             .Orientation = xlRowField
             .Subtotals = Array(False, False, False, False, False, False, False, False, False, False, False, False)
        End With
        With .PivotFields("Destination")
             .Orientation = xlRowField
             .Subtotals = Array(False, False, False, False, False, False, False, False, False, False, False, False)
        End With
        With .PivotFields("Sponsorship")
             .Orientation = xlRowField
             .Subtotals = Array(False, False, False, False, False, False, False, False, False, False, False, False)
        End With
        With .PivotFields("Passion Pillar (Campaign-Level)")
             .Orientation = xlRowField
             .Subtotals = Array(False, False, False, False, False, False, False, False, False, False, False, False)
        End With
        With .PivotFields("Initiative (Campaign-Level)")
             .Orientation = xlRowField
             .Subtotals = Array(False, False, False, False, False, False, False, False, False, False, False, False)
        End With
        With .PivotFields("Notes")
             .Orientation = xlRowField
             .Subtotals = Array(False, False, False, False, False, False, False, False, False, False, False, False)
        End With
        .RowAxisLayout xlTabularRow
    End With
    Call SubFunc_PivotRepeatLabel(ptDeliverData, True) ' On/Show

    Set rngMT = ptMT.TableRange1
    Set rngDeliverData = ptDeliverData.TableRange1
    Set dictMT = CreateObject("Scripting.Dictionary")
    Set dictDeliverData = CreateObject("Scripting.Dictionary")
    
    Set dictMT_0 = CreateObject("Scripting.Dictionary")
    Set dictDeliverData_0 = CreateObject("Scripting.Dictionary")

    ' Build dictMT with key = first column, value = array of cell addresses
    For i = 2 To rngMT.Rows.Count
        If rngMT.Cells(i, 1).Value <> "Grand Total" Then
            keyMT = UCase(Trim(CStr(rngMT.Cells(i, 1).Value)))
            Set dictMT_0.item(keyMT) = rngMT.Cells(i, 1) '
            ' Store addresses, not values
            dictMT.item(keyMT) = Array(rngMT.Cells(i, 2).Address, _
                                       rngMT.Cells(i, 3).Address, _
                                       rngMT.Cells(i, 4).Address, _
                                       rngMT.Cells(i, 5).Address, _
                                       rngMT.Cells(i, 6).Address, _
                                       rngMT.Cells(i, 7).Address)
        End If
    Next i

    ' Build dictDeliverData
    For j = 2 To rngDeliverData.Rows.Count
        If rngDeliverData.Cells(j, 1).Value <> "Grand Total" Then
            keyDeliverData = UCase(Trim(CStr(rngDeliverData.Cells(j, 1).Value)))
            Set dictDeliverData_0.item(keyDeliverData) = rngDeliverData.Cells(j, 1) '
            ' Store addresses, not values
            dictDeliverData.item(keyDeliverData) = Array(rngDeliverData.Cells(j, 2).Address, _
                                          rngDeliverData.Cells(j, 4).Address, _
                                          rngDeliverData.Cells(j, 6).Address, _
                                          rngDeliverData.Cells(j, 8).Address, _
                                          rngDeliverData.Cells(j, 9).Address, _
                                          rngDeliverData.Cells(j, 10).Address)
        End If
    Next j

    ' Compare matching keys
    For Each k In dictMT.Keys
        If dictDeliverData.Exists(k) Then
            SubValues_MT = dictMT.item(k)
            SubValues_DeliverData = dictDeliverData.item(k)
            ' Compare each corresponding cell
            For idx = LBound(SubValues_MT) To UBound(SubValues_MT)
                valMT = UCase(Trim(CStr(wsNew.Range(SubValues_MT(idx)).Value)))
                valDeliverData = UCase(Trim(CStr(wsNew.Range(SubValues_DeliverData(idx)).Value)))
                ' Match MT Country value South Korea to Korea
                If valMT = "SOUTH KOREA" Then valMT = "KOREA"
                ' Match MT Spnsorship value Not Sponsorship to N/A
                If valMT = "NOT SPONSORSHIP" Then valMT = "N/A"
                ' Now compare if each of them match
                If valMT <> valDeliverData Then
                    wsNew.Range(SubValues_MT(idx)).Interior.Color = ERR_HIG
                    errorMess1 = "Cell " & wsNew.Range(SubValues_MT(idx)).Address & ": " & "must match with DeliverData (Overview)"
                    errorList1.Add errorMess1
                    wsNew.Range(SubValues_DeliverData(idx)).Interior.Color = ERR_HIG
                    errorMess2 = "Cell " & wsNew.Range(SubValues_DeliverData(idx)).Address & ": " & "must match with UniqueCloud1 (Overview)"
                    errorList2.Add errorMess2
                End If
            Next idx
        Else
            ' Key exists in MT but not in DeliverData
            dictMT_0(k).Interior.Color = ERR_HIG
            errorMess1 = "Plan Name, Cell " & dictMT_0(k).Address & ": " & "Plan Name must match with Plan Name in DeliverData (Overview)"
            errorList1.Add errorMess1
            
'            SubValues_MT = dictMT.item(k)
'            For idx = LBound(SubValues_MT) To UBound(SubValues_MT)
'                wsNew.Range(SubValues_MT(idx)).Interior.Color = ERR_HIG
'                errorMess1 = "Cell " & wsNew.Range(SubValues_MT(idx)).Address & ": " & "Plan Name does not exist in DeliverData (Overview)"
'                errorList1.Add errorMess1
'            Next idx
        End If
    Next k
    ' Keys that exist in DeliverData but not in MT
    For Each p In dictDeliverData.Keys
        If Not dictMT.Exists(p) Then
            dictDeliverData_0(p).Interior.Color = ERR_HIG
            errorMess2 = "Plan Name, Cell " & dictDeliverData_0(p).Address & ": " & "Plan Name must match with Plan Name in UniqueCloud1 (Overview)"
            errorList2.Add errorMess2
                
'            SubValues_DeliverData = dictDeliverData.item(p)
'            For idx = LBound(SubValues_DeliverData) To UBound(SubValues_DeliverData)
'                wsNew.Range(SubValues_DeliverData(idx)).Interior.Color = ERR_HIG
'                errorMess2 = "Cell " & wsNew.Range(SubValues_DeliverData(idx)).Address & ": " & "Plan Name does not exist in UniqueCloud1 (Overview)"
'                errorList2.Add errorMess2
'            Next idx
        End If
    Next p

    Call SubFunc_WriteErrorList(errorList1, 10, True) ' Col 10, start block true
    Call SubFunc_WriteErrorList(errorList2, 11, True) ' Col 11, start block true

    ' Appy additional complex conditional formatting logic
    Call SubFunc_ConFor_Classification_DeliverData_Overview(ptDeliverData) ' Lots of conditions for DeliverData Breakdown table
    Call SubFunc_ConFor_Classification_MultipleRows(ptMT, 10, False) ' Col 10, start block false
    Call SubFunc_ConFor_Classification_MultipleRows(ptDeliverData, 11, False) ' Col 11, start block false

    wsNew.UsedRange.Font.Name = "Arial"
    wsNew.UsedRange.Font.Size = 8
    wsNew.Columns.AutoFit
    Call SubFunc_SetZoomLevel(ThisWorkbook)
    wsNew.Select
    totalRunTime = Round((Timer - BenchMark) / 60, 2)
    Application.ScreenUpdating = True
    If Not showMsg Then MsgBox "Compare UniqueCloud1 vs DeliverData Classification" & vbCrLf & _
           "Total run time: " & totalRunTime & " minutes", vbInformation, "Complete"
End Sub

' Conditional Formatting Logic for Classification DeliverData Overview table
Public Sub SubFunc_ConFor_Classification_DeliverData_Overview(ptDeliverData As PivotTable)
    Dim rngDeliverData As Range, errorMess As String, errorList As New Collection

    ' Set up all the dictionaries from the Guidelines sheet, based on the header name
    Set dictCountry = SubFunc_GetDictionary("Country Full")
    Set dictCampaignLead = SubFunc_GetDictionary("UniqueClientName Campaign Lead Full")
    Set dictFundingSource = SubFunc_GetDictionary("Funding Source Full")
    Set dictFund = SubFunc_GetDictionary("Fund Full")
    Set dictInitiative = SubFunc_GetDictionary("Initiative Full")
    Set dictPCB = SubFunc_GetDictionary("Profitable Consumer Behaviour Full")
    Set dictProductMessage = SubFunc_GetDictionary("Product Message Full")
    Set dictDestination = SubFunc_GetDictionary("Destination Full")
    Set dictSponsorship = SubFunc_GetDictionary("Sponsorship Full")
    Set dictPassionPillarCampaign = SubFunc_GetDictionary("Passion Pillar (Campaign-Level) Full")
    Set dictInitiativeCampaign = SubFunc_GetDictionary("Initiative (Campaign-Level) Full")

    Set rngDeliverData = ptDeliverData.TableRange1
    Set colIndex = SubFunc_GetColIndex(rngDeliverData) ' Initilize the column index of the table range
    ' Start from row 2 (skip the header) to last row (skip the Grand Total)
    For r = 2 To rngDeliverData.Rows.Count - 1
    
        ' Set up the cell name by row and column
        Set cellName_PlanName = rngDeliverData.Cells(r, colIndex("Plan Name"))
        Set cellName_Country = rngDeliverData.Cells(r, colIndex("Country"))
        Set cellName_CampaignLead = rngDeliverData.Cells(r, colIndex("UniqueClientName Campaign Lead"))
        Set cellName_FundingSource = rngDeliverData.Cells(r, colIndex("Funding Source"))
        Set cellName_Fund = rngDeliverData.Cells(r, colIndex("Fund"))
        Set cellName_Initiative = rngDeliverData.Cells(r, colIndex("Initiative"))
        Set cellName_PCB = rngDeliverData.Cells(r, colIndex("Profitable Consumer Behaviour"))
        Set cellName_ProductMessage = rngDeliverData.Cells(r, colIndex("Product Message"))
        Set cellName_Destination = rngDeliverData.Cells(r, colIndex("Destination"))
        Set cellName_Sponsorship = rngDeliverData.Cells(r, colIndex("Sponsorship"))
        Set cellName_PassionPillarCampaign = rngDeliverData.Cells(r, colIndex("Passion Pillar (Campaign-Level)"))
        Set cellName_InitiativeCampaign = rngDeliverData.Cells(r, colIndex("Initiative (Campaign-Level)"))
        Set cellName_Notes = rngDeliverData.Cells(r, colIndex("Notes"))
        
        ' Set up the cell value with some cleaning
        cellValue_PlanName = UCase(Trim(CStr(cellName_PlanName.Value)))
        cellValue_Country = UCase(Trim(CStr(cellName_Country.Value)))
        cellValue_CampaignLead = UCase(Trim(CStr(cellName_CampaignLead.Value)))
        cellValue_FundingSource = UCase(Trim(CStr(cellName_FundingSource.Value)))
        cellValue_Fund = UCase(Trim(CStr(cellName_Fund.Value)))
        cellValue_Initiative = UCase(Trim(CStr(cellName_Initiative.Value)))
        cellValue_PCB = UCase(Trim(CStr(cellName_PCB.Value)))
        cellValue_ProductMessage = UCase(Trim(CStr(cellName_ProductMessage.Value)))
        cellValue_Destination = UCase(Trim(CStr(cellName_Destination.Value)))
        cellValue_Sponsorship = UCase(Trim(CStr(cellName_Sponsorship.Value)))
        cellValue_PassionPillarCampaign = UCase(Trim(CStr(cellName_PassionPillarCampaign.Value)))
        cellValue_InitiativeCampaign = UCase(Trim(CStr(cellName_InitiativeCampaign.Value)))
        cellValue_Notes = UCase(Trim(CStr(cellName_Notes.Value)))

        ' Find Plan Name value blank, empty
        If cellValue_PlanName = "" Or cellValue_PlanName = "(BLANK)" Or cellValue_PlanName = "#N/A" Or cellValue_PlanName = "N/A" Then
            cellName_PlanName.Interior.Color = ERR_HIG
            errorMess = "Plan Name, Cell " & cellName_PlanName.Address & ": " & "Plan Name cannot be blank"
            errorList.Add errorMess
        End If

        ' Find Country value not exist in dictionary
        If Not dictCountry.Exists(cellValue_Country) Then 
            cellName_Country.Interior.Color = ERR_HIG
            errorMess = "Country, Cell " & cellName_Country.Address & ": " & "Country must exist in Dictionary under Country"
            errorList.Add errorMess
        End If

        ' Find Campaign Lead value not exist in dictionary
        If Not dictCampaignLead.Exists(cellValue_CampaignLead) Then 
            cellName_CampaignLead.Interior.Color = ERR_HIG
            errorMess = "UniqueClientName Campaign Lead, Cell " & cellName_CampaignLead.Address & ": " & "UniqueClientName Campaign Lead must exist in Dictionary under UniqueClientName Campaign Lead"
            errorList.Add errorMess
        End If

        ' Find Funding Source value not exist in dictionary
        If Not dictFundingSource.Exists(cellValue_FundingSource) Then 
            cellName_FundingSource.Interior.Color = ERR_HIG
            errorMess = "Funding Source, Cell " & cellName_FundingSource.Address & ": " & "Funding Source must exist in Dictionary under Funding Source"
            errorList.Add errorMess
        End If

        ' Find Fund value not exist in dictionary
        If Not dictFund.Exists(cellValue_Fund) Then 
            cellName_Fund.Interior.Color = ERR_HIG
            errorMess = "Fund, Cell " & cellName_Fund.Address & ": " & "Fund must exist in Dictionary under Fund"
            errorList.Add errorMess
        End If

        ' Find Initiative value not exist in dictionary
        If Not dictInitiative.Exists(cellValue_Initiative) Then 
            cellName_Initiative.Interior.Color = ERR_HIG
            errorMess = "Initiative, Cell " & cellName_Initiative.Address & ": " & "Initiative must exist in Dictionary under Initiative"
            errorList.Add errorMess
        End If

        ' Find Destination = Domestic, Initiative contains Inbound or Outbound
        If cellValue_Destination = "DOMESTIC" Then
            If (InStr(1, cellValue_Initiative, "INBOUND", vbTextCompare) > 0 Or InStr(1, cellValue_Initiative, "OUTBOUND", vbTextCompare) > 0) Then
                cellName_Initiative.Interior.Color = ERR_HIG
                errorMess = "Initiative, Cell " & cellName_Initiative.Address & ": " & "Initiative cannot contain Inbound or Outbound if Destination is Domestic"
                errorList.Add errorMess
            End If
        End If

        ' Find Destination = Inbound, Initiative contains Domestic or Outbound
        If cellValue_Destination = "INBOUND" Then
            If (InStr(1, cellValue_Initiative, "DOMESTIC", vbTextCompare) > 0 Or InStr(1, cellValue_Initiative, "OUTBOUND", vbTextCompare) > 0) Then
                cellName_Initiative.Interior.Color = ERR_HIG
                errorMess = "Initiative, Cell " & cellName_Initiative.Address & ": " & "Initiative cannot contain Domestic or Outbound if Destination is Inbound"
                errorList.Add errorMess
            End If
        End If

        ' Find Destination = Outbound, Initiative contains Domestic or Inbound
        If cellValue_Destination = "OUTBOUND" Then
            If (InStr(1, cellValue_Initiative, "DOMESTIC", vbTextCompare) > 0 Or InStr(1, cellValue_Initiative, "INBOUND", vbTextCompare) > 0) Then
                cellName_Initiative.Interior.Color = ERR_HIG
                errorMess = "Initiative, Cell " & cellName_Initiative.Address & ": " & "Initiative cannot contain Inbound or Domestic if Destination is Outbound"
                errorList.Add errorMess
            End If
        End If

        ' Find PCB value not exist in dictionary
        If Not dictPCB.Exists(cellValue_PCB) Then 
            cellName_PCB.Interior.Color = ERR_HIG
            errorMess = "Profitable Consumer Behaviour, Cell " & cellName_PCB.Address & ": " & "Profitable Consumer Behaviour must exist in Dictionary under Profitable Consumer Behaviour"
            errorList.Add errorMess
        End If

        ' Find Initiative doesn't contain CMS, VAS, Product Platform Application & Services, PCB = N/A
        If InStr(1, cellValue_Initiative, "CMS", vbTextCompare) = 0 And InStr(1, cellValue_Initiative, "VAS", vbTextCompare) = 0 And InStr(1, cellValue_Initiative, "PRODUCT PLATFORM APPLICATION & SERVICES", vbTextCompare) = 0 Then
            If cellValue_PCB = "N/A" Then
                cellName_PCB.Interior.Color = ERR_HIG
                errorMess = "Profitable Consumer Behaviour, Cell " & cellName_PCB.Address & ": " & "Profitable Consumer Behaviour cannot be N/A if Initiative does not contain CMS ,VAS or Product Platform Application & Services"
                errorList.Add errorMess
            End If
        End If

        ' Find Product Message value not exist in dictionary
        If Not dictProductMessage.Exists(cellValue_ProductMessage) Then 
            cellName_ProductMessage.Interior.Color = ERR_HIG
            errorMess = "Product Message, Cell " & cellName_ProductMessage.Address & ": " & "Product Message must exist in Dictionary under Product Message"
            errorList.Add errorMess
        End If

        ' Find Destination value not exist in dictionary
        If Not dictDestination.Exists(cellValue_Destination) Then 
            cellName_Destination.Interior.Color = ERR_HIG
            errorMess = "Destination, Cell " & cellName_Destination.Address & ": " & "Destination must exist in Dictionary under Destination"
            errorList.Add errorMess
        End If

        ' Find Initiative contain Domestic and Destination not Domestic
        If InStr(1, cellValue_Initiative, "DOMESTIC", vbTextCompare) > 0 And cellValue_Destination <> "DOMESTIC" Then
            cellName_Destination.Interior.Color = ERR_HIG
            errorMess = "Destination, Cell " & cellName_Destination.Address & ": " & "Destination must be Domestic if Initiative contains Domestic"
            errorList.Add errorMess
        End If

        ' Find Initiative contain Inbound and Destination not Inbound
        If InStr(1, cellValue_Initiative, "INBOUND", vbTextCompare) > 0 And cellValue_Destination <> "INBOUND" Then
            cellName_Destination.Interior.Color = ERR_HIG
            errorMess = "Destination, Cell " & cellName_Destination.Address & ": " & "Destination must be Inbound if Initiative contains Inbound"
            errorList.Add errorMess
        End If

        ' Find Initiative contain Outbound and Destination not Outbound
        If InStr(1, cellValue_Initiative, "OUTBOUND", vbTextCompare) > 0 And cellValue_Destination <> "OUTBOUND" Then
            cellName_Destination.Interior.Color = ERR_HIG
            errorMess = "Destination, Cell " & cellName_Destination.Address & ": " & "Destination must be Outbound if Initiative contains Outbound"
            errorList.Add errorMess
        End If

        ' Find Sponsorship value not exist in dictionary
        If Not dictSponsorship.Exists(cellValue_Sponsorship) Then 
            cellName_Sponsorship.Interior.Color = ERR_HIG
            errorMess = "Sponsorship, Cell " & cellName_Sponsorship.Address & ": " & "Sponsorship must exist in Dictionary under Sponsorship"
            errorList.Add errorMess
        End If

        ' Find Passion Pillar Campaign value not exist in dictionary
        If Not dictPassionPillarCampaign.Exists(cellValue_PassionPillarCampaign) Then 
            cellName_PassionPillarCampaign.Interior.Color = ERR_HIG
            errorMess = "Passion Pillar Campaign, Cell " & cellName_PassionPillarCampaign.Address & ": " & "Passion Pillar (Campaign-Level) must exist in Dictionary under Passion Pillar (Campaign-Level)"
            errorList.Add errorMess
        End If

        ' Find Initiative Campaign value not exist in dictionary
        If Not dictInitiativeCampaign.Exists(cellValue_InitiativeCampaign) Then 
            cellName_InitiativeCampaign.Interior.Color = ERR_HIG
            errorMess = "Initiative Campaign, Cell " & cellName_InitiativeCampaign.Address & ": " & "Initiative (Campaign-Level) must exist in Dictionary under Initiative (Campaign-Level)"
            errorList.Add errorMess
        End If

        ' Find PCB = Other, blank Notes
        If cellValue_PCB = "OTHER" And cellValue_Notes = "(BLANK)" Then
            cellName_Notes.Interior.Color = ERR_HIG
            errorMess = "Notes, Cell " & cellName_Notes.Address & ": " & "Notes cannot be blank if PCB is Other. Hypothesis PCB must be provided in Notes."
            errorList.Add errorMess
        End If

        ' Find Sponsorship = Other, blank Notes
        If cellValue_Sponsorship = "OTHER" And cellValue_Notes = "(BLANK)" Then
            cellName_Notes.Interior.Color = ERR_HIG
            errorMess = "Notes, Cell " & cellName_Notes.Address & ": " & "Notes cannot be blank if Sponsorship is Other. Sponsorship must be provided in Notes."
            errorList.Add errorMess
        End If
    Next r
    Call SubFunc_WriteErrorList(errorList, 11, False) ' Col 11, start block false
End Sub

' Conditional Formatting multiple rows Logic for Classification MT & DeliverData Overview table
Public Sub SubFunc_ConFor_Classification_MultipleRows(ptTempt As PivotTable, writeCol_Index As Long, start_block As Boolean)
    Dim rngTable As Range, r As Long, c As Long, k As Long, errorMess As String, errorList As New Collection
    Dim firstCol As Long, startCol As Long, lastCol As Long, lastRow As Long

    Set rngTable = ptTempt.TableRange1
    firstCol = 1 ' Just the first col of the table
    startCol = firstCol + 1 ' Next col after the first col
    lastCol = rngTable.Columns.Count ' Count to the last col of the table
    lastRow = rngTable.Rows.Count - 1 ' Count to the last row of the table minus Grand Total row
    For r = 2 To lastRow ' From row 2 to last row except Grand Total
        For k = r + 1 To lastRow ' From the subsequent row under r
            ' In the first col, if there are duplicate values
            If UCase(Trim(CStr(rngTable.Cells(k, firstCol).Value))) = UCase(Trim(CStr(rngTable.Cells(r, firstCol).Value))) Then
                For c = startCol To lastCol ' Start going over each subsequent col to check
                    ' If detect a differece, highlight both values
                    If UCase(Trim(CStr(rngTable.Cells(r, c).Value))) <> UCase(Trim(CStr(rngTable.Cells(k, c).Value))) Then
                        rngTable.Cells(r, c).Interior.Color = ERR_HIG
                        rngTable.Cells(k, c).Interior.Color = ERR_HIG
                        errorMess = "Cell " & rngTable.Cells(r, c).Address & " and " & rngTable.Cells(k, c).Address & ": " & "are different, must have only one unique value for each Plan Name"
                        errorList.Add errorMess
                    End If
                Next c
            End If
        Next k
    Next r
    Call SubFunc_WriteErrorList(errorList, writeCol_Index, start_block)
End Sub
"""

In [ ]:
Module_Macro_7 = """
Public showMsg As Boolean ' A global variable as a silent mode flag
Sub MT_vs_DeliverData_All()
    BenchMark = Timer
    ' Set up 1 button to run all 3 scripts
    showMsg = True ' This will block all msgbox
    Call MT_vs_DeliverData_Plan
    Call MT_vs_DeliverData_Spend
    Call MT_vs_DeliverData_Classification
    showMsg = False ' This will allow all msgbox again
    totalRunTime = Round((Timer - BenchMark) / 60, 2)
    MsgBox "All 3 buttons completed successfully!" & vbCrLf & _
           "Total run time: " & totalRunTime & " minutes", vbInformation, "Complete"
End Sub

Sub DeliverData_All()
    Dim wsDeliverData As Worksheet, wsNew As Worksheet, wsNew2 As Worksheet
    Dim ptDeliverData As PivotTable, ptDeliverData_2 As PivotTable, ptDeliverData_3 As PivotTable, pcDeliverData As PivotCache, pcDeliverData_2 As PivotCache, pcDeliverData_3 As PivotCache
    Dim dataDeliverData As Range, pivotStartCellDeliverData As Range, pivotStartCellDeliverData_2 As Range, pivotStartCellDeliverData_3 As Range, rngDeliverData As Range
    Dim lastRowDeliverData As Long, lastColDeliverData As Long
    BenchMark = Timer
    Application.ScreenUpdating = False
    Set wsDeliverData = ThisWorkbook.Sheets("DeliverData")
    wsDeliverData.AutoFilterMode = False
    Call SubFunc_CreateNewSheet("DeliverData Spend", wsNew)
    Call SubFunc_CreateNewSheet("DeliverData Classification", wsNew2)
    wsNew.Range("A1").Value = "DeliverData (Overview)"
    wsNew.Range("J1").Value = "DeliverData (Breakdown)"
    With wsNew.Range("AL2") ' Set up the CPM column for DeliverData Breakdown
        .Value = "CPM/CPC/CPP"
        .Font.Bold = True
        .Interior.ThemeColor = xlThemeColorAccent1
        .Interior.TintAndShade = 0.8  ' 80%
    End With
    wsNew2.Range("A1").Value = "DeliverData Classification"
    lastRowDeliverData = wsDeliverData.Cells(Rows.Count, 1).End(xlUp).Row
    lastColDeliverData = wsDeliverData.Cells(1, Columns.Count).End(xlToLeft).Column
    Set headerRowDeliverData = wsDeliverData.Rows(1)
    Set dataDeliverData = wsDeliverData.Range(wsDeliverData.Cells(1, 1), wsDeliverData.Cells(lastRowDeliverData, lastColDeliverData))
    Set pivotStartCellDeliverData = wsNew.Range("A2")
    Set pivotStartCellDeliverData_2 = wsNew.Range("J2")
    Set pivotStartCellDeliverData_3 = wsNew2.Range("A2")
    ' Safe guard check for Date columns inside DeliverData sheet
    col_StartDate = Application.Match("Start Date", headerRowDeliverData, 0)
    col_EndDate = Application.Match("End Date", headerRowDeliverData, 0)
    For r = 2 To lastRowDeliverData
        Set cell_StartDate = wsDeliverData.Cells(r, col_StartDate)
        Set cell_EndDate = wsDeliverData.Cells(r, col_EndDate)
        ' Check if Start Date / End Date is not a valid date format
        If Not IsDate(cell_StartDate.Value) Then
            cell_StartDate.Interior.Color = ERR_HIG
            wsDeliverData.Select
            MsgBox "DeliverData Cell " & cell_StartDate.Address & ": " & "Start Date is invalid!", vbExclamation, "Date Validation Error"
            Exit Sub
        End If
        If Not IsDate(cell_EndDate.Value) Then
            cell_EndDate.Interior.Color = ERR_HIG
            wsDeliverData.Select
            MsgBox "DeliverData Cell " & cell_EndDate.Address & ": " & "End Date is invalid!", vbExclamation, "Date Validation Error"
            Exit Sub
        End If
        ' Check if both are valid dates and End Date < Start Date
        If CDate(cell_EndDate.Value) < CDate(cell_StartDate.Value) Then
            cell_EndDate.Interior.Color = ERR_HIG
            wsDeliverData.Select
            MsgBox "DeliverData Cell " & cell_EndDate.Address & ": " & "End Date is smaller than Start Date!", vbExclamation, "Date Validation Error"
            Exit Sub
        End If
    Next r
    ' Create PivotTable for DeliverData
    Set pcDeliverData = ThisWorkbook.PivotCaches.Create(SourceType:=xlDatabase, SourceData:=dataDeliverData)
    Set ptDeliverData = pcDeliverData.CreatePivotTable(TableDestination:=pivotStartCellDeliverData, TableName:="PivotDeliverData")
    DoEvents
    With ptDeliverData
        With .PivotFields("Plan Name")
             .Orientation = xlRowField
             .Subtotals = Array(True, False, False, False, False, False, False, False, False, False, False, False)
             .Caption = "Plan Name"
        End With
        With .PivotFields("Publisher/Vendor")
             .Orientation = xlRowField
             .Subtotals = Array(False, False, False, False, False, False, False, False, False, False, False, False)
        End With
        With .PivotFields("Category")
             .Orientation = xlRowField
             .Subtotals = Array(False, False, False, False, False, False, False, False, False, False, False, False)
        End With
        With .PivotFields("YQ")
             .Orientation = xlRowField
             .Subtotals = Array(False, False, False, False, False, False, False, False, False, False, False, False)
        End With
        With .PivotFields("Start Date")
             .Orientation = xlDataField
             .Function = xlMin
             .NumberFormat = "dd-mmm-yyyy"
             .Name = "Earliest Start Date"
        End With
        With .PivotFields("End Date")
             .Orientation = xlDataField
             .Function = xlMax
             .NumberFormat = "dd-mmm-yyyy"
             .Name = "Latest End Date"
        End With
        With .PivotFields("Start Date")
             .Orientation = xlDataField
             .Function = xlCount
             .NumberFormat = "#,##0"
             .Name = "Count of Start Date"
        End With
        With .PivotFields("Media Cost (LC)")
             .Orientation = xlDataField
             .Function = xlSum
             .NumberFormat = "#,##0"
        End With
        .RowAxisLayout xlTabularRow
    End With
    Call SubFunc_PivotRepeatLabel(ptDeliverData, True) ' On/Show

    ' Create PivotTable for DeliverData (Full Breakdown)
    Set pcDeliverData_2 = ThisWorkbook.PivotCaches.Create(SourceType:=xlDatabase, SourceData:=dataDeliverData)
    Set ptDeliverData_2 = pcDeliverData_2.CreatePivotTable(TableDestination:=pivotStartCellDeliverData_2, TableName:="PivotDeliverData_2")
    DoEvents ' Ensure PivotTable is ready before accessing fields
    With ptDeliverData_2
        With .PivotFields("Country")
             .Orientation = xlRowField
             .Subtotals = Array(False, False, False, False, False, False, False, False, False, False, False, False)
        End With
        With .PivotFields("Plan Name")
             .Orientation = xlRowField
             .Subtotals = Array(False, False, False, False, False, False, False, False, False, False, False, False)
        End With
        With .PivotFields("Campaign")
             .Orientation = xlRowField
             .Subtotals = Array(False, False, False, False, False, False, False, False, False, False, False, False)
        End With
        With .PivotFields("Campaign Objective")
             .Orientation = xlRowField
             .Subtotals = Array(False, False, False, False, False, False, False, False, False, False, False, False)
        End With
        With .PivotFields("Cohorts")
             .Orientation = xlRowField
             .Subtotals = Array(False, False, False, False, False, False, False, False, False, False, False, False)
        End With
        With .PivotFields("Publisher/Vendor")
             .Orientation = xlRowField
             .Subtotals = Array(False, False, False, False, False, False, False, False, False, False, False, False)
        End With
        With .PivotFields("Vehicle Name")
             .Orientation = xlRowField
             .Subtotals = Array(False, False, False, False, False, False, False, False, False, False, False, False)
        End With
        With .PivotFields("Media Channel")
             .Orientation = xlRowField
             .Subtotals = Array(False, False, False, False, False, False, False, False, False, False, False, False)
        End With
        With .PivotFields("Inventory (Distribution) Type")
             .Orientation = xlRowField
             .Subtotals = Array(False, False, False, False, False, False, False, False, False, False, False, False)
        End With
        With .PivotFields("Primary Target Strategy")
             .Orientation = xlRowField
             .Subtotals = Array(False, False, False, False, False, False, False, False, False, False, False, False)
        End With
        With .PivotFields("Promotion Incentive")
             .Orientation = xlRowField
             .Subtotals = Array(False, False, False, False, False, False, False, False, False, False, False, False)
        End With
        With .PivotFields("Passion Pillar (Placement-Level)")
             .Orientation = xlRowField
             .Subtotals = Array(False, False, False, False, False, False, False, False, False, False, False, False)
        End With
        With .PivotFields("Initiative (Placement-Level)")
             .Orientation = xlRowField
             .Subtotals = Array(False, False, False, False, False, False, False, False, False, False, False, False)
        End With
        With .PivotFields("Media Market")
             .Orientation = xlRowField
             .Subtotals = Array(False, False, False, False, False, False, False, False, False, False, False, False)
        End With
        With .PivotFields("Passion Pillar (Creative-Level)")
             .Orientation = xlRowField
             .Subtotals = Array(False, False, False, False, False, False, False, False, False, False, False, False)
        End With
        With .PivotFields("Initiative (Creative-Level)")
             .Orientation = xlRowField
             .Subtotals = Array(False, False, False, False, False, False, False, False, False, False, False, False)
        End With
        With .PivotFields("Category")
             .Orientation = xlRowField
             .Subtotals = Array(False, False, False, False, False, False, False, False, False, False, False, False)
        End With
        With .PivotFields("Notes")
             .Orientation = xlRowField
             .Subtotals = Array(False, False, False, False, False, False, False, False, False, False, False, False)
        End With
        With .PivotFields("Does Media Cost include Production Fees?")
             .Orientation = xlRowField
             .Subtotals = Array(False, False, False, False, False, False, False, False, False, False, False, False)
        End With
        With .PivotFields("Currency")
             .Orientation = xlRowField
             .Subtotals = Array(False, False, False, False, False, False, False, False, False, False, False, False)
        End With
        With .PivotFields("Media Cost (LC)")
             .Orientation = xlDataField
             .Function = xlSum
             .NumberFormat = "#,##0"
        End With
        With .PivotFields("Paid Impressions")
             .Orientation = xlDataField
             .Function = xlSum
             .NumberFormat = "#,##0"
        End With
        With .PivotFields("Organic Impressions")
             .Orientation = xlDataField
             .Function = xlSum
             .NumberFormat = "#,##0"
        End With
        With .PivotFields("Clicks")
             .Orientation = xlDataField
             .Function = xlSum
             .NumberFormat = "#,##0"
        End With
        With .PivotFields("Video Views")
             .Orientation = xlDataField
             .Function = xlSum
             .NumberFormat = "#,##0"
        End With
        With .PivotFields("Video Completes")
             .Orientation = xlDataField
             .Function = xlSum
             .NumberFormat = "#,##0"
        End With
        With .PivotFields("TV National GRPs")
             .Orientation = xlDataField
             .Function = xlSum
             .NumberFormat = "#,##0.00"
        End With
        With .PivotFields("TV Duration (seconds)")
             .Orientation = xlDataField
             .Function = xlAverage
             .NumberFormat = "#,##0"
        End With
        .RowAxisLayout xlTabularRow
    End With
    Call SubFunc_PivotRepeatLabel(ptDeliverData_2, True) ' On/Show

    'Set rngDeliverData = ptDeliverData_2.TableRange1
    ' Get Min and Max Date Range from the file name
    'Dim output_WeekStartRange As Date, output_WeekEndRange As Date, output_DayStartRange As Date, output_DayEndRange As Date
    'Call SubFunc_ExtractFileDateRange(output_WeekStartRange, output_WeekEndRange, output_DayStartRange, output_DayEndRange)
    ' Highlight Earliest Start Date and Latest End Date that outside of the range
    'For p = 2 To rngDeliverData.Rows.Count ' Start from row 2
        'cellValue_EarliestStartDate = rngDeliverData.Cells(p, 24).Value
        'cellValue_LatestEndDate = rngDeliverData.Cells(p, 25).Value
        'If cellValue_EarliestStartDate < output_DayStartRange Then rngDeliverData.Cells(p, 24).Interior.Color = ERR_HIG
        'If cellValue_LatestEndDate > output_DayEndRange Then rngDeliverData.Cells(p, 25).Interior.Color = ERR_HIG
    'Next p
    
    ' Create PivotTable for DeliverData Classification
    Set pcDeliverData_3 = ThisWorkbook.PivotCaches.Create(SourceType:=xlDatabase, SourceData:=dataDeliverData)
    Set ptDeliverData_3 = pcDeliverData_3.CreatePivotTable(TableDestination:=pivotStartCellDeliverData_3, TableName:="PivotDeliverData_3")
    DoEvents
    With ptDeliverData_3
        With .PivotFields("Plan Name")
             .Orientation = xlRowField
             .Subtotals = Array(False, False, False, False, False, False, False, False, False, False, False, False)
             .Caption = "Plan Name"
        End With
        With .PivotFields("Country")
             .Orientation = xlRowField
             .Subtotals = Array(False, False, False, False, False, False, False, False, False, False, False, False)
        End With
        With .PivotFields("UniqueClientName Campaign Lead")
             .Orientation = xlRowField
             .Subtotals = Array(False, False, False, False, False, False, False, False, False, False, False, False)
        End With
        With .PivotFields("Funding Source")
             .Orientation = xlRowField
             .Subtotals = Array(False, False, False, False, False, False, False, False, False, False, False, False)
        End With
        With .PivotFields("Fund")
             .Orientation = xlRowField
             .Subtotals = Array(False, False, False, False, False, False, False, False, False, False, False, False)
        End With
        With .PivotFields("Initiative")
             .Orientation = xlRowField
             .Subtotals = Array(False, False, False, False, False, False, False, False, False, False, False, False)
        End With
        With .PivotFields("Profitable Consumer Behaviour")
             .Orientation = xlRowField
             .Subtotals = Array(False, False, False, False, False, False, False, False, False, False, False, False)
        End With
        With .PivotFields("Product Message")
             .Orientation = xlRowField
             .Subtotals = Array(False, False, False, False, False, False, False, False, False, False, False, False)
        End With
        With .PivotFields("Destination")
             .Orientation = xlRowField
             .Subtotals = Array(False, False, False, False, False, False, False, False, False, False, False, False)
        End With
        With .PivotFields("Sponsorship")
             .Orientation = xlRowField
             .Subtotals = Array(False, False, False, False, False, False, False, False, False, False, False, False)
        End With
        With .PivotFields("Passion Pillar (Campaign-Level)")
             .Orientation = xlRowField
             .Subtotals = Array(False, False, False, False, False, False, False, False, False, False, False, False)
        End With
        With .PivotFields("Initiative (Campaign-Level)")
             .Orientation = xlRowField
             .Subtotals = Array(False, False, False, False, False, False, False, False, False, False, False, False)
        End With
        With .PivotFields("Notes")
             .Orientation = xlRowField
             .Subtotals = Array(False, False, False, False, False, False, False, False, False, False, False, False)
        End With
        .RowAxisLayout xlTabularRow
    End With
    Call SubFunc_PivotRepeatLabel(ptDeliverData_3, True) ' On/Show

    ' Appy additional complex conditional formatting logic
    Call SubFunc_ConFor_Spend_DeliverData_Overview(ptDeliverData) ' Few conditions for DeliverData Overview table
    Call SubFunc_ConFor_Spend_DeliverData_Breakdown(ptDeliverData_2) ' Lots of conditions for DeliverData Breakdown table
    Call SubFunc_ConFor_Spend_DeliverData_Breakdown_IDT(ptDeliverData_2) ' Extra condition for DeliverData Breakdown table, Inventory Type

    Call SubFunc_ConFor_Classification_DeliverData_Overview(ptDeliverData_3) ' Lots of conditions for DeliverData Breakdown table
    Call SubFunc_ConFor_Classification_MultipleRows(ptDeliverData_3, 11, False) ' Col 11, start block false

    Set wsIL = ThisWorkbook.Sheets("Issue List")
    Application.DisplayAlerts = False ' Disable delete confirmation prompt
    wsIL.Delete ' Delete the worksheet
    Application.DisplayAlerts = True ' Re-enable alerts

    wsNew.Columns(37).NumberFormat = "#,##0.00" ' Apply extra format for DeliverData Breakdown, CPM column

    wsNew.UsedRange.Font.Name = "Arial"
    wsNew.UsedRange.Font.Size = 8
    wsNew.Columns.AutoFit

    wsNew2.UsedRange.Font.Name = "Arial"
    wsNew2.UsedRange.Font.Size = 8
    wsNew2.Columns.AutoFit
    Call SubFunc_SetZoomLevel(ThisWorkbook)
    wsNew.Select
    totalRunTime = Round((Timer - BenchMark) / 60, 2)
    Application.ScreenUpdating = True
    If Not showMsg Then MsgBox "DeliverData All is ready!" & vbCrLf & _
           "Total run time: " & totalRunTime & " minutes", vbInformation, "Complete"
End Sub
"""

In [ ]:
Module_Macro_8 = """
Sub DeliverData_Generate_WeekMonth()
    Dim wsDeliverData As Worksheet, headerRowDeliverData As Range, lastRowDeliverData As Long, i As Long, j As Long
    Dim colCategory As Long, colStart As Long, colEnd As Long, colBeginning As Long, colEnding As Long
    Dim dtDate As Date, beginDate As Date, endingDate As Date
    BenchMark = Timer
    Application.ScreenUpdating = False
    Set wsDeliverData = ThisWorkbook.Sheets("DeliverData")
    wsDeliverData.AutoFilterMode = False
    lastRowDeliverData = wsDeliverData.Cells(wsDeliverData.Rows.Count, 1).End(xlUp).Row
    Set headerRowDeliverData = wsDeliverData.Rows(1)
    colStart = Application.Match("Start Date", headerRowDeliverData, 0)
    colEnd = Application.Match("End Date", headerRowDeliverData, 0)
    Call SubFunc_CreateNewColumnRight(colEnd, colBeginning, "Week/Month Beginning", wsDeliverData)
    Call SubFunc_CreateNewColumnRight(colBeginning, colEnding, "Week/Month Ending", wsDeliverData)
    colCategory = Application.Match("Category", headerRowDeliverData, 0)

    ' Generate Week/Month Beginning
    For i = 2 To lastRowDeliverData ' Populate formulas row by row
        Call SubFunc_EnforceCorrectDate(wsDeliverData, i, colStart, dtDate)
        If dtDate = NA Then
            wsDeliverData.Cells(i, colBeginning).Value = NA
            GoTo NextRow
        End If
        ' Determine beginning date
        If InStr(1, wsDeliverData.Cells(i, colCategory).Value, "Offline-Monthly", vbTextCompare) > 0 Then
            beginDate = DateSerial(Year(dtDate), Month(dtDate), 1)
        Else
            beginDate = dtDate - Weekday(dtDate, vbMonday) + 1
        End If
        wsDeliverData.Cells(i, colBeginning).Value = beginDate
NextRow:
    Next i
    
    ' Generate Week/Month Ending
    For j = 2 To lastRowDeliverData
        Call SubFunc_EnforceCorrectDate(wsDeliverData, j, colBeginning, dtDate)
        If dtDate = NA Then
            wsDeliverData.Cells(j, colEnding).Value = NA
            GoTo NextE
        End If
        If InStr(1, wsDeliverData.Cells(j, colCategory).Value, "Offline-Monthly", vbTextCompare) > 0 Then
            endingDate = DateSerial(Year(dtDate), Month(dtDate) + 1, 0) ' Last day of month via zero-day trick
        Else
            endingDate = dtDate + 6 ' Last day of the week
        End If
        wsDeliverData.Cells(j, colEnding).Value = endingDate
NextE:
    Next j

    Call SubFunc_UnifyMetricCol(wsDeliverData)
    wsDeliverData.UsedRange.Font.Name = "Arial"
    wsDeliverData.UsedRange.Font.Size = 8
    wsDeliverData.Select
    'totalRunTime = Round((Timer - BenchMark) / 60, 2)
    Application.ScreenUpdating = True
    'MsgBox "Generate Week/Month Beginning/Ending completed successfully!" & vbCrLf & _
           '"Total run time: " & totalRunTime & " minutes", vbInformation, "Complete"
End Sub

Sub DeliverData_Generate_CPM()
    Dim wsDeliverData As Worksheet, lastRowDeliverData As Long
    BenchMark = Timer
    Application.ScreenUpdating = False
    Set wsDeliverData = ThisWorkbook.Sheets("DeliverData")
    lastRowDeliverData = wsDeliverData.Cells(Rows.Count, 1).End(xlUp).Row
    Set headerRowDeliverData = wsDeliverData.Rows(1)
    col_MediaChannel = Application.Match("Media Channel", headerRowDeliverData, 0)
    col_MediaCostLC = Application.Match("Media Cost (LC)", headerRowDeliverData, 0)
    col_PaidImpressions = Application.Match("Paid Impressions", headerRowDeliverData, 0)
    col_Clicks = Application.Match("Clicks", headerRowDeliverData, 0)
    col_TVGRP = Application.Match("TV National GRPs", headerRowDeliverData, 0)
    col_CPMCPCCPP = Application.Match("CPM/CPC/CPP", headerRowDeliverData, 0)

    For r = 2 To lastRowDeliverData
        cellValue_MediaChannel = Trim(CStr(wsDeliverData.Cells(r, col_MediaChannel).Value))
        cellValue_MediaCostLC = Trim(wsDeliverData.Cells(r, col_MediaCostLC).Value)
        cellValue_PaidImpressions = Trim(wsDeliverData.Cells(r, col_PaidImpressions).Value)
        cellValue_Clicks = Trim(wsDeliverData.Cells(r, col_Clicks).Value)
        cellValue_TVGRP = Trim(wsDeliverData.Cells(r, col_TVGRP).Value)

        ' Calculation logic for CPM/CPC/CPP column
        If UCase(cellValue_MediaChannel) = "TV" Then ' Media Channel = TV
            If IsNumeric(cellValue_MediaCostLC) Then ' For numeric Media Cost (LC)
                If IsNumeric(cellValue_TVGRP) And cellValue_TVGRP <> 0 Then ' For numeric and non-zero GRP
                    ' CPP = Media Cost (LC) / TV National GRPs
                    cellValue_CPMCPCCPP = cellValue_MediaCostLC / cellValue_TVGRP
                Else ' For non-numeric and zero GRP
                    cellValue_CPMCPCCPP = "#DIV/0!" ' Write error
                End If
            Else ' For non-numeric Media Cost (LC)
                cellValue_CPMCPCCPP = "#N/A" ' Write error
            End If
        ElseIf UCase(cellValue_MediaChannel) = "SEARCH" Then ' Media Channel = Search
            If IsNumeric(cellValue_MediaCostLC) Then ' For numeric Media Cost (LC)
                If IsNumeric(cellValue_Clicks) And cellValue_Clicks <> 0 Then ' For numeric and non-zero Clicks
                    ' CPC = Media Cost (LC) / Clicks
                    cellValue_CPMCPCCPP = cellValue_MediaCostLC / cellValue_Clicks
                Else ' For non-numeric and zero Clicks
                    cellValue_CPMCPCCPP = "#DIV/0!" ' Write error
                End If
            Else ' For non-numeric Media Cost (LC)
                cellValue_CPMCPCCPP = "#N/A" ' Write error
            End If
        Else ' Everything else
            If IsNumeric(cellValue_MediaCostLC) Then ' For numeric Media Cost (LC)
                If IsNumeric(cellValue_PaidImpressions) And cellValue_PaidImpressions <> 0 Then ' For numeric and non-zero Paid Impressions
                    ' CPM = (Media Cost (LC) / Paid Impressions) * 1000
                    cellValue_CPMCPCCPP = (cellValue_MediaCostLC / cellValue_PaidImpressions) * 1000
                Else ' For non-numeric and zero Paid Impressions
                    cellValue_CPMCPCCPP = "#DIV/0!" ' Write error
                End If
            Else ' For non-numeric Media Cost (LC)
                cellValue_CPMCPCCPP = "#N/A" ' Write error
            End If
        End If
        ' Write the final CPM value back into the cell
        wsDeliverData.Cells(r, col_CPMCPCCPP).Value = cellValue_CPMCPCCPP
    Next r

    Call SubFunc_UnifyMetricCol(wsDeliverData)
    wsDeliverData.UsedRange.Font.Name = "Arial"
    wsDeliverData.UsedRange.Font.Size = 8
    wsDeliverData.Select
    totalRunTime = Round((Timer - BenchMark) / 60, 2)
    Application.ScreenUpdating = True
    MsgBox "Calculate CPM/CPC/CPP completed successfully!" & vbCrLf & _
           "Total run time: " & totalRunTime & " minutes", vbInformation, "Complete"
End Sub

Public Sub SubFunc_CreateDateDifference(ws As Worksheet)
    ' Helper function to generate the Date Difference column
    Dim lastRow As Long, colCategory As Long, colStart As Long, colEnd As Long, colDiff As Long, i As Long
    Dim valStart As Variant, valEnd As Variant, diffVal As Variant, headerRow As Range
    ws.AutoFilterMode = False
    lastRow = ws.Cells(ws.Rows.Count, 1).End(xlUp).Row
    Set headerRow = ws.Rows(1)
    colCategory = Application.Match("Category", headerRow, 0)
    colStart = Application.Match("Start Date", headerRow, 0)
    colEnd = Application.Match("End Date", headerRow, 0)
    ' Create new column for Date Difference
    Call SubFunc_CreateNewColumnRight(colEnd, colDiff, "Date Difference", ws)
    ' Generate Date Difference values row by row
    For i = 2 To lastRow
        valStart = ws.Cells(i, colStart).Value
        valEnd = ws.Cells(i, colEnd).Value
        ' Calculate date difference: (End Date – Start Date) + 1
        If IsDate(valStart) And IsDate(valEnd) Then
            diffVal = DateValue(valEnd) - DateValue(valStart) + 1
        Else
            diffVal = "N/A"
        End If
        ws.Cells(i, colDiff).Value = diffVal
        ws.Cells(i, colDiff).NumberFormat = "#,##0"
    Next i
End Sub
"""

In [ ]:
Module_Macro_9 = """
Sub DeliverData_Channel_Social()
    BenchMark = Timer
    Application.ScreenUpdating = False
    Set wsDeliverData = ThisWorkbook.Sheets("DeliverData")
    wsDeliverData.AutoFilterMode = False

    lastRowDeliverData = wsDeliverData.Cells(Rows.Count, 1).End(xlUp).Row
    Set headerRow = wsDeliverData.Rows(1)

    colMediaChannel = Application.Match("Media Channel", headerRow, 0)
    colChannel = Application.Match("Channel", headerRow, 0)
    colPublisher = Application.Match("Publisher/Vendor", headerRow, 0)
    colVehicleName = Application.Match("Vehicle Name", headerRow, 0)
    colSocial = Application.Match("Social?", headerRow, 0)

    ' Build Channel_Map: Media Channel to Channel
    Set Channel_Map = CreateObject("Scripting.Dictionary")
    With Channel_Map
        .Add "Digital Audio", "Digital"
        .Add "Digital Display (Non-Paid Social)", "Digital"
        .Add "Digital OOH", "OOH"
        .Add "Digital Video (Non-CTV, OTT, Paid Social)", "Digital"
        .Add "Direct Mail", "Digital"
        .Add "Email", "Digital"
        .Add "Influencer/Creator Display", "Digital"
        .Add "Influencer/Creator Video", "Digital"
        .Add "Non-Linear TV (VOD, OTT & CTV)", "Digital"
        .Add "OTHER (I.E. EVENT SIGN.)", "Digital"
        .Add "Paid Social Display", "Digital"
        .Add "Paid Social Video", "Digital"
        .Add "Search", "Digital"
        .Add "Airport", "OOH"
        .Add "OOH-Other", "OOH"
        .Add "Cinema", "Cinema"
        .Add "Radio", "Radio"
        .Add "Magazines", "Print"
        .Add "Newspapers", "Print"
        '.Add "DM/eDM", "DM/eDM"
        .Add "Instore/POS", "Instore/POS"
        .Add "TV", "TV"
    End With

    ' Define SocialPublisher_List as an array
    SocialPublisher_List = Array("Meta Facebook", "Meta Instagram", "LinkedIn", "Threads", "TikTok", "Snapchat", "WeChat", "X (Twitter)", "Kakao", "Kakao Bizboard", "Kakao Moment", "Kakaotalk")

    ' Apply the mappings
    If colMediaChannel > 0 And colChannel > 0 And colSocial > 0 Then
        For r = 2 To lastRowDeliverData
            cellMediaChannel = Trim(wsDeliverData.Cells(r, colMediaChannel).Value)
            cellPublisher = Trim(wsDeliverData.Cells(r, colPublisher).Value)
            cellVehicleName = Trim(wsDeliverData.Cells(r, colVehicleName).Value)

            ' Look up and write Channel value
            If Channel_Map.Exists(cellMediaChannel) Then
                wsDeliverData.Cells(r, colChannel).Value = Channel_Map(cellMediaChannel)
            End If

            ' Check if (Publisher or Vehicle Name) is in SocialPublisher_List
            If IsInArray(cellPublisher, SocialPublisher_List) Or IsInArray(cellVehicleName, SocialPublisher_List) Then
                wsDeliverData.Cells(r, colSocial).Value = "Social"
            Else
                wsDeliverData.Cells(r, colSocial).Value = "Non-Social"
            End If
        Next r
    End If
    
    ' Add the 5th button for modelled market
    Dim btn5 As Button, rng5 As Range
    Set wsGuidelines = ThisWorkbook.Worksheets("Guidelines")
    On Error Resume Next
    wsGuidelines.Shapes("btnRun5").Delete
    On Error GoTo 0
    Set rng5 = wsGuidelines.Range(wsGuidelines.Cells(60, 1), wsGuidelines.Cells(61, 3))
    With wsGuidelines
        Set btn5 = wsGuidelines.Buttons.Add(rng5.Left, rng5.Top, rng5.Width, rng5.Height)
        btn5.Name = "btnRun5"
        btn5.OnAction = "Split_DeliverData_Templates_1a"
        btn5.Characters.Text = "Refresh Data Breakout, Weekly, Monthly sheets"
    End With

    ' Format the worksheet
    wsDeliverData.UsedRange.Font.Name = "Arial"
    wsDeliverData.UsedRange.Font.Size = 8
    wsDeliverData.Columns.AutoFit
    Call SubFunc_SetZoomLevel(ThisWorkbook)
    wsDeliverData.Select

    totalRunTime = Round((Timer - BenchMark) / 60, 2)
    Application.ScreenUpdating = True
    MsgBox "Generate Channel & Social completed successfully!" & vbCrLf & _
           "Total run time: " & totalRunTime & " minutes", vbInformation, "Complete"
End Sub

Sub DeliverData_Extract_Campaign()
    Dim ws As Worksheet
    Dim lastRow As Long, i As Long, firstPos As Long, lastPos As Long
    Dim colPlanName As Variant, colCampaign As Variant
    Dim planNameText As String, campaignText As String

    Set ws = ThisWorkbook.Worksheets("DeliverData")
    ws.AutoFilterMode = False
    lastRow = ws.Cells(ws.Rows.Count, "A").End(xlUp).Row
    colPlanName = Application.Match("Plan Name", ws.Rows(1), 0)
    colCampaign = Application.Match("Campaign", ws.Rows(1), 0)

    If IsError(colCampaign) Then
        ws.Columns(colPlanName + 1).Insert Shift:=xlToRight
        colCampaign = colPlanName + 1
        ws.Cells(1, colCampaign).Value = "Campaign"
    Else
    End If

    For i = 2 To lastRow
        planNameText = CStr(ws.Cells(i, colPlanName).Value)
        If Len(planNameText) > 0 Then
            firstPos = InStr(1, planNameText, "_")
            lastPos = InStrRev(planNameText, "_")
            If firstPos > 0 Then
                If lastPos > firstPos Then
                    campaignText = Left(planNameText, firstPos - 1) & " " & Mid(planNameText, lastPos + 1)
                Else
                    campaignText = Left(planNameText, firstPos - 1) & " " & Mid(planNameText, firstPos + 1)
                End If
            End If
        ws.Cells(i, colCampaign).Value = campaignText
        End If
    Next i
    ws.UsedRange.Font.Name = "Arial"
    ws.UsedRange.Font.Size = 8
End Sub
"""

In [ ]:
Module_Macro_10 = """
Sub DeliverData_Date_Granularity()
    Dim wsDeliverData As Worksheet, wsDG As Worksheet, errorMess As String
    Application.ScreenUpdating = False
    BenchMark = Timer
    Set wsDeliverData = ThisWorkbook.Worksheets("DeliverData")
    wsDeliverData.AutoFilterMode = False
    Call SubFunc_CustomizeWholeSheet(Array("DeliverData"))
    Call SubFunc_UnifyMetricCol(wsDeliverData)

    ' Delete sheet "DeliverData_Date_Granularity" if it exists to ensure a fresh copy
    On Error Resume Next
    Application.DisplayAlerts = False
    ThisWorkbook.Worksheets("DeliverData_Date_Granularity").Delete
    Application.DisplayAlerts = True
    On Error GoTo 0

    ' Create a copy of the "DeliverData" sheet
    wsDeliverData.Copy After:=wsDeliverData
    Set wsDG = ActiveSheet
    wsDG.Name = "DeliverData_Date_Granularity"

    ' Create and apply conditional formating for Date Difference
    Call SubFunc_CreateDateDifference(wsDG)

    ' Row Splitting Logic for Daily/Weekly & Monthly
    Call SubFunc_DateGranularityLogic(wsDG, errorMess)

    If Len(errorMess) > 0 Then
        ' If there is errorMess, show MsgBox and stop
        MsgBox "Issues detected: " & errorMess, vbExclamation, "Date Validation Error"
        Exit Sub
    End If

    ' If no errorMess, the Splitting is finished, check the Date Granularity output
    Call SubFunc_DateGranularityValidation(wsDG, errorMess)
    If Len(errorMess) > 0 Then
        ' If there is errorMess, show MsgBox and stop
        MsgBox "Issues detected: " & errorMess, vbExclamation, "Date Validation Error"
        Exit Sub
    Else
        wsDG.Columns(Application.Match("Date Difference", wsDG.Rows(1), 0)).Delete
    End If

    ' Re-format all the important columns
    Call SubFunc_UnifyMetricCol(wsDG)
    Call SubFunc_SetZoomLevel(ThisWorkbook)
    'totalRunTime = Round((Timer - BenchMark) / 60, 2)
    Application.ScreenUpdating = True
    'MsgBox "DeliverData Date Granularity completed successfully!" & vbCrLf & _
           '"Total run time: " & totalRunTime & " minutes", vbInformation, "Complete"
    wsDG.Select
End Sub

' Complicated row splitting logic for Daily/Weekly and Monthly campaigns
Private Sub SubFunc_DateGranularityLogic(ws As Worksheet, ByRef errorMess As String)
    Dim currentRow As Long, originalDateDiff As Long, weekDateDiff As Long, numWeeks As Long, weekNum As Long, insertRow As Long, tempDayOfWeek As Long, dayOfWeek As Long
    Dim colCategory As Long, colStart As Long, colEnd As Long, i As Long, j As Long, colBeginning As Long
    Dim categoryVal As String
    Dim originalStartDate As Date, originalEndDate As Date, weekStartDate As Date, weekEndDate As Date, tempStartDate As Date
    Dim originalMediaCost As Double, originalPaidImpressions As Double, weekMediaCost As Double, weekPaidImpressions As Double
    Dim dtDate As Date, beginDate As Date, endingDate As Date

    Set dataRange = ws.Range("A1").CurrentRegion
    lastRow = dataRange.Rows.Count
    lastCol = dataRange.Columns.Count
    Set headerRow = ws.Rows(1)
    colCategory = Application.Match("Category", headerRow, 0)
    colStart = Application.Match("Start Date", headerRow, 0)
    colEnd = Application.Match("End Date", headerRow, 0)
    colDiff = Application.Match("Date Difference", headerRow, 0)
    colMediaCostLC = Application.Match("Media Cost (LC)", headerRow, 0)
    colMediaCostUSD = Application.Match("Media Cost (USD)", headerRow, 0)
    colPaidImpressions = Application.Match("Paid Impressions", headerRow, 0)
    colOrganicImpressions = Application.Match("Organic Impressions", headerRow, 0)
    colClicks = Application.Match("Clicks", headerRow, 0)
    colVideoViews = Application.Match("Video Views", headerRow, 0)
    colVideoComplete = Application.Match("Video Completes", headerRow, 0)

    ' Loop from bottom to top so insertions don't affect loop counter
    currentRow = lastRow
    Do While currentRow >= 2
        categoryVal = ws.Cells(currentRow, colCategory).Value
        ' Safety check numeric
        If IsNumeric(ws.Cells(currentRow, colDiff).Value) Then
            originalDateDiff = ws.Cells(currentRow, colDiff).Value
            ' Safety check negative
            If originalDateDiff < 0 Then
                Debug.Print "Row " & currentRow & ": Negative Date Difference " & originalDateDiff
                errorMess = "Cell " & ws.Cells(currentRow, colDiff).Address & ": " & "Date Difference is negative!"
                ws.Cells(currentRow, colDiff).Interior.Color = ERR_HIG
                Exit Sub
            End If

            ' Safety check more than 120 (4 months)
            If originalDateDiff > 120 Then
                Debug.Print "Row " & currentRow & ": Date Difference " & originalDateDiff & " > 120"
                errorMess = "Cell " & ws.Cells(currentRow, colDiff).Address & ": " & "Date Difference is more than 120!"
                ws.Cells(currentRow, colDiff).Interior.Color = ERR_HIG
                Exit Sub
            End If

            Call SubFunc_EnforceCorrectDate(ws, currentRow, colStart, originalStartDate)
            Call SubFunc_EnforceCorrectDate(ws, currentRow, colEnd, originalEndDate)
            ' Safety check valid Start Date & End Date
            If originalStartDate = NA Or originalEndDate = NA Then
                Debug.Print "Row " & currentRow & ": Invalid Start/End Date"
                errorMess = "Cell " & ws.Cells(currentRow, colStart).Address & ": " & "Start/End Date are invalid!"
                ws.Cells(currentRow, colStart).Interior.Color = ERR_HIG
                ws.Cells(currentRow, colEnd).Interior.Color = ERR_HIG
                Exit Sub
            End If

            ' Check if this row needs splitting (Daily or Weekly category AND Date Difference > 7)
            If InStr(1, categoryVal, "Daily", vbTextCompare) > 0 Or InStr(1, categoryVal, "Weekly", vbTextCompare) > 0 Then
                If originalDateDiff > 7 Then
                    ' This row needs to be split into smaller weekly rows
                    originalMediaCost = ws.Cells(currentRow, colMediaCostLC).Value
                    originalMediaCostUSD = ws.Cells(currentRow, colMediaCostUSD).Value
                    originalPaidImpressions = ws.Cells(currentRow, colPaidImpressions).Value
                    originalOrganicImpressions = ws.Cells(currentRow, colOrganicImpressions).Value
                    originalClicks = ws.Cells(currentRow, colClicks).Value
                    originalVideoViews = ws.Cells(currentRow, colVideoViews).Value
                    originalVideoComplete = ws.Cells(currentRow, colVideoComplete).Value

                    ' Calculate number of weeks needed by simulating the week splits
                    numWeeks = 0
                    tempStartDate = originalStartDate

                    Do While tempStartDate <= originalEndDate
                        numWeeks = numWeeks + 1
                        ' Calculate how many days until Sunday from tempStartDate
                        tempDayOfWeek = Weekday(tempStartDate, vbSunday)
                        If tempDayOfWeek = 1 Then
                            ' Sunday - move to next Monday
                            tempStartDate = tempStartDate + 1
                        ElseIf tempDayOfWeek = 2 Then
                            ' Monday - move to next Monday (7 days)
                            tempStartDate = tempStartDate + 7
                        Else
                            ' Any other day - move to next Monday after this week's Sunday
                            tempStartDate = tempStartDate + (7 - tempDayOfWeek + 1) + 1
                        End If
                    Loop

                    ' Process each week
                    weekStartDate = originalStartDate
                    Debug.Print "Row " & currentRow & ": Original Daily/Weekly Start Date " & originalStartDate
                    Debug.Print "Row " & currentRow & ": Original Daily/Weekly End Date " & originalEndDate

                    For weekNum = 1 To numWeeks
                        ' Calculate week end date based on what day weekStartDate is
                        ' Weekday returns: 1=Sunday, 2=Monday, 3=Tuesday, ..., 7=Saturday
                        dayOfWeek = Weekday(weekStartDate, vbSunday)
                        If dayOfWeek = 1 Then
                            ' If Sunday, end date is the same day
                            weekEndDate = weekStartDate
                        ElseIf dayOfWeek = 2 Then
                            ' If Monday, end date is 6 days later (Sunday)
                            weekEndDate = weekStartDate + 6
                        Else
                            ' For any other day, calculate days until Sunday
                            weekEndDate = weekStartDate + (7 - dayOfWeek + 1)
                        End If
                        ' Don't exceed the original end date
                        If weekEndDate > originalEndDate Then
                            weekEndDate = originalEndDate
                        End If

                        ' Calculate actual days in this week
                        weekDateDiff = weekEndDate - weekStartDate + 1

                        ' Calculate proportional costs and impressions
                        weekMediaCost = (originalMediaCost / originalDateDiff) * weekDateDiff
                        weekMediaCostUSD = (originalMediaCostUSD / originalDateDiff) * weekDateDiff
                        weekPaidImpressions = (originalPaidImpressions / originalDateDiff) * weekDateDiff
                        weekOrganicImpressions = (originalOrganicImpressions / originalDateDiff) * weekDateDiff
                        weekClicks = (originalClicks / originalDateDiff) * weekDateDiff
                        weekVideoViews = (originalVideoViews / originalDateDiff) * weekDateDiff
                        weekVideoComplete = (originalVideoComplete / originalDateDiff) * weekDateDiff

                        ' For first week, update the existing row
                        If weekNum = 1 Then
                            ws.Cells(currentRow, colStart).Value = weekStartDate
                            ws.Cells(currentRow, colEnd).Value = weekEndDate
                            ws.Cells(currentRow, colDiff).Value = weekDateDiff
                            ws.Cells(currentRow, colMediaCostLC).Value = weekMediaCost
                            ws.Cells(currentRow, colMediaCostUSD).Value = weekMediaCostUSD
                            ws.Cells(currentRow, colPaidImpressions).Value = weekPaidImpressions
                            ws.Cells(currentRow, colOrganicImpressions).Value = weekOrganicImpressions
                            ws.Cells(currentRow, colClicks).Value = weekClicks
                            ws.Cells(currentRow, colVideoViews).Value = weekVideoViews
                            ws.Cells(currentRow, colVideoComplete).Value = weekVideoComplete
                            'ws.Cells(currentRow, colCategory).Interior.Color = vbGreen ' Highlight to detect easily
                        Else
                            ' For subsequent weeks, insert a new row below
                            insertRow = currentRow + weekNum - 1
                            ws.Rows(insertRow).Insert Shift:=xlDown

                            ' Copy the entire row from the original
                            ws.Rows(currentRow).Copy
                            ws.Rows(insertRow).PasteSpecial xlPasteAll
                            Application.CutCopyMode = False

                            ' Update the split-specific values
                            ws.Cells(insertRow, colStart).Value = weekStartDate
                            ws.Cells(insertRow, colEnd).Value = weekEndDate
                            ws.Cells(insertRow, colDiff).Value = weekDateDiff
                            ws.Cells(insertRow, colMediaCostLC).Value = weekMediaCost
                            ws.Cells(insertRow, colMediaCostUSD).Value = weekMediaCostUSD
                            ws.Cells(insertRow, colPaidImpressions).Value = weekPaidImpressions
                            ws.Cells(insertRow, colOrganicImpressions).Value = weekOrganicImpressions
                            ws.Cells(insertRow, colClicks).Value = weekClicks
                            ws.Cells(insertRow, colVideoViews).Value = weekVideoViews
                            ws.Cells(insertRow, colVideoComplete).Value = weekVideoComplete
                            'ws.Cells(insertRow, colCategory).Interior.Color = vbGreen ' Highlight to detect easily
                        End If
                        ' Move to next week (start on Monday)
                        weekStartDate = weekEndDate + 1
                    Next weekNum
                    ' Update lastRow to account for inserted rows
                    lastRow = lastRow + (numWeeks - 1)
                End If
            ' Check if this row needs splitting (Monthly category AND Month difference > 1)
            ElseIf InStr(1, categoryVal, "Monthly", vbTextCompare) > 0 Then
                YearMonthOfStartDate = DateSerial(Year(originalStartDate), Month(originalStartDate), 1)
                YearMonthOfEndDate = DateSerial(Year(originalEndDate), Month(originalEndDate), 1)
                If YearMonthOfStartDate <> YearMonthOfEndDate Then
                    originalMediaCost = ws.Cells(currentRow, colMediaCostLC).Value
                    originalMediaCostUSD = ws.Cells(currentRow, colMediaCostUSD).Value
                    originalPaidImpressions = ws.Cells(currentRow, colPaidImpressions).Value
                    originalOrganicImpressions = ws.Cells(currentRow, colOrganicImpressions).Value
                    originalClicks = ws.Cells(currentRow, colClicks).Value
                    originalVideoViews = ws.Cells(currentRow, colVideoViews).Value
                    originalVideoComplete = ws.Cells(currentRow, colVideoComplete).Value

                    numMonths = DateDiff("m", YearMonthOfStartDate, YearMonthOfEndDate) + 1
                    ' Process each month
                    monthStartDate = originalStartDate
                    Debug.Print "Row " & currentRow & ": Original Monthly Start Date " & originalStartDate
                    Debug.Print "Row " & currentRow & ": Original Monthly End Date " & originalEndDate

                    For monthNum = 1 To numMonths
                        ' Calculate month end date based on Month(originalStartDate)
                        monthEndDate = DateSerial(Year(monthStartDate), Month(monthStartDate) + 1, 0)
                        ' Don't exceed the original end date
                        If monthEndDate > originalEndDate Then
                            monthEndDate = originalEndDate
                        End If

                        ' Calculate actual days in this month
                        monthDateDiff = monthEndDate - monthStartDate + 1
                        ' Calculate proportional costs and impressions
                        monthMediaCost = (originalMediaCost / originalDateDiff) * monthDateDiff
                        monthMediaCostUSD = (originalMediaCostUSD / originalDateDiff) * monthDateDiff
                        monthPaidImpressions = (originalPaidImpressions / originalDateDiff) * monthDateDiff
                        monthOrganicImpressions = (originalOrganicImpressions / originalDateDiff) * monthDateDiff
                        monthClicks = (originalClicks / originalDateDiff) * monthDateDiff
                        monthVideoViews = (originalVideoViews / originalDateDiff) * monthDateDiff
                        monthVideoComplete = (originalVideoComplete / originalDateDiff) * monthDateDiff

                        ' For first month, update the existing row
                        If monthNum = 1 Then
                            ws.Cells(currentRow, colStart).Value = monthStartDate
                            ws.Cells(currentRow, colEnd).Value = monthEndDate
                            ws.Cells(currentRow, colDiff).Value = monthDateDiff
                            ws.Cells(currentRow, colMediaCostLC).Value = monthMediaCost
                            ws.Cells(currentRow, colMediaCostUSD).Value = monthMediaCostUSD
                            ws.Cells(currentRow, colPaidImpressions).Value = monthPaidImpressions
                            ws.Cells(currentRow, colOrganicImpressions).Value = monthOrganicImpressions
                            ws.Cells(currentRow, colClicks).Value = monthClicks
                            ws.Cells(currentRow, colVideoViews).Value = monthVideoViews
                            ws.Cells(currentRow, colVideoComplete).Value = monthVideoComplete
                            'ws.Cells(currentRow, colCategory).Interior.Color = vbGreen ' Highlight to detect easily
                        Else
                            ' For subsequent months, insert a new row below
                            insertRow = currentRow + monthNum - 1
                            ws.Rows(insertRow).Insert Shift:=xlDown
                            ' Copy the entire row from the original
                            ws.Rows(currentRow).Copy
                            ws.Rows(insertRow).PasteSpecial xlPasteAll
                            Application.CutCopyMode = False
                            ' Update the split-specific values
                            ws.Cells(insertRow, colStart).Value = monthStartDate
                            ws.Cells(insertRow, colEnd).Value = monthEndDate
                            ws.Cells(insertRow, colDiff).Value = monthDateDiff
                            ws.Cells(insertRow, colMediaCostLC).Value = monthMediaCost
                            ws.Cells(insertRow, colMediaCostUSD).Value = monthMediaCostUSD
                            ws.Cells(insertRow, colPaidImpressions).Value = monthPaidImpressions
                            ws.Cells(insertRow, colOrganicImpressions).Value = monthOrganicImpressions
                            ws.Cells(insertRow, colClicks).Value = monthClicks
                            ws.Cells(insertRow, colVideoViews).Value = monthVideoViews
                            ws.Cells(insertRow, colVideoComplete).Value = monthVideoComplete
                            'ws.Cells(insertRow, colCategory).Interior.Color = vbGreen ' Highlight to detect easily
                        End If
                        ' Move to next month (start on the beginning of the month)
                        monthStartDate = monthEndDate + 1
                    Next monthNum
                    ' Update lastRow to account for inserted rows
                    lastRow = lastRow + (numMonths - 1)
                End If
            Else
                errorMess = "Cell " & ws.Cells(currentRow, colCategory).Address & ": " & "Category is invalid!"
                Exit Sub
            End If
        Else
            errorMess = "Cell " & ws.Cells(currentRow, colDiff).Address & ": " & "Date Difference is not numeric!"
            Exit Sub
        End If
        currentRow = currentRow - 1
    Loop
End Sub

' A validation function to check the results of Date Granularity
Public Sub SubFunc_DateGranularityValidation(ws As Worksheet, ByRef errorMess As String)
    Dim lastRow As Long, col_StartDate As Long, col_EndDate As Long, col_Difference As Long, col_Beginning As Long, col_Ending As Long, col_Category As Long, headerRow As Range
    ' Create and apply conditional formating for Date Difference
    Call SubFunc_CreateDateDifference(ws)
    lastRow = ws.Cells(Rows.Count, 1).End(xlUp).Row
    Set headerRow = ws.Rows(1)
    col_StartDate = Application.Match("Start Date", headerRow, 0)
    col_EndDate = Application.Match("End Date", headerRow, 0)
    col_Difference = Application.Match("Date Difference", headerRow, 0)
'    col_Beginning = Application.Match("Week/Month Beginning", headerRow, 0)
'    col_Ending = Application.Match("Week/Month Ending", headerRow, 0)
    col_Category = Application.Match("Category", headerRow, 0)
    For r = 2 To lastRow
        Set cell_StartDate = ws.Cells(r, col_StartDate)
        Set cell_EndDate = ws.Cells(r, col_EndDate)
        Set cell_Difference = ws.Cells(r, col_Difference)
'        Set cell_Beginning = ws.Cells(r, col_Beginning)
'        Set cell_Ending = ws.Cells(r, col_Ending)
        Set cell_Category = ws.Cells(r, col_Category)

        If Not IsDate(cell_StartDate.Value) Then
            cell_StartDate.Interior.Color = ERR_HIG
            errorMess = "Cell " & cell_StartDate.Address & ": " & "Start Date is invalid!"
            Exit Sub
        End If

        If Not IsDate(cell_EndDate.Value) Then
            cell_EndDate.Interior.Color = ERR_HIG
            errorMess = "Cell " & cell_EndDate.Address & ": " & "End Date is invalid!"
            Exit Sub
        End If

        If CDate(cell_EndDate.Value) < CDate(cell_StartDate.Value) Then
            cell_EndDate.Interior.Color = ERR_HIG
            errorMess = "Cell " & cell_EndDate.Address & ": " & "End Date is smaller than Start Date!"
            Exit Sub
        End If

'        If Not IsDate(cell_Beginning.Value) Then
'            cell_Beginning.Interior.Color = ERR_HIG
'            errorMess = "Cell " & cell_Beginning.Address & ": " & "Week/Month Beginning is invalid!"
'            Exit Sub
'        End If
'
'        If Not IsDate(cell_Ending.Value) Then
'            cell_Ending.Interior.Color = ERR_HIG
'            errorMess = "Cell " & cell_Ending.Address & ": " & "Week/Month Ending is invalid!"
'            Exit Sub
'        End If
'
'        If CDate(cell_Ending.Value) < CDate(cell_Beginning.Value) Then
'            cell_Ending.Interior.Color = ERR_HIG
'            errorMess = "Cell " & cell_Ending.Address & ": " & "Week/Month Ending is smaller than Beginning!"
'            Exit Sub
'        End If

        If InStr(1, cell_Category.Value, "Daily", vbTextCompare) = 0 And InStr(1, cell_Category.Value, "Weekly", vbTextCompare) = 0 And InStr(1, cell_Category.Value, "Monthly", vbTextCompare) = 0 Then
            cell_Category.Interior.Color = ERR_HIG
            errorMess = "Cell " & cell_Category.Address & ": " & "Category is invalid!"
            Exit Sub
        End If

        If IsNumeric(cell_Difference.Value) Then
            If cell_Difference.Value < 0 Then
                cell_Difference.Interior.Color = ERR_HIG
                errorMess = "Cell " & cell_Difference.Address & ": " & "Date Difference is negative!"
                Exit Sub
            End If

            If cell_Difference.Value > 120 Then
                cell_Difference.Interior.Color = ERR_HIG
                errorMess = "Cell " & cell_Difference.Address & ": " & "Date Difference is more than 120!"
                Exit Sub
            End If

            If InStr(1, cell_Category.Value, "Daily", vbTextCompare) > 0 Or InStr(1, cell_Category.Value, "Weekly", vbTextCompare) > 0 Then
                If cell_Difference.Value > 7 Then
                    cell_Difference.Interior.Color = ERR_HIG
                    errorMess = "Cell " & cell_Difference.Address & ": " & "Category Daily/Weekly and Date Difference is more than 7!"
                    Exit Sub
                End If
            End If

            If InStr(1, cell_Category.Value, "Monthly", vbTextCompare) > 0 Then
                YearMonthOfStartDate = DateSerial(Year(cell_StartDate.Value), Month(cell_StartDate.Value), 1)
                YearMonthOfEndDate = DateSerial(Year(cell_EndDate.Value), Month(cell_EndDate.Value), 1)
                If YearMonthOfStartDate <> YearMonthOfEndDate Then
                    cell_Difference.Interior.Color = ERR_HIG
                    errorMess = "Cell " & cell_Difference.Address & ": " & "Category Monthly and Month Difference is more than 1!"
                    Exit Sub
                End If
            End If
        Else
            cell_Difference.Interior.Color = ERR_HIG
            errorMess = "Cell " & cell_Difference.Address & ": " & "Date Difference is not numeric!"
            Exit Sub
        End If
    Next r
End Sub
"""

In [ ]:
Module_Macro_11 = """
Sub DeliverData_Check_Impression_Pivot()
    Dim wsDeliverData As Worksheet, wsNew As Worksheet
    Dim ptDeliverData As PivotTable, pcDeliverData As PivotCache
    Dim lastRowDeliverData As Long, lastColDeliverData As Long
    Dim dataDeliverData As Range, pivotStartCellDeliverData As Range, rngDeliverData As Range
    Application.ScreenUpdating = False
    Set wsDeliverData = ThisWorkbook.Sheets("DeliverData")
    wsDeliverData.AutoFilterMode = False
    Call SubFunc_CreateNewSheet("DeliverData_Check_Impression_Pivot", wsNew)
    lastRowDeliverData = wsDeliverData.Cells(Rows.Count, 1).End(xlUp).Row
    lastColDeliverData = wsDeliverData.Cells(1, Columns.Count).End(xlToLeft).Column
    Set dataDeliverData = wsDeliverData.Range(wsDeliverData.Cells(1, 1), wsDeliverData.Cells(lastRowDeliverData, lastColDeliverData))
    Set pivotStartCellDeliverData = wsNew.Range("A1")
    Set pcDeliverData = ThisWorkbook.PivotCaches.Create(SourceType:=xlDatabase, SourceData:=dataDeliverData)
    Set ptDeliverData = pcDeliverData.CreatePivotTable(TableDestination:=pivotStartCellDeliverData, TableName:="DeliverDataCheckImpression")
    DoEvents
    On Error Resume Next

    With ptDeliverData
        With .PivotFields("Campaign")
             .Orientation = xlRowField
             .Subtotals = Array(False, False, False, False, False, False, False, False, False, False, False, False)
        End With
        With .PivotFields("Publisher/Vendor")
             .Orientation = xlRowField
             .Subtotals = Array(False, False, False, False, False, False, False, False, False, False, False, False)
        End With
        With .PivotFields("Vehicle Name")
             .Orientation = xlRowField
             .Subtotals = Array(False, False, False, False, False, False, False, False, False, False, False, False)
        End With
        With .PivotFields("Media Channel")
             .Orientation = xlRowField
             .Subtotals = Array(False, False, False, False, False, False, False, False, False, False, False, False)
        End With
        With .PivotFields("Inventory (Distribution) Type")
             .Orientation = xlRowField
             .Subtotals = Array(False, False, False, False, False, False, False, False, False, False, False, False)
        End With
        With .PivotFields("Category")
             .Orientation = xlRowField
             .Subtotals = Array(False, False, False, False, False, False, False, False, False, False, False, False)
        End With
        With .PivotFields("Week/Month Beginning")
             .Orientation = xlRowField
             .Subtotals = Array(False, False, False, False, False, False, False, False, False, False, False, False)
             .NumberFormat = "dd-mmm-yyyy"
        End With
        With .PivotFields("Paid Impressions")
             .Orientation = xlDataField
             .Function = xlSum
             .NumberFormat = "#,##0"
        End With
        With .PivotFields("TV National GRPs")
             .Orientation = xlDataField
             .Function = xlSum
             .NumberFormat = "#,##0.00"
        End With
    .RowAxisLayout xlTabularRow
    .RefreshTable
    End With
    Call SubFunc_PivotRepeatLabel(ptDeliverData, True) ' On/Show

    ' Prepare dictionaries for tracking duplicates
    Set dictTV = CreateObject("Scripting.Dictionary")
    Set dictPaidImpression = CreateObject("Scripting.Dictionary")
    ' Identify header row and column indexes
    headerRow = pivotStartCellDeliverData.Row
    col_MediaChannel = 0: colTVGRPs = 0: colPaidImpression = 0

    ' Find column positions more efficiently
    For c = 1 To ptDeliverData.TableRange1.Columns.Count
        Select Case wsNew.Cells(headerRow, c).Value
            Case "Media Channel": col_MediaChannel = c
            Case "Sum of TV National GRPs": colTVGRPs = c
            Case "Sum of Paid Impressions": colPaidImpression = c
        End Select
        ' Exit early if all columns found
        If col_MediaChannel > 0 And colTVGRPs > 0 And colPaidImpression > 0 Then Exit For
    Next c
    Debug.Print "Media Channel col no " & col_MediaChannel
    Debug.Print "TV col no " & colTVGRPs
    Debug.Print "Paid Impressions col no " & colPaidImpression
    ' Validate that required columns were found
    If col_MediaChannel = 0 Then
        MsgBox "Media Channel column not found!", vbCritical
        Exit Sub
    End If
    Debug.Print "Last row " & ptDeliverData.TableRange1.Rows.Count
    For i = pivotStartCellDeliverData.Row + 1 To ptDeliverData.TableRange1.Rows.Count
        mediaVal = Trim(CStr(wsNew.Cells(i, col_MediaChannel).Value))
        If UCase(mediaVal) = "TV" Then
            ' Process TV data
            If colTVGRPs > 0 Then
                Set cellTV = wsNew.Cells(i, colTVGRPs)
                tvValue = cellTV.Value
                ' Reset formatting first
                cellTV.Interior.ColorIndex = xlColorIndexNone
                cellTV.Font.ColorIndex = xlColorIndexAutomatic
                ' Check for blank/zero values (red fill)
                If IsEmpty(tvValue) Or tvValue = 0 Or tvValue = "" Then
                    cellTV.Interior.Color = ERR_HIG
                ' Check for duplicates (red font)
                ElseIf dictTV.Exists(CStr(tvValue)) Then
                    cellTV.Interior.Color = ERR_HIG
                Else
                    ' Add to dictionary for duplicate tracking
                    dictTV.Add CStr(tvValue), True
                End If
            End If

        Else
            ' Process non-TV data (Paid Impressions)
            If colPaidImpression > 0 Then
                Set cellPaidImpression = wsNew.Cells(i, colPaidImpression)
                paidimpValue = cellPaidImpression.Value
                ' Reset formatting first
                cellPaidImpression.Interior.ColorIndex = xlColorIndexNone
                cellPaidImpression.Font.ColorIndex = xlColorIndexAutomatic
                ' Check for blank/zero values (red fill)
                If IsEmpty(paidimpValue) Or paidimpValue = 0 Or paidimpValue = "" Or paidimpValue < 100 Then
                    cellPaidImpression.Interior.Color = ERR_HIG
                ' Check for duplicates (red font)
                ElseIf dictPaidImpression.Exists(CStr(paidimpValue)) Then
                    cellPaidImpression.Interior.Color = ERR_HIG
                Else
                    ' Add to dictionary for duplicate tracking
                    dictPaidImpression.Add CStr(paidimpValue), True
                End If
            End If
        End If
    Next i
    Set dictTV = Nothing
    Set dictPaidImpression = Nothing
    Set cellTV = Nothing
    Set cellPaidImpression = Nothing
    'Call SubFunc_PivotRepeatLabel(ptDeliverData, False) ' Off/Hide
    wsNew.UsedRange.Font.Name = "Arial"
    wsNew.UsedRange.Font.Size = 8
    wsNew.Columns.AutoFit
    Call SubFunc_SetZoomLevel(ThisWorkbook)
    On Error GoTo 0
    Application.ScreenUpdating = True
    MsgBox "DeliverData Check Impression Pivot is ready!", vbInformation, "Complete"
    wsNew.Select
End Sub
"""

In [ ]:
Module_Macro_12 = """
Sub DeliverData_Redistribute_Cost()
    Dim wsRC As Worksheet, origWS As Worksheet
    Dim dataRange As Range, headerRow As Range
    Dim lastRow As Long, lastCol As Long, i As Long, j As Long, k As Long
    Dim col_PlanName As Long, col_Publisher As Long, col_Difference As Long, col_MediaChannel As Long, col_InventoryDistributionType As Long
    Dim col_MediaCostLC As Long, col_PaidImpressions As Long, col_Clicks As Long, col_CPMCPCPP As Long, col_Category As Long, col_VehicleName As Long
    Dim groupDict_Digital As Object, groupDict_Search As Object, groupDict_NonDigital As Object
    Dim uniqueKey_Digital As String, uniqueKey_Search As String, uniqueKey_NonDigital As String
    Dim groupTotalSums As Variant, match_col_CPMCPCPP As Variant
    Dim groupTotalCost As Double, groupTotalImp As Double, groupAvgCPM As Double, groupTotalClicks As Double, groupAvgCPC As Double, groupTotalDiff As Double
    Dim row_MediaCostLC As Double, row_PaidImpressions As Double, row_Clicks As Double, row_Diff As Double
    BenchMark = Timer
    Application.ScreenUpdating = False

    Set origWS = ThisWorkbook.Worksheets("DeliverData")
    origWS.AutoFilterMode = False
    ' Delete sheet "DeliverData_Redistribute_Cost" if it exists to ensure a fresh copy
    On Error Resume Next
    Application.DisplayAlerts = False
    ThisWorkbook.Worksheets("DeliverData_Redistribute_Cost").Delete
    Application.DisplayAlerts = True
    On Error GoTo 0

    ' Create a copy of the "DeliverData" sheet
    origWS.Copy After:=origWS
    Set wsRC = ActiveSheet
    wsRC.Name = "DeliverData_Redistribute_Cost"

    ' Create and apply conditional formating for Date Difference
    Call SubFunc_CreateDateDifference(wsRC)

    ' Identify column positions based on header names in row 1
    Set headerRow = wsRC.Rows(1)
    col_PlanName = Application.Match("Plan Name", headerRow, 0)
    col_Publisher = Application.Match("Publisher/Vendor", headerRow, 0)
    col_Difference = Application.Match("Date Difference", headerRow, 0)
    col_MediaChannel = Application.Match("Media Channel", headerRow, 0)
    col_InventoryDistributionType = Application.Match("Inventory (Distribution) Type", headerRow, 0)
    col_MediaCostLC = Application.Match("Media Cost (LC)", headerRow, 0)
    col_PaidImpressions = Application.Match("Paid Impressions", headerRow, 0)
    col_Clicks = Application.Match("Clicks", headerRow, 0)
    col_Category = Application.Match("Category", headerRow, 0)
    col_VehicleName = Application.Match("Vehicle Name", headerRow, 0)

    ' Now work on the new sheet "DeliverData_Redistribute_Cost"
    Set dataRange = wsRC.Range("A1").CurrentRegion
    lastRow = dataRange.Rows.Count
    lastCol = dataRange.Columns.Count

    match_col_CPMCPCPP = Application.Match("CPM/CPC/CPP", headerRow, 0)
    If IsError(match_col_CPMCPCPP) Then
        ' Not found: insert new column after the last column
        col_CPMCPCPP = lastCol + 1
        wsRC.Cells(1, col_CPMCPCPP).Value = "CPM/CPC/CPP"
        lastCol = lastCol + 1
    Else
        ' Found: reuse that column, clear old values
        col_CPMCPCPP = match_col_CPMCPCPP
        wsRC.Range(wsRC.Cells(2, col_CPMCPCPP), wsRC.Cells(lastRow, col_CPMCPCPP)).Clear
    End If

    ' Define Digital Media Channel without Search List as an array
    DigitalMediaChannel_List_NoSearch = Array("Digital Audio", "Digital Display (Non-Paid Social)", "Digital Video (Non-CTV, OTT, Paid Social)", "Direct Mail", "Email", "Influencer/Creator Display", "Influencer/Creator Video", "Non-Linear TV (VOD, OTT & CTV)", "OTHER (I.E. EVENT SIGN.)", "Paid Social Display", "Paid Social Video")

    ' Define Non Digital Media Channel List as an array
    NonDigitalMediaChannel_List = Array("Digital OOH", "Airport", "OOH-Other", "Cinema", "Radio", "Magazines", "Newspapers", "Instore/POS", "TV")

    ' Create a dictionary to store group totals
    Set groupDict_Digital = CreateObject("Scripting.Dictionary")
    Set groupDict_Search = CreateObject("Scripting.Dictionary")
    Set groupDict_NonDigital = CreateObject("Scripting.Dictionary")

    ' 1) Redistribute Media Cost (LC) proportionally based on Paid Impressions
    ' Apply AutoFilter: For Media Channel that are considered Digital without Search
    dataRange.AutoFilter Field:=col_MediaChannel, Criteria1:=DigitalMediaChannel_List_NoSearch, Operator:=xlFilterValues

    ' 1a) sum Media Cost (LC) and Paid Impressions for each unique group
    For i = 2 To lastRow
        If Not wsRC.Rows(i).Hidden Then
            uniqueKey_Digital = UCase(Trim(CStr(wsRC.Cells(i, col_Category).Value))) & "|" & _
                                UCase(Trim(CStr(wsRC.Cells(i, col_PlanName).Value))) & "|" & _
                                UCase(Trim(CStr(wsRC.Cells(i, col_Publisher).Value))) & "|" & _
                                UCase(Trim(CStr(wsRC.Cells(i, col_VehicleName).Value))) & "|" & _
                                UCase(Trim(CStr(wsRC.Cells(i, col_MediaChannel).Value))) & "|" & _
                                UCase(Trim(CStr(wsRC.Cells(i, col_InventoryDistributionType).Value)))

            If groupDict_Digital.Exists(uniqueKey_Digital) Then
                groupTotalSums = groupDict_Digital(uniqueKey_Digital)
                groupTotalSums(0) = groupTotalSums(0) + wsRC.Cells(i, col_MediaCostLC).Value ' Total Media Cost (LC)
                groupTotalSums(1) = groupTotalSums(1) + wsRC.Cells(i, col_PaidImpressions).Value  ' Total Paid Impressions
                groupDict_Digital(uniqueKey_Digital) = groupTotalSums
            Else
                ReDim groupTotalSums(0 To 1)
                groupTotalSums(0) = wsRC.Cells(i, col_MediaCostLC).Value  ' Total Media Cost (LC)
                groupTotalSums(1) = wsRC.Cells(i, col_PaidImpressions).Value  ' Total Paid Impressions
                groupDict_Digital.Add uniqueKey_Digital, groupTotalSums
            End If
        End If
    Next i
    ' 1b) Calculate the Average CPM, then multiple it to each row Paid Impressions to get Media Cost (LC)
    For i = 2 To lastRow
        If Not wsRC.Rows(i).Hidden Then
            uniqueKey_Digital = UCase(Trim(CStr(wsRC.Cells(i, col_Category).Value))) & "|" & _
                                UCase(Trim(CStr(wsRC.Cells(i, col_PlanName).Value))) & "|" & _
                                UCase(Trim(CStr(wsRC.Cells(i, col_Publisher).Value))) & "|" & _
                                UCase(Trim(CStr(wsRC.Cells(i, col_VehicleName).Value))) & "|" & _
                                UCase(Trim(CStr(wsRC.Cells(i, col_MediaChannel).Value))) & "|" & _
                                UCase(Trim(CStr(wsRC.Cells(i, col_InventoryDistributionType).Value)))

            groupTotalSums = groupDict_Digital(uniqueKey_Digital)
            groupTotalCost = groupTotalSums(0)
            groupTotalImp = groupTotalSums(1)
            row_PaidImpressions = wsRC.Cells(i, col_PaidImpressions).Value

            If IsNumeric(groupTotalCost) And IsNumeric(groupTotalImp) And groupTotalImp <> 0 Then
                groupAvgCPM = groupTotalCost / groupTotalImp ' Usually *1000 but later also /1000 so no need
                ' Writing the group Average CPM value for debugging
                'wsRC.Cells(i, col_CPMCPCPP).Value = groupAvgCPM
                ' Override Media Cost (LC) for the row
                row_MediaCostLC = groupAvgCPM * row_PaidImpressions
                wsRC.Cells(i, col_MediaCostLC).Value = row_MediaCostLC
            End If
        End If
    Next i

    ' Remove the filter
    wsRC.AutoFilterMode = False
    ' Reset dataRange after filter removal
    Set dataRange = wsRC.Range("A1").CurrentRegion
    lastRow = dataRange.Rows.Count
    lastCol = dataRange.Columns.Count

    ' 2) Redistribute Media Cost (LC) proportionally based on Clicks
    ' Apply AutoFilter: For only Media Channel = Search (Search is part of Digital)
    dataRange.AutoFilter Field:=col_MediaChannel, Criteria1:=Array("Search"), Operator:=xlFilterValues

    ' 2a) sum Media Cost (LC) and Clicks for each unique group
    For j = 2 To lastRow
        If Not wsRC.Rows(j).Hidden Then
            uniqueKey_Search = UCase(Trim(CStr(wsRC.Cells(j, col_Category).Value))) & "|" & _
                                UCase(Trim(CStr(wsRC.Cells(j, col_PlanName).Value))) & "|" & _
                                UCase(Trim(CStr(wsRC.Cells(j, col_Publisher).Value))) & "|" & _
                                UCase(Trim(CStr(wsRC.Cells(j, col_VehicleName).Value))) & "|" & _
                                UCase(Trim(CStr(wsRC.Cells(j, col_MediaChannel).Value))) & "|" & _
                                UCase(Trim(CStr(wsRC.Cells(j, col_InventoryDistributionType).Value)))

            If groupDict_Search.Exists(uniqueKey_Search) Then
                groupTotalSums = groupDict_Search(uniqueKey_Search)
                groupTotalSums(0) = groupTotalSums(0) + wsRC.Cells(j, col_MediaCostLC).Value ' Total Media Cost (LC)
                groupTotalSums(1) = groupTotalSums(1) + wsRC.Cells(j, col_Clicks).Value  ' Total Clicks
                groupDict_Search(uniqueKey_Search) = groupTotalSums
            Else
                ReDim groupTotalSums(0 To 1)
                groupTotalSums(0) = wsRC.Cells(j, col_MediaCostLC).Value  ' Total Media Cost (LC)
                groupTotalSums(1) = wsRC.Cells(j, col_Clicks).Value  ' Total Clicks
                groupDict_Search.Add uniqueKey_Search, groupTotalSums
            End If
        End If
    Next j
    ' 2b) Calculate the Average CPC, then multiple it to each row Clicks to get Media Cost (LC)
    For j = 2 To lastRow
        If Not wsRC.Rows(j).Hidden Then
            uniqueKey_Search = UCase(Trim(CStr(wsRC.Cells(j, col_Category).Value))) & "|" & _
                                UCase(Trim(CStr(wsRC.Cells(j, col_PlanName).Value))) & "|" & _
                                UCase(Trim(CStr(wsRC.Cells(j, col_Publisher).Value))) & "|" & _
                                UCase(Trim(CStr(wsRC.Cells(j, col_VehicleName).Value))) & "|" & _
                                UCase(Trim(CStr(wsRC.Cells(j, col_MediaChannel).Value))) & "|" & _
                                UCase(Trim(CStr(wsRC.Cells(j, col_InventoryDistributionType).Value)))

            groupTotalSums = groupDict_Search(uniqueKey_Search)
            groupTotalCost = groupTotalSums(0)
            groupTotalClicks = groupTotalSums(1)
            row_Clicks = wsRC.Cells(j, col_Clicks).Value

            If IsNumeric(groupTotalCost) And IsNumeric(groupTotalClicks) And groupTotalClicks <> 0 Then
                groupAvgCPC = groupTotalCost / groupTotalClicks
                ' Writing the group Average CPC value for debugging
                'wsRC.Cells(j, col_CPMCPCPP).Value = groupAvgCPC
                ' Override Media Cost (LC) for the row
                row_MediaCostLC = groupAvgCPC * row_Clicks
                wsRC.Cells(j, col_MediaCostLC).Value = row_MediaCostLC
            End If
        End If
    Next j

    ' Remove the filter
    wsRC.AutoFilterMode = False
    ' Reset dataRange after filter removal
    Set dataRange = wsRC.Range("A1").CurrentRegion
    lastRow = dataRange.Rows.Count
    lastCol = dataRange.Columns.Count

    ' 3) Redistribute Media Cost (LC) proportionally based on Date Difference
    ' Apply AutoFilter: For Media Channel that are NOT considered Digital
    dataRange.AutoFilter Field:=col_MediaChannel, Criteria1:=NonDigitalMediaChannel_List, Operator:=xlFilterValues

    ' 3a) sum Media Cost (LC) and Date Difference for each unique group
    For k = 2 To lastRow
        If Not wsRC.Rows(k).Hidden Then
            uniqueKey_NonDigital = UCase(Trim(CStr(wsRC.Cells(k, col_Category).Value))) & "|" & _
                                    UCase(Trim(CStr(wsRC.Cells(k, col_PlanName).Value))) & "|" & _
                                    UCase(Trim(CStr(wsRC.Cells(k, col_Publisher).Value))) & "|" & _
                                    UCase(Trim(CStr(wsRC.Cells(k, col_VehicleName).Value))) & "|" & _
                                    UCase(Trim(CStr(wsRC.Cells(k, col_MediaChannel).Value))) & "|" & _
                                    UCase(Trim(CStr(wsRC.Cells(k, col_InventoryDistributionType).Value)))

            If groupDict_NonDigital.Exists(uniqueKey_NonDigital) Then
                groupTotalSums = groupDict_NonDigital(uniqueKey_NonDigital)
                groupTotalSums(0) = groupTotalSums(0) + wsRC.Cells(k, col_MediaCostLC).Value ' Total Media Cost (LC)
                groupTotalSums(1) = groupTotalSums(1) + wsRC.Cells(k, col_Difference).Value  ' Total Date Difference
                groupDict_NonDigital(uniqueKey_NonDigital) = groupTotalSums
            Else
                ReDim groupTotalSums(0 To 1)
                groupTotalSums(0) = wsRC.Cells(k, col_MediaCostLC).Value  ' Total Media Cost (LC)
                groupTotalSums(1) = wsRC.Cells(k, col_Difference).Value  ' Total Date Difference
                groupDict_NonDigital.Add uniqueKey_NonDigital, groupTotalSums
            End If
        End If
    Next k
    ' 3b) Calculate the Average CPM, then multiple it to each row Date Difference to get Media Cost (LC)
    For k = 2 To lastRow
        If Not wsRC.Rows(k).Hidden Then
            uniqueKey_NonDigital = UCase(Trim(CStr(wsRC.Cells(k, col_Category).Value))) & "|" & _
                                    UCase(Trim(CStr(wsRC.Cells(k, col_PlanName).Value))) & "|" & _
                                    UCase(Trim(CStr(wsRC.Cells(k, col_Publisher).Value))) & "|" & _
                                    UCase(Trim(CStr(wsRC.Cells(k, col_VehicleName).Value))) & "|" & _
                                    UCase(Trim(CStr(wsRC.Cells(k, col_MediaChannel).Value))) & "|" & _
                                    UCase(Trim(CStr(wsRC.Cells(k, col_InventoryDistributionType).Value)))

            groupTotalSums = groupDict_NonDigital(uniqueKey_NonDigital)
            groupTotalCost = groupTotalSums(0)
            groupTotalDiff = groupTotalSums(1)
            row_Diff = wsRC.Cells(k, col_Difference).Value

            If IsNumeric(groupTotalCost) And IsNumeric(groupTotalDiff) And groupTotalDiff <> 0 Then
                groupAvgCPM = groupTotalCost / groupTotalDiff
                ' Writing the group Average CPM value for debugging
                'wsRC.Cells(k, col_CPMCPCPP).Value = groupAvgCPM
                ' Override Media Cost (LC) for the row
                row_MediaCostLC = groupAvgCPM * row_Diff
                wsRC.Cells(k, col_MediaCostLC).Value = row_MediaCostLC
            End If
        End If
    Next k

    ' Remove the filter
    wsRC.AutoFilterMode = False

    ' Additional calculation for CPM/CPC/CPP
    Call SubFunc_UnifyMetricCol(wsRC)
    ' Delete the Date Difference column
    wsRC.Columns(col_Difference).Delete
    wsRC.UsedRange.Font.Name = "Arial"
    wsRC.UsedRange.Font.Size = 8
    wsRC.Columns.AutoFit
    Call SubFunc_SetZoomLevel(ThisWorkbook)
    wsRC.Select
    totalRunTime = Round((Timer - BenchMark) / 60, 2)
    Application.ScreenUpdating = True
    MsgBox "Redistribute Cost completed successfully!" & vbCrLf & _
           "Total run time: " & totalRunTime & " minutes", vbInformation, "Complete"
End Sub
"""

In [ ]:
Module_Macro_13 = """

Sub Split_DeliverData_Templates_1a()
    BenchMark = Timer
    Dim wsDeliverData As Worksheet, wsDBTemp As Worksheet, wsDB As Worksheet, wsWeekly As Worksheet, wsMonthly As Worksheet
    Dim ptDeliverData As PivotTable, pcDeliverData As PivotCache
    Dim dataDeliverData As Range, rngDeliverData As Range, pivotStartCellDBTemp As Range
    Dim lastRowDeliverData As Long, lastColDeliverData As Long, i As Long, targetSheetArray As Variant
    Dim colPublisherVendor As Long, colVehicleName As Long, colProperty As Long
    Dim pfCategory As PivotField, pfMonth As PivotField, foundItem As Boolean, item As PivotItem, pi As PivotItem
    Application.ScreenUpdating = False

    If ThisWorkbook.Worksheets("DeliverData").Visible = xlSheetHidden Then ThisWorkbook.Worksheets("DeliverData").Visible = xlSheetVisible
    Set wsDeliverData = ThisWorkbook.Sheets("DeliverData")

    Call SubFunc_CreateNewSheet("Data Breakout Temp", wsDBTemp)
    Call SubFunc_CreateNewSheet("Data Breakout", wsDB)
    Call SubFunc_CreateNewSheet("Weekly", wsWeekly)
    Call SubFunc_CreateNewSheet("Monthly", wsMonthly)

    Set dataDeliverData = wsDeliverData.Range("A1").CurrentRegion
    lastRowDeliverData = dataDeliverData.Rows.Count
    lastColDeliverData = dataDeliverData.Columns.Count
    colPublisherVendor = Application.Match("Publisher/Vendor", wsDeliverData.Rows(1), 0)
    colVehicleName = Application.Match("Vehicle Name", wsDeliverData.Rows(1), 0)
    Call SubFunc_CreateNewColumnRight(colVehicleName, colProperty, "Property (Website or Partner)", wsDeliverData)
    For i = 2 To lastRowDeliverData
        PublisherVendor_Value = Trim(wsDeliverData.Cells(i, colPublisherVendor).Value)
        VehicleName_Value = Trim(wsDeliverData.Cells(i, colVehicleName).Value)
        If IsEmpty(VehicleName_Value) Or VehicleName_Value = "" Or VehicleName_Value = " " Or VehicleName_Value = "(blank)" Or VehicleName_Value = "#N/A" Then
            wsDeliverData.Cells(i, colProperty).Value = PublisherVendor_Value
        Else
            wsDeliverData.Cells(i, colProperty).Value = wsDeliverData.Cells(i, colPublisherVendor).Value & " (" & wsDeliverData.Cells(i, colVehicleName).Value & ")"
        End If
    Next i

    Set pivotStartCellDBTemp = wsDBTemp.Range("A3")
    Set pcDeliverData = ThisWorkbook.PivotCaches.Create(SourceType:=xlDatabase, SourceData:=dataDeliverData)
    Set ptDeliverData = pcDeliverData.CreatePivotTable(TableDestination:=pivotStartCellDBTemp, TableName:="DeliverDataDataBreakout")
    DoEvents
    On Error Resume Next
    With ptDeliverData
        With .PivotFields("Campaign")
             .Orientation = xlRowField
             .Subtotals = Array(False, False, False, False, False, False, False, False, False, False, False, False)
             .Caption = "Campaign"
        End With
        With .PivotFields("Plan Name")
             .Orientation = xlRowField
             .Subtotals = Array(False, False, False, False, False, False, False, False, False, False, False, False)
        End With
        With .PivotFields("UniqueClientName Campaign Lead")
             .Orientation = xlRowField
             .Subtotals = Array(False, False, False, False, False, False, False, False, False, False, False, False)
        End With
        With .PivotFields("Funding Source")
             .Orientation = xlRowField
             .Subtotals = Array(False, False, False, False, False, False, False, False, False, False, False, False)
        End With
        With .PivotFields("Fund")
             .Orientation = xlRowField
             .Subtotals = Array(False, False, False, False, False, False, False, False, False, False, False, False)
        End With
        With .PivotFields("Initiative")
             .Orientation = xlRowField
             .Subtotals = Array(False, False, False, False, False, False, False, False, False, False, False, False)
        End With
        With .PivotFields("Profitable Consumer Behaviour")
             .Orientation = xlRowField
             .Subtotals = Array(False, False, False, False, False, False, False, False, False, False, False, False)
        End With
        With .PivotFields("Product Message")
             .Orientation = xlRowField
             .Subtotals = Array(False, False, False, False, False, False, False, False, False, False, False, False)
        End With
        With .PivotFields("Destination")
             .Orientation = xlRowField
             .Subtotals = Array(False, False, False, False, False, False, False, False, False, False, False, False)
        End With
        With .PivotFields("Sponsorship")
             .Orientation = xlRowField
             .Subtotals = Array(False, False, False, False, False, False, False, False, False, False, False, False)
        End With
        With .PivotFields("Property (Website or Partner)")
             .Orientation = xlRowField
             .Subtotals = Array(False, False, False, False, False, False, False, False, False, False, False, False)
        End With
        With .PivotFields("Channel")
             .Orientation = xlRowField
             .Subtotals = Array(False, False, False, False, False, False, False, False, False, False, False, False)
        End With
        With .PivotFields("Media Channel")
             .Orientation = xlRowField
             .Subtotals = Array(False, False, False, False, False, False, False, False, False, False, False, False)
        End With
        With .PivotFields("Social?")
             .Orientation = xlRowField
             .Subtotals = Array(False, False, False, False, False, False, False, False, False, False, False, False)
        End With
        With .PivotFields("Inventory (Distribution) Type")
             .Orientation = xlRowField
             .Subtotals = Array(False, False, False, False, False, False, False, False, False, False, False, False)
        End With
        With .PivotFields("Promotion Incentive")
             .Orientation = xlRowField
             .Subtotals = Array(False, False, False, False, False, False, False, False, False, False, False, False)
        End With
        With .PivotFields("Media Market")
             .Orientation = xlRowField
             .Subtotals = Array(False, False, False, False, False, False, False, False, False, False, False, False)
        End With
        With .PivotFields("Does Media Cost include Production Fees?")
             .Orientation = xlRowField
             .Subtotals = Array(False, False, False, False, False, False, False, False, False, False, False, False)
        End With
        With .PivotFields("Currency")
             .Orientation = xlRowField
             .Subtotals = Array(False, False, False, False, False, False, False, False, False, False, False, False)
        End With
        With .PivotFields("Notes")
             .Orientation = xlRowField
             .Subtotals = Array(False, False, False, False, False, False, False, False, False, False, False, False)
        End With
        With .PivotFields("Media Cost (LC)")
             .Orientation = xlDataField
             .Function = xlSum
             .NumberFormat = "#,##0"
        End With
        With .PivotFields("Paid Impressions")
             .Orientation = xlDataField
             .Function = xlSum
             .NumberFormat = "#,##0"
        End With
        With .PivotFields("Organic Impressions")
             .Orientation = xlDataField
             .Function = xlSum
             .NumberFormat = "#,##0"
        End With
        With .PivotFields("Clicks")
             .Orientation = xlDataField
             .Function = xlSum
             .NumberFormat = "#,##0"
        End With
        With .PivotFields("TV National GRPs")
             .Orientation = xlDataField
             .Function = xlSum
             .NumberFormat = "#,##0.00"
        End With
        With .PivotFields("TV Duration (seconds)")
             .Orientation = xlDataField
             .Function = xlAverage
             .NumberFormat = "#,##0"
        End With
        With .PivotFields("CPM/CPC/CPP")
             .Orientation = xlDataField
             .Function = xlAverage
             .NumberFormat = "#,##0.00"
        End With
    .RowAxisLayout xlTabularRow
    .ColumnGrand = False
    End With
    Call SubFunc_PivotRepeatLabel(ptDeliverData, True) ' Turn Repeat All Item Labels on
    On Error GoTo 0

    Set rngDeliverData = ptDeliverData.TableRange1 ' Set the pivot table data range

    rngDeliverData.Copy ' Copy pivot table range and paste values + formatting into wsDB starting at A1
    With wsDB.Range("A1")
        .PasteSpecial Paste:=xlPasteValues
        .PasteSpecial Paste:=xlPasteFormats
    End With
    Application.CutCopyMode = False ' Clear the clipboard to remove selection outline marching ants
    'wsDB.Rows(1).Delete ' Delete the extra first row
    wsDB.UsedRange.Replace What:="(blank)", Replacement:=""

    ' Reorder certain columns
    colNote_DB = Application.Match("Notes", wsDB.Rows(1), 0)
    colCPM_DB = Application.Match("Average of CPM/CPC/CPP", wsDB.Rows(1), 0)

    wsDB.Columns(colNote_DB).Cut
    wsDB.Columns(colCPM_DB + 1).Insert Shift:=xlToRight
    Application.CutCopyMode = False

    Call SubFunc_CreateNewSheet("Data Breakout Temp", wsDBTemp) ' Refresh the sheet
    Set pivotStartCellDBTemp = wsDBTemp.Range("A3") ' Set up the pivot table again
    Set pcDeliverData = ThisWorkbook.PivotCaches.Create(SourceType:=xlDatabase, SourceData:=dataDeliverData)
    Set ptDeliverData = pcDeliverData.CreatePivotTable(TableDestination:=pivotStartCellDBTemp, TableName:="DeliverDataDataBreakout")
    DoEvents
    On Error Resume Next
    With ptDeliverData
        .PivotFields("Category").Orientation = xlPageField
        With .PivotFields("Week/Month Beginning")
             .Orientation = xlRowField
             .Subtotals = Array(False, False, False, False, False, False, False, False, False, False, False, False)
             .Caption = "Week Beginning" ' Rename for Weekly sheet
        End With
        With .PivotFields("Media Market")
             .Orientation = xlRowField
             .Subtotals = Array(False, False, False, False, False, False, False, False, False, False, False, False)
        End With
        With .PivotFields("Funding Source")
             .Orientation = xlRowField
             .Subtotals = Array(False, False, False, False, False, False, False, False, False, False, False, False)
        End With
        With .PivotFields("Sponsorship")
             .Orientation = xlRowField
             .Subtotals = Array(False, False, False, False, False, False, False, False, False, False, False, False)
        End With
        With .PivotFields("Destination")
             .Orientation = xlRowField
             .Subtotals = Array(False, False, False, False, False, False, False, False, False, False, False, False)
        End With
        With .PivotFields("Product Message")
             .Orientation = xlRowField
             .Subtotals = Array(False, False, False, False, False, False, False, False, False, False, False, False)
        End With
        With .PivotFields("Initiative")
             .Orientation = xlRowField
             .Subtotals = Array(False, False, False, False, False, False, False, False, False, False, False, False)
        End With
        With .PivotFields("Profitable Consumer Behaviour")
             .Orientation = xlRowField
             .Subtotals = Array(False, False, False, False, False, False, False, False, False, False, False, False)
        End With
        With .PivotFields("Campaign")
             .Orientation = xlRowField
             .Subtotals = Array(False, False, False, False, False, False, False, False, False, False, False, False)
        End With
        With .PivotFields("Promotion Incentive")
             .Orientation = xlRowField
             .Subtotals = Array(False, False, False, False, False, False, False, False, False, False, False, False)
        End With
        With .PivotFields("Media Channel")
             .Orientation = xlRowField
             .Subtotals = Array(False, False, False, False, False, False, False, False, False, False, False, False)
        End With
        With .PivotFields("Property (Website or Partner)")
             .Orientation = xlRowField
             .Subtotals = Array(False, False, False, False, False, False, False, False, False, False, False, False)
        End With
        With .PivotFields("Inventory (Distribution) Type")
             .Orientation = xlRowField
             .Subtotals = Array(False, False, False, False, False, False, False, False, False, False, False, False)
        End With
        With .PivotFields("Currency")
             .Orientation = xlRowField
             .Subtotals = Array(False, False, False, False, False, False, False, False, False, False, False, False)
        End With
        With .PivotFields("Notes")
             .Orientation = xlRowField
             .Subtotals = Array(False, False, False, False, False, False, False, False, False, False, False, False)
        End With
        With .PivotFields("Media Cost (LC)")
             .Orientation = xlDataField
             .Function = xlSum
             .NumberFormat = "#,##0"
        End With
        With .PivotFields("Paid Impressions")
             .Orientation = xlDataField
             .Function = xlSum
             .NumberFormat = "#,##0"
        End With
        'With .PivotFields("Organic Impressions")
             '.Orientation = xlDataField
             '.Function = xlSum
             '.NumberFormat = "#,##0"
        'End With
        'With .PivotFields("Viral Impressions")
             '.Orientation = xlDataField
             '.Function = xlSum
             '.NumberFormat = "#,##0"
        'End With
        With .PivotFields("TV National GRPs")
             .Orientation = xlDataField
             .Function = xlSum
             .NumberFormat = "#,##0.00"
        End With
        With .PivotFields("Clicks")
             .Orientation = xlDataField
             .Function = xlSum
             .NumberFormat = "#,##0"
        End With
    .RowAxisLayout xlTabularRow
    .ColumnGrand = False ' Remove Grand Total
    End With
    On Error GoTo 0
    Call SubFunc_PivotRepeatLabel(ptDeliverData, True) ' Turn Repeat All Item Labels on
    Set rngDeliverData = ptDeliverData.TableRange1
    Set pfCategory = ptDeliverData.PivotFields("Category") ' Reference the PivotFields
    Set pfMonth = ptDeliverData.PivotFields("Week Beginning")
    foundItem = False ' Check if "Offline-Monthly" exists
    For Each item In pfCategory.PivotItems
        If item.Name = "Offline-Monthly" Then
            foundItem = True
            Exit For
        End If
    Next item

    If foundItem Then ' If found, copy sheets normally
        ' Exclude "Offline-Monthly" from Category Filter
        With pfCategory
            .ClearAllFilters
            For Each pi In .PivotItems
                If pi.Value = "Offline-Monthly" Then pi.Visible = False
            Next pi
        End With
        rngDeliverData.Copy ' Copy data into the Weekly sheet
        With wsWeekly.Range("A1")
            .PasteSpecial Paste:=xlPasteValues
        End With
        Application.CutCopyMode = False
        'wsWeekly.Rows(1).Delete
        wsWeekly.UsedRange.Replace What:="(blank)", Replacement:=""

        ' Reorder certain columns
        colNotes_Weekly = Application.Match("Notes", wsWeekly.Rows(1), 0)
        colClicks_Weekly = Application.Match("Sum of Clicks", wsWeekly.Rows(1), 0)

        wsWeekly.Columns(colNotes_Weekly).Cut
        wsWeekly.Columns(colClicks_Weekly + 1).Insert Shift:=xlToRight
        Application.CutCopyMode = False

        ' Keep only "Offline-Monthly" from Category Filter
        With pfCategory
            .ClearAllFilters
            .CurrentPage = "Offline-Monthly"
        End With
        pfMonth.Caption = "Month" ' Rename for Monthly sheet
        rngDeliverData.Copy ' Copy data into the Monthly sheet
        With wsMonthly.Range("A1")
            .PasteSpecial Paste:=xlPasteValues
        End With
        Application.CutCopyMode = False
        'wsMonthly.Rows(1).Delete
        wsMonthly.UsedRange.Replace What:="(blank)", Replacement:=""

        ' Reorder certain columns
        colNotes_Monthly = Application.Match("Notes", wsMonthly.Rows(1), 0)
        colClicks_Monthly = Application.Match("Sum of Clicks", wsMonthly.Rows(1), 0)

        wsMonthly.Columns(colNotes_Monthly).Cut
        wsMonthly.Columns(colClicks_Monthly + 1).Insert Shift:=xlToRight
        Application.CutCopyMode = False
    Else
        pfCategory.ClearAllFilters
        rngDeliverData.Copy ' Copy data into the Weekly sheet
        With wsWeekly.Range("A1")
            .PasteSpecial Paste:=xlPasteValues
        End With
        Application.CutCopyMode = False
        'wsWeekly.Rows(1).Delete
        wsWeekly.UsedRange.Replace What:="(blank)", Replacement:=""
        ' Reorder certain columns
        colNotes_Weekly = Application.Match("Notes", wsWeekly.Rows(1), 0)
        colClicks_Weekly = Application.Match("Sum of Clicks", wsWeekly.Rows(1), 0)

        wsWeekly.Columns(colNotes_Weekly).Cut
        wsWeekly.Columns(colClicks_Weekly + 1).Insert Shift:=xlToRight
        Application.CutCopyMode = False
        Debug.Print "'Offline-Monthly' not found in Category field"
    End If

    ' If Data Breakout Temp worksheet exists, delete it safely
    If Not wsDBTemp Is Nothing Then
        Application.DisplayAlerts = False ' Disable delete confirmation prompt
        wsDBTemp.Delete ' Delete the worksheet
        Application.DisplayAlerts = True ' Re-enable alerts
    End If

    ' Apply formatting for 3 sheets
    Call SubFunc_CustomizeWholeSheet(Array("Data Breakout", "Weekly", "Monthly"))
    ' Rename Original headers to DeliverData Submission headers
    Call SubFunc_RenameColHeaders(wsDB)
    Call SubFunc_RenameColHeaders(wsWeekly)
    Call SubFunc_RenameColHeaders(wsMonthly)
    Call SubFunc_UnifyMetricCol(wsDB)
    Call SubFunc_UnifyMetricCol(wsWeekly)
    Call SubFunc_UnifyMetricCol(wsMonthly)
    wsDB.Columns.AutoFit
    
    ' Set tab color
    wsDB.Tab.Color = TAB_HIG
    wsWeekly.Tab.Color = TAB_HIG
    wsMonthly.Tab.Color = TAB_HIG
    'wsDB.Tab.ColorIndex = xlColorIndexNone ' To reset the tab back to no color, default grey

    ' Delete "Property (Website or Partner)" column in DeliverData sheet
    wsDeliverData.Columns(colProperty).Delete

    ' Hide all the unnecessary sheets
    targetSheetArray = Array("Data Breakout", "Weekly", "Monthly", "JCR vs DeliverData", "Brand Tracker vs DeliverData")
    Call SubFunc_SheetVisible(targetSheetArray)

    Call SubFunc_SetZoomLevel(ThisWorkbook)
    totalRunTime = Round((Timer - BenchMark) / 60, 2)
    Application.ScreenUpdating = True
    MsgBox "Split DeliverData Templates 1a completed successfully!" & vbCrLf & _
           "Total run time: " & totalRunTime & " minutes", vbInformation, "Complete"
    wsDB.Select
End Sub

Sub Split_DeliverData_Templates_1b()
    Dim ws As Worksheet, wbDB As Workbook
    Dim WorksheetExists As Boolean, NewFileName As String
    Dim SavePath As Variant
    WorksheetExists = False
    ' Check if "Data Breakout" worksheet exists
    For Each ws In ThisWorkbook.Worksheets
        If LCase(ws.Name) = LCase("JCR vs DeliverData Spend") Then
            ws.Tab.Color = TAB_HIG
        End If
    Next ws
    For Each ws In ThisWorkbook.Worksheets
        If LCase(ws.Name) = LCase("Data Breakout") Then
            WorksheetExists = True
            ws.Tab.Color = TAB_HIG
            Exit For
        End If
    Next ws
    ' If worksheet not found, exit
    If Not WorksheetExists Then
        MsgBox "Data Breakout not found!", vbExclamation, "Data Validation Error"
        Exit Sub
    End If
    ' Prompt user to select save location
    SavePath = Application.GetSaveAsFilename( _
        InitialFileName:=Replace(ThisWorkbook.Name, ".xlsm", " (Data Breakout).xlsm"), _
        FileFilter:="Excel Macro-Enabled Workbook (*.xlsm), *.xlsm", _
        Title:="Save Data Breakout File As")
    ' Check if user cancelled
    If SavePath = False Then
        MsgBox "Operation cancelled!", vbInformation, "Cancelled"
        Exit Sub
    End If
    NewFileName = SavePath
    ' Save a copy with new name
    ThisWorkbook.SaveCopyAs Filename:=NewFileName
    ' Wait 2 seconds
    Application.Wait Now + TimeValue("00:00:05")
    ' Open the new file
    Set wbDB = Workbooks.Open(NewFileName)
    ' Turn off alerts to prevent deletion prompts
    Application.DisplayAlerts = False
    ' Delete all sheets except "Data Breakout"
    For Each ws In wbDB.Worksheets
        If LCase(ws.Name) <> LCase("Data Breakout") Then
            If LCase(ws.Name) <> LCase("JCR vs DeliverData Spend") Then
                ws.Delete
            End If
        End If
    Next ws
    ' Turn alerts back on
    Application.DisplayAlerts = True
    ' Save and close
    wbDB.Save
    MsgBox "Data Breakout file creation completed successfully!", vbInformation, "Complete"
End Sub
"""

In [ ]:
Module_Macro_14 = """
Sub Split_DeliverData_Templates_2a()
    BenchMark = Timer
    Dim wsDeliverData As Worksheet, ws As Worksheet, wsMPR As Worksheet, wsHS As Worksheet, wsCO As Worksheet
    Dim dataDeliverData As Range, headerRow As Range
    Dim lastRowDeliverData As Long, lastRowMPR As Long, lastRowHS As Long, i As Long
    Dim colChannel As Long, colMediaChannel As Long, colMediaMarket As Long
    Dim colCampaign As Long, colFundingSource As Long, colProfitableConsumerBehaviour As Long
    Dim targetSheetArray As Variant, Template2Array As Variant, sheetName As Variant
    Application.ScreenUpdating = False

    Template2Array = Array("Media Penetration Rate", "Household Size")
    If ThisWorkbook.Worksheets("DeliverData").Visible = xlSheetHidden Then ThisWorkbook.Worksheets("DeliverData").Visible = xlSheetVisible
    Set wsDeliverData = ThisWorkbook.Sheets("DeliverData")
    Set dataDeliverData = wsDeliverData.Range("A1").CurrentRegion
    lastRowDeliverData = dataDeliverData.Rows.Count
    Set headerRow = wsDeliverData.Rows(1)
    colChannel = Application.Match("Channel", headerRow, 0)
    colMediaChannel = Application.Match("Media Channel", headerRow, 0)
    colMediaMarket = Application.Match("Media Market", headerRow, 0)

    ' Loop through templates
    For Each sheetName In Template2Array
        On Error Resume Next ' Try to get existing sheet
        Set ws = ThisWorkbook.Worksheets(sheetName)
        On Error GoTo 0
        If ws Is Nothing Then
            ' Sheet does not exist, create and populate data
            Set ws = ThisWorkbook.Worksheets.Add(After:=ThisWorkbook.Worksheets(ThisWorkbook.Worksheets.Count))
            ws.Name = sheetName
            Debug.Print "Created sheet: " & sheetName
            ' Generate content based on sheet name
            Select Case sheetName
                Case "Media Penetration Rate"
                    Set wsMPR = ws
                    wsMPR.Range("A:B").Value = wsDeliverData.Range(wsDeliverData.Cells(1, colChannel), wsDeliverData.Cells(lastRowDeliverData, colMediaChannel)).Value
                    ' Remove duplicates
                    wsMPR.Range("A:B").RemoveDuplicates Columns:=Array(1, 2), Header:=xlYes
                    wsMPR.Range("C1").Value = "FY23"
                    wsMPR.Range("D1").Value = "FY24"
                    wsMPR.Range("E1").Value = "FY25"
                    wsMPR.Range("F1").Value = "FY26"
                    ' Get the last row again
                    lastRowMPR = wsMPR.Range("A1").CurrentRegion.Rows.Count
                    ' Sort A-Z
                    With wsMPR.Sort
                        .SortFields.Clear
                        .SortFields.Add key:=wsMPR.Range("A1"), Order:=xlAscending
                        .SortFields.Add key:=wsMPR.Range("B1"), Order:=xlAscending
                        .SetRange wsMPR.Range("A1:B" & lastRowMPR)
                        .Header = xlYes
                        .Apply
                    End With
                    wsMPR.Range("A" & (lastRowMPR + 2)).Value = "Based on Target Audience Group"
                    wsMPR.Range("A" & (lastRowMPR + 3)).Value = "Source"
                    wsMPR.Range("A" & (lastRowMPR + 4)).Value = "Notes"
                    ' Clean the #N/A values
                    wsMPR.UsedRange.Replace What:="#N/A", Replacement:=""
                    Call SubFunc_RenameColHeaders(wsMPR)
                    Call SubFunc_CustomizeWholeSheet(Array("Media Penetration Rate"))
                    wsMPR.Tab.Color = TAB_HIG
                Case "Household Size"
                    Set wsHS = ws
                    wsHS.Range("A:A").Value = wsDeliverData.Range(wsDeliverData.Cells(1, colMediaMarket), wsDeliverData.Cells(lastRowDeliverData, colMediaMarket)).Value
                    ' Remove duplicates
                    wsHS.Range("A:A").RemoveDuplicates Columns:=1, Header:=xlYes
                    wsHS.Range("B1").Value = "FY23"
                    wsHS.Range("C1").Value = "FY24"
                    wsHS.Range("D1").Value = "FY25"
                    wsHS.Range("E1").Value = "FY26"
                    ' Get the last row again
                    lastRowHS = wsHS.Range("A1").CurrentRegion.Rows.Count
                    ' Sort A-Z
                    With wsHS.Sort
                        .SortFields.Clear
                        .SortFields.Add key:=wsHS.Range("A1"), Order:=xlAscending
                        .SetRange wsHS.Range("A1:A" & lastRowHS)
                        .Header = xlYes
                        .Apply
                    End With
                    wsHS.Range("A" & (lastRowHS + 2)).Value = "# of People per Household"
                    wsHS.Range("A" & (lastRowHS + 3)).Value = "Based on Target Audience Group"
                    wsHS.Range("A" & (lastRowHS + 4)).Value = "Source"
                    wsHS.Range("A" & (lastRowHS + 5)).Value = "Notes"
                    ' Clean the #N/A values
                    wsHS.UsedRange.Replace What:="#N/A", Replacement:=""
                    wsHS.Range("A1").Value = "Media Market"
                    Call SubFunc_CustomizeWholeSheet(Array("Household Size"))
                    wsHS.Tab.Color = TAB_HIG
            End Select
        Else
            ' Sheet already exists, skip generation
            Debug.Print "Skipped existing sheet: " & sheetName
        End If
        Set ws = Nothing ' Reset reference before next iteration
    Next sheetName

    ' Hide all the unnecessary sheets
    targetSheetArray = Array("Media Penetration Rate", "Household Size", "Creatives Objectives")
    'Call SubFunc_SheetVisible(targetSheetArray)
    
    Call SubFunc_SetZoomLevel(ThisWorkbook)
    totalRunTime = Round((Timer - BenchMark) / 60, 2)
    Application.ScreenUpdating = True
    MsgBox "Split DeliverData Templates 2a completed successfully!" & vbCrLf & _
           "Total run time: " & totalRunTime & " minutes", vbInformation, "Complete"
End Sub

Sub Split_DeliverData_Templates_2b()
    BenchMark = Timer
    Dim wsDeliverData As Worksheet, ws As Worksheet, wsMPR As Worksheet, wsHS As Worksheet, wsCO As Worksheet
    Dim dataDeliverData As Range, headerRow As Range
    Dim lastRowDeliverData As Long, lastRowMPR As Long, lastRowHS As Long, i As Long
    Dim colChannel As Long, colMediaChannel As Long, colMediaMarket As Long
    Dim colCampaign As Long, colFundingSource As Long, colProfitableConsumerBehaviour As Long
    Dim targetSheetArray As Variant, Template2Array As Variant, sheetName As Variant
    Application.ScreenUpdating = False

    Template2Array = Array("Creatives Objectives")
    If ThisWorkbook.Worksheets("DeliverData").Visible = xlSheetHidden Then ThisWorkbook.Worksheets("DeliverData").Visible = xlSheetVisible
    Set wsDeliverData = ThisWorkbook.Sheets("DeliverData")
    Set dataDeliverData = wsDeliverData.Range("A1").CurrentRegion
    lastRowDeliverData = dataDeliverData.Rows.Count
    Set headerRow = wsDeliverData.Rows(1)
    colCampaign = Application.Match("Campaign", headerRow, 0)
    colPlanName = Application.Match("Plan Name", headerRow, 0)
    colFundingSource = Application.Match("Funding Source", headerRow, 0)
    colProfitableConsumerBehaviour = Application.Match("Profitable Consumer Behaviour", headerRow, 0)

    ' Loop through templates
    For Each sheetName In Template2Array
        On Error Resume Next ' Try to get existing sheet
        Set ws = ThisWorkbook.Worksheets(sheetName)
        On Error GoTo 0
        If ws Is Nothing Then
            ' Sheet does not exist, create and populate data
            Set ws = ThisWorkbook.Worksheets.Add(After:=ThisWorkbook.Worksheets(ThisWorkbook.Worksheets.Count))
            ws.Name = sheetName
            Debug.Print "Created sheet: " & sheetName
            ' Generate content based on sheet name
            Select Case sheetName
                Case "Creatives Objectives"
                    Set wsCO = ws
                    wsCO.Range("A:A").Value = wsDeliverData.Range(wsDeliverData.Cells(1, colCampaign), wsDeliverData.Cells(lastRowDeliverData, colCampaign)).Value
                    wsCO.Range("B:B").Value = wsDeliverData.Range(wsDeliverData.Cells(1, colPlanName), wsDeliverData.Cells(lastRowDeliverData, colPlanName)).Value
                    wsCO.Range("C:C").Value = wsDeliverData.Range(wsDeliverData.Cells(1, colFundingSource), wsDeliverData.Cells(lastRowDeliverData, colFundingSource)).Value
                    wsCO.Range("D:D").Value = wsDeliverData.Range(wsDeliverData.Cells(1, colProfitableConsumerBehaviour), wsDeliverData.Cells(lastRowDeliverData, colProfitableConsumerBehaviour)).Value
                     ' Remove duplicates
                    wsCO.Range("A:D").RemoveDuplicates Columns:=Array(1, 2, 3, 4), Header:=xlYes
                    wsCO.Range("E1").Value = "Marketing KPI"
                    wsCO.Range("F1").Value = "Objective/Campaign Description"
                    wsCO.Range("G1").Value = "Target Audience"
                    wsCO.Range("H1").Value = "Creative"
                    ' Get the last row again
                    lastRowCO = wsCO.Range("A1").CurrentRegion.Rows.Count
                     ' Sort A-Z
                     With wsCO.Sort
                        .SortFields.Clear
                        .SortFields.Add key:=wsCO.Range("A1"), Order:=xlAscending
                        .SortFields.Add key:=wsCO.Range("B1"), Order:=xlAscending
                        .SortFields.Add key:=wsCO.Range("C1"), Order:=xlAscending
                        .SortFields.Add key:=wsCO.Range("D1"), Order:=xlAscending
                        .SetRange wsCO.Range("A1:D" & lastRowCO)
                        .Header = xlYes
                        .Apply
                    End With
                    ' Clean the #N/A values
                    wsCO.UsedRange.Replace What:="#N/A", Replacement:=""
                    Call SubFunc_RenameColHeaders(wsCO)
                    Call SubFunc_CustomizeWholeSheet(Array("Creatives Objectives"))
                    lastRowCO = wsCO.Range("A1").CurrentRegion.Rows.Count
                    ' Set Row Height for rows 2 to lastRowCO
                    wsCO.Rows("2:" & lastRowCO).RowHeight = 130
                    ' Set Column Width for columns A to G
                    wsCO.Range("A1:G" & lastRowCO).EntireColumn.ColumnWidth = 30
                    ' Set Column Width for column H
                    wsCO.Range("H1:H" & lastRowCO).EntireColumn.ColumnWidth = 160
                    wsCO.Range("A2:G" & lastRowCO).WrapText = True
                    wsCO.Tab.Color = TAB_HIG
            End Select
            ' Prompt user to select save location
            SavePath = Application.GetSaveAsFilename( _
                InitialFileName:=Replace(ThisWorkbook.Name, ".xlsm", " (Paid Media Creatives Objectives).xlsm"), _
                FileFilter:="Excel Macro-Enabled Workbook (*.xlsm), *.xlsm", _
                Title:="Save Creatives Objectives File As")
            ' Check if user cancelled
            If SavePath = False Then
                MsgBox "Saving cancelled!", vbInformation, "Cancelled"
                Exit Sub
            End If
            NewFileName = SavePath
            ' Save a copy with new name
            ThisWorkbook.SaveCopyAs Filename:=NewFileName
            ' Wait 5 seconds
            Application.Wait Now + TimeValue("00:00:05")
            ' Open the new file
            Set wbCO = Workbooks.Open(NewFileName)
            ' Turn off alerts to prevent deletion prompts
            Application.DisplayAlerts = False
            ' Delete all sheets except
            For Each ws In wbCO.Worksheets
                If LCase(ws.Name) <> LCase("Creatives Objectives") Then
                    ws.Delete
                End If
            Next ws
            ' Turn alerts back on
            Application.DisplayAlerts = True
            ' Save and close
            wbCO.Save
        Else
            ' Sheet already exists, skip generation
            Debug.Print "Skipped existing sheet: " & sheetName
            ws.Tab.Color = TAB_HIG
            ' Prompt user to select save location
            SavePath = Application.GetSaveAsFilename( _
                InitialFileName:=Replace(ThisWorkbook.Name, ".xlsm", " (Paid Media Creatives Objectives).xlsm"), _
                FileFilter:="Excel Macro-Enabled Workbook (*.xlsm), *.xlsm", _
                Title:="Save Creatives Objectives File As")
            ' Check if user cancelled
            If SavePath = False Then
                MsgBox "Saving cancelled!", vbInformation, "Cancelled"
                Exit Sub
            End If
            NewFileName = SavePath
            ' Save a copy with new name
            ThisWorkbook.SaveCopyAs Filename:=NewFileName
            ' Wait 5 seconds
            Application.Wait Now + TimeValue("00:00:05")
            ' Open the new file
            Set wbCO = Workbooks.Open(NewFileName)
            ' Turn off alerts to prevent deletion prompts
            Application.DisplayAlerts = False
            ' Delete all sheets except
            For Each ws In wbCO.Worksheets
                If LCase(ws.Name) <> LCase("Creatives Objectives") Then
                    ws.Delete
                End If
            Next ws
            ' Turn alerts back on
            Application.DisplayAlerts = True
            ' Save and close
            wbCO.Save
        End If
        Set ws = Nothing ' Reset reference before next iteration
    Next sheetName

    ' Hide all the unnecessary sheets
    targetSheetArray = Array("Media Penetration Rate", "Household Size", "Creatives Objectives")
    'Call SubFunc_SheetVisible(targetSheetArray)

    Call SubFunc_SetZoomLevel(ThisWorkbook)
    totalRunTime = Round((Timer - BenchMark) / 60, 2)
    Application.ScreenUpdating = True
    MsgBox "Split DeliverData Templates 2b completed successfully!" & vbCrLf & _
           "Total run time: " & totalRunTime & " minutes", vbInformation, "Complete"
End Sub
"""

In [ ]:
Module_Macro_15 = """
Sub Validation_Results()
    Dim wsMT As Worksheet, wsRMD As Worksheet, wsNew As Worksheet, wsVR As Worksheet
    Dim ptMT As PivotTable, ptRMD_Current As PivotTable, ptRMD_Overlapping As PivotTable
    Dim pcMT As PivotCache, pcRMD_Current As PivotCache, pcRMD_Overlapping As PivotCache
    Dim dataMT As Range, dataRMD_Current As Range, dataRMD_Overlapping As Range
    Dim lastRowMT As Long, lastColMT As Long, lastRowVR As Long
    Dim lastRowRMD_Current As Long, lastColRMD_Current As Long
    Dim lastRowRMD_Overlapping As Long, lastColRMD_Overlapping As Long
    Dim pivotStartCellMT As Range, pivotStartCellRMD_Current As Range, pivotStartCellRMD_Overlapping As Range
    Dim rngMT As Range, rngRMD_Current As Range, rngRMD_Overlapping As Range
    Dim fixed_quarter As String, fixed_country As String
    BenchMark = Timer
    Application.ScreenUpdating = False

    ' Set up UniqueCloud2 input data and adjust some format
    Set wsRMD = ThisWorkbook.Sheets("Regional DeliverData Dashboards")
    wsRMD.AutoFilterMode = False
    wsRMD.Range("A1").Value = "Data in current quarter"
    wsRMD.Range("F1").Value = "Data in period overlapping with previous quarter"
    With wsRMD.Range("A1:I2").Interior
        .ThemeColor = xlThemeColorAccent1
        .TintAndShade = 0.8  ' 80%
    End With
    wsRMD.Range("E1:E2").Interior.ColorIndex = xlNone
    wsRMD.Range("A1:D1").Merge
    wsRMD.Range("F1:I1").Merge

    ' Set up UniqueCloud1 sheet
    Set wsMT = ThisWorkbook.Sheets("UniqueCloud1")
    wsMT.AutoFilterMode = False

    ' Set up the Pivot table sheet
    Call SubFunc_CreateNewSheet("Validation Results Pivot", wsNew)
    wsNew.Range("A1").Value = "Regional DeliverData Dashboard"
    wsNew.Range("A2").Value = "Data in current quarter"
    wsNew.Range("F1").Value = "Regional DeliverData Dashboard"
    wsNew.Range("F2").Value = "Data in period overlapping with previous quarter"
    wsNew.Range("K1").Value = "UniqueCloud1"

    Set headerRowwsRMD = wsRMD.Rows(2)
    ' Define data range for RMD Current
    lastRowRMD_Current = wsRMD.Cells(Rows.Count, 1).End(xlUp).Row
    lastColRMD_Current = 4 ' Column D
    ' Starting from row 2
    Set dataRMD_Current = wsRMD.Range(wsRMD.Cells(2, 1), wsRMD.Cells(lastRowRMD_Current, lastColRMD_Current))

    ' Define data range for RMD Overlapping
    lastRowRMD_Overlapping = wsRMD.Cells(Rows.Count, 6).End(xlUp).Row
    lastColRMD_Overlapping = 9 ' Column I
    ' Starting from row 2
    Set dataRMD_Overlapping = wsRMD.Range(wsRMD.Cells(2, 6), wsRMD.Cells(lastRowRMD_Overlapping, lastColRMD_Overlapping))

    ' Define data range for UniqueCloud1
    lastRowMT = wsMT.Cells(Rows.Count, 1).End(xlUp).Row
    lastColMT = wsMT.Cells(1, Columns.Count).End(xlToLeft).Column
    Set headerRowMT = wsMT.Rows(1)
    Set dataMT = wsMT.Range(wsMT.Cells(1, 1), wsMT.Cells(lastRowMT, lastColMT))
    ' Safe guard for South Korea value
    'wsMT.Range(wsMT.Cells(1, 1), wsMT.Cells(lastRowMT, 1)).Replace What:="South Korea", Replacement:="Korea", LookAt:=xlPart, MatchCase:=False

    ' Define PivotTable start positions
    Set pivotStartCellRMD_Current = wsNew.Range("A4")
    Set pivotStartCellRMD_Overlapping = wsNew.Range("F4")
    Set pivotStartCellMT = wsNew.Range("K4")

    ' Create PivotTable for RMD Current
    Set pcRMD_Current = ThisWorkbook.PivotCaches.Create(SourceType:=xlDatabase, SourceData:=dataRMD_Current)
    Set ptRMD_Current = pcRMD_Current.CreatePivotTable(TableDestination:=pivotStartCellRMD_Current, TableName:="PivotRMD_Current")
    DoEvents
    With ptRMD_Current
        With .PivotFields("Country")
             .Orientation = xlRowField
             .Subtotals = Array(False, False, False, False, False, False, False, False, False, False, False, False)
        End With
        With .PivotFields("Data Stream")
             .Orientation = xlRowField
             .Subtotals = Array(False, False, False, False, False, False, False, False, False, False, False, False)
        End With
        With .PivotFields("Plan ID")
             .Orientation = xlRowField
             .Subtotals = Array(False, False, False, False, False, False, False, False, False, False, False, False)
             .Caption = "Plan ID"
        End With
        With .PivotFields("Media Cost (USD)")
             .Orientation = xlDataField
             .Function = xlSum
             .NumberFormat = "#,##0"
        End With
    .RowAxisLayout xlTabularRow
    .TableStyle2 = "PivotStyleLight15"
    End With
    Call SubFunc_PivotRepeatLabel(ptRMD_Current, True)

    ' Create PivotTable for RMD Overlapping
    Set pcRMD_Overlapping = ThisWorkbook.PivotCaches.Create(SourceType:=xlDatabase, SourceData:=dataRMD_Overlapping)
    Set ptRMD_Overlapping = pcRMD_Overlapping.CreatePivotTable(TableDestination:=pivotStartCellRMD_Overlapping, TableName:="PivotRMD_Overlapping")
    DoEvents
    With ptRMD_Overlapping
        With .PivotFields("Country")
             .Orientation = xlRowField
             .Subtotals = Array(False, False, False, False, False, False, False, False, False, False, False, False)
        End With
        With .PivotFields("Data Stream")
             .Orientation = xlRowField
             .Subtotals = Array(False, False, False, False, False, False, False, False, False, False, False, False)
        End With
        With .PivotFields("Plan ID")
             .Orientation = xlRowField
             .Subtotals = Array(False, False, False, False, False, False, False, False, False, False, False, False)
             .Caption = "Plan ID"
        End With
        With .PivotFields("Media Cost (USD)")
             .Orientation = xlDataField
             .Function = xlSum
             .NumberFormat = "#,##0"
        End With
    .RowAxisLayout xlTabularRow
    .TableStyle2 = "PivotStyleLight15"
    End With
    Call SubFunc_PivotRepeatLabel(ptRMD_Overlapping, True)

    ' Create PivotTable for UniqueCloud1
    Set pcMT = ThisWorkbook.PivotCaches.Create(SourceType:=xlDatabase, SourceData:=dataMT)
    Set ptMT = pcMT.CreatePivotTable(TableDestination:=pivotStartCellMT, TableName:="PivotUniqueCloud1")
    Call SubFunc_GetFixedQuarter(fixed_quarter)
    Call SubFunc_GetFixedCountry(fixed_country)
    
    DoEvents
    With ptMT
        With .PivotFields("Quarter")
            .Orientation = xlPageField
            On Error Resume Next
            .CurrentPage = fixed_quarter
            If Err.Number <> 0 Then
                Err.Clear
                On Error GoTo 0
                wsMT.Select
                MsgBox fixed_quarter & " not found in Quarter column, UniqueCloud1 sheet", vbExclamation, "Date Validation Error"
                Exit Sub
            End If
            On Error GoTo 0
        End With
        ' Set MT filters for the current country from file name
        With .PivotFields("Country")
            .Orientation = xlPageField
            .CurrentPage = fixed_country
        End With
        With .PivotFields("Country")
             .Orientation = xlRowField
             .Subtotals = Array(False, False, False, False, False, False, False, False, False, False, False, False)
        End With
        With .PivotFields("Offline vs. Online")
             .Orientation = xlRowField
             .Subtotals = Array(False, False, False, False, False, False, False, False, False, False, False, False)
        End With
        With .PivotFields("Plan ID")
             .Orientation = xlRowField
             .Subtotals = Array(False, False, False, False, False, False, False, False, False, False, False, False)
             .Caption = "Plan ID"
        End With
        With .PivotFields("Media Cost (USD)")
             .Orientation = xlDataField
             .Function = xlSum
             .NumberFormat = "#,##0"
        End With
    .RowAxisLayout xlTabularRow
    .TableStyle2 = "PivotStyleLight20"
    End With
    Call SubFunc_PivotRepeatLabel(ptMT, True) ' On/Show

    ' Set the pivot table ranges
    Set rngRMD_Current = ptRMD_Current.TableRange1
    Set rngRMD_Overlapping = ptRMD_Overlapping.TableRange1
    Set rngMT = ptMT.TableRange1

    ' Build the final result sheet
    Call SubFunc_CreateNewSheet("Validation Results", wsVR)
    Call SubFunc_OuterJoinLogic(rngRMD_Current, rngRMD_Overlapping, rngMT, wsVR)
    
    ' Build the nice format for the final sheet
    lastRowVR = wsVR.Cells(Rows.Count, 1).End(xlUp).Row
    wsVR.Range("A1").Value = "Validation Results"
    wsVR.Range("D3:J3").Value = Array("Regional DeliverData Dashboard", "Regional DeliverData Dashboard", "Regional DeliverData Dashboard", "UniqueCloud1", "Regional DeliverData Dashboard vs UniqueCloud1", "", "Regional DeliverData Dashboard Links")
    wsVR.Range("D4:F4").Value = Array("Data in current quarter", "Data in period overlapping with previous quarter", "Data in current quarter excluding Data in period overlapping with previous quarter")
    wsVR.Range("D4:F4").WrapText = True
    wsVR.Range("A5:H5").Value = Array("Country", "Offline / Online", "Plan ID", "Media Cost (USD)", "Media Cost (USD)", "Media Cost (USD)", "Media Cost (USD)", "Variance (%)")
    wsVR.Range("J6:J15").Value = Application.Transpose(Array("AP Online", "AP Offline", "LAC Online", "LAC Offline", "EU Online", "EU Offline", "CEMEA Online", "CEMEA Offline", "NA Online", "NA Offline"))
    wsVR.Range("A1:K5").Font.Bold = True
    wsVR.Range("G3:G4").Merge
    wsVR.Range("H3:H4").Merge
    wsVR.Range("J3:K5").Merge

    ' Format the final table
    With wsVR
        With .Cells
        .Font.Name = "Arial"
        .Font.Size = 8
        .HorizontalAlignment = xlCenter
        .VerticalAlignment = xlCenter
        End With
        ' Format for Plan ID
        With .Range(.Cells(6, 3), .Cells(lastRowVR, 3))
            .Value = .Value
            .NumberFormat = "@"
        End With
        ' Format for Media Cost (USD)
        With .Range(.Cells(6, 4), .Cells(lastRowVR, 7))
            .Value = .Value
            .NumberFormat = "#,##0"
        End With
        ' Color fill for grey headers
        With .Range(.Cells(3, 4), .Cells(5, 5)).Interior
            .ThemeColor = xlThemeColorDark1
            .TintAndShade = -0.25 ' 25%
        End With
        ' Color fill for blue headers
        With .Range(.Cells(3, 6), .Cells(5, 6)).Interior
            .ThemeColor = xlThemeColorAccent1
            .TintAndShade = 0.8  ' 80%
        End With
        ' Color fill for pink headers
        With .Range(.Cells(3, 7), .Cells(5, 7)).Interior
            .ThemeColor = xlThemeColorAccent5
            .TintAndShade = 0.8  ' 80%
        End With
        ' Color fill for green headers
        With .Range(.Cells(3, 8), .Cells(5, 8)).Interior
            .ThemeColor = xlThemeColorAccent3
            .TintAndShade = 0.8  ' 80%
        End With
        ' Color fill for blue headers
        With .Range(.Cells(3, 10), .Cells(5, 11)).Interior
            .ThemeColor = xlThemeColorAccent1
            .TintAndShade = 0.8  ' 80%
        End With
        ' Tab color
        With .Tab
            .Color = TAB_HIG
        End With
        ' Set row 4 height a bit taller
        .Rows("4:4").RowHeight = 35
        .Columns.AutoFit
        ' Apply border on cell with value
        .UsedRange.SpecialCells(xlCellTypeConstants).Borders.LineStyle = xlContinuous
    End With
    
    wsNew.UsedRange.Font.Name = "Arial"
    wsNew.UsedRange.Font.Size = 8
    wsNew.Columns.AutoFit
    ' If Validation Results Pivot worksheet exists, delete it safely
    If Not wsNew Is Nothing Then
        Application.DisplayAlerts = False ' Disable delete confirmation prompt
        wsNew.Delete ' Delete the worksheet
        Application.DisplayAlerts = True ' Re-enable alerts
    End If

    Call SubFunc_SetZoomLevel(ThisWorkbook)

    ' Prompt user to select save location
    SavePath = Application.GetSaveAsFilename( _
        InitialFileName:=Replace(ThisWorkbook.Name, "UniqueClientName - AP - DeliverData -", "UniqueClientName - AP - Regional DeliverData Dashboards-UniqueCloud1 Validation -"), _
        FileFilter:="Excel Macro-Enabled Workbook (*.xlsm), *.xlsm", _
        Title:="Save Validation File As")
    ' Check if user cancelled
    If SavePath = False Then
        MsgBox "Saving cancelled!", vbInformation, "Cancelled"
        Exit Sub
    End If
    NewFileName = SavePath
    ' Save a copy with new name
    ThisWorkbook.SaveCopyAs Filename:=NewFileName
    ' Wait 5 seconds
    Application.Wait Now + TimeValue("00:00:05")
    ' Open the new file
    Set wbVR = Workbooks.Open(NewFileName)
    ' Turn off alerts to prevent deletion prompts
    Application.DisplayAlerts = False
    ' Delete all sheets except "Data Breakout"
    For Each ws In wbVR.Worksheets
        If LCase(ws.Name) <> LCase("Validation Results") Then
            ws.Delete
        End If
    Next ws
    ' Turn alerts back on
    Application.DisplayAlerts = True
    ' Save and close
    wbVR.Save
    totalRunTime = Round((Timer - BenchMark) / 60, 2)
    Application.ScreenUpdating = True
    If Not showMsg Then MsgBox "Validation Results completed successfully!" & vbCrLf & _
           "Total run time: " & totalRunTime & " minutes", vbInformation, "Complete"
End Sub

Private Sub SubFunc_OuterJoinLogic(rng1 As Range, rng2 As Range, rng3 As Range, wsOut As Worksheet)
    Dim main_dict As Object
    Dim input_ranges As Variant, key As Variant, val As Variant, keyParts As Variant
    Dim r As Long, i As Integer, outputRow As Long
    Dim val_RMD_Current As Double, val_RMD_Overlapping As Double, val_RMD_Excluded As Double, val_MT As Double, perc As Double

    ' Create main master list/dictionary
    Set main_dict = CreateObject("Scripting.Dictionary")
    ' Put each whole pivot table inside the array
    input_ranges = Array(rng1, rng2, rng3)

    ' Loop through all 3 ranges to find every unique 3-column combination
    For i = 0 To 2
        For r = 2 To input_ranges(i).Rows.Count ' Start from row 2 to skip the header to the last row of each table
            ' Create the 3-column composite key from each Pivot Table
            If input_ranges(i).Cells(r, 1).Value <> "Grand Total" Then ' Skip the Grand Total row
                cellValue_Country = Trim(CStr(input_ranges(i).Cells(r, 1).Value))
                cellValue_Category = Trim(CStr(input_ranges(i).Cells(r, 2).Value))
                cellValue_PlanID = Trim(CStr(input_ranges(i).Cells(r, 3).Value))
                ' Match MT Country value South Korea to Korea
                If cellValue_Country = "South Korea" Then cellValue_Country = "Korea"

                key = cellValue_Country & "|" & cellValue_Category & "|" & cellValue_PlanID
                ' If it's a new key, initialize an array of size 0 to 2 to store 3 possible values from the 3 pivot tables
                ' Default Array(0, 0, 0) creates an array of Integers
                If Not main_dict.Exists(key) Then main_dict.Add key, Array(0#, 0#, 0#) ' 0# = CDbl(0) a type shortcut for Double float type
                ' Put the 4th column value into the correct slot (0, 1, or 2)
                ' In VBA, when store an array inside a dictionary, cannot modify the array "inside" the dictionary directly
                ' Taking a copy of the array out of the dictionary and putting it into a temporary variable
                val = main_dict(key)
                ' Update the specific slot in that temporary copy
                ' Immediate If function IIf(Check, TruePart, FalsePart)
                val(i) = CDbl(IIf(IsNumeric(input_ranges(i).Cells(r, 4).Value), input_ranges(i).Cells(r, 4).Value, 0))
                ' Put the updated copy back into the dictionary, overwriting the old version
                main_dict(key) = val
            End If
        Next r ' Next row
    Next i ' Next pivot table range

    ' Writing the data back and do calculation
    outputRow = 6 ' Write data at row 6 because header row 5
    For Each key In main_dict.Keys ' For each unique composite key
        keyParts = Split(key, "|") ' Break back into 3 parts as combined originally
        val_RMD_Current = main_dict(key)(0) ' Get Media Cost (USD) for data current
        val_RMD_Overlapping = main_dict(key)(1) ' Get Media Cost (USD) for data overlapping
        ' Calculate the UniqueCloud2 Excluded data
        val_RMD_Excluded = val_RMD_Current - val_RMD_Overlapping ' Do the subtraction
        val_MT = main_dict(key)(2) ' Get Media Cost (USD) for UniqueCloud1
        ' Rewrite the 3 columns
        wsOut.Cells(outputRow, 1).Resize(1, 3).Value = keyParts ' Country - Offline/Online - Plan ID level
        ' Save the actual values
        wsOut.Cells(outputRow, 4).Value = val_RMD_Current
        wsOut.Cells(outputRow, 5).Value = val_RMD_Overlapping
        wsOut.Cells(outputRow, 6).Value = val_RMD_Excluded
        wsOut.Cells(outputRow, 7).Value = val_MT
        ' Calculate Variance percentage
        If IsNumeric(val_RMD_Excluded) And IsNumeric(val_MT) Then
            If val_RMD_Excluded <> 0 Then ' For numeric and non-zero
                perc = Abs((val_RMD_Excluded - val_MT)) / val_RMD_Excluded
                wsOut.Cells(outputRow, 8).Value = perc ' Write the percentage difference
                wsOut.Cells(outputRow, 8).NumberFormat = "0.00%"
                If perc > 0.05 Then
                    wsOut.Cells(outputRow, 8).Interior.Color = ERR_HIG
                End If
            Else
                wsOut.Cells(outputRow, 8).Value = "#DIV/0!" ' Write error
            End If
        Else
            wsOut.Cells(outputRow, 8).Value = "#N/A" ' Write error
        End If
        ' Move on to the next output row
        outputRow = outputRow + 1
    Next key
End Sub
"""

### 1d. Helper Functions

In [ ]:
# Set up the 1st extraction function - From Code to Full Text
def map_code_to_text(entry: str, mapping_dict: dict[str, str]) -> str | None:
    """
    Scans `entry` for any mapping key (e.g. "MK~AU_") and returns the associated full text value
    Returns "Unknown" Or "Unclassified" if `entry` is NaN/None or if no key is found
    """
    # Handle missing values
    if pd.isna(entry):
        return "Unknown"

    search_text = str(entry) # Coerce to string
    for code_key, full_text in mapping_dict.items():
        # If the code key appears anywhere in the entry text, return its full text value
        if str(code_key).strip() in search_text:
            return full_text

    # No match no key found
    return "Unclassified"

# Set up the 2nd extraction function - From Full Text to Code
def map_text_to_code(entry: str, mapping_dict: dict[str, str]) -> str | None:
    """
    Scans `entry` (with all whitespace removed and lowercased) for any full text value
    (also whitespace-stripped & lowercased) and returns the associated code key
    Returns None Or None if `entry` is NaN/None or if no text value substring is found
    """
    if pd.isna(entry):
        return None

    # Normalize the input: remove all whitespace (\s+) and lowercase
    # normalized_search_text = str(re.sub(r"\s+", "", str(entry)).lower())
    normalized_search_text = str(entry).lower().strip()
    # Iterate through each mapping pair
    for code_key, full_text in mapping_dict.items():
        # Normalize the full_text value the same way
        # normalized_full_text = str(re.sub(r"\s+", "", full_text).lower())
        normalized_full_text = str(full_text).lower().strip()

        # Check if the normalized full text appears in the normalized search text
        if normalized_full_text in normalized_search_text:
            return code_key
    # No match
    return None

In [ ]:
# Sub function to strip trailing and leading spaces and lowercase string value and convert non-string value
def strp_low_str(val):
    if isinstance(val, str): # Already string
        return val.strip().lower()
    elif pd.isnull(val): # Blank
        return ''
    else: # Convert to string
        return str(val).strip().lower()

In [ ]:
# Sub function to detect rows with multiple specific patterns inside it, scan certain multiple columns
def check_contain_any(row, patterns, cols):
    row_str = row[cols].astype(str)
    return any(row_str.str.contains(p, case=False, na=False).any() for p in patterns)

In [ ]:
# Sub function to detect the first Header Row
def detect_header_row(df, required_keywords, max_rows=200):
    """
    The detect_header_row function checks the first 200 rows of the file (can adjust max_rows as needed) 
    For each row, it counts how many of the required keywords appear (ignoring case) and returns the row index with exact matches
    """
    # Normalize required keywords to lower-case and stripped
    required_keywords = [kw.strip().lower() for kw in required_keywords]
    best_row = None
    max_matches = 0
    # Iterate over the first `max_rows` rows of the DataFrame
    for i in range(min(max_rows, len(df))):
        # Get row values as stripped, lower-case strings
        row_vals = df.iloc[i].apply(strp_low_str).tolist()
        # Count how many required keywords appear in any cell of the row
        matches = sum(1 for keyword in required_keywords if any(keyword in cell for cell in row_vals))
        # Update the best row if more matches are found
        if matches > max_matches:
            max_matches = matches
            best_row = i
        # If a row matches all required keywords, return immediately
        if matches == len(required_keywords):
            return i
    return best_row

In [ ]:
# Sub function to add modules into Excel VBA
# Define VBA scripts in a list of tuples (module name inside Excel, VBA code stored in Python)
vba_scripts = [
    ("Module1", Module_Macro_1),
    ("Module2", Module_Macro_2),
    ("Module3", Module_Macro_3),
    ("Module4", Module_Macro_4),
    ("Module5", Module_Macro_5),
    ("Module6", Module_Macro_6),
    ("Module7", Module_Macro_7),
    ("Module8", Module_Macro_8),
    ("Module9", Module_Macro_9),
    ("Module10", Module_Macro_10),
    ("Module11", Module_Macro_11),
    ("Module12", Module_Macro_12),
    ("Module13", Module_Macro_13),
    ("Module14", Module_Macro_14),
    ("Module15", Module_Macro_15),
]
def vb_project(workbook_xlwings):
    # Access the VBA project (make sure programmatic access is enabled in Excel)
    vb_proj = workbook_xlwings.api.VBProject
    for module_name, vba_code in vba_scripts:
        existing_module = None # Check if the module already exists
        for vb_comp in vb_proj.VBComponents:
            if vb_comp.Name == module_name:
                existing_module = vb_comp
                break
        
        if existing_module is not None:
            # print(f"  - Reupdating '{module_name}'")
            # Optionally clear existing code:
            existing_module.CodeModule.DeleteLines(1, existing_module.CodeModule.CountOfLines)
            target_module = existing_module
        else:
            # Create a new module
            target_module = vb_proj.VBComponents.Add(1)  # 1 = standard module
            target_module.Name = module_name
            print(f"  - Created new '{module_name}'")
        # Add or update the VBA code in the module
        target_module.CodeModule.AddFromString(vba_code)
    return workbook_xlwings

# xlwings sub function to reapply formats and conditions
def xlwings_reapply_format(workbook_xlwings):
    workbook_xlwings.activate() # Activate the workbook
    workbook_xlwings.sheets["DeliverData"].activate() # Activate the sheet
    workbook_xlwings.macro("Apply_DeliverData_Format")() # xlwings helper to call the main macro
    workbook_xlwings.save()
    time.sleep(10)
    # workbook_xlwings.close() # for offline run only
    return 

In [ ]:
# Sub function to standardise date parsing to the default py pandas (YYYY MM DD) using dateutil.parser
def clean_date_col(x, day_first_for_date, day_first_for_text):
    # dayfirst=False - first ## digits are for Month
    # dayfirst=True - first ## digits are for Day
    try:
        if pd.isna(x): # For blank
            return pd.NaT
        
        # If cell is already in datetime64ns
        if isinstance(x, (pd.Timestamp, datetime)): # Often Month first, day_first_for_date=False
            return parser.parse(str(x), dayfirst=day_first_for_date, ignoretz=True)
        
        # If cell is in string
        if isinstance(x, str):
            return parser.parse(str(x), dayfirst=day_first_for_text, ignoretz=True)
        
        # If cell is in Excel serial numbers, use Excel calculation way
        if isinstance(x, (int, float)) or str(x).isdigit():
            return pd.to_datetime(float(x), unit="D", origin="1899-12-30")
    except Exception:
        return pd.NaT

In [ ]:
# Sub function to detect invalid taxonomy strings
def detect_invalid_taxonomy(t: str, required_keys: List[str]):
    if not isinstance(t, str) or not t.strip(): # This is case sensitive
        return ["Invalid Input"], []
    
    invalid_keys = []
    duplicate_keys = []

    for k in required_keys:
        valid_pattern = rf'(^|_){re.escape(k)}~\S+' # Set up the correct key with ~
        space_pattern = rf'(^|_){re.escape(k)}\s+\S+' # Set up semi correct key with space instead of ~
        any_pattern = rf'(^|_){re.escape(k)}(?:~|\s)' # Set up duplicate keys in any form

        valid_matches = re.findall(valid_pattern, t)
        space_matches = re.findall(space_pattern, t)
        any_matches   = re.findall(any_pattern, t)

        # Save semi correct key anywhere
        if space_matches:
            invalid_keys.append(k)

        # Save the missing valid key~
        if not valid_matches and k not in invalid_keys:
            invalid_keys.append(k)

        # Save any duplicate key in any form
        if len(any_matches) > 1:
            if k not in invalid_keys:
                invalid_keys.append(k)
            duplicate_keys.append(k)

    return invalid_keys, duplicate_keys

# Sub function to match taxonomy string row by row with unique taxonomy cache saved
def write_taxonomy_issues(t, cache):
    # t is taxonomy row by row, cache is the unique taxonomy cache saved earlier
    if t not in cache:
        return None

    invalid_keys, duplicate_keys = cache[t]

    if not invalid_keys:
        return None

    parts = []
    # Then output the corresponding invalid_keys and duplicate_keys
    missing_only = [k for k in invalid_keys if k not in duplicate_keys]
    if missing_only:
        parts.append(f"Missing keys: {', '.join(missing_only)}")    

    if duplicate_keys:
        parts.append(f"Duplicate keys: {', '.join(duplicate_keys)}")
    
    return " | ".join(parts)

In [ ]:
# Support sub function to convert spend_cost only for Platform data
def convert_currency_platform(row):
    # For a given row, check if the 'Currency' matches the Expected Currency from the currency_dict dictionary Media Cost (LC)
    # If not, perform the appropriate conversion on 'Media Cost (LC)' and update 'Currency'
    country = str(row.get("Country", "")).strip()  # Get the current Country value
    current_curr = str(row.get("Currency", "")).strip().upper()  # Get the current Currency value
    file_name = str(row.get("File Name", "")).strip()  # Get File Name value
    try:
        spend_cost = float(row.get("Media Cost (LC)", 0))  # Get the current Cost value
        spend_cost_usd = float(row.get("Media Cost (USD)", 0))  # Get the current Cost USD value
    except (ValueError, TypeError):
        return row  # Skip if spend_cost is not a valid number
    
    # Skip if country not in our dictionary
    if country not in currency_dict:
        return row
    
    expected_curr = currency_dict[country]  # Get the correct Expected Currency value
    
    # If current Currency matches the correct Expected Currency, nothing to do
    if current_curr == expected_curr:
        return row
    
    # Case a): For Group A countries (should be in Local Currency) but row is in USD
    if country in group_A and current_curr == "USD":
        rate = exchange_dict.get(country, None)
        if rate and rate != 0:
            # Convert from USD to local Currency: new_cost = spend_cost * exchange_rate
            row["Media Cost (LC)"] = spend_cost * rate  # Multiply direction: To convert from USD to LC
            row["Currency"] = expected_curr  # Set back the correct LC
            print(f"    > {file_name} {country}: converted {spend_cost:.2f} USD to {row['Media Cost (LC)']:.2f} {expected_curr}")
    
    elif country in group_A and (current_curr == "" or pd.isna(row.get("Currency"))):  # For blank Currency case
        rate = exchange_dict.get(country, None)
        if rate and rate != 0:
            # Check if spend_cost equals spend_cost_usd
            if abs(spend_cost - spend_cost_usd) < 0.01:  # Using small epsilon for float comparison
                # If yes, do conversion from USD to local Currency
                row["Media Cost (LC)"] = spend_cost * rate
                row["Currency"] = expected_curr
                print(f"    > {file_name} {country}: converted ({spend_cost:.2f} USD to {row['Media Cost (LC)']:.2f} {expected_curr}), with {spend_cost_usd:.2f} USD")
            else:
                # If no, just set currency without conversion
                row["Currency"] = expected_curr
                print(f"    > {file_name} {country}: keep {spend_cost:.2f}, updated {current_curr} to {expected_curr}, with {spend_cost_usd:.2f} USD")
    
    # Case b): For Group B countries (should be in USD) but row is in Local Currency
    elif country in group_B and current_curr in ["IDR", "PHP", "VND"]:
        rate = exchange_dict.get(country, None)
        if rate and rate != 0:
            # Convert from local Currency to USD: new_cost = spend_cost / exchange_rate
            row["Media Cost (LC)"] = spend_cost / rate  # Divide direction: To convert from LC to USD
            row["Currency"] = expected_curr  # Set back to USD
            print(f"    > {file_name} {country}: converted {spend_cost:.2f} {current_curr} to {row['Media Cost (LC)']:.2f} {expected_curr}")
    
    elif country in group_B and (current_curr == "" or pd.isna(row.get("Currency"))):  # For blank Currency case
        rate = exchange_dict.get(country, None)
        if rate and rate != 0:
            # Check if spend_cost equals spend_cost_usd
            if abs(spend_cost - spend_cost_usd) < 0.01:  # Using small epsilon for float comparison
                # If yes, just set currency without conversion
                row["Currency"] = expected_curr
                print(f"    > {file_name} {country}: keep {spend_cost:.2f}, updated {current_curr} to {expected_curr}, with {spend_cost_usd:.2f} USD")
            else:
                # If no, do conversion from local Currency to USD
                row["Media Cost (LC)"] = spend_cost / rate
                row["Currency"] = expected_curr
                print(f"    > {file_name} {country}: converted {spend_cost:.2f} {current_curr} to {row['Media Cost (LC)']:.2f} {expected_curr}, with {spend_cost_usd:.2f} USD")
    return row

In [ ]:
# Sub function to move a column after a reference column inside df
def move_col_after(df, col_to_move, reference_col):
    all_cols = df.columns.tolist()
    all_cols.insert(all_cols.index(reference_col) + 1, # Add an empty col
                    all_cols.pop(all_cols.index(col_to_move))) # Move the col there
    return df[all_cols]

In [ ]:
# Sub function to reindex the dataframe to match the given header
def match_header_cols(df, header):
    # Reindex will set the DataFrame's columns to exactly match header
    # Extra columns are dropped, missing columns will be added with NaN
    return df.reindex(columns=header)

In [ ]:
# Validate PO Number format: optional "P" or "#", then exactly 5 or 6 digits
def validate_po_number(po):
    po = str(po).strip()
    # Remove optional prefix
    clean_po = po[1:] if po and po[0] in ['P', '#'] else po
    # Check if remaining characters are exactly 5 or 6 digits (no spaces or other characters)
    if re.match(r'^\d{5,6}$', clean_po):
        return po
    else:
        return "P112233" # dummy value

In [ ]:
# Sub function to get the FY and Q from the file name
def calculatefiscal_fyq(master_file, verbose=True):
    """
    Extract Q#FY## pattern(s) from filename and convert to FY##Q# format
    Cases:
    - Single pattern (Q1FY26) -> returns ["FY26Q1"]
    - Range pattern same year (Q1FY26-Q4FY26) -> returns ["FY26Q1", "FY26Q2", "FY26Q3", "FY26Q4"]
    - Range pattern cross-year (Q3FY25-Q2FY26) -> returns ["FY25Q3", "FY25Q4", "FY26Q1", "FY26Q2"]
    - No pattern found -> returns empty list
    """
    year_quarters = []
    # Pattern to match Q#FY## format
    pattern = r'Q(\d)FY(\d{2})'
    
    # Find all matches
    matches = re.findall(pattern, master_file)
    
    if not matches:
        if verbose:
            print(f"\n> No Q#FY## pattern found in {master_file} file name")
        return year_quarters
    
    # Case 1: Single Q#FY## pattern
    if len(matches) == 1:
        quarter, year = matches[0]
        fy_format = f"FY{year}Q{quarter}"
        year_quarters.append(fy_format)
        if verbose:
            print(f"\n+ Found {fy_format} in {master_file} file name")
    
    # Case 2: Two Q#FY## patterns (range)
    elif len(matches) == 2:
        start_quarter, start_year = matches[0]
        end_quarter, end_year = matches[1]
        
        start_q = int(start_quarter)
        end_q = int(end_quarter)
        start_y = int(start_year)
        end_y = int(end_year)
        
        # Same year range
        if start_year == end_year:
            for q in range(start_q, end_q + 1):
                year_quarters.append(f"FY{start_year}Q{q}")
            if verbose:
                print(f"\n+ Found FY{start_year}Q{start_quarter}-FY{end_year}Q{end_quarter} in {master_file} file name")
        
        # Cross-year range
        else:
            # Add quarters from start year (start_q to Q4)
            for q in range(start_q, 5):  # Q1-Q4
                year_quarters.append(f"FY{start_year}Q{q}")
            
            # Add quarters from intermediate years (if any)
            for y in range(start_y + 1, end_y):
                for q in range(1, 5):  # Q1-Q4
                    year_quarters.append(f"FY{y:02d}Q{q}")
            
            # Add quarters from end year (Q1 to end_q)
            for q in range(1, end_q + 1):
                year_quarters.append(f"FY{end_year}Q{q}")
            if verbose:
                print(f"\n+ Found FY{start_year}Q{start_quarter}-FY{end_year}Q{end_quarter} in {master_file} file name")
    return year_quarters

# Save the list of values
year_quarters = calculatefiscal_fyq(master_file)

In [ ]:
# Sub function to apply some basic formatting to a worksheet
def excel_apply_sheet_format(ws):
    font_name = "Arial"
    font_size = 8
    cell_font = Font(name=font_name, size=font_size)
    center_alignment = Alignment(horizontal="center", vertical="center")
    # Find each column's index from the header row
    header = {cell.value: cell.column for cell in ws[1]}
    # Apply font and alignment to all cells
    for row in ws.iter_rows(min_row=1, max_row=ws.max_row):
        for cell in row:
            cell.font = cell_font
            cell.alignment = center_alignment
    # Apply number formats from excel_format_map
    for col_name, fmt in excel_format_map.items():
        if col_name not in header:
            continue
        col_idx = header[col_name]
        col_letter = get_column_letter(col_idx)
        # Apply format to every data row (skip row 1 = header)
        for row in range(2, ws.max_row + 1):
                ws[f"{col_letter}{row}"].number_format = fmt

In [ ]:
# Sub function to write df to an Excel file, apply some formats, mainly for saving
def excel_writer_customize(df, file_path, sheet_name_val):
    file_path_pa = Path(file_path)
    if not os.path.exists(file_path):
        print(f"  - Creating new {file_path} file")
        # Save Type 1b: Brand new xlsx with format
        if file_path_pa.suffix == '.xlsx':
            df.to_excel(file_path, sheet_name=sheet_name_val, index=False)
            wb = load_workbook(file_path, keep_vba=False)
            ws = wb[sheet_name_val]
            excel_apply_sheet_format(ws)
            wb.save(file_path)
        # Save Type 2b: Brand new xlsm with format
        elif file_path_pa.suffix == '.xlsm':
            # Create .xlsx first, then save as .xlsm
            temp_xlsx = file_path_pa.with_suffix('.xlsx')
            df.to_excel(temp_xlsx, sheet_name=sheet_name_val, index=False)
            wb = load_workbook(temp_xlsx, keep_vba=True) # True here to prepare for xlsm
            ws = wb[sheet_name_val]
            excel_apply_sheet_format(ws)
            # Save as .xlsm (openpyxl will handle the conversion)
            wb.save(file_path)
            excel_app = xw.App(visible=False, add_book=False)
            workbook_xlwings = excel_app.books.open(file_path)
            workbook_xlwings = vb_project(workbook_xlwings)
            workbook_xlwings.save()
            os.remove(temp_xlsx)  # Clean up temp file
        else:
            raise ValueError(f"Unsupported file type: {file_path.suffix}")
    else:
        print(f"  - Writing data to '{sheet_name_val}' sheet of the existing file")
        # Save Type 1c: Overwrite existing xlsx with format
        if file_path_pa.suffix == '.xlsx':
            # Using mode='a' with if_sheet_exists='replace'
            with pd.ExcelWriter(file_path, engine='openpyxl', mode='a', if_sheet_exists='replace') as writer:
                df.to_excel(writer, sheet_name=sheet_name_val, index=False)
                ws = writer.sheets[sheet_name_val]
                excel_apply_sheet_format(ws)
        # Save Type 2c: Overwrite existing xlsm
        elif file_path_pa.suffix == '.xlsm':
            wb = load_workbook(file_path, keep_vba=True)
            # Check if sheet exists, if so delete it
            if sheet_name_val in wb.sheetnames:
                ws = wb[sheet_name_val]
                # Delete all rows in the sheet (including header)
                ws.delete_rows(1, ws.max_row)
            else:
                # Recreate new sheet
                ws = wb.create_sheet(sheet_name_val)
                
            # Write dataframe using dataframe_to_rows
            for row in dataframe_to_rows(df, index=False, header=True):
                ws.append(row)
            # Save the workbook (keeps VBA)
            wb.save(file_path)
            wb.close # for offline run only
        else:
            raise ValueError(f"Unsupported file type: {file_path.suffix}")

### 1d. Update Guidelines

In [ ]:
# Sub function to help write df_tax_values back into df_guidelines
def update_tax_back_guide(df_guidelines: pd.DataFrame, start_col: int, df_headerbox: pd.DataFrame, df_tax_values: pd.DataFrame, header_rows: int = 2) -> pd.DataFrame:
    # Keep header rows (in Guidelines Sheet) and replace values below with df_tax_values (from TaxonomyTool Dictionary sheet), clear any extra rows below the inserted values
    if df_tax_values.shape[1] != df_headerbox.shape[1]: # [1] is the total columns
        raise ValueError("      > df_headerbox and df_tax_values must have same number of columns")
    needed_rows = header_rows + len(df_tax_values) # Set up new total rows
    current_rows = len(df_guidelines) # Current total rows
    # Expand df_guidelines if needed
    if needed_rows > current_rows:
        rows_added = needed_rows - current_rows # New additional rows
        print(f"  - Expanding df_guidelines from {current_rows} to {needed_rows}, total {rows_added}")
        df_guidelines = df_guidelines.reindex(range(needed_rows))
    col_range_slice = slice(start_col, start_col+df_headerbox.shape[1]) # Should be a range of 2 cols
    # Clear everything below the header first
    df_guidelines.iloc[header_rows:, col_range_slice] = pd.NA
    # Write headers back (preserve header content)
    df_guidelines.iloc[:header_rows, col_range_slice] = df_headerbox.values
    # Write new values under headers
    if len(df_tax_values) > 0:
        df_guidelines.iloc[header_rows: header_rows+len(df_tax_values), col_range_slice] = df_tax_values.values
    return df_guidelines

In [ ]:
# Sub function to process TaxonomyTool dictionaries
def process_taxonomy_dictionary(classifications, df_taxonomy, df_guidelines, header_rows=2):
    missing_classifications = [] 
    for classification in classifications:
        print(f"\n+ Processing '{classification}'")
        try:
            # Find the classification column index in Guidelines sheet
            start_idx = df_guidelines.columns.get_loc(classification)
        except KeyError:
            print(f"      > {classification} not found in Guidelines, skipping")
            missing_classifications.append(classification)
            continue
        # Determine the Guidelines headerbox: first 3 rows and 2 columns of that Classification column
        df_headerbox = (df_guidelines.iloc[:header_rows, start_idx:start_idx+2].copy().reset_index(drop=True)) # Because Python slices include the start index but exclude the end index

        if df_headerbox.empty or df_headerbox.shape[1] < 2: # The number of columns is less than 2
            print(f"      > {classification} headerblock empty/invalid, skipping")
            missing_classifications.append(classification)
            continue
        print(f"  - Headerbox {df_headerbox.shape}")

        # Map Guidelines classification column to TaxonomyTool dictionary 
        if classification in ("Publisher/Vendor (Online)", "Vehicle Name (Online)"):
            search_classification = "Publisher"
        elif classification == "Promotion Incentive":
            search_classification = "Promotion"
        elif classification == "Profitable Consumer Behaviour":
            search_classification = "Profitable Consumer Behaviors"
        elif classification == "Primary Target Strategy":
            search_classification = "Primary Target"
        else: 
            search_classification = classification
        # Find that TaxonomyTool classification col
        matched_classifications = df_taxonomy.columns[df_taxonomy.columns.str.contains(search_classification, regex=False)]
        if len(matched_classifications) == 0:
            print(f"      > {search_classification} not found in TaxonomyTool, skipping")
            missing_classifications.append(classification)
            continue
        if len(matched_classifications) >= 2:
            print(f"      > {search_classification} found more than 1 time in TaxonomyTool, skipping")
            missing_classifications.append(classification)
            continue
        # Get that TaxonomyTool classification col index
        start_idx_tax = df_taxonomy.columns.get_loc(matched_classifications[0])

        # Build df_tax_values to insert under df_headerbox (No headers included)
        if classification in ("Promotion Incentive", "Apex"):
            # One source col, create second col as key+value
            df_tax_target = df_taxonomy.iloc[:, start_idx_tax:start_idx_tax+1].copy()
            taxonomy_key = str(df_tax_target.iat[0, 0]) # Get Taxonomy Key PRM~
            df_tax_values = df_tax_target.iloc[2:].dropna(how="all").reset_index(drop=True)
            print(f"  - Dictionary values found {df_tax_values.shape}")
            if df_tax_values.empty:
                print(f"      > {classification} empty, just clear old values")
                df_tax_values = pd.DataFrame(columns=df_headerbox.columns)
            else:
                # Create a 2nd column for key+value
                df_tax_values = pd.DataFrame({
                    df_headerbox.columns[0]: df_tax_values.iloc[:, 0].astype(str), # The value col
                    df_headerbox.columns[1]: df_tax_values.iloc[:, 0].astype(str).map(lambda x: taxonomy_key + x)}) # Duplicate value col to create PRM~Value
                print(f"  - {taxonomy_key} + Value finished")
        else:
            # Normal two sources col
            df_tax_target = df_taxonomy.iloc[:, start_idx_tax:start_idx_tax+2].copy() # Python slicing only keep +1
            if df_tax_target.shape[1] < 2:
                print(f"      > {classification} expected 2 columns in TaxonomyTool, skipping")
                missing_classifications.append(classification)
                continue
            # Set Taxonomy Key: special case for Vehicle Name (Online) first then 0:0 position for everything else
            taxonomy_key = "PD~" if classification == "Vehicle Name (Online)" else str(df_tax_target.iat[0, 0])
            # Clean blank values first
            df_tax_values = df_tax_target.iloc[2:].dropna(how="all").reset_index(drop=True)
            print(f"  - Dictionary values found {df_tax_values.shape}")

            # Additional special manual classification adjustments
            if classification == 'Initiative':
                df_tax_values = df_tax_values[~(df_tax_values.iloc[:, 0]== "Consumer Payments: Brand Preference B2B")]
                df_tax_values.iloc[:, 0] = df_tax_values.iloc[:, 0].str.strip()

            if classification == 'Profitable Consumer Behaviour':
                df_tax_values.iloc[:, 0] = df_tax_values.iloc[:, 0].str.replace('Other, please make sure you have an hypothesis PCB', 'Other', regex=False)
                df_tax_values.iloc[:, 0] = df_tax_values.iloc[:, 0].str.replace('# N/A (only for B2B, CMS , VAS and Product Platform Application & Services campaigns)', 'N/A', regex=False)
                df_tax_values.iloc[:, 0] = df_tax_values.iloc[:, 0].str.strip()

            if classification == 'Sponsorship':
                df_tax_values.iloc[:, 0] = df_tax_values.iloc[:, 0].str.replace('N/A: If there is no sponsorship. Also, use for all B2B efforts', 'N/A', regex=False)
                df_tax_values.iloc[:, 0] = df_tax_values.iloc[:, 0].str.replace('Other: If sponsoring an event outside of the above list', 'Other', regex=False)
                df_tax_values.iloc[:, 0] = df_tax_values.iloc[:, 0].str.strip()

            if classification == 'Campaign Objective':
                df_tax_values.iloc[:, 0] = df_tax_values.iloc[:, 0].str.replace('Acquisition/Leads/Conversions/Revenue/Sales', 'Acquisition', regex=False)
                df_tax_values.iloc[:, 0] = df_tax_values.iloc[:, 0].str.replace('Reach/Views/Branding/Awareness', 'Awareness', regex=False)
                df_tax_values.iloc[:, 0] = df_tax_values.iloc[:, 0].str.replace('Site Traffic/Clicks/Consideration', 'Consideration', regex=False)
                df_tax_values.iloc[:, 0] = df_tax_values.iloc[:, 0].str.strip()

            if classification == 'Cohorts':
                df_tax_values.iloc[:, 0] = df_tax_values.iloc[:, 0].str.replace('Not Applicable', 'N/A', regex=False)
                df_tax_values.iloc[:, 0] = df_tax_values.iloc[:, 0].str.strip()

            if classification == 'Publisher/Vendor (Online)':
                # df_tax_values = df_tax_values[~(df_tax_values.iloc[:, 0]=="---")]
                exclude_values = ["---", "Meta", "Tik Tok", "Apex Programmatic", "Apple Search", "Trip Advisor", "Trucaller", "Naver M.Branding", "Naver Timeboard"]
                df_tax_values = df_tax_values[~df_tax_values.iloc[:, 0].isin(exclude_values)]
                
            if classification == "Vehicle Name (Online)":
                exclude_values = ["---", "Meta", "Tik Tok", "Apex Programmatic", "Apple Search", "Trip Advisor", "Trucaller", "Naver", "Naver Brand Search", "Naver GFA", "Naver SEM", "Naver Webtoon"]
                df_tax_values = df_tax_values[~df_tax_values.iloc[:, 0].isin(exclude_values)]

                blank_row = pd.DataFrame([["(blank)", "nan"]], columns=df_tax_values.columns)
                df_tax_values = pd.concat([df_tax_values, blank_row], ignore_index=True)
            
            if classification == "Media Channel":
                df_tax_values.iloc[:, 0] = df_tax_values.iloc[:, 0].str.replace('Digital Display(Non - Paid Social)', 'Digital Display (Non-Paid Social)', regex=False)
                df_tax_values.iloc[:, 0] = df_tax_values.iloc[:, 0].str.replace('Digital Video (Non -  CTV, OTT, Paid Social)', 'Digital Video (Non-CTV, OTT, Paid Social)', regex=False)
                df_tax_values.iloc[:, 0] = df_tax_values.iloc[:, 0].str.replace('Influencer/creator Display', 'Influencer/Creator Display', regex=False)
                df_tax_values.iloc[:, 0] = df_tax_values.iloc[:, 0].str.replace('Influencer/creator Video', 'Influencer/Creator Video', regex=False)
                additional_values = ["Airport", "OOH-Other", "Cinema", "Radio", "Magazines", "Newspapers", "Instore/POS", "TV", "Digital OOH"]
                additional_rows = pd.DataFrame({df_tax_values.columns[0]: additional_values})
                df_tax_values = pd.concat([df_tax_values, additional_rows], ignore_index=True)
                df_tax_values.iloc[:, 0] = df_tax_values.iloc[:, 0].str.strip()

            # Combine Taxonomy Key to 2nd column and align with df_headerbox columns
            if df_tax_values.empty:
                print(f"      > {classification} empty, just clear old values")
                df_tax_values = pd.DataFrame(columns=df_headerbox.columns)
            else:
                # Ensure string concat is safe, key+code
                df_tax_values.iloc[:, 1] = taxonomy_key + df_tax_values.iloc[:, 1].astype(str)
                print(f"  - {taxonomy_key} + Code finished")

                # Rename columns to guideline headerbox
                df_tax_values.columns = df_headerbox.columns

        # Write df_tax_values back into df_guidelines
        df_guidelines = update_tax_back_guide(df_guidelines=df_guidelines, start_col=start_idx, df_headerbox=df_headerbox, df_tax_values=df_tax_values, header_rows=header_rows)
        print(f"  - Updated {len(df_tax_values)} rows")

    # Update the YQ value from the file name into this list
    if 'YQ' in df_guidelines.columns:
        yq_val = df_guidelines['YQ'].values.tolist()
        yq_val = [v for v in yq_val[2:] if pd.notna(v)]
        df_guidelines.iloc[header_rows:, df_guidelines.columns.get_loc('YQ')] = pd.NA
        df_guidelines.iloc[header_rows:header_rows+len(year_quarters), df_guidelines.columns.get_loc('YQ')] = year_quarters
        print(f"\n  - Updated YQ column with {year_quarters}")

    # Print out classifications issues
    if missing_classifications:
        print("      > Classifications skipped: ", missing_classifications)
    return df_guidelines, missing_classifications

In [ ]:
# Sub function to help write data partially into Excel sheet 
def overwrite_guidelines_values_in_place(ap_DeliverData_file:str, df_updated:pd.DataFrame, sheet_name:str="Guidelines", excel_start_col:int=4, excel_header_rows:int=3, clear_buffer_rows:int=70):
    # start col D, keep the top 3 rows

    file_path = Path(ap_DeliverData_file)
    if file_path.suffix == '.xlsx':
        wb = load_workbook(ap_DeliverData_file, keep_vba=False)
    elif file_path.suffix == '.xlsm':
        wb = load_workbook(ap_DeliverData_file, keep_vba=True)
    else:
        raise ValueError(f"Unsupported file type: {file_path.suffix}")
    
    ws = wb[sheet_name]
    nrows, ncols = df_updated.shape
    # Start writing Below the existing Excel headers
    write_start_row = 1 + excel_header_rows   # Excel row index where data starts
    max_clear_row = write_start_row + (nrows - excel_header_rows) + clear_buffer_rows
    max_clear_col = excel_start_col + ncols - 1
    # Clear old values only below headers
    for row in ws.iter_rows(min_row=write_start_row, max_row=max_clear_row, min_col=excel_start_col, max_col=max_clear_col):
        for cell in row:
            if isinstance(cell, MergedCell):
                continue
            cell.value = None
    # Write df values below headers
    row_output = write_start_row
    for i in range(excel_header_rows -1, nrows): # shift back to pd index
        for j in range(ncols):
            output_value = df_updated.iat[i, j]
            if pd.isna(output_value):
                output_value = None
            cell = ws.cell(row=row_output, column=excel_start_col + j) # start from 4th col
            if isinstance(cell, MergedCell):
                raise AttributeError(f"      > Merged Cell detected at {cell.coordinate}")
            cell.value = output_value
        row_output += 1
    wb.save(ap_DeliverData_file)
    # wb.close()
    print(f"\n~ Saved to {ap_DeliverData_file}")

In [ ]:
# Main function to update Guidelines sheet with TaxonomyTool dictionaries
def update_guidelines():
    print("\n─────➤ 1d. Update Guidelines")

    # Load taxonomy file
    taxonomy_files = glob.glob("*UniqueClientNameTaxonomyTool*")
    if len(taxonomy_files) != 1:
        print(f"\n> Expected 1 Taxonomy file, found {len(taxonomy_files)}, aborting")
        return
    taxonomy_file = taxonomy_files[0]
    df_taxonomy = pd.read_excel(taxonomy_file, sheet_name="Dictionary", engine="openpyxl", keep_default_na=False, na_values=[''], header=3) # From row 4
    df_taxonomy = df_taxonomy.iloc[:, 4:]  # Drop A-D columns
    print(f"\n+ Loading file '{taxonomy_file}'")

    # Load AP file Guidelines sheet
    ap_DeliverData_any_list = glob.glob("*UniqueClientName - AP - DeliverData - Q*")
    if len(ap_DeliverData_any_list) != 1:
        print(f"\n> Expected 1 AP DeliverData file, found {len(ap_DeliverData_any_list)}, aborting")
        return
    ap_DeliverData_file = ap_DeliverData_any_list[0]
    df_guidelines = pd.read_excel(ap_DeliverData_file, sheet_name="Guidelines", engine="openpyxl", keep_default_na=False, na_values=['']) 
    df_guidelines = df_guidelines.iloc[:, 3:] # Drop A-C columns
    print(f"\n+ Loading file '{ap_DeliverData_file}'")

    # Set up the Classification columns from Guidelines to be updated
    classifications = ['Funding Source', 'Initiative', 'Profitable Consumer Behaviour', 'Product Message', 'Destination', 'Sponsorship', 'Campaign Objective', 'Cohorts','Publisher/Vendor (Online)', 'Vehicle Name (Online)', 'Media Channel', 'Inventory (Distribution) Type', 'Primary Target Strategy', 'Promotion Incentive', 
                       'Merchant Code', 'Placement Rate Type', 'Package Type', 'Apex', 'Language', 'Creative Size']

    # Update df_taxonomy to df_guidelines
    df_guidelines_updated, missing_classifications = process_taxonomy_dictionary(classifications, df_taxonomy, df_guidelines)

    if missing_classifications:
        print(f"\n> Classification issues detected, skip updating")
    else: 
        overwrite_guidelines_values_in_place(ap_DeliverData_file=ap_DeliverData_file, df_updated=df_guidelines_updated, sheet_name="Guidelines", excel_start_col=4, excel_header_rows=3)

### 1e. Update Guidelines Offline

In [ ]:
# Update Guidelines sheet (Offline only) from the SubtypeListReport input file
def update_guidelines_offline():
    print("\n─────➤ 1e. Update Guidelines Offline")

    # Load SubtypeListReport input file
    subtype_files = glob.glob("*SubtypeListReport*")
    if len(subtype_files) != 1:
        print(f"\n> Expected 1 SubtypeListReport file, found {len(subtype_files)}, aborting")
        return
    subtype_file = subtype_files[0]
    df_subtype = pd.read_excel(subtype_file, header=0, engine='openpyxl')
    print(f"\n+ Loading file '{subtype_file}'")

    # Load AP file
    ap_DeliverData_any_list = glob.glob("*UniqueClientName - AP - DeliverData - Q*")
    if len(ap_DeliverData_any_list) != 1:
        print(f"\n> Expected 1 AP DeliverData file, found {len(ap_DeliverData_any_list)}, aborting")
        return
    ap_DeliverData_file = ap_DeliverData_any_list[0]
    print(f"\n+ Loading file '{ap_DeliverData_file}'")

    # Media Type Name exclusion list
    excluded_media_types = [
        "Creator, Content, Influencer Fee", "Digital Audio", "Digital Display (Non-Paid Social)", "Digital Fee",
        "Digital Video (Non-CTV, OTT, Paid Social)", "Direct Mail", "Direct Mail/Electronic DM", "Email",
        "Influencer/creator Display", "Influencer/creator Video", "Non-Linear TV (VOD, OTT & CTV)", "Offline Fee",
        "Other (i.e. Content, Influencer, Event Sign.)", "Paid Social Display", "Paid Social Video", "Search"
    ]

    # Name value exclusion list
    excluded_names = ["Haneda", "Hwaeun Media", "Media One", "MediaOneSpace", "Mediaworks", "Mediaworks OOH", "StellarAce", "Times Of India"]

    # Filter rows where Media Type Name is NOT in the excluded list
    df_filtered = df_subtype[~df_subtype["Media Type Name"].isin(excluded_media_types)]

    # Clean up Name column: drop blanks, drop duplicates, drop excluded names
    name_series = df_filtered["Name"].dropna()
    name_series = name_series[name_series.astype(str).str.strip() != ""]
    name_series = name_series.drop_duplicates()
    name_series = name_series[~name_series.isin(excluded_names)]

    # Check that Pay TV/Cable and Broadcast are present
    for required_name in ["Pay TV/Cable", "Broadcast"]:
        if required_name not in name_series.values:
            print(f"\n  - {required_name} not found under in Name Series")

    name_list = name_series.tolist()
    print(f"\n+ Found {len(name_list)} unique Name values to write")

    # Load AP DeliverData workbook
    file_path = Path(ap_DeliverData_file)
    if file_path.suffix == '.xlsx':
        wb = load_workbook(ap_DeliverData_file, keep_vba=False)
    elif file_path.suffix == '.xlsm':
        wb = load_workbook(ap_DeliverData_file, keep_vba=True)
    else:
        raise ValueError(f"Unsupported file type: {file_path.suffix}")

    ws = wb["Guidelines"]

    # Find the target columns by header name in row 1
    publisher_col = None
    vehicle_col = None
    for cell in ws[1]:
        if cell.value == "Publisher/Vendor (Offline)":
            publisher_col = cell.column
        elif cell.value == "Vehicle Name (Offline)":
            vehicle_col = cell.column

    if publisher_col is None or vehicle_col is None:
        print("\n> Could not find one or both target columns in row 1, aborting")
        return

    # Clear existing values from row 4 to the last row in both columns
    max_row = ws.max_row
    if max_row >= 4:
        for row in range(4, max_row + 1):
            ws.cell(row=row, column=publisher_col).value = None
            ws.cell(row=row, column=vehicle_col).value = None

    # Write the Name list starting at row 4 in both columns
    for i, name in enumerate(name_list):
        row = 4 + i
        ws.cell(row=row, column=publisher_col).value = name
        ws.cell(row=row, column=vehicle_col).value = name

    wb.save(ap_DeliverData_file)
    print(f"\n~ Saved to {ap_DeliverData_file}")

### 2. Combine Platform

In [ ]:
# Platform headers dictionary - all synonyms into one main
platform_col_mapping = {
    "Date": ["Date", "Day", "By Day", "Start Date (in UTC)"],
    "Account Name": ["Advertiser", "Ad Account Name", "Account name", "Account Name"],
    "Campaign Taxonomy": ["Campaign", "Campaign Name", "Campaign name"],
    "Placement Taxonomy": ["Line Item", "Ad group", "Ad Group", "Campaign Group Name", "Ad set name"],
    "Creative Taxonomy": ["YouTube Ad", "YouTube Ad/Creative Name", "Creative"],
    "Publisher/Vendor": ["Property", "Property ", "Platform", "Publisher/Vendor"],
    "Currency": ["Partner Currency", "Advertiser Currency", "Currency code", "Currency", "Currency ", "Advertiser Currency Code"],
    "Media Cost (LC)": ["Revenue (Partner Currency)", "Revenue (Adv Currency)", "Spent (USD)", "Cost", "Total Spent"],
    "Paid Impressions": ["Impressions", "Impr.", "Paid Impressions"],
    "Clicks": ["Clicks", "Link clicks", "Clicks (all)", "Clicks (destination)"],
    "Video Views": ["TrueView: Views", "TrueView views", "Video Views"],
    "Video Completes": ["Complete Views (Video)", "Completed Views", "Video played to 100%"] 
}

In [ ]:
# Process each Excel file: read all sheets and extract/rename columns based on platform_col_mapping
def extract_platform_data(file_path):
    print(f"\n+ Processing file '{file_path}'")
    compiled_frames = []
    try:
        # Load the Excel file
        xls = pd.ExcelFile(file_path, engine='openpyxl')
        for sheet in xls.sheet_names:
            print(f"  - Processing sheet '{sheet}'")
            df = pd.read_excel(file_path, sheet_name=sheet, header=0, engine='openpyxl')
            
            # Build a mapping from the sheet's columns to main headers
            mapping_found = {}
            for main_header, synonyms in platform_col_mapping.items():
                found = False
                for col in df.columns:
                    if col in synonyms:
                        mapping_found[col] = main_header
                        found = True
                        break  # Only one match per main_header column is expected
                if not found:
                    print(f"    > '{main_header}' not found: {synonyms}")
            
            if mapping_found:
                # Extract matched columns and rename them to main_header names
                df_extracted = df[list(mapping_found.keys())].copy()
                df_extracted.rename(columns=mapping_found, inplace=True)

                # Save the file name and sheet name
                df_extracted["File Name"] = file_path
                # df_extracted["Sheet Name"] = sheet
                
                # Append only if the DataFrame is not completely empty
                if not df_extracted.dropna(how="all").empty:
                    compiled_frames.append(df_extracted)
                else:
                    print(f"    > No data found in matched columns for '{file_path}' - '{sheet}'")
            else:
                print(f"    > No matching columns found in '{file_path}' - '{sheet}'")
    except Exception as e:
        print(f"\n> Error processing '{file_path}': {e}")
    return compiled_frames

In [ ]:
# Compile data from multiple Excel files into one consolidated file
def combine_platform():
    print("\n─────➤ 2. Consolidate Platform Files")

    # Find all Excel files in the current directory
    file_paths = glob.glob("*.xlsx")

    # Set up the output file
    platform_consolidation_output_file = "Platform_Consolidation_Output.xlsx"

    # Set up empty data frame
    all_frames = []

    # Build a list of files to remove, using glob for patterns, and listing specific filenames
    excludes = (
        glob.glob("*_Output.xlsx*") + # Previous Output files
        glob.glob("*UniqueCloud3*.xlsx") + # UniqueCloud3 file
        glob.glob("*UniqueClientName - AP - DeliverData - Q*.xlsx") + # Main AP DeliverData file
        ["Online Delivery Summary.xlsx", # UniqueCloud2 file
         "Raw Extract.xlsx", # UniqueCloud2 Raw Table file
         "FY25 APAC Report.xlsm"] # UniqueCloud1 file
    )

    # Filter out the excluded files from the main file list
    file_paths = [file_path for file_path in file_paths if file_path not in excludes]
    
    # Process each file and collect all DataFrames
    for file_path in file_paths:
        frames = extract_platform_data(file_path)
        all_frames.extend(frames)
    
    if all_frames:
        # Remove any empty DataFrames
        frames_with_data = [frame for frame in all_frames if not frame.dropna(how="all").empty]
        # Concatenate all DataFrames with data into one DataFrame
        if frames_with_data:
            compiled_df = pd.concat(frames_with_data, ignore_index=True)
            # Safeguard: Drop rows that are completely empty
            compiled_df = compiled_df.dropna(how='all')
            # Remove rows that contain "Total of" in any cell in one column (case-insensitive)
            compiled_df = compiled_df[~compiled_df.apply(check_contain_any, args=(["Total of"], ["Date"]), axis=1)]
            print("  - Removed 'Total of' rows")
            
            # day_first_for_date=False py pandas (YYYY MM DD), day_first_for_text=True (YYYY DD MM)
            mask_filename_ggads = compiled_df['File Name'].str.lower().str.contains('google ads', na=False, regex=True)
            compiled_df.loc[mask_filename_ggads, 'Date'] = compiled_df.loc[mask_filename_ggads, 'Date'].apply(clean_date_col, args=(False, True))
            print("  - Date formatting: Google Ads")
            
            # day_first_for_date=False py pandas (YYYY MM DD), day_first_for_text=False (YYYY MM DD)
            mask_filename_dv360 = compiled_df['File Name'].str.lower().str.contains('dv360|meta|tiktok|uber', na=False, regex=True)
            compiled_df.loc[mask_filename_dv360, 'Date'] = compiled_df.loc[mask_filename_dv360, 'Date'].apply(clean_date_col, args=(False, False))
            print("  - Date formatting: DV360, Meta, Tiktok, Uber")

            # day_first_for_date=True (YYYY DD MM) switch back to (YYYY MM DD), day_first_for_text=False (YYYY MM DD)
            mask_filename_linkedin = compiled_df['File Name'].str.lower().str.contains('linkedin|ttd', na=False, regex=True)
            compiled_df.loc[mask_filename_linkedin, 'Date'] = compiled_df.loc[mask_filename_linkedin, 'Date'].apply(clean_date_col, args=(True, False))
            print("  - Date formatting: Linkedin")

            # Clean Media Cost (LC) to remove symbol and keep only numbers
            compiled_df["Media Cost (LC)"] = (compiled_df["Media Cost (LC)"].astype(str).str.replace(r'[^\d.]', '', regex=True).str.replace(r'\.(?=.*\.)', '', regex=True)) # Keep digits and dot, then keep only last dot
            compiled_df["Media Cost (LC)"] = pd.to_numeric(compiled_df["Media Cost (LC)"], errors='coerce').combine_first(compiled_df["Media Cost (LC)"])
            print("  - Cleaned 'Media Cost (LC)'")

            # Reorder the column order
            platform_col_order = ['File Name'] + list(platform_col_mapping.keys())
            compiled_df = match_header_cols(compiled_df, platform_col_order)

            # Save Type 1b or 1c
            excel_writer_customize(compiled_df, platform_consolidation_output_file, "Consolidation")
        else:
            print("\n> No data frames found to compile")
    else:
        print("\n> No data found to compile")

### 3. Clean Platform

In [ ]:
# Clean the Platform Consolidation file
def clean_platform():
    print("\n─────➤ 3. Clean Platform Consolidation")

    # Set up the input file and output file names
    platform_consolidation_output_file = "Platform_Consolidation_Output.xlsx"
    platform_cleaned_output_file = "Platform_Cleaned_Output.xlsx"
    if not os.path.exists(platform_consolidation_output_file):
        print(f"\n> No Platform Consolidation file found for cleaning")
        return
    print(f"\n+ Processing file '{platform_consolidation_output_file}'")
    try:
        # Load the consolidation output file
        df = pd.read_excel(platform_consolidation_output_file, header=0, engine='openpyxl', keep_default_na=False, na_values=['']) # At this point “N/A” remains the literal string "N/A" 

        # 
        df.rename(columns={"Date": "Start Date"}, inplace=True)

        # Match the current df to DeliverData header + File Name from combine_platform
        header_DeliverData_temp = header_DeliverData + ['File Name']
        df = match_header_cols(df, header_DeliverData_temp)

        # Fill YQ column with the last value of year_quarters
        df["YQ"] = year_quarters[-1]
        print("  - Fill value in 'YQ' column")

        # Clean and then copy the "Start Date" column to "End Date"
        if "Start Date" in df.columns:
            df["End Date"] = df["Start Date"]
            print("  - Copied 'Start Date' to 'End Date'")
        
        # Create new fixed-value / static columns
        df["Category"] = "Online-Daily-Platform"
        df["Does Media Cost include Production Fees?"] = "No"

        if "Campaign Taxonomy" in df.columns:
            # Check each unique Campaign Taxonomy for any issues
            # campaign_taxonomy_issues_cache = {taxonomy: detect_invalid_taxonomy(taxonomy, required_campaign_taxonomy_keys) for taxonomy in df['Campaign Taxonomy'].drop_duplicates()} # Only store unique taxonomy strings
            # # Print out the issues in the console
            # print("  - Checking Campaign Taxonomy for any issues...")
            # for taxonomy, (invalid_keys, duplicate_keys) in campaign_taxonomy_issues_cache.items():
            #     if invalid_keys:
            #         print(f"    > Detected: {taxonomy}")
            #         print(f"      > Missing keys: {', '.join(invalid_keys)}")
            #         if duplicate_keys:
            #             print(f"      > Duplicate keys: {', '.join(duplicate_keys)}")
            # Create a new column inside the df for taxonomy issues
            # df['Campaign_Taxonomy_Issues'] = df['Campaign Taxonomy'].apply(lambda x: write_taxonomy_issues(x, campaign_taxonomy_issues_cache))
            # Move the taxonomy issues column after the main taxonomy column
            # df = move_col_after(df, 'Campaign_Taxonomy_Issues', 'Campaign Taxonomy')
            # print("  - Added 'Campaign_Taxonomy_Issues'")

            # Extract "Plan ID" from "Campaign Taxonomy": value between "_MT~" and "_"
            df["Plan ID"] = df["Campaign Taxonomy"].str.extract(r'_MT~(.*?)_', expand=False).str.strip().fillna("").astype(str)
            print("  - Extracted 'Plan ID' from 'Campaign Taxonomy'")

            # Extract "Campaign" from "Campaign Taxonomy": value between "_CN~" and "_"
            df["Campaign"] = df["Campaign Taxonomy"].str.extract(r'_CN~(.*?)_', expand=False).str.strip().fillna("").astype(str)
            print("  - Extracted 'Campaign' from 'Campaign Taxonomy'")

            # Extract "Vehicle Name" from "Campaign Taxonomy": value after "_U3~"
            df["Vehicle Name"] = df["Campaign Taxonomy"].str.extract(r'_U3~(.*)', expand=False).str.strip().fillna("FAULTY").astype(str)
            print("  - Extracted 'Vehicle Name' from 'Campaign Taxonomy' 1st time")

            df["Country"] = df["Campaign Taxonomy"].apply(lambda x: map_code_to_text(x, country_mapping))
            print("  - Extracted 'Country' from 'Campaign Taxonomy'")

            df["Funding Source"] = df["Campaign Taxonomy"].apply(lambda x: map_code_to_text(x, funding_source_mapping))
            print("  - Extracted 'Funding Source' from 'Campaign Taxonomy'")

            df["Initiative"] = df["Campaign Taxonomy"].apply(lambda x: map_code_to_text(x, initiative_mapping))
            print("  - Extracted 'Initiative' from 'Campaign Taxonomy'")

            df["Profitable Consumer Behaviour"] = df["Campaign Taxonomy"].apply(lambda x: map_code_to_text(x, profitable_consumer_behaviour_mapping))
            print("  - Extracted 'Profitable Consumer Behaviour' from 'Campaign Taxonomy'")

            df["Product Message"] = df["Campaign Taxonomy"].apply(lambda x: map_code_to_text(x, product_message_mapping))
            print("  - Extracted 'Product Message' from 'Campaign Taxonomy'")

            df["Destination"] = df["Campaign Taxonomy"].apply(lambda x: map_code_to_text(x, destination_mapping))
            print("  - Extracted 'Destination' from 'Campaign Taxonomy'")

            df["Sponsorship"] = df["Campaign Taxonomy"].apply(lambda x: map_code_to_text(x, sponsorship_mapping))
            print("  - Extracted 'Sponsorship' from 'Campaign Taxonomy'")

            df["Passion Pillar (Campaign-Level)"] = df["Campaign Taxonomy"].apply(lambda x: map_code_to_text(x, passion_pillar_campaign_mapping))
            print("  - Extracted 'Passion Pillar (Campaign-Level)' from 'Campaign Taxonomy'")

            df["Initiative (Campaign-Level)"] = df["Campaign Taxonomy"].apply(lambda x: map_code_to_text(x, initiative_campaign_mapping))
            print("  - Extracted 'Initiative (Campaign-Level)' from 'Campaign Taxonomy'")

        if "Placement Taxonomy" in df.columns:
            # Check each unique Placement Taxonomy for any issues
            # placement_taxonomy_issues_cache = {taxonomy: detect_invalid_taxonomy(taxonomy, required_placement_taxonomy_keys) for taxonomy in df['Placement Taxonomy'].drop_duplicates()}
            # # Print out the issues in the console
            # print("  - Checking Placement Taxonomy for any issues")
            # for taxonomy, (invalid_keys, duplicate_keys) in placement_taxonomy_issues_cache.items():
            #     if invalid_keys:
            #         print(f"    > Detected: {taxonomy}")
            #         print(f"      > Missing keys: {', '.join(invalid_keys)}")
            #         if duplicate_keys:
            #             print(f"      > Duplicate keys: {', '.join(duplicate_keys)}")
            # Create a new column inside the df for taxonomy issues
            # df['Placement_Taxonomy_Issues'] = df['Placement Taxonomy'].apply(lambda x: write_taxonomy_issues(x, placement_taxonomy_issues_cache))
            # Move the taxonomy issues column after the main taxonomy column
            # df = move_col_after(df, 'Placement_Taxonomy_Issues', 'Placement Taxonomy')
            # print("  - Added 'Placement_Taxonomy_Issues'")

            # If Vehicle Name = FAULTY, Extract "Vehicle Name" from "Placement Taxonomy": value between "_PD~" and "_"
            mask_u3_veh = df["Vehicle Name"] == "FAULTY"
            if mask_u3_veh.any():
                df.loc[mask_u3_veh, "Vehicle Name"] = df.loc[mask_u3_veh, "Placement Taxonomy"].str.extract(r'_PD~(.*?)_', expand=False).str.strip().fillna("").astype(str)
                print("  - Extracted 'Vehicle Name' from 'Placement Taxonomy' 2nd time")

            # Extract "Campaign Objective" from "Placement Taxonomy" based on campaignobjective_mapping
            df["Campaign Objective"] = df["Placement Taxonomy"].apply(lambda x: map_code_to_text(x, campaignobjective_mapping))
            print("  - Extracted 'Campaign Objective' from 'Placement Taxonomy'")

            # Extract "Cohorts" from "Placement Taxonomy" based on cohorts_mapping
            df["Cohorts"] = df["Placement Taxonomy"].apply(lambda x: map_code_to_text(x, cohorts_mapping))
            print("  - Extracted 'Cohorts' from 'Placement Taxonomy'")

            # Extract "Publisher/Vendor" from "Placement Taxonomy" based on publisher_mapping. Skip if file name contains Meta
            mask_pub_meta = ~df["File Name"].str.lower().str.contains('meta', na=False, regex=True)
            df.loc[mask_pub_meta, "Publisher/Vendor"] = df.loc[mask_pub_meta, "Placement Taxonomy"].apply(lambda x: map_code_to_text(x, publisher_mapping))
            print("  - Extracted 'Publisher/Vendor' from 'Placement Taxonomy'")

            # Extract "Media Channel" from "Placement Taxonomy" based on media_channel_mapping
            df["Media Channel"] = df["Placement Taxonomy"].apply(lambda x: map_code_to_text(x, media_channel_mapping))
            print("  - Extracted 'Media Channel' from 'Placement Taxonomy'")

            # Extract "Inventory (Distribution) Type" from "Placement Taxonomy" based on inventory_distribution_type_mapping
            df["Inventory (Distribution) Type"] = df["Placement Taxonomy"].apply(lambda x: map_code_to_text(x, inventory_distribution_type_mapping))
            print("  - Extracted 'Inventory (Distribution) Type' from 'Placement Taxonomy'")

            # Extract "Primary Target Strategy" from "Placement Taxonomy" based on primary_target_strategy_mapping
            df["Primary Target Strategy"] = df["Placement Taxonomy"].apply(lambda x: map_code_to_text(x, primary_target_strategy_mapping))
            print("  - Extracted 'Primary Target Strategy' from 'Placement Taxonomy'")

            df["Promotion Incentive"] = df["Placement Taxonomy"].apply(lambda x: map_code_to_text(x, promotion_incentive_mapping))
            print("  - Extracted 'Promotion Incentive' from 'Placement Taxonomy'")

            # df["Passion Pillar (Placement-Level)"] = df["Placement Taxonomy"].str.extract(r'_U1~(.*?)_', expand=False).str.strip().fillna("").astype(str)
            df["Passion Pillar (Placement-Level)"] = df["Placement Taxonomy"].apply(lambda x: map_code_to_text(x, passion_pillar_placement_mapping))
            print("  - Extracted 'Passion Pillar (Placement-Level)' from 'Placement Taxonomy'")

            # df["Initiative (Placement-Level)"] = df["Placement Taxonomy"].str.extract(r'_U2~(.*?)_', expand=False).str.strip().fillna("").astype(str)
            df["Initiative (Placement-Level)"] = df["Placement Taxonomy"].apply(lambda x: map_code_to_text(x, initiative_placement_mapping))
            print("  - Extracted 'Initiative (Placement-Level)' from 'Placement Taxonomy'")
        
        if "Creative Taxonomy" in df.columns:
            df["Passion Pillar (Creative-Level)"] = df["Creative Taxonomy"].apply(lambda x: map_code_to_text(x, passion_pillar_creative_mapping))
            df["Passion Pillar (Creative-Level)"] = df["Passion Pillar (Creative-Level)"].str.strip().fillna("FAULTY").astype(str)
            print("  - Extracted 'Passion Pillar (Creative-Level)' from 'Creative Taxonomy'")

            df["Initiative (Creative-Level)"] = df["Creative Taxonomy"].apply(lambda x: map_code_to_text(x, initiative_creative_mapping))
            df["Initiative (Creative-Level)"] = df["Initiative (Creative-Level)"].str.strip().fillna("FAULTY").astype(str)
            print("  - Extracted 'Initiative (Creative-Level)' from 'Creative Taxonomy'")

            # 
            mask_u1_cre = (df["File Name"].str.lower().str.contains('linkedin|google ads|dv360', na=False, regex=True)) & (df["Passion Pillar (Creative-Level)"] == "FAULTY")
            if mask_u1_cre.any():
                df.loc[mask_u1_cre, "Passion Pillar (Creative-Level)"] = df.loc[mask_u1_cre, "Passion Pillar (Placement-Level)"]
                print("  - Extracted 'Passion Pillar (Creative-Level)' from 'Placement Taxonomy'")

            #
            mask_u2_crea = (df["File Name"].str.lower().str.contains('linkedin|google ads|dv360', na=False, regex=True)) & (df["Initiative (Creative-Level)"] == "FAULTY")
            if mask_u2_crea.any():
                df.loc[mask_u2_crea, "Initiative (Creative-Level)"] = df.loc[mask_u2_crea, "Initiative (Placement-Level)"]
                print("  - Extracted 'Initiative (Creative-Level)' from 'Placement Taxonomy'")
        
        # Case-insensitive rename for Publisher
        rename_list_publisher = {"facebook": "Meta Facebook", "instagram": "Meta Instagram", "instagram.com": "Meta Instagram", "messenger": "Meta Messenger", "audience network": "Meta Audience Network"}
        for previous_val, new_val in rename_list_publisher.items():
            mask_pub_rename = df["Publisher/Vendor"].fillna("").str.lower() == previous_val
            if mask_pub_rename.any():
                df.loc[mask_pub_rename, "Publisher/Vendor"] = new_val
                print(f"  - Renamed {previous_val} to {new_val} for Publisher")
        
        # 
        mask_veh_rename1 = df["Vehicle Name"].str.lower().str.contains("demand gen", na=False, regex=True)
        if mask_veh_rename1.any():
            df.loc[mask_veh_rename1, "Vehicle Name"] = "Demand Gen"
            df.loc[mask_veh_rename1, "Media Channel"] = "OTHER (I.E. EVENT SIGN.)"
            print(f"  - Renamed string contains demand gen to only Demand Gen for Vehicle Name")
        
        #
        mask_veh_rename2 = df["Vehicle Name"].str.lower().str.contains(r"\byt\b|youtube", na=False, regex=True)
        if mask_veh_rename2.any():
            df.loc[mask_veh_rename2, "Vehicle Name"] = "YouTube"
            print(f"  - Renamed yt|youtube to YouTube for Vehicle Name")

        mask_veh_rename3 = ~mask_veh_rename1 & ~mask_veh_rename2
        if mask_veh_rename3.any():
            df.loc[mask_veh_rename3, "Vehicle Name"] = ""
            print(f"  - Clear other values not Demand Gen and not Youtube")

        # Case-insensitive rename for Media Channel
        rename_list_medcha = {"display": "Digital Display (Non-Paid Social)", "video": "Digital Video (Non-CTV, OTT, Paid Social)"}
        for previous_val, new_val in rename_list_medcha.items():
            mask_medcha_rename = df["Media Channel"].str.lower() == previous_val
            if mask_medcha_rename.any():
                df.loc[mask_medcha_rename, "Media Channel"] = new_val
                print(f"  - Renamed {previous_val} to {new_val} for Media Channel")

        # Delete all Platform data for CN, JP
        df = df[~(df["Country"].isin(["China", "Japan"]))].reset_index(drop=True)

        # Only keep rows with Paid Impressions more than 10
        before_paid_imp = len(df)
        df = df[df["Paid Impressions"] > 10]
        after_paid_imp = len(df)
        print(f"  - From {before_paid_imp} rows drop down to {after_paid_imp} rows")

        # Drop rows that are completely empty
        df = df.dropna(how='all')
        print("  - Deleted all fully-empty-rows")

        # Check and correct the Platform Currency
        df = df.apply(convert_currency_platform, axis=1)

        # Drop File Name to return original header_DeliverData
        df = df.drop(columns=['File Name'])
        print("  - Dropped File Name column")

        # Save Type 1a: Brand new xlsx with no format
        df.to_excel(platform_cleaned_output_file, index=False)
        print(f"\n~ Saved to '{platform_cleaned_output_file}'")
    except Exception as e:
        print(f"\n> Error processing '{platform_consolidation_output_file}' : {e}")

### 4. Clean UniqueCloud3

In [ ]:
# Clean the UniqueCloud3 file
def clean_UniqueCloud3():
    print("\n─────➤ 4. Clean UniqueCloud3")

    # Set up the input file and output file names
    UniqueCloud3_input_files = glob.glob("*UniqueCloud3_APAC_*.xlsx")
    UniqueCloud3_input_file = UniqueCloud3_input_files[0]
    UniqueCloud3_output_file = "UniqueCloud3_Cleaned_Output.xlsx"

    # If UniqueCloud3 files are not found, skip the cleaning process
    if not os.path.exists(UniqueCloud3_input_file):
        print(f"\n> No UniqueCloud3 file found for cleaning")
        return

    # Required keywords to detect the header row
    required_keywords = ["Advertiser", "Site (UniqueCloud3)", "Campaign", "Campaign ID", "Placement", "Placement ID"] # Add more keywords as needed
    
    print(f"\n+ Processing file '{UniqueCloud3_input_file}'")
    try:
        # Read the file without header to detect the header row
        df_raw = pd.read_excel(UniqueCloud3_input_file, header=None, engine='openpyxl', keep_default_na=False, na_values=[''])

        # Detect the header row based on required keywords
        header_row = detect_header_row(df_raw, required_keywords)

        if header_row is None:
            print(f"    > Header row not found in '{UniqueCloud3_input_file}', skipping")
            return
        print(f"  - Found header row at {header_row+1}")
        
        # Re-read the file using the detected header row
        df = pd.read_excel(UniqueCloud3_input_file, header=header_row, engine='openpyxl', keep_default_na=False, na_values=[''])

        # Rename specific columns
        rename_mapping = {
            "Site (UniqueCloud3)": "Publisher/Vendor",
            "Date": "Start Date",
            "Campaign": "Campaign Taxonomy",
            "Placement": "Placement Taxonomy",
            "Creative": "Creative Taxonomy",
            "Video Completions": "Video Completes",
            "Impressions": "Paid Impressions",
        }
        df.rename(columns=rename_mapping, inplace=True)
        print("  - Renamed some column headers")

        # Match the current df to DeliverData header
        df = match_header_cols(df, header_DeliverData)
        
        # Remove rows that contain "Grand Total" in any cell in one column (case-insensitive)
        df = df[~df.apply(check_contain_any, args=(["Grand Total"], ["Start Date"]), axis=1)]
        print("  - Removed 'Grand Total:' rows")

        # Delete Site Name in the exclusion list
        df = df[~(df["Publisher/Vendor"].isin(["Facebook", "Instagram", "instagram.com", "TikTok", "Linkedin", "Precision", "The Trade Desk"]))].reset_index(drop=True)

        # Fill YQ column with the last value of year_quarters
        df["YQ"] = year_quarters[-1]
        print("  - Fill value in 'YQ' column")

        # Clean and then copy the "Start Date" column to "End Date"
        if "Start Date" in df.columns:
            df["End Date"] = df["Start Date"]
            print("  - Copied 'Start Date' to 'End Date'")

        # Create new fixed-value / static columns
        df["Category"] = "Online-Daily-UniqueCloud3"

        # Set all values in the "Media Cost (LC)" column to zero
        if "Media Cost (LC)" in df.columns:
            df["Media Cost (LC)"] = 0
            print("  - Set 'Media Cost (LC)' to 0")

        if "Campaign Taxonomy" in df.columns:
            # Check each unique Campaign Taxonomy for any issues
            campaign_taxonomy_issues_cache = {taxonomy: detect_invalid_taxonomy(taxonomy, required_campaign_taxonomy_keys) for taxonomy in df['Campaign Taxonomy'].drop_duplicates()} # Only store unique taxonomy strings
            # Print out the issues in the console
            print("  - Checking Campaign Taxonomy for any issues...")
            for taxonomy, (invalid_keys, duplicate_keys) in campaign_taxonomy_issues_cache.items():
                if invalid_keys:
                    print(f"    > Detected: {taxonomy}")
                    print(f"      > Missing keys: {', '.join(invalid_keys)}")
                    if duplicate_keys:
                        print(f"      > Duplicate keys: {', '.join(duplicate_keys)}")
            # Create a new column inside the df for taxonomy issues
            # df['Campaign_Taxonomy_Issues'] = df['Campaign Taxonomy'].apply(lambda x: write_taxonomy_issues(x, campaign_taxonomy_issues_cache))
            # Move the taxonomy issues column after the main taxonomy column
            # df = move_col_after(df, 'Campaign_Taxonomy_Issues', 'Campaign Taxonomy')
            # print("  - Added 'Campaign_Taxonomy_Issues'")

            # Extract "Plan ID" from "Campaign Taxonomy": value between "_MT~" and "_"
            df["Plan ID"] = df["Campaign Taxonomy"].str.extract(r'_MT~(.*?)_', expand=False).str.strip().fillna("").astype(str)
            print("  - Extracted 'Plan ID' from 'Campaign Taxonomy'")

            # Extract "Campaign" from "Campaign Taxonomy": value between "_CN~" and "_"
            df["Campaign"] = df["Campaign Taxonomy"].str.extract(r'_CN~(.*?)_', expand=False).str.strip().fillna("").astype(str)
            print("  - Extracted 'Campaign' from 'Campaign Taxonomy'")

            # Extract "Vehicle Name" from "Campaign Taxonomy": value after "_U3~"
            df["Vehicle Name"] = df["Campaign Taxonomy"].str.extract(r'_U3~(.*)', expand=False).str.strip().fillna("FAULTY").astype(str)
            print("  - Extracted 'Vehicle Name' from 'Campaign Taxonomy' 1st time")

            df["Country"] = df["Campaign Taxonomy"].apply(lambda x: map_code_to_text(x, country_mapping))
            print("  - Extracted 'Country' from 'Campaign Taxonomy'")

            df["Funding Source"] = df["Campaign Taxonomy"].apply(lambda x: map_code_to_text(x, funding_source_mapping))
            print("  - Extracted 'Funding Source' from 'Campaign Taxonomy'")

            df["Initiative"] = df["Campaign Taxonomy"].apply(lambda x: map_code_to_text(x, initiative_mapping))
            print("  - Extracted 'Initiative' from 'Campaign Taxonomy'")

            df["Profitable Consumer Behaviour"] = df["Campaign Taxonomy"].apply(lambda x: map_code_to_text(x, profitable_consumer_behaviour_mapping))
            print("  - Extracted 'Profitable Consumer Behaviour' from 'Campaign Taxonomy'")

            df["Product Message"] = df["Campaign Taxonomy"].apply(lambda x: map_code_to_text(x, product_message_mapping))
            print("  - Extracted 'Product Message' from 'Campaign Taxonomy'")

            df["Destination"] = df["Campaign Taxonomy"].apply(lambda x: map_code_to_text(x, destination_mapping))
            print("  - Extracted 'Destination' from 'Campaign Taxonomy'")

            df["Sponsorship"] = df["Campaign Taxonomy"].apply(lambda x: map_code_to_text(x, sponsorship_mapping))
            print("  - Extracted 'Sponsorship' from 'Campaign Taxonomy'")

            df["Passion Pillar (Campaign-Level)"] = df["Campaign Taxonomy"].apply(lambda x: map_code_to_text(x, passion_pillar_campaign_mapping))
            print("  - Extracted 'Passion Pillar (Campaign-Level)' from 'Campaign Taxonomy'")

            df["Initiative (Campaign-Level)"] = df["Campaign Taxonomy"].apply(lambda x: map_code_to_text(x, initiative_campaign_mapping))
            print("  - Extracted 'Initiative (Campaign-Level)' from 'Campaign Taxonomy'")

        if "Placement Taxonomy" in df.columns:
            # Check each unique Placement Taxonomy for any issues
            placement_taxonomy_issues_cache = {taxonomy: detect_invalid_taxonomy(taxonomy, required_placement_taxonomy_keys) for taxonomy in df['Placement Taxonomy'].drop_duplicates()}
            # Print out the issues in the console
            print("  - Checking Placement Taxonomy for any issues")
            for taxonomy, (invalid_keys, duplicate_keys) in placement_taxonomy_issues_cache.items():
                if invalid_keys:
                    print(f"    > Detected: {taxonomy}")
                    print(f"      > Missing keys: {', '.join(invalid_keys)}")
                    if duplicate_keys:
                        print(f"      > Duplicate keys: {', '.join(duplicate_keys)}")
            # Create a new column inside the df for taxonomy issues
            # df['Placement_Taxonomy_Issues'] = df['Placement Taxonomy'].apply(lambda x: write_taxonomy_issues(x, placement_taxonomy_issues_cache))
            # Move the taxonomy issues column after the main taxonomy column
            # df = move_col_after(df, 'Placement_Taxonomy_Issues', 'Placement Taxonomy')
            # print("  - Added 'Placement_Taxonomy_Issues'")

            # If Vehicle Name = FAULTY, Extract "Vehicle Name" from "Placement Taxonomy": value between "_PD~" and "_"
            mask_u3_veh = df["Vehicle Name"] == "FAULTY"
            if mask_u3_veh.any():
                df.loc[mask_u3_veh, "Vehicle Name"] = df.loc[mask_u3_veh, "Placement Taxonomy"].str.extract(r'_PD~(.*?)_', expand=False).str.strip().fillna("").astype(str)
                print("  - Extracted 'Vehicle Name' from 'Placement Taxonomy' 2nd time")

            # Extract "Campaign Objective" from "Placement Taxonomy" based on campaignobjective_mapping
            df["Campaign Objective"] = df["Placement Taxonomy"].apply(lambda x: map_code_to_text(x, campaignobjective_mapping))
            print("  - Extracted 'Campaign Objective' from 'Placement Taxonomy'")
                
            # Extract "Cohorts" from "Placement Taxonomy" based on cohorts_mapping
            df["Cohorts"] = df["Placement Taxonomy"].apply(lambda x: map_code_to_text(x, cohorts_mapping))
            print("  - Extracted 'Cohorts' from 'Placement Taxonomy'")

            # Extract "Publisher/Vendor" from "Placement Taxonomy" based on publisher_mapping
            df["Publisher/Vendor"] = df["Placement Taxonomy"].apply(lambda x: map_code_to_text(x, publisher_mapping))
            print("  - Extracted 'Publisher/Vendor' from 'Placement Taxonomy'")

            # Extract "Media Channel" from "Placement Taxonomy" based on media_channel_mapping
            df["Media Channel"] = df["Placement Taxonomy"].apply(lambda x: map_code_to_text(x, media_channel_mapping))
            print("  - Extracted 'Media Channel' from 'Placement Taxonomy'")

            # Extract "Inventory (Distribution) Type" from "Placement Taxonomy" based on inventory_distribution_type_mapping
            df["Inventory (Distribution) Type"] = df["Placement Taxonomy"].apply(lambda x: map_code_to_text(x, inventory_distribution_type_mapping))
            print("  - Extracted 'Inventory (Distribution) Type' from 'Placement Taxonomy'")

            # Extract "Primary Target Strategy" from "Placement Taxonomy" based on primary_target_strategy_mapping
            df["Primary Target Strategy"] = df["Placement Taxonomy"].apply(lambda x: map_code_to_text(x, primary_target_strategy_mapping))
            print("  - Extracted 'Primary Target Strategy' from 'Placement Taxonomy'")

            df["Promotion Incentive"] = df["Placement Taxonomy"].apply(lambda x: map_code_to_text(x, promotion_incentive_mapping))
            print("  - Extracted 'Promotion Incentive' from 'Placement Taxonomy'")

            df["Passion Pillar (Placement-Level)"] = df["Placement Taxonomy"].apply(lambda x: map_code_to_text(x, passion_pillar_placement_mapping))
            print("  - Extracted 'Passion Pillar (Placement-Level)' from 'Placement Taxonomy'")

            df["Initiative (Placement-Level)"] = df["Placement Taxonomy"].apply(lambda x: map_code_to_text(x, initiative_placement_mapping))
            print("  - Extracted 'Initiative (Placement-Level)' from 'Placement Taxonomy'")
        
        if "Creative Taxonomy" in df.columns:
            df["Passion Pillar (Creative-Level)"] = df["Creative Taxonomy"].apply(lambda x: map_code_to_text(x, passion_pillar_creative_mapping))
            df["Passion Pillar (Creative-Level)"] = df["Passion Pillar (Creative-Level)"].str.strip().fillna("").astype(str)
            print("  - Extracted 'Passion Pillar (Creative-Level)' from 'Creative Taxonomy'")

            df["Initiative (Creative-Level)"] = df["Creative Taxonomy"].apply(lambda x: map_code_to_text(x, initiative_creative_mapping))
            df["Initiative (Creative-Level)"] = df["Initiative (Creative-Level)"].str.strip().fillna("").astype(str)
            print("  - Extracted 'Initiative (Creative-Level)' from 'Creative Taxonomy'")
        
        # Case-insensitive rename for Publisher
        rename_list_publisher = {"facebook": "Meta Facebook", "instagram": "Meta Instagram", "instagram.com": "Meta Instagram", "messenger": "Meta Messenger", "audience network": "Meta Audience Network"}
        for previous_val, new_val in rename_list_publisher.items():
            mask_pub_rename = df["Publisher/Vendor"].fillna("").str.lower() == previous_val
            if mask_pub_rename.any():
                df.loc[mask_pub_rename, "Publisher/Vendor"] = new_val
                print(f"  - Renamed {previous_val} to {new_val} for Publisher")
        
        # 
        mask_veh_rename1 = df["Vehicle Name"].str.lower().str.contains("demand gen", na=False, regex=True)
        if mask_veh_rename1.any():
            df.loc[mask_veh_rename1, "Vehicle Name"] = "Demand Gen"
            df.loc[mask_veh_rename1, "Media Channel"] = "OTHER (I.E. EVENT SIGN.)"
            print(f"  - Renamed string contains demand gen to only Demand Gen for Vehicle Name")
        
        #
        mask_veh_rename2 = df["Vehicle Name"].str.lower().str.contains(r"\byt\b|youtube", na=False, regex=True)
        if mask_veh_rename2.any():
            df.loc[mask_veh_rename2, "Vehicle Name"] = "YouTube"
            print(f"  - Renamed yt|youtube to YouTube for Vehicle Name")

        mask_veh_rename3 = ~mask_veh_rename1 & ~mask_veh_rename2
        if mask_veh_rename3.any():
            df.loc[mask_veh_rename3, "Vehicle Name"] = ""
            print(f"  - Clear other values not Demand Gen and not Youtube")
        
        # Case-insensitive rename for Media Channel
        rename_list_medcha = {"display": "Digital Display (Non-Paid Social)", "video": "Digital Video (Non-CTV, OTT, Paid Social)"}
        for previous_val, new_val in rename_list_medcha.items():
            mask_medcha_rename = df["Media Channel"].str.lower() == previous_val
            if mask_medcha_rename.any():
                df.loc[mask_medcha_rename, "Media Channel"] = new_val
                print(f"  - Renamed {previous_val} to {new_val} for Media Channel")

        # Delete all UniqueCloud3 data for CN, JP, KR
        df = df[~(df["Country"].isin(["China", "Japan", "Korea"]))].reset_index(drop=True)

        # Clear Vehicle Name values that duplicate with Publisher 
        df.loc[df["Vehicle Name"] == df["Publisher/Vendor"], "Vehicle Name"] = ""

        # Only keep rows with Paid Impressions more than 10
        before_paid_imp = len(df)
        df = df[df["Paid Impressions"] > 10]
        after_paid_imp = len(df)
        print(f"  - From {before_paid_imp} rows drop down to {after_paid_imp} rows")

        # Drop rows that are completely empty
        df = df.dropna(how='all')
        print("  - Deleted all fully-empty-rows")

        # Save Type 1a: Brand new xlsx with no format
        df.to_excel(UniqueCloud3_output_file, index=False)
        print(f"\n~ Saved to '{UniqueCloud3_output_file}'")
    except Exception as e:
        print(f"\n> Error processing {UniqueCloud3_input_file} : {e}")

### 5. Clean UniqueCloud2

In [ ]:
# Clean the Raw Extract files from UniqueCloud2 Dashboard
def clean_UniqueCloud2():
    print("\n─────➤ 5. Clean UniqueCloud2")

    # Find all Raw Extract Excel files in the current directory
    file_paths = glob.glob("Raw Extract*.xlsx")
    if not file_paths:
        print("\n> No Raw Extract files found, aborting")
        return
    
    # Set up the output file
    UniqueCloud2_output_file = "UniqueCloud2_Cleaned_Output.xlsx"

    # Initialize empty list to collect Online UniqueCloud2 DataFrames
    df_list = []

    # Required keywords to detect the UniqueCloud2 Online header row
    online_required_keywords = ["Media Cost (LC)", "Media Cost (USD)", "Paid Impressions", "Clicks", "Video Views", "Video Completes"] 

    for file_path in file_paths:
        df_temp = pd.read_excel(file_path, header=4, engine='openpyxl', keep_default_na=False, na_values=[''])
        header_current = df_temp.columns.tolist()
        
        # Check if all required keywords are in the current header
        if all(keyword in header_current for keyword in online_required_keywords):
            df_list.append(df_temp)
            print(f"  - Detected UniqueCloud2 Online {file_path}")
        else:
            print(f"  - Skipped {file_path}")
    
    # Combine all DataFrames
    if df_list:
        df = pd.concat(df_list, ignore_index=True)
        print(f"  - Combined {len(df_list)} files into DataFrame with {len(df)} rows")
    df.rename(columns={"Data Stream": "Category"}, inplace=True)

    # Match the current df to DeliverData header
    df = match_header_cols(df, header_DeliverData)

    # Rename Net New data stream into Online-Daily-Other
    if (df["Category"] == "UniqueDataStream1").any():
        df["Category"] = df["Category"].replace("UniqueDataStream1", "Online-Daily-Other")
        other_row_count = (df["Category"] == "Online-Daily-Other").sum()
        print(f"  - Renamed {other_row_count:,} Net New rows into 'Online-Daily-Other'")

    # Delete Online-Daily-Other rows since we don't need this at the start
    if (df["Category"] == "Online-Daily-Other").any():
        df = df[df["Category"] != "Online-Daily-Other"]
        print(f"  - Detected and deleted {other_row_count:,} 'Online-Daily-Other' rows")

    # Rename DFA data stream to Online-Daily-UniqueCloud3
    if (df["Category"] == "UniqueDataStream2").any():
        df["Category"] = df["Category"].replace("UniqueDataStream2", "Online-Daily-UniqueCloud3")
        UniqueCloud3_row_count = (df["Category"] == "Online-Daily-UniqueCloud3").sum()
        print(f"  - Renamed {UniqueCloud3_row_count:,} DFA rows into 'Online-Daily-UniqueCloud3'")

    # Change all other categories to Online-Daily-Platform
    if (~df["Category"].isin(["Online-Daily-Other", "Online-Daily-UniqueCloud3"])).any():
        df.loc[~df["Category"].isin(["Online-Daily-Other", "Online-Daily-UniqueCloud3"]), "Category"] = "Online-Daily-Platform"
        df.loc[~df["Category"].isin(["Online-Daily-Other", "Online-Daily-UniqueCloud3"]), "Does Media Cost include Production Fees?"] = "No"
        platform_row_count = (df["Category"] == "Online-Daily-Platform").sum()
        print(f"  - Renamed {platform_row_count:,} other rows into 'Online-Daily-Platform'")

    # Remove rows that contain "Totals" in any cell in one column (case-insensitive)
    df = df[~df.apply(check_contain_any, args=(["Totals"], ["Country"]), axis=1)]
    print("  - Removed 'Totals' rows")

    # This builds two masks, combines them with | (OR), then uses ~ to invert and keep only the rows that don't match either condition. The reset_index(drop=True) just cleans up the index after deletion
    mask_drop_platform = (df["Category"] == "Online-Daily-Platform") & (df["Country"].isin(["China", "Japan"]))
    mask_drop_UniqueCloud3 = (df["Category"] == "Online-Daily-UniqueCloud3") & (df["Country"].isin(["China", "Japan", "Korea"])) 
    df = df[~(mask_drop_platform | mask_drop_UniqueCloud3)].reset_index(drop=True)

    if "Campaign Taxonomy" in df.columns:
        # Check each unique Campaign Taxonomy for any issues
        campaign_taxonomy_issues_cache = {taxonomy: detect_invalid_taxonomy(taxonomy, required_campaign_taxonomy_keys) for taxonomy in df['Campaign Taxonomy'].drop_duplicates()} # Only store unique taxonomy strings
        # Print out the issues in the console
        print("  - Checking Campaign Taxonomy for any issues...")
        for taxonomy, (invalid_keys, duplicate_keys) in campaign_taxonomy_issues_cache.items():
            if invalid_keys:
                print(f"    > Detected: {taxonomy}")
                print(f"      > Missing keys: {', '.join(invalid_keys)}")
                if duplicate_keys:
                    print(f"      > Duplicate keys: {', '.join(duplicate_keys)}")

    if "Placement Taxonomy" in df.columns:
        # Check each unique Placement Taxonomy for any issues
        placement_taxonomy_issues_cache = {taxonomy: detect_invalid_taxonomy(taxonomy, required_placement_taxonomy_keys) for taxonomy in df['Placement Taxonomy'].drop_duplicates()}
        # Print out the issues in the console
        print("  - Checking Placement Taxonomy for any issues")
        for taxonomy, (invalid_keys, duplicate_keys) in placement_taxonomy_issues_cache.items():
            if invalid_keys:
                print(f"    > Detected: {taxonomy}")
                print(f"      > Missing keys: {', '.join(invalid_keys)}")
                if duplicate_keys:
                    print(f"      > Duplicate keys: {', '.join(duplicate_keys)}")

        # Clear Vehicle Name values that duplicate with Publisher 
        df.loc[df["Vehicle Name"] == df["Publisher/Vendor"], "Vehicle Name"] = ""

    # Change Media Cost LC to zero for only Category Online-Daily-UniqueCloud3
    UniqueCloud3_mask = df["Category"] == "Online-Daily-UniqueCloud3"
    if UniqueCloud3_mask.any():
        df.loc[UniqueCloud3_mask, "Media Cost (LC)"] = 0
        print("  - Set 'Media Cost (LC)' to zero for 'Online-Daily-UniqueCloud3' rows")

    # Only keep rows with Paid Impressions more than 10
    before_paid_imp = len(df)
    df = df[df["Paid Impressions"] > 10]
    after_paid_imp = len(df)
    print(f"  - From {before_paid_imp} rows drop down to {after_paid_imp} rows")

    # Drop rows that are completely empty
    df = df.dropna(how='all')
    print("  - Deleted all fully-empty-rows")

    # Save Type 1a: Brand new xlsx with no format
    df.to_excel(UniqueCloud2_output_file, index=False)
    print(f"\n~ Saved to '{UniqueCloud2_output_file}'")

### 6. Combine into AP

In [ ]:
# For combining all the dataframes back into one AP DeliverData file
def combine_ap():
    print("\n─────➤ 6. Consolidate into AP DeliverData")

    # Set up file names
    platform_output_file = "Platform_Cleaned_Output.xlsx"
    UniqueCloud3_output_file = "UniqueCloud3_Cleaned_Output.xlsx"
    UniqueCloud2_output_file = "UniqueCloud2_Cleaned_Output.xlsx"
    
    ap_DeliverData_list_xlsx = glob.glob("*UniqueClientName - AP - DeliverData - Q*.xlsx")
    # Check that exactly one ap DeliverData file is found
    if len(ap_DeliverData_list_xlsx) != 1:
        print(f"\n> Expected 1 AP DeliverData file, found {len(ap_DeliverData_list_xlsx)}, aborting")
        return
    ap_DeliverData_file = ap_DeliverData_list_xlsx[0]

    if os.path.exists(platform_output_file) and os.path.exists(UniqueCloud3_output_file) and not os.path.exists(UniqueCloud2_output_file):
        # Load the cleaned Platform file
        df_platform = pd.read_excel(platform_output_file, header=0, engine='openpyxl', keep_default_na=False, na_values=[''])
        print(f"+ Loading file '{platform_output_file}'")
        # Load the cleaned UniqueCloud3 file
        df_UniqueCloud3 = pd.read_excel(UniqueCloud3_output_file, header=0, engine='openpyxl', keep_default_na=False, na_values=[''])
        print(f"+ Loading '{UniqueCloud3_output_file}'")
        # Match each DataFrame's columns to the target sheet header order
        df_platform_matched = match_header_cols(df_platform, header_DeliverData)
        df_UniqueCloud3_matched = match_header_cols(df_UniqueCloud3, header_DeliverData)

        # day_first_for_date=False py pandas (YYYY MM DD), day_first_for_text=False (YYYY MM DD)
        df_platform_matched["Start Date"] = df_platform_matched["Start Date"].apply(clean_date_col, args=(False, False))
        df_platform_matched["End Date"] = df_platform_matched["End Date"].apply(clean_date_col, args=(False, False))
        print("  - Cleaned Platform date formatting")

        # day_first_for_date=False py pandas (YYYY MM DD), day_first_for_text=False (YYYY MM DD)
        df_UniqueCloud3_matched["Start Date"] = df_UniqueCloud3_matched["Start Date"].apply(clean_date_col, args=(False, False))
        df_UniqueCloud3_matched["End Date"] = df_UniqueCloud3_matched["End Date"].apply(clean_date_col, args=(False, False))
        print("  - Cleaned UniqueCloud3 date formatting")

        # For the DeliverData sheet, concatenate Platform and UniqueCloud3 data (keeping both sets of rows)
        df_DeliverData_combined = pd.concat([df_platform_matched, df_UniqueCloud3_matched], ignore_index=True)

        # Clean Media Cost (LC) to remove symbol and keep only numbers
        df_DeliverData_combined["Media Cost (LC)"] = (df_DeliverData_combined["Media Cost (LC)"].astype(str).str.replace(r'[^\d.]', '', regex=True).str.replace(r'\.(?=.*\.)', '', regex=True))
        df_DeliverData_combined["Media Cost (LC)"] = pd.to_numeric(df_DeliverData_combined["Media Cost (LC)"], errors='coerce').combine_first(df_DeliverData_combined["Media Cost (LC)"])
        print("  - Cleaned 'Media Cost (LC)'")

    elif not os.path.exists(platform_output_file) and not os.path.exists(UniqueCloud3_output_file) and os.path.exists(UniqueCloud2_output_file):
        # Load the cleaned UniqueCloud2 file
        df_UniqueCloud2 = pd.read_excel(UniqueCloud2_output_file, header=0, engine='openpyxl', keep_default_na=False, na_values=[''])
        print(f"+ Loading file '{UniqueCloud2_output_file}'")
        # Match each DataFrame's columns to the target sheet header order
        df_UniqueCloud2_matched = match_header_cols(df_UniqueCloud2, header_DeliverData)

        # Might need to review once the dashboard is updated
        # day_first_for_date=False py pandas (YYYY MM DD), day_first_for_text=False (YYYY MM DD)
        df_UniqueCloud2_matched["Start Date"] = df_UniqueCloud2_matched["Start Date"].apply(clean_date_col, args=(False, False))
        df_UniqueCloud2_matched["End Date"] = df_UniqueCloud2_matched["End Date"].apply(clean_date_col, args=(False, False))
        print("  - Cleaned UniqueCloud2 date formatting")

        # For the DeliverData sheet, concatenate Platform and UniqueCloud3 data (keeping both sets of rows)
        df_DeliverData_combined = pd.concat([df_UniqueCloud2_matched], ignore_index=True)

        # Clean Media Cost (LC) to remove symbol and keep only numbers
        df_DeliverData_combined["Media Cost (LC)"] = (df_DeliverData_combined["Media Cost (LC)"].astype(str).str.replace(r'[^\d.]', '', regex=True).str.replace(r'\.(?=.*\.)', '', regex=True))
        df_DeliverData_combined["Media Cost (LC)"] = pd.to_numeric(df_DeliverData_combined["Media Cost (LC)"], errors='coerce').combine_first(df_DeliverData_combined["Media Cost (LC)"])
        print("  - Cleaned 'Media Cost (LC)'")
        
    else:
        print(f"\n> Platform={os.path.exists(platform_output_file)}, UniqueCloud3={os.path.exists(UniqueCloud3_output_file)}, UniqueCloud2={os.path.exists(UniqueCloud2_output_file)}, aborting")
        return
    # Save Type 1c: Overwrite existing xlsx with format
    excel_writer_customize(df_DeliverData_combined, ap_DeliverData_file, "DeliverData")

### 7a. Add Macro

In [ ]:
# Add VBA directly into the AP DeliverData output file
"""
Need to access Excel’s underlying COM object model via the .api attribute to manipulate the VBA project
Programmatic access to the VBA project must be enabled in Excel’s Trust Center
Access the VBProject and add a module using VBComponents.Add(1) (where 1 represents a standard module).
Excel often blocks programmatic access to the VBA project for security reasons.
Enabled "Trust access to the VBA project object model" in the Trust Center settings.
After adding macros, must save the workbook in a macro-enabled format (e.g., .xlsm).

The function first checks for an existing macro‑enabled file. 
If one exists, it sets a flag (keep_vba) so that later you only add the VBA code without converting the file format. 
If only an xlsx file exists, the function will add the code and then save a new xlsm file.

Depending on the flag, the workbook is either saved as the existing file (if it's already macro‑enabled) or as a new xlsm file.
"""
def add_macro():
    print("\n─────➤ 7a. Add Macro")
    # Check for macro-enabled files (*.xlsm) first, then (*.xlsx)
    ap_DeliverData_list_xlsm = glob.glob("*UniqueClientName - AP - DeliverData - Q*.xlsm")
    ap_DeliverData_list_xlsx = glob.glob("*UniqueClientName - AP - DeliverData - Q*.xlsx")
    ap_DeliverData_file = None
    keep_vba = False

    if len(ap_DeliverData_list_xlsm) == 1:
        ap_DeliverData_file = ap_DeliverData_list_xlsm[0]
        keep_vba = True
        print(f"\n+ Processing macro-enabled '{ap_DeliverData_file}'")
    elif len(ap_DeliverData_list_xlsx) == 1:
        ap_DeliverData_file = ap_DeliverData_list_xlsx[0]
        print(f"\n+ Processing xlsx '{ap_DeliverData_file}")
    else: 
        total_found = len(ap_DeliverData_list_xlsm) + len(ap_DeliverData_list_xlsx)
        print(f"\n> Expected 1 AP DeliverData file, found {total_found}, aborting")
        return # Stop if the condition is not met
    
    try:
        excel_app = xw.App(visible=True, add_book=False)
        # Run add modules function
        workbook_xlwings = excel_app.books.open(ap_DeliverData_file)
        workbook_xlwings = vb_project(workbook_xlwings)
        # Save the workbook appropriately
        if keep_vba:
            # For xlsm, simply save the changes to the existing file
            workbook_xlwings.save()
            print(f"\n~ Saved to '{ap_DeliverData_file}'")
            # Actually run the Macro scripts
            workbook_xlwings.macro("SubFunc_AddButtons")(True) # True to keep button0
            workbook_xlwings.save()
        else:
            # For xlsx, save a new macro-enabled file
            output_file = ap_DeliverData_file.replace(".xlsx", ".xlsm")
            workbook_xlwings.save(output_file)
            print(f"\n~ Saved to '{output_file}'")
            # Actually run the Macro scripts
            workbook_xlwings.macro("SubFunc_AddButtons")(True) # True to keep button0
            workbook_xlwings.save(output_file)
        # excel_app.quit()
    except Exception as e:
        print(f"\n> Error adding Macro AP DeliverData file: {e}")
        return
    print(f"\n+ Next: manually activate VBA Refresh_UniqueCloud1")

### 7b. Update Macro

In [ ]:
# To easily update VBA changes back into lots of Excel files
def update_macro():
    print("\n─────➤ 7b. Update Macro")
    file_paths = glob.glob("*.xlsm")

    # Build a list of files to remove, using glob for patterns, and listing specific filenames
    excludes = (
        # glob.glob("UniqueClientName - AU - DeliverData - Q*.xlsm") #+ glob.glob("UniqueClientName - CN - DeliverData - Q*.xlsm")
        # ["UniqueClientName - IN - DeliverData - Q3FY24-Q4FY25.xlsm",
        #  "FY25 APAC Report.xlsm"]
    )
    # Filter out the excluded files from the main file list
    file_paths = [file_path for file_path in file_paths if file_path not in excludes]
    
    for file_path in file_paths:
        try:
            excel_app = xw.App(visible=True, add_book=False)
            workbook_xlwings = excel_app.books.open(file_path)
            print(f"\n+ Processing file '{file_path}'")
            workbook_xlwings = vb_project(workbook_xlwings)
            # workbook_xlwings.macro("SubFunc_AddButtons")(True) # True to keep button0
            workbook_xlwings.activate()
            workbook_xlwings.sheets["DeliverData"].activate()
            # workbook_xlwings.macro("Apply_DeliverData_Format")()
            # workbook_xlwings.macro("MT_vs_DeliverData_All")()
            workbook_xlwings.save()
            # xlwings_reapply_format(workbook_xlwings)
            print(f"  - Updated successfully")
            # excel_app.quit()
        except Exception as e:
            print(f"\n> Error reapplying macro : {e}")
        time.sleep(5)

### 8. Get Plan Name

In [ ]:
# Support sub function to clean Plan ID 
def clean_plan_id_value(value):
    # Convert to string and remove whitespace
    string_value = str(value).strip()

    # If string_value is blank or 'nan', return "N/A"
    if not string_value or string_value.lower() == 'nan':
        return "N/A"
    
    # If string_value does not contain any digits, return "N/A"
    if not any(char.isdigit() for char in string_value):
        return "N/A"

    # If string_value contains alphabetical letters, return "N/A"
    if any(char.isalpha() for char in string_value):
        return "N/A"

    # Try to convert to integer numeric values
    try:
        # Extract only digits (removes any remaining special characters)
        # numeric_str = ''.join(char for char in string_value if char.isdigit())
        # if not numeric_str:  # If no digits were found
        #     return "N/A"
        integer_value = int(float(string_value))
        return integer_value
    except (ValueError, TypeError):
        return "N/A"

# To extract "Plan Name" from "Plan ID" from UniqueCloud1 sheet
def get_plan_name():
    print("\n─────➤ 8. Get Plan Name")
    ap_DeliverData_list_xlsm = glob.glob("*UniqueClientName - AP - DeliverData - Q*.xlsm")
    if len(ap_DeliverData_list_xlsm) != 1:
        print(f"\n> Expected 1 AP DeliverData file, found {len(ap_DeliverData_list_xlsm)}, aborting")
        return
    ap_DeliverData_file = ap_DeliverData_list_xlsm[0]
    print(f"\n+ Processing file '{ap_DeliverData_file}'")
    try:
        workbook = load_workbook(ap_DeliverData_file, keep_vba=True)
        if "DeliverData" not in workbook.sheetnames:
            print(f"    > 'DeliverData' sheet not found in {ap_DeliverData_file}, aborting")
            workbook.close()
            return
        if "UniqueCloud1" not in workbook.sheetnames:
            print(f"    > 'UniqueCloud1' sheet not found in {ap_DeliverData_file}, aborting")
            workbook.close()
            return
        worksheet_DeliverData = workbook["DeliverData"]

        df_DeliverData = pd.read_excel(ap_DeliverData_file, sheet_name="DeliverData", engine='openpyxl', keep_default_na=False, na_values=[''])
        df_UniqueCloud1 = pd.read_excel(ap_DeliverData_file, sheet_name="UniqueCloud1", engine='openpyxl', keep_default_na=False, na_values=[''])
        valid_category = ["Online-Daily-Platform", "Online-Daily-UniqueCloud3"]
        mask_valid_category = df_DeliverData["Category"].isin(valid_category)
        df_DeliverData_temp = df_DeliverData[mask_valid_category].copy()
        if df_DeliverData_temp.empty:
            print("\n> No data rows with valid templates found")
            return
        idx_to_update = df_DeliverData_temp.index.tolist()
        
        # Check Plan ID & Plan Name columns
        if 'Plan ID' not in df_UniqueCloud1.columns or 'Plan Name' not in df_UniqueCloud1.columns:
            print("\n> 'Plan ID' or 'Plan Name' column missing in MT")
            return
        
        # Clean MT Plan ID
        df_UniqueCloud1['Plan ID (Clean)'] = df_UniqueCloud1['Plan ID'].apply(clean_plan_id_value)
        
        # Filter out rows with "N/A" Plan ID before building dictionary
        valid_UniqueCloud1 = df_UniqueCloud1[df_UniqueCloud1['Plan ID (Clean)'] != "N/A"].copy()

        # Build the mapping dictionary with cleaned Plan IDs
        plan_dict_UniqueCloud1 = (valid_UniqueCloud1[['Plan ID (Clean)', 'Plan Name']]
                        .drop_duplicates()
                        .set_index('Plan ID (Clean)')['Plan Name']
                        .to_dict())
        print("  - Created look up dictionary 'Plan ID' & 'Plan Name'")

        if 'Plan ID' in df_DeliverData_temp.columns:
            # Clean DeliverData Plan ID
            df_DeliverData_temp['Plan ID (Clean)'] = df_DeliverData_temp['Plan ID'].apply(clean_plan_id_value)
            # Use the cleaned Plan ID for lookup
            df_DeliverData_temp['Plan Name'] = df_DeliverData_temp['Plan ID (Clean)'].map(plan_dict_UniqueCloud1).fillna("N/A") # Since there is no N/A in the dictionary, it outputs blank, fill blank with N/A again
            print("  - Updated 'Plan Name' from 'Plan ID' for DeliverData sheet")
        else:
            print("\n> 'Plan ID' column not found in DeliverData sheet")
            return

        # Save Type 2d: Overwrite existing xlsm with no format, only specific data location
        # Get header row from worksheet to find column positions
        header_row_DeliverData = [cell.value for cell in worksheet_DeliverData[1]]
        # Find column indices for the columns we need to update
        plan_name_col_DeliverData = header_row_DeliverData.index("Plan Name") + 1 # +1 for Excel 1-based indexing
        print("  - Updating DeliverData sheet with new Plan Name...")
        # Update DeliverData sheet - only the specific rows that were processed
        for df_row_idx in idx_to_update:
            excel_row = df_row_idx + 2  # +2 because pandas index starts at 0, Excel at 1, and row 1 is header
            worksheet_DeliverData.cell(row=excel_row, column=plan_name_col_DeliverData, value=df_DeliverData_temp.loc[df_row_idx, "Plan Name"]) # Only one column updated
        workbook.save(ap_DeliverData_file)
        excel_app = xw.App(visible=True, add_book=False)
        workbook_xlwings = excel_app.books.open(ap_DeliverData_file)
        xlwings_reapply_format(workbook_xlwings)
        print(f"\n~ Saved to '{ap_DeliverData_file}'")
    except Exception as e:
        print(f"\n> Error processing files : {e}")

### 9. Split Markets

In [ ]:
# Support sub function to split a market file by Funding Source
def split_by_funding_source(main_market_code, sub_market_code, exclude_fs):
    # Creates a new file excluding rows that match the specified Funding Source
    main_market_list = glob.glob(f"*UniqueClientName - {main_market_code} - DeliverData - Q*.xlsm")
    if len(main_market_list) != 1:
        print(f"\n> Expected only 1 {main_market_code} DeliverData file, found {len(main_market_list)}, aborting")
        return
    main_market_file = main_market_list[0]
    sub_market_file = main_market_file.replace(main_market_code, sub_market_code, 1)
    print(f"\n+ Processing file '{sub_market_file}'")
    # Lower-case and strip extra spaces
    exclude_fs_cleaned = exclude_fs.strip().lower()
    try: # Perform copy and filter
        shutil.copy(main_market_file, sub_market_file) # for file duplication
        print(f"  - Splitting {sub_market_code} from {main_market_code} by excluding {exclude_fs}")
        df = pd.read_excel(sub_market_file, sheet_name="DeliverData", engine='openpyxl', keep_default_na=False, na_values=[''])
        if "Funding Source" not in df.columns:
            print(f"    > 'Funding Source' column not found in DeliverData sheet")
            return
        # Filter out rows matching the exclude_fs value
        # df_filtered = df[df["Funding Source"].apply(lambda x: str(x).strip().lower() != exclude_fs_cleaned)]
        df_filtered = df[df["Funding Source"].apply(strp_low_str) != exclude_fs_cleaned]
        print(f"    > Removed {len(df) - len(df_filtered)} rows, kept {len(df_filtered)} rows")
        # Save Type 2c: Overwrite existing xlsm
        excel_writer_customize(df_filtered, sub_market_file, "DeliverData")
        time.sleep(5)
        excel_app = xw.App(visible=True, add_book=False)
        workbook_xlwings = excel_app.books.open(sub_market_file)
        xlwings_reapply_format(workbook_xlwings)
        print(f"  - Split complete for '{sub_market_code}'")
    except Exception as e:
        print(f"\n> Error splitting {sub_market_code} from {main_market_code}: {e}")
        return
    time.sleep(5)

In [ ]:
# Create multiple individual market files from 1 AP DeliverData file
def split_markets():
    print("\n─────➤ 9a. Split Markets")
    ap_DeliverData_list_xlsm = glob.glob("*UniqueClientName - AP - DeliverData - Q*.xlsm")
    if len(ap_DeliverData_list_xlsm) != 1:
        print(f"\n> Expected 1 AP DeliverData file, found {len(ap_DeliverData_list_xlsm)}, aborting")
        return
    ap_DeliverData_file = ap_DeliverData_list_xlsm[0]
    # Reference country_code_to_full mapping
    for country_code, country_full in country_code_to_full.items():
        # Ignore the blank/empty values
        if not country_code or not country_full or pd.isna(country_code) or pd.isna(country_full):
            continue
        # Replace the first occurrence of "AP" in the file name with the current country code
        market_file = ap_DeliverData_file.replace("AP", country_code, 1)
        print(f"\n+ Processing file '{market_file}'")
        # Lower-case and strip extra spaces
        country_full_cleaned = country_full.strip().lower()
        try: # Perform copy and filter
            shutil.copy(ap_DeliverData_file, market_file) # for file duplication
            # Do the split for 2 sheets
            for sheet_name in ["UniqueCloud1", "DeliverData"]:
                print(f"  - Filtering '{country_full}' in '{sheet_name}' sheet")
                df = pd.read_excel(market_file, sheet_name=sheet_name, engine='openpyxl', keep_default_na=False, na_values=[''])
                if "Country" not in df.columns:
                    print(f"    > 'Country' column not found in sheet '{sheet_name}'")
                    return
                # Filter the df using the country_full_cleaned
                df_filtered = df[df["Country"].apply(strp_low_str) == country_full_cleaned]
                print(f"  - '{sheet_name}': {country_full} {len(df_filtered)} rows")
                # Save Type 2c: Overwrite existing xlsm
                excel_writer_customize(df_filtered, market_file, sheet_name)
                time.sleep(5)
            # Now we run VBA to reapply the formatting again
            try:
                excel_app = xw.App(visible=True, add_book=False)
                workbook_xlwings = excel_app.books.open(market_file)
                workbook_xlwings.macro("SubFunc_AddButtons")(False) # False to delete button0
                xlwings_reapply_format(workbook_xlwings)
            except Exception as e:
                print(f"\n> Error reapplying macro for '{country_code}': {e}")
        except Exception as e:
            print(f"\n> Error during copy/filter for '{country_code}': {e}")
            continue # Skip to the next Country 
    time.sleep(5)
    print("\n─────➤ 9b. Split Markets by Funding Source")
    split_by_funding_source("HK", "HK-VIK", exclude_fs="UniqueClientName/Core Marketing")
    split_by_funding_source("HK", "HK-UniqueClientName", exclude_fs="Value In Kind (VIK)")

### 10a. Package Google Trends

In [ ]:
# Reference Link: https://github.com/GeneralMills/pytrends?tab=readme-ov-file#api-methods
def connect_ggtrend(search_term, timeframe, geo, cat=67, sleep=20, verbose=True, max_retries=3):
    # Connect to Google, hl host language, tz timezone offset
    pytrends = TrendReq(hl='en-US')
    # cat: Find available categories by inspecting the url when manually using Google Trends. Set Category default Travel 67
    # geo: Two letter country abbreviation
    def process_ggtrend():
        pytrends.build_payload([search_term], timeframe=timeframe, geo=geo, cat=cat)
        # Interest Over Time: returns historical, indexed data for when the keyword was searched most as shown on Google Trends' Interest Over Time section
        df = pytrends.interest_over_time()
        if df is None or df.empty:
            return None
        df = df.reset_index()
        # Remove partial data
        if "isPartial" in df.columns:
            df = df[df["isPartial"] == False].drop(columns=["isPartial"])
        df = df.rename(columns={"date": "Start Date", search_term: "Daily Variance"})
        # Add country column
        df["Country"] = geo
        return df
    
    attempts = 0
    while attempts <= max_retries:
        try:
            if verbose:
                attempt_msg = "1 time" if attempts == 0 else f"Retry {attempts} times"
                # print(f"    - Trying to download data for '{geo}' '{search_term}' ({attempt_msg})")
                print(f"    - Trying ({attempt_msg})")
            df_daily = process_ggtrend()
            if df_daily is not None:
                if verbose:
                    # print(f"    - {len(df_daily)} rows downloaded for '{geo}' '{search_term}'")
                    print(f"    - Downloaded {len(df_daily)} rows")
                return df_daily
            if verbose:
                print(f"    > No data for '{geo}' '{search_term}'")
            return None
        except Exception as e:
            if "429" in str(e) and attempts < max_retries:
                if verbose:
                    print(f"    - Rate limited on '{geo}' '{search_term}', cooling down 60s before retry...")
                time.sleep(60)
                attempts += 1
                continue
            if verbose:
                print(f"    > Failed downloading data for '{geo}' '{search_term}': {e}")
            return None
        finally:
            time.sleep(sleep)
    return None

In [ ]:
# Detect and adjust duplicate Paid Impressions values within each campaign group using Google Trends variance as a guide for redistribution
def process_duplicate_impressions(df_imp, verbose=True):
    # verbose : bool, Whether to print detailed processing information
    df_imp_adjusted = df_imp.copy()
    df_imp_adjusted["Paid Impressions (New)"] = df_imp_adjusted["Paid Impressions"]
    df_imp_adjusted["Adjustment Note"] = ""
    
    # Group by granularity level
    group_cols = ["Campaign", "Publisher/Vendor", "Vehicle Name", "Media Channel", "Inventory (Distribution) Type", "Category"]
    
    for group_name, group_df in df_imp_adjusted.groupby(group_cols):
        if len(group_df) <= 1:
            continue  # No duplicates possible with only 1 row
        
        # Find duplicate Paid Impressions within this group
        impressions_counts = group_df["Paid Impressions"].value_counts()
        impressions_duplicates = impressions_counts[impressions_counts > 1]
        
        if impressions_duplicates.empty:
            continue  # No duplicates in this group
        
        if verbose:
            print(f"\n   - Found duplicates in group: {group_name}")
        
        # Process each duplicate value
        for impressions_duplicate_value in impressions_duplicates.index:
            # Get all rows with this duplicate paid impression value
            duplicate_mask = (df_imp_adjusted[group_cols] == list(group_name)).all(axis=1) & \
                       (df_imp_adjusted["Paid Impressions"] == impressions_duplicate_value)
            duplicate_indices = df_imp_adjusted[duplicate_mask].index
            duplicate_rows = df_imp_adjusted.loc[duplicate_indices].copy()
            num_duplicates = len(duplicate_rows)
            
            if verbose:
                targetted_dates = duplicate_rows["Week/Month Beginning"].dt.strftime('%Y-%m-%d').tolist()
                print(f"      - Duplicate value: {impressions_duplicate_value:,} appears {num_duplicates} times on {targetted_dates}")
            
            # Check if we have GT variance data for all duplicates
            gt_variances = duplicate_rows["GT Corresponding Variance"].values
            
            if all(pd.notna(gt_variances)) and all(gt_variances > 0):
                """
                # Check if GT variances have any duplicates (partial or full)
                unique_variances = len(set(gt_variances))
                if unique_variances < num_duplicates:
                    # Some or all GT variances are duplicated - need to offset them
                    variances_duplicate_count = num_duplicates - unique_variances
                    if verbose:
                        if unique_variances == 1:
                            print(f"      - All GT variances are identical ({gt_variances[0]}), applying offset...")
                        else:
                            print(f"      - Found {variances_duplicate_count} duplicate GT variance value(s), applying offset...")
                            print(f"      - Original GT variances: {[f'{v:.2f}' for v in gt_variances]}")
                    
                    # Sort indices by variance value to maintain relative ordering
                    sorted_indices = np.argsort(gt_variances)
                    # Create small offsets to differentiate the variances
                    offset_step = 0.03 # Use a 3% step offset based on position
                    gt_variances_offset = np.array(gt_variances, dtype=float)

                    for rank, idx in enumerate(sorted_indices):
                        actual_offset = 1 + (rank * offset_step) - ((num_duplicates - 1) * offset_step / 2)
                        gt_variances_offset[idx] = gt_variances[idx] * actual_offset
                    gt_variances = gt_variances_offset
                    
                    if verbose:
                        print(f"      - Offset variances: {[f'{v:.2f}' for v in gt_variances]}")
                else:
                    # All variances are unique - no offset needed
                    if verbose:
                        print(f"      - All GT variances are unique, using original values")
                """
                # Calculate total Paid Impressions
                total_paid_imp = impressions_duplicate_value * num_duplicates
                
                # Calculate total GT variance
                total_variance = gt_variances.sum()
                
                # Calculate proportion (total impressions / total variance)
                percentage_proportion = total_paid_imp / total_variance
                
                # Redistribute impressions proportionally based on GT variance
                for idx, variance in zip(duplicate_indices, gt_variances):
                    imp_new_value = variance * percentage_proportion
                    df_imp_adjusted.loc[idx, "Paid Impressions (New)"] = round(imp_new_value, 2)
                    # df_imp_adjusted.loc[idx, "Adjustment Note"] = f"Previous value: {impressions_duplicate_value:,}, Total of: {total_paid_imp:,}, Corresponding Variance: {variance:.2f}"
                    df_imp_adjusted.loc[idx, "Adjustment Note"] = "To apply back in"
                
                if verbose:
                    print(f"       - Total Paid Impressions: {total_paid_imp:,}, Total Variance: {total_variance:.2f}")
                    for idx, variance in zip(duplicate_indices, gt_variances):
                        targetted_date = df_imp_adjusted.loc[idx, "Week/Month Beginning"].strftime('%Y-%m-%d')
                        imp_new_value_print = df_imp_adjusted.loc[idx, "Paid Impressions (New)"]
                        print(f"       + {targetted_date}: {impressions_duplicate_value:,} → {imp_new_value_print:,.2f}, Variance: {variance:.2f}")
            else:
                # Missing or zero GT variance - flag for manual review
                missing_count = sum(pd.isna(gt_variances))
                zero_count = sum(gt_variances == 0) if not any(pd.isna(gt_variances)) else 0
                if verbose:
                    print(f"    > Cannot redistribute: Missing GT data ({missing_count} missing, {zero_count} zeros)")
                for idx in duplicate_indices:
                    df_imp_adjusted.loc[idx, "Adjustment Note"] = "Duplicate detected but missing/zero GT variance - needs manual review"
    # Flag Paid Impressions (New) below threshhold 101
    low_impression_mask = (df_imp_adjusted["Media Channel"] != "TV") & (df_imp_adjusted["Paid Impressions (New)"] < 101)
    low_impression_count = low_impression_mask.sum()
    if low_impression_count > 0:
        df_imp_adjusted.loc[low_impression_mask, "Adjustment Note"] = "To delete"
        # df_imp_adjusted.loc[low_impression_mask, "Adjustment Note"] = df_imp_adjusted.loc[low_impression_mask, "Adjustment Note"] + " | To delete"
        if verbose:
            print(f"    > Found {low_impression_count} rows with Paid Impressions (New) < 101, flagged 'To delete'")
    # Make sure skip Media Channel TV
    tv_mask = df_imp_adjusted["Media Channel"] == "TV"
    df_imp_adjusted.loc[tv_mask, "Adjustment Note"] = ""
    return df_imp_adjusted

In [ ]:
# Main function to download and process GT data
def package_ggtrend():
    print("\n─────➤ 10a. Process Google Trends data")
    # modelled_country = ["AU", "JP", "ID"] # Set the targetted country each time
    modelled_country = ["JP"] # Set the targetted country each time
    ggtrend_output_file = "Google Trends - FY24Q2-FY26Q3.xlsx" # Only 1 file to store all the downloaded data

    if not os.path.exists(ggtrend_output_file):
        print(f"  - No Google Trends file found, begin downloading data...")
        # Set up the search term / keyword list
        search_term = [""]  # blank for generic search

        # Set up timeframe ranges
        timeframe_ranges = [ # YYYY-MM-DD
            "2024-01-01 2024-06-30", # FY24Q2-FY24Q3
            "2024-07-01 2024-12-30", # FY24Q4-FY25Q1
            "2025-01-01 2025-06-30", # FY25Q2-FY25Q3
            "2025-07-01 2025-12-31", # FY25Q4-FY26Q1
        ]

        # Initialize lists to collect DataFrames
        all_daily_list = []

        # Start the 3 loops
        for t in search_term:
            for mod_cou in modelled_country:
                country_daily_list = []
                # Loop through each timeframe range
                for timeframe in timeframe_ranges:
                    print(f"   - {timeframe} {mod_cou} {t}")
                    df_daily = connect_ggtrend(search_term=t, timeframe=timeframe, geo=mod_cou)
                    if df_daily is not None:
                        country_daily_list.append(df_daily)
                
                # Concatenate all daily data for this country and search term
                if country_daily_list:
                    df_all_country_daily = pd.concat(country_daily_list, ignore_index=True)
                    all_daily_list.append(df_all_country_daily)
        
        # After all daily data is collected, create weekly and monthly aggregations
        if all_daily_list:
            # Combine all daily data
            df_all_daily = pd.concat(all_daily_list, ignore_index=True)
            # Create weekly aggregation
            df_weekly = df_all_daily.copy()
            df_weekly["Week Beginning"] = df_weekly["Start Date"].dt.to_period("W").dt.start_time # Monday of the week
            df_weekly = df_weekly.groupby(["Country", "Week Beginning"], as_index=False)["Daily Variance"].sum() 
            df_weekly = df_weekly.rename(columns={"Daily Variance": "Weekly Variance"})

            # Create monthly aggregation
            df_monthly = df_all_daily.copy()
            df_monthly["Month Beginning"] = df_monthly["Start Date"].dt.to_period("M").dt.start_time # First day of the month
            df_monthly = df_monthly.groupby(["Country", "Month Beginning"], as_index=False)["Daily Variance"].sum()
            df_monthly = df_monthly.rename(columns={"Daily Variance": "Monthly Variance"})
            
            # # Save Type 1a: Brand new xlsx with no format, multiple sheets
            with pd.ExcelWriter(ggtrend_output_file, engine='openpyxl') as writer:
                df_all_daily.to_excel(writer, sheet_name='Daily', index=False)
                df_weekly.to_excel(writer, sheet_name='Weekly', index=False)
                df_monthly.to_excel(writer, sheet_name='Monthly', index=False)
            print(f"\n~ Saved to '{ggtrend_output_file}' with Daily, Weekly, and Monthly sheets")
        else:
            print("    > No data collected")
    else:
        print(f"  - Google Trends file found, continuing processing data...")
        df_weekly = pd.read_excel(ggtrend_output_file, sheet_name='Weekly', engine='openpyxl', keep_default_na=False, na_values=[''])
        df_monthly= pd.read_excel(ggtrend_output_file, sheet_name='Monthly', engine='openpyxl', keep_default_na=False, na_values=[''])

        for mod_cou in modelled_country:
            # Search for files matching the pattern for the current country code
            market_files = glob.glob(f"*UniqueClientName - {mod_cou} - DeliverData - Q*.xlsm")
            if len(market_files) != 1:
                print(f"\n> Expected 1 file for {mod_cou}, found {len(market_files)}, skipping")
                continue
            market_file = market_files[0]
            print(f"\n+ Processing {mod_cou} file '{market_file}'")

            df_DeliverData = pd.read_excel(market_file, sheet_name="DeliverData", engine='openpyxl', keep_default_na=False, na_values=[''])
            # Set up the Check Impressions df
            df_imp = df_DeliverData.copy()
            for col in ["Campaign", "Publisher/Vendor", "Vehicle Name", "Media Channel", "Inventory (Distribution) Type", "Category"]:
                df_imp[col] = df_imp[col].fillna("Blank_Placeholder").astype(str) # Since any blank in Groupby while crash the df
            df_imp = df_imp.groupby(["Campaign", "Publisher/Vendor", "Vehicle Name", "Media Channel", "Inventory (Distribution) Type", "Category", "Week/Month Beginning"], as_index=False)["Paid Impressions"].sum()
            # df_imp["Week/Month Beginning"] = pd.to_datetime(df_imp["Week/Month Beginning"]) # Ensure Week/Month Beginning is datetime
            # Initialize the Google Trends Corresponding Variance column
            df_imp["GT Corresponding Variance"] = None

            # Filter the current Modelled Country
            df_weekly_country = df_weekly[df_weekly["Country"] == mod_cou].copy()
            df_monthly_country = df_monthly[df_monthly["Country"] == mod_cou].copy()

            # For non-Offline-Monthly categories: match with Weekly data
            non_monthly_mask = df_imp["Category"] != "Offline-Monthly"
            for idx in df_imp[non_monthly_mask].index: # For all Categories that are not Offline-Monthly
                week_beginning = df_imp.loc[idx, "Week/Month Beginning"]
                matching_weekly = df_weekly_country[df_weekly_country["Week Beginning"] == week_beginning] # Match inside GT data, Weekly sheet
                if not matching_weekly.empty:
                    df_imp.loc[idx, "GT Corresponding Variance"] = matching_weekly.iloc[0]["Weekly Variance"] # Get the Variance value
            
            # For Offline-Monthly category: match with monthly data
            monthly_mask = df_imp["Category"] == "Offline-Monthly"
            for idx in df_imp[monthly_mask].index: # For only Offline-Monthly category
                month_beginning = df_imp.loc[idx, "Week/Month Beginning"]
                matching_monthly = df_monthly_country[df_monthly_country["Month Beginning"] == month_beginning] # Match inside GT data, Monthly sheet
                if not matching_monthly.empty:
                    df_imp.loc[idx, "GT Corresponding Variance"] = matching_monthly.iloc[0]["Monthly Variance"] # Get the Variance value

            # Check for duplicate impressions and redistribute them using GT variance
            print(f"  - Checking duplicate impressions for {mod_cou} at Weekly/Monthly levels...")
            df_imp = process_duplicate_impressions(df_imp, verbose=True)
            
            # Save Type 1a: Brand new xlsx with no format
            df_imp.to_excel(f"{mod_cou}_DeliverData_Check_Impressions.xlsx", index=False)
            print(f"\n~ Saved results to '{mod_cou}_DeliverData_Check_Impressions.xlsx'")

            # Save Type 2c: Overwrite existing xlsm
            # excel_writer_customize(df_imp, market_file, "DeliverData_Check_Impressions")
            # time.sleep(1)
            # excel_app = xw.App(visible=True, add_book=False)
            # workbook_xlwings = excel_app.books.open(market_file)
            # workbook_xlwings = vb_project(workbook_xlwings)
            # xlwings_reapply_format(workbook_xlwings)

### 10b. Apply Google Trends

In [ ]:
# Second main function to apply the Paid Impressions (New) back into DeliverData sheet
def apply_ggtrend():
    print("\n─────➤ 10b. Apply new Impressions back")
    # modelled_country = ["AU", "JP", "ID"]
    modelled_country = ["JP"]
    
    for mod_cou in modelled_country:
        # Search for files matching the pattern for the current country code
        market_files = glob.glob(f"*UniqueClientName - {mod_cou} - DeliverData - Q*.xlsm")
        if len(market_files) != 1:
            print(f"\n> Expected 1 file for {mod_cou}, found {len(market_files)}, skipping")
            continue
        market_file = market_files[0]
        print(f"\n+ Processing {mod_cou} file '{market_file}'")
        
        df_DeliverData = pd.read_excel(market_file, sheet_name="DeliverData", engine='openpyxl', keep_default_na=False, na_values=[''])
        df_imp = pd.read_excel(market_file, sheet_name="DeliverData_Check_Impressions", engine='openpyxl', keep_default_na=False, na_values=[''])
        
        # Fill NaN and convert to string for grouping columns
        group_cols = ["Campaign", "Publisher/Vendor", "Vehicle Name", "Media Channel", "Inventory (Distribution) Type", "Category"]
        for col in group_cols:
            df_imp[col] = df_imp[col].fillna("Blank_Placeholder").astype(str)
            df_DeliverData[col] = df_DeliverData[col].fillna("Blank_Placeholder").astype(str)
        
        # Ensure date columns are datetime
        # df_imp["Week/Month Beginning"] = pd.to_datetime(df_imp["Week/Month Beginning"])
        # df_DeliverData["Week/Month Beginning"] = pd.to_datetime(df_DeliverData["Week/Month Beginning"])
        
        # Track rows to delete
        rows_to_delete = []
        delete_count = 0
        apply_count = 0
        
        # a) Process deletions, groups with "To delete"
        print("\n  - Processing 'To delete' rows")
        delete_mask = df_imp["Adjustment Note"] == "To delete"
        df_to_delete = df_imp[delete_mask]
        
        if not df_to_delete.empty:
            for group_name, group_df in df_to_delete.groupby(group_cols + ["Week/Month Beginning"]):
                # Find matching group in df_DeliverData
                match_mask = (df_DeliverData[group_cols + ["Week/Month Beginning"]] == list(group_name)).all(axis=1)
                matching_DeliverData = df_DeliverData[match_mask]
                if matching_DeliverData.empty:
                    print(f"    > No matching group found in DeliverData for deletion: {group_name}")
                    continue
                
                group_total_rows = len(matching_DeliverData)
                DeliverData_total_impressions = matching_DeliverData["Paid Impressions"].sum()
                imp_total_impressions = group_df["Paid Impressions"].iloc[0]  # Should be same value
                print(f"     - {group_name} | {group_total_rows} rows | DeliverData Total Paid Impressions: {DeliverData_total_impressions:,.0f} | Total Check Impressions: {imp_total_impressions:,.0f}")
                
                # Double check totals match and value is < 101
                if abs(DeliverData_total_impressions - imp_total_impressions) < 0.01 and DeliverData_total_impressions < 101:
                    rows_to_delete.extend(matching_DeliverData.index.tolist())
                    delete_count += group_total_rows
                    print(f"       - Marked {group_total_rows} rows for deletion")
                else:
                    print(f"       > Validation failed, skip deletion")
        
        # b) Process applications, groups with "To apply back in"
        print("\n  - Processing 'To apply back in' rows")
        apply_mask = df_imp["Adjustment Note"] == "To apply back in"
        df_to_apply = df_imp[apply_mask]
        
        if not df_to_apply.empty:
            # Check if "Paid Impressions (New)" column exists
            if "Paid Impressions (New)" not in df_to_apply.columns:
                print("    > 'Paid Impressions (New)' column not found in DeliverData_Check_Impressions, skipping applications")
            else:
                for group_name, group_df in df_to_apply.groupby(group_cols + ["Week/Month Beginning"]):
                    # Find matching group in df_DeliverData
                    match_mask = (df_DeliverData[group_cols + ["Week/Month Beginning"]] == list(group_name)).all(axis=1)
                    matching_DeliverData_indices = df_DeliverData[match_mask].index
                    if len(matching_DeliverData_indices) == 0:
                        print(f"    > No matching group found in DeliverData for application: {group_name}")
                        continue
                    
                    group_total_rows = len(matching_DeliverData_indices)
                    DeliverData_total_impressions = df_DeliverData.loc[matching_DeliverData_indices, "Paid Impressions"].sum()
                    imp_old_total = group_df["Paid Impressions"].iloc[0]
                    imp_new_total = group_df["Paid Impressions (New)"].iloc[0]
                    print(f"     - {group_name} | {group_total_rows} rows | DeliverData Total Paid Impressions: {DeliverData_total_impressions:,.0f} | Old Total Check Impressions: {imp_old_total:,.0f} | New Total Check Impressions: {imp_new_total:,.0f}")
                    
                    # Double check totals match
                    if abs(DeliverData_total_impressions - imp_old_total) < 0.01:
                        # Apply new impressions evenly across all rows in the group
                        new_value_per_row = imp_new_total / group_total_rows
                        df_DeliverData.loc[matching_DeliverData_indices, "Paid Impressions"] = new_value_per_row
                        apply_count += group_total_rows
                        print(f"       - Applied {imp_new_total:,.2f} across {group_total_rows} rows ({new_value_per_row:,.2f} per row)")
                    else:
                        print(f"       > Validation failed, skip application")
        
        # Delete marked rows
        if rows_to_delete:
            print(f"\n  - Deleting {delete_count} rows from DeliverData sheet...")
            df_DeliverData = df_DeliverData.drop(index=rows_to_delete).reset_index(drop=True)
        
        # Restore blank values
        for col in group_cols:
            df_DeliverData[col] = df_DeliverData[col].replace("Blank_Placeholder", '')
        
        # Summary
        print(f"\n  - Summary: {delete_count} rows deleted, {apply_count} rows updated Paid Impressions")

        # Save Type 1a: Brand new xlsx with no format
        df_DeliverData.to_excel(f"{mod_cou}_finished.xlsx", index=False)
        print(f"\n~ Saved results to '{mod_cou}_finished.xlsx'")
        
        # Save Type 2c: Overwrite existing xlsm
        excel_writer_customize(df_DeliverData, market_file, "DeliverData")
        time.sleep(1)
        excel_app = xw.App(visible=True, add_book=False)
        workbook_xlwings = excel_app.books.open(market_file)
        workbook_xlwings = vb_project(workbook_xlwings)
        xlwings_reapply_format(workbook_xlwings)

### 11. Combine Markets

In [ ]:
# Combines multiple country-specific 'UniqueClientName - {code} - DeliverData - Q*.xlsm' files into a single 'UniqueClientName - AP - DeliverData - Q*.xlsm' file 
def combine_markets():
    print("\n─────➤ 11. Combine Markets")

    # Initialize lists to collect DataFrames from each country file
    all_frames_1 = []

    ap_DeliverData_list_xlsm = glob.glob("*UniqueClientName - AP - DeliverData - Q*.xlsm")
    if len(ap_DeliverData_list_xlsm) != 1:
        print(f"\n> Expected 1 AP DeliverData file, found {len(ap_DeliverData_list_xlsm)}, aborting")
        return
    ap_DeliverData_file = ap_DeliverData_list_xlsm[0]

    for code in country_codes_expand:
        # Search for files matching the pattern for the current country code
        market_files = glob.glob(f"*UniqueClientName - {code} - DeliverData - Q*.xlsm")
        if len(market_files) != 1:
            print(f"\n> Expected 1 file for {code}, found {len(market_files)}, skipping")
            continue
        market_file = market_files[0]
        print(f"\n+ Processing {code} file '{market_file}'")

        # Load the 'DeliverData' sheet into a pandas DataFrame
        df = pd.read_excel(market_file, sheet_name="DeliverData", engine='openpyxl', keep_default_na=False, na_values=[''])
        # Add column existence check
        if "YQ" not in df.columns:
            print(f"    > 'YQ' column not found in {market_file}, skipping")
            continue

        # Drop all completely empty rows
        df.dropna(how='all', inplace=True)

        # Safe guard filter to combine only the current period data
        # original_rows = len(df)
        # normalized_yq = df["YQ"].astype(str).str.strip().str.lower()
        # normalized_quarters = [q.lower() for q in year_quarters]
        # df = df[normalized_yq.isin(normalized_quarters)].copy()
        # filtered_rows = len(df)

        # if filtered_rows == 0:
        #     print(f"    > No data for {year_quarters} in {market_file}, skipping")
        #     continue
        # print(f"    > Filtered from {original_rows} to {filtered_rows} rows for {year_quarters}")

        # Append the cleaned DataFrame to the list
        all_frames_1.append(df)

    # Concatenate all DataFrames into a single DataFrame
    if all_frames_1:
        df_DeliverData = pd.concat(all_frames_1, ignore_index=True)
    else:
        print("\n> No DeliverData data frames to concatenate, aborting")
        return
    
    # Save Type 2c: Overwrite existing xlsm
    excel_writer_customize(df_DeliverData, ap_DeliverData_file, "DeliverData")
    time.sleep(1)
    excel_app = xw.App(visible=True, add_book=False)
    workbook_xlwings = excel_app.books.open(ap_DeliverData_file)
    workbook_xlwings = vb_project(workbook_xlwings)
    workbook_xlwings.macro("SubFunc_AddButtons")(False) # False to delete button0
    xlwings_reapply_format(workbook_xlwings)
    print(f"\n~ Successfully combined data into '{ap_DeliverData_file}'")

### 12. Convert Cost to USD

In [ ]:
# Sub function to convert media_cost_lc to USD
def convert_cost_to_USD_sub(row):
    row_index = row.name # Get the index label (useful for diagnostics)
    row_label = f"Row {row_index}" if row_index is not None else "Row (unknown index)"
    country = str(row.get("Country", "")).strip() # Get the current Country value
    current_curr = str(row.get("Currency", "")).strip().upper() # Get the current Currency
    expected_curr = currency_dict[country] # Get the initial correct Expected Currency value
    try:
        media_cost_lc = float(row.get("Media Cost (LC)", 0)) # Get the current Cost value
    except (ValueError, TypeError):
        print(f"> {row_label}: {row.get("Media Cost (LC)", 0)} is not a valid number, skipping row")
        return row # Print and skip if media_cost_lc is not a valid number

    if country not in currency_dict:
        print(f"> {row_label}: {country} is not the correct Country, skipping row")
        return row # Print and skip if country not in dictionary
    
    if current_curr != expected_curr:
        print(f"> {row_label}: {country} - {current_curr} does not match {expected_curr}, skipping row")
        return row # If current Currency doesn't match the initial correct Expected Currency, print and skip
    
    if country in group_B and current_curr == expected_curr:
        row["Media Cost (USD)"] = media_cost_lc # For Group B countries, currency is already in USD, simply copy the values over
        print(f"  - {country}: {media_cost_lc:.2f} ({current_curr}) copied to 'Media Cost (USD)' column")
        return row
    
    # For Group A countries, currency is in Local Currency, want to convert the Cost to USD
    if country in group_A and current_curr == expected_curr:
        rate = exchange_dict.get(country, None)
        if rate and rate != 0:
            # Convert from Local Currency to USD: new_cost = media_cost_lc / exchange_rate
            row["Media Cost (USD)"] = media_cost_lc / rate # Divide direction: to convert from LC to USD
            print(f"  - {country}: {media_cost_lc:.2f} ({current_curr}) -> {row['Media Cost (USD)']:.2f} (USD)")
        return row

In [ ]:
# Main function to convert cost to usd
def convert_cost_to_USD():
    print("\n─────➤ 12. Convert Cost To USD")
    ap_DeliverData_list_xlsm = glob.glob("*UniqueClientName - AP - DeliverData - Q*.xlsm")
    if len(ap_DeliverData_list_xlsm) != 1:
        print(f"\n> Expected 1 AP DeliverData file, found {len(ap_DeliverData_list_xlsm)}, aborting")
        return
    ap_DeliverData_file = ap_DeliverData_list_xlsm[0]
    print(f"\n+ Processing file '{ap_DeliverData_file}'")
    try:
        df_DeliverData = pd.read_excel(ap_DeliverData_file, sheet_name="DeliverData", engine='openpyxl', keep_default_na=False, na_values=[''])
        
        # Safe guard for Country Korea
        if (df_DeliverData["Country"] == "South Korea").any():
            print("  - Found 'South Korea' -> normalized to 'Korea'")
            df_DeliverData["Country"] = df_DeliverData["Country"].replace("South Korea", "Korea")

        df_DeliverData_temp = df_DeliverData[(df_DeliverData["Country"].astype(str).str.strip() != "") & (df_DeliverData["Currency"].astype(str).str.strip() != "")]
        if df_DeliverData_temp.empty:
            print("\n> No data rows with valid 'Country' and 'Currency' found")
            return
        idx_to_update = df_DeliverData_temp.index.tolist()
        df_DeliverData.loc[idx_to_update] = df_DeliverData.loc[idx_to_update].apply(convert_cost_to_USD_sub, axis=1)

        # Save Type 2c: Overwrite existing xlsm
        excel_writer_customize(df_DeliverData, ap_DeliverData_file, "DeliverData")
        time.sleep(1)
        excel_app = xw.App(visible=True, add_book=False)
        workbook_xlwings = excel_app.books.open(ap_DeliverData_file)
        workbook_xlwings = vb_project(workbook_xlwings)
        xlwings_reapply_format(workbook_xlwings)
        print(f"\n~ Convert cost to USD finished in '{ap_DeliverData_file}'")
    except Exception as e:
        print(f"\n> Error convert cost to USD : {e}")

### 15. UniqueCloud2 Templates

In [ ]:
# From DeliverData sheet, split to 3 UniqueCloud2 templates
def create_UniqueCloud2_templates():
    print("\n─────➤ 15. Split DeliverData to UniqueCloud2 Templates")
    ap_DeliverData_list_xlsm = glob.glob("*UniqueClientName - AP - DeliverData - Q*.xlsm")
    if len(ap_DeliverData_list_xlsm) != 1:
        print(f"    > Expected 1 AP DeliverData file, found {len(ap_DeliverData_list_xlsm)}, aborting")
        return
    ap_DeliverData_file = ap_DeliverData_list_xlsm[0]
    
    # Extract quarter_year from filename, "Q1FY24" or "Q1FY24-Q4FY24"
    match = re.search(r'(Q\d+FY\d+(?:-Q\d+FY\d+)?)', ap_DeliverData_file)
    if not match:
        print(f"    > Could not extract quarter_year from {ap_DeliverData_file}, aborting")
        return
    quarter_year = match.group(1)
    
    df_all = pd.read_excel(ap_DeliverData_file, sheet_name="DeliverData", engine="openpyxl", keep_default_na=False, na_values=[""])
    print(f"\n+ Loading file '{ap_DeliverData_file}'")
    df_all.dropna(how='all', inplace=True)
    
    DeliverData_columns = ["Category", "Week/Month Beginning", "Week/Month Ending", "Campaign ID (New)", "Campaign Taxonomy (UniqueCloud2)", "Placement ID (New)", "Placement Taxonomy (UniqueCloud2)", 
        "Creative ID (New)", "Creative Taxonomy (UniqueCloud2)", "Currency", "Media Cost (USD)", "Paid Impressions", "Clicks", "Video Views", "Video Completes", "Plan ID", "Plan Name", "Media Channel", "TV National GRPs", "Notes"]
    for col in DeliverData_columns:
        if col not in df_all.columns:
            print(f"    > '{col}' column not found in {ap_DeliverData_file}, aborting")
            return
    
    # Filter and select columns 
    df_net_new = df_all[df_all["Category"].astype(str).str.contains("Online", case=False, na=False)].copy()
    df_offline_weekly = df_all[df_all["Category"].astype(str).str.contains("Offline-Weekly", case=False, na=False)].copy()
    df_offline_monthly = df_all[df_all["Category"].astype(str).str.contains("Offline-Monthly", case=False, na=False)].copy()
    
    net_new_columns_to_keep = ["Week/Month Beginning", "Week/Month Ending", "Campaign ID (New)", "Campaign Taxonomy (UniqueCloud2)", "Placement ID (New)", "Placement Taxonomy (UniqueCloud2)", "Creative ID (New)", "Creative Taxonomy (UniqueCloud2)", "Currency", "Media Cost (USD)", "Paid Impressions", "Clicks", "Video Views", "Video Completes"]
    df_net_new = df_net_new[net_new_columns_to_keep]

    offline_columns_to_keep = ["Week/Month Beginning", "Week/Month Ending", "Plan ID", "Plan Name", "Media Channel", "Currency", "Media Cost (USD)", "Paid Impressions", "TV National GRPs", "Notes"]
    df_offline_weekly = df_offline_weekly[offline_columns_to_keep]
    df_offline_monthly = df_offline_monthly[offline_columns_to_keep]
    
    rename_net_new = {
        "Week/Month Beginning": "Start Date", 
        "Week/Month Ending": "End Date", 
        "Campaign ID (New)": "Campaign ID",
        "Campaign Taxonomy (UniqueCloud2)": "Campaign Name", 
        "Placement ID (New)": "Placement ID", 
        "Placement Taxonomy (UniqueCloud2)": "Placement Name",
        "Creative ID (New)": "Creative ID", 
        "Creative Taxonomy (UniqueCloud2)": "Creative Name", 
        "Media Cost (USD)": "Media Cost", 
        "Paid Impressions": "Impressions",
        "Video Views": "Starts", 
        "Video Completes": "Completes",
    }
    df_net_new.rename(columns=rename_net_new, inplace=True)

    rename_offline = {
        "Week/Month Beginning": "Start Date Data Frame", 
        "Week/Month Ending": "End Date Data Frame",
        "Media Channel": "Media Type", 
        "Media Cost (USD)": "Cost",
        "Paid Impressions": "Impressions",
        "TV National GRPs": "Grps",
        "Notes": "Notes Column",
    }
    df_offline_weekly.rename(columns=rename_offline, inplace=True)
    df_offline_monthly.rename(columns=rename_offline, inplace=True)

    net_new_additional_columns = ["25% Completed", "50% Completed", "75% Complete"]
    for col in net_new_additional_columns:
        df_net_new[col] = ""
    df_net_new["Currency"] = "USD"

    offline_additional_columns = ["Circulation", "Foot Traffic", "Impacts"]
    for col in offline_additional_columns:
        df_offline_weekly[col] = ""
        df_offline_monthly[col] = ""
    df_offline_weekly["Currency"] = "USD"
    df_offline_monthly["Currency"] = "USD"
    
    # Save Type 1b: Brand new xlsx with format
    net_new_file = f"UniqueClientName - AP - Net New - {quarter_year}.xlsx"
    excel_writer_customize(df_net_new, net_new_file, "Net New")
    offline_weekly_file = f"UniqueClientName - AP - Offline-Weekly - {quarter_year}.xlsx"
    excel_writer_customize(df_offline_weekly, offline_weekly_file, "Offline Weekly")
    offline_monthly_file = f"UniqueClientName - AP - Offline-Monthly - {quarter_year}.xlsx"
    excel_writer_customize(df_offline_monthly, offline_monthly_file, "Offline Monthly")
    print(f"\n~ Split DeliverData to UniqueCloud2 templates finished'")

### 16. UniqueCloud2 Submission

In [ ]:
# Find UniqueCloud2 submission files, send them through email
def email_UniqueCloud2_submission():
    print("\n─────➤ 16. Send emails to UniqueCloud2 Submission")
    net_new_files = glob.glob("UniqueClientName - AP - Net New - Q*.xlsx")
    offline_weekly_files = glob.glob("UniqueClientName - AP - Offline-Weekly - Q*.xlsx")
    offline_monthly_files = glob.glob("UniqueClientName - AP - Offline-Monthly - Q*.xlsx")
    
    # Create Outlook connection
    try:
        outlook = win32.Dispatch('Outlook.Application')
        print("  - Outlook connection established")
    except Exception as e:
        print(f"    > Error connecting to Outlook: {e}")
        return

    # Implemented file size check > 25MB (25 x 1024 x 1024 bytes)
    for files in [net_new_files, offline_weekly_files, offline_monthly_files]:
        for file in files:
            if os.path.getsize(file) > 25 * 1024 * 1024:
                print(f"    > File {file} is > 25MB email limit, aborting")
                return

    for file in net_new_files:
        print(f"\n+ Processing net new file: {file}")
        mail = outlook.CreateItem(0) # 0 = Mail item
        file_name = os.path.basename(file) # extract just the filename from the path
        mail.To = "" # Main recipient, Net New Data Stream
        mail.CC = "" # CC recipient
        mail.Subject = file_name
        mail.Body = file_name
        mail.Attachments.Add(os.path.abspath(file)) # use the actual file path
        # mail.Attachments.Add(r'C:\Path\To\Your\File.pdf')
        mail.Send()
        # mail.Display() # Show the email before sending
        print(f"~ Email sent for '{file_name}'")
    
    for file in offline_weekly_files:
        print(f"\n+ Processing offline weekly file: {file}")
        mail = outlook.CreateItem(0)
        file_name = os.path.basename(file)
        mail.To = "" # Offline Weekly Data Stream
        mail.CC = ""
        mail.Subject = file_name
        mail.Body = file_name
        mail.Attachments.Add(os.path.abspath(file))
        mail.Send()
        print(f"~ Email sent for '{file_name}'")

    for file in offline_monthly_files:
        print(f"\n+ Processing offline monthly file: {file}")
        mail = outlook.CreateItem(0)
        file_name = os.path.basename(file)
        mail.To = "" # Offline Monthly Data Stream
        mail.CC = ""
        mail.Subject = file_name
        mail.Body = file_name
        mail.Attachments.Add(os.path.abspath(file))
        mail.Send()
        print(f"~ Email sent for '{file_name}'")
    print(f"\n~ Send the UniqueCloud2 submissions through email finished'")

### 17. UniqueCloud2 Validation

In [ ]:
# Collect and combine UniqueCloud2 Raw Extract files
def validate_UniqueCloud2():
    print("\n─────➤ 17. Validate UniqueCloud2")
    # Initialize empty lists to collect UniqueCloud2 DataFrames
    df_list_online = []
    df_list_offline = []

    # Required keywords to detect the UniqueCloud2 Online/Offline header row 
    online_required_keywords = ["Publisher/Vendor", "Vehicle Name", "Clicks", "Video Views", "Video Completes"]
    offline_required_keywords = ["Spend", "Impressions", "GRPs"]

    # Identify subfolders
    for entry in os.scandir(main_working_folder):
        if entry.is_dir():
            folder_name = entry.name
            folder_path = entry.path
            print(f"\n+ Searching inside '{folder_name}' folder")
            # Construct the search pattern for this specific subfolder
            search_pattern = os.path.join(folder_path, "Raw Extract*.xlsx")
            # Find matches and add them to list
            file_paths = glob.glob(search_pattern)
            if not file_paths:
                print("\n> No Raw Extract files found, aborting")
                # return
            # Loop through each file in the subfolder
            for file_path in file_paths:
                df_temp = pd.read_excel(file_path, header=4, engine='openpyxl', keep_default_na=False, na_values=[''])
                header_current = df_temp.columns.tolist()
                # Save the folder name into the column
                df_temp['Folder Name'] = folder_name
                # Check if all required keywords are in the current header
                if all(keyword in header_current for keyword in online_required_keywords):
                    # Save the Offline Online indicator into the column
                    df_temp["Offline / Online"] = "Online"
                    df_list_online.append(df_temp)
                    print(f"  - Detected UniqueCloud2 Online {file_path}")
                elif all(keyword in header_current for keyword in offline_required_keywords):
                    df_temp["Offline / Online"] = "Offline"
                    df_list_offline.append(df_temp)
                    print(f"  - Detected UniqueCloud2 Offline {file_path}")
                else:
                    print(f"  - Skipped {file_path}")

    # Combine all Online DataFrames
    if df_list_online:
        df_online = pd.concat(df_list_online, ignore_index=True)
        print(f"  - Combined {len(df_list_online)} files into Online DataFrame with {len(df_online)} rows")

    # Combine all Offline DataFrames
    if df_list_offline:
        df_offline = pd.concat(df_list_offline, ignore_index=True)
        # Rename certain columns for later matching
        df_offline.rename(columns={"Spend": "Media Cost (USD)", "Impressions": "Paid Impressions"}, inplace=True)
        print(f"  - Combined {len(df_list_offline)} files into Offline DataFrame with {len(df_offline)} rows")
    # Concatenate vertically, axis=0 is default
    df_UniqueCloud2 = pd.concat([df_online, df_offline], axis=0, ignore_index=True)
    # Remove rows that contain "Totals" in any cell in one column (case-insensitive)
    df_UniqueCloud2 = df_UniqueCloud2[~df_UniqueCloud2.apply(check_contain_any, args=(["Totals"], ["Country"]), axis=1)]
    print(f"  - Combined UniqueCloud2 with {len(df_UniqueCloud2)} rows")

    # Transform the values under Data Stream to Online or Offline
    mask_weekly = (df_UniqueCloud2["Offline / Online"] == "Offline") & (df_UniqueCloud2["Data Stream"] == "UniqueDataStream3")
    mask_monthly = (df_UniqueCloud2["Offline / Online"] == "Offline") & (df_UniqueCloud2["Data Stream"] == "UniqueDataStream4")
    mask_net_new = (df_UniqueCloud2["Offline / Online"] == "Online") & (df_UniqueCloud2["Data Stream"] == "UniqueDataStream1")
    mask_dfa = (df_UniqueCloud2["Offline / Online"] == "Online") & (df_UniqueCloud2["Data Stream"] == "UniqueDataStream2")
    mask_else = (df_UniqueCloud2["Offline / Online"] == "Online") & (df_UniqueCloud2["Data Stream"] != "Online")

    if mask_weekly.any():
        df_UniqueCloud2.loc[mask_weekly, "Data Stream"] = "Offline"
    if mask_monthly.any():
        df_UniqueCloud2.loc[mask_monthly, "Data Stream"] = "Offline"
    if mask_net_new.any():
        df_UniqueCloud2.loc[mask_net_new, "Data Stream"] = "Online"
    if mask_dfa.any():
        df_UniqueCloud2.loc[mask_dfa, "Data Stream"] = "Online"
    if mask_else.any():
        df_UniqueCloud2.loc[mask_else, "Data Stream"] = "Online"
    
    # Create 2 sub DataFrames from UniqueCloud2 data set
    df_UniqueCloud2_current = df_UniqueCloud2[df_UniqueCloud2['Folder Name'].apply(strp_low_str) == "current"].reset_index(drop=True)
    df_UniqueCloud2_overlapping = df_UniqueCloud2[df_UniqueCloud2['Folder Name'].apply(strp_low_str) == "overlapping"].reset_index(drop=True)

    # Keep only certain columns
    df_UniqueCloud2_current = df_UniqueCloud2_current[['Country', 'Plan ID', 'Data Stream', 'Media Cost (USD)']]
    df_UniqueCloud2_overlapping = df_UniqueCloud2_overlapping[['Country', 'Plan ID', 'Data Stream', 'Media Cost (USD)']]
    # df_UniqueCloud2_current.to_excel("df_UniqueCloud2_current.xlsx")
    # df_UniqueCloud2_overlapping.to_excel("df_UniqueCloud2_overlapping.xlsx")

    ap_DeliverData_list_xlsm = glob.glob("*UniqueClientName - AP - DeliverData - Q*.xlsm")
    if len(ap_DeliverData_list_xlsm) != 1:
        print(f"\n> Expected 1 AP DeliverData file, found {len(ap_DeliverData_list_xlsm)}, aborting")
        return
    ap_DeliverData_file = ap_DeliverData_list_xlsm[0]
    print(f"+ Loading file '{ap_DeliverData_file}'")

    # Use 'overlay' so both DataFrames can write to the same sheet without overwriting
    with pd.ExcelWriter(ap_DeliverData_file, engine='openpyxl', mode='a', if_sheet_exists='overlay', engine_kwargs={'keep_vba': True}) as writer:
        df_UniqueCloud2_current.to_excel(writer, sheet_name='Regional DeliverData Dashboards', startrow=1, startcol=0, index=False) # From cell A2
        df_UniqueCloud2_overlapping.to_excel(writer, sheet_name='Regional DeliverData Dashboards', startrow=1, startcol=(df_UniqueCloud2_current.shape[1]+1), index=False)

    excel_app = xw.App(visible=True, add_book=False)
    workbook_xlwings = excel_app.books.open(ap_DeliverData_file)
    workbook_xlwings = vb_project(workbook_xlwings)
    # workbook_xlwings.activate()
    # workbook_xlwings.sheets["DeliverData"].activate()
    # workbook_xlwings.macro("Apply_DeliverData_Format")()
    # workbook_xlwings.macro("Validation_Results")()
    workbook_xlwings.save()
    print(f"\n~ Saved to '{ap_DeliverData_file}'")

### 18. Combine Storage

In [ ]:
# Sub function to get the first FY##Q# value in the period range of the filename
def get_initial_fyq(file_name, verbose=True):
    fyq_vals = calculatefiscal_fyq(file_name, verbose=verbose)
    if not fyq_vals:
        return (9999, 9) # dummy value
    initial_fyq_val = fyq_vals[0]
    return (int(initial_fyq_val[2:4]), int(initial_fyq_val[5:])) # Output (Year, Quarter)

# Combine files with previous periods and rename them to latest period range in DeliverData Reports folder
def combine_DeliverData_storage():
    print("\n─────➤ 18. Combine DeliverData Storage")
    for code in country_codes_expand:
        market_files = glob.glob(f"*UniqueClientName - {code} - DeliverData - Q*.xlsm")
        # Exclude files with "Paid Media Creatives Objectives" in the name
        print(f"  - {code} found {len(market_files)} files before exclusion")
        market_files = [market_file for market_file in market_files if "Paid Media Creatives Objectives" not in market_file]
        print(f"  - {code} found {len(market_files)} files after exclusion")
        if not market_files:
            print(f"    > No files found for storage, skipping")
            continue
        if len(market_files) == 1:
            print(f"    > Only one file found for {code} market, skipping")
            continue
        else:
            print(f"  - Multiple files found for {code} market, proceeding with concatenation")

        # Using get_initial_fyq to get the first fyq value from each file name, then sort all the files based on that order
        market_files = sorted(market_files, key=lambda fmarket_file: get_initial_fyq(market_file, verbose=False))

        # Get all fiscal year quarter values
        all_fyq_vals = []
        for market_file in market_files:
            all_fyq_vals.extend(calculatefiscal_fyq(market_file))

        # Remove duplicates, then convert to list (to use in sort function)
        all_fyq_vals2 = list(set(all_fyq_vals))
        # 
        if len(all_fyq_vals) != len(all_fyq_vals2):
            raise ValueError(f"    > Overlapping FYQ values {all_fyq_vals}, aborting")
        
        # Sort by increasing FY34Q6 digit order 
        all_fyq_vals = sorted(all_fyq_vals, key=lambda fyq_val: (int(fyq_val[2:4]), int(fyq_val[5:])))
        if not all_fyq_vals:
            print("    > No fiscal year quarter data found, skipping")
            continue

        # Extract the starting FY##Q# value
        start_fyq_val = all_fyq_vals[0]
        start_fy_val = start_fyq_val[0:4] # FY##
        start_q_val = start_fyq_val[4:6] # Q#

        # Extract the ending FY##Q# value
        end_fyq_val = all_fyq_vals[-1]
        end_fy_val = end_fyq_val[0:4]
        end_q_val = end_fyq_val[4:6]

        # Copy the original file to the final storage file
        original_file = market_files[0]
        storage_file = f"UniqueClientName - {code} - DeliverData - {start_q_val}{start_fy_val}-{end_q_val}{end_fy_val}.xlsm"
        shutil.copy(original_file, storage_file)
        print(f"  - Created {storage_file} from {original_file}")

        # Read all files in sorted order
        df_list = []
        for market_file in market_files:
            print(f"\n+ Processing {code} file '{market_file}'")
            df = pd.read_excel(market_file, sheet_name="DeliverData", engine='openpyxl', keep_default_na=False, na_values=[''])
            print(f"  - Read {len(df)} rows from {market_file}")
            df_list.append(df)
        if not df_list:
            print("    > No valid data, skipping")
            continue
        df_merged = pd.concat(df_list, ignore_index=True)
        df_merged = df_merged.dropna(how='all')
        print(f"  - Merged DataFrame {len(df_merged)} rows")

        # Save Type 2c: Overwrite existing xlsm
        excel_writer_customize(df_merged, storage_file, "DeliverData")
        time.sleep(1)
        excel_app = xw.App(visible=True, add_book=False)
        workbook_xlwings = excel_app.books.open(storage_file)
        workbook_xlwings = vb_project(workbook_xlwings)
        xlwings_reapply_format(workbook_xlwings)
        print(f"\n~ Successfully combined data into '{storage_file}'")

### ▶ Main Execution

In [ ]:
if __name__ == "__main__":
    # update_guidelines()
    # update_guidelines_offline()
    # combine_platform()
    # clean_platform()
    # clean_UniqueCloud3()
    # clean_UniqueCloud2()
    # combine_ap()
    # add_macro()
    # get_plan_name()
    # split_markets()
    # update_macro()
    # package_ggtrend()
    # apply_ggtrend()
    # combine_markets()
    # convert_cost_to_USD()
    # create_UniqueCloud2_templates()
    # email_UniqueCloud2_submission()
    validate_UniqueCloud2()
    # combine_DeliverData_storage()
completion_timestamp = datetime.now().strftime("%I:%M %p, %d-%b-%Y")
print(f"\nLast Execution ✅ {completion_timestamp}")